# IJDAR Parameter Sensitivity and Robustness Analysis

**Paper:** *Topology-Aware Structural Parsing of Hand-Drawn Diagrams via Learning-Aligned Decoding*

This notebook contains the complete reproduction/inference implementation plus
the deterministic-assembler sensitivity experiments added during revision.

It supports both stages used in the revised manuscript:

1. **Stage 1:** five functional parameter groups, with relaxed/strict
   perturbations and coordinated all-relaxed/all-strict stress tests.
2. **Stage 2:** individual-parameter perturbations inside the Stage-1-identified
   evidence-admission group.

Neural predictions are shared across parameter variants; only deterministic
assembly decisions are perturbed. The public evaluator uses mBox-IoU and nSEE.


In [ ]:
# ============================================================
# MASTER CONFIG — PUBLICATION RELEASE
# ============================================================
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("IJDAR_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
WORKING_ROOT = Path(os.environ.get("IJDAR_WORKING_ROOT", PROJECT_ROOT / "workspace")).expanduser().resolve()

# -----------------------------
# Dataset layout
# -----------------------------
DATASET_ROOT = os.environ.get("IJDAR_DATASET_ROOT", str(WORKING_ROOT / "datasets_converted"))
DATASET_FAMILIES = ("fa", "fca", "fcb")
TRAIN_SPLITS = ("train",)
VAL_SPLITS = ("val",)
TEST_SPLITS = ("test",)
USE_EXPLICIT_DATASET_SPLITS = True

# Optional dataset reconstruction. The release does not redistribute original
# source images. Obtain them from SOURCE_DATASET_GITHUB_URL and either point
# SOURCE_DATASETS_ROOT to the local clone or enable ALLOW_SOURCE_DOWNLOAD.
PREPARE_DATASET_IF_MISSING = False
ALLOW_SOURCE_DOWNLOAD = False
SOURCE_DATASETS_ROOT = os.environ.get("IJDAR_SOURCE_DATASETS_ROOT", "")
SOURCE_DATASET_GITHUB_URL = "https://github.com/bernhardschaefer/handwritten-diagram-datasets"
WORKING_CONVERTED_DATASET_ROOT = str(WORKING_ROOT / "datasets_converted")

# -----------------------------
# Trained checkpoints
# -----------------------------
GRAPH_MODEL_CHECKPOINT = os.environ.get(
    "IJDAR_GRAPH_CHECKPOINT",
    str(PROJECT_ROOT / "trained_models" / "fcinkml_best.pt"),
)
NODE_MODEL_CHECKPOINT = os.environ.get(
    "IJDAR_NODE_CHECKPOINT",
    str(PROJECT_ROOT / "trained_models" / "resnet_34.pth"),
)
GRAPH_CHECKPOINT_FILENAMES = ("fcinkml_best.pt", "best.pt")
NODE_CHECKPOINT_FILENAMES = ("resnet_34.pth", "best.pth")

# Training is opt-in. Missing checkpoints do not silently trigger training.
TRAIN_GRAPH_MODEL_IF_MISSING = False
TRAIN_NODE_MODEL_IF_MISSING = False
FORCE_TRAIN_GRAPH_MODEL = False
FORCE_TRAIN_NODE_MODEL = False

NODE_MASK_DATASET_ROOT = os.environ.get(
    "IJDAR_NODE_MASK_ROOT",
    str(PROJECT_ROOT / "node_mask_dataset"),
)

# -----------------------------
# Run policy
# -----------------------------
# auto | full_inference | evaluate_existing | prepare_dataset_only
RUN_MODE = "auto"
REUSE_EXISTING_PREDICTIONS = True

# "revision" is the public evaluator: mBox-IoU, nSEE, conditional geometry
# coverage, and pooled endpoint/centerline diagnostics.
METRIC_PROFILE = "revision"

OUTPUT_ROOT = str(PROJECT_ROOT / "outputs" / "main_reproduction")
PREDICTIONS_ZIP = str(Path(OUTPUT_ROOT) / "predictions.zip")

# -----------------------------
# Execution
# -----------------------------
PARALLEL_INFERENCE = True
PARALLEL_BACKEND = "threads"
MAX_GPU_WORKERS = 2
CPU_WORKERS_FOR_IO = min(4, os.cpu_count() or 1)
NUM_DATALOADER_WORKERS = min(4, os.cpu_count() or 1)
NUM_PREPROCESS_WORKERS = min(4, os.cpu_count() or 1)
PRINT_TIMING_DIAGNOSTICS = True
PROGRESS_EVERY_N = 10
RANDOM_SEED = 39

QUICK_TEST = False
QUICK_TEST_N = 12

# -----------------------------
# Frozen paper inference settings
# -----------------------------
INFERENCE_SPLITS = TEST_SPLITS
TARGET_W = 512
TARGET_H = 512
IMAGE_MODE = "grayscale"
NORMALIZE_IMAGE = True
APPLY_OTSU_THRESHOLD = True
GRAPH_BASE_CHANNELS = 16
NORM_TYPE = "auto"

NODE_SEG_INPUT_SIZE = 320
NODE_SEG_THRESHOLD = 0.46
NODE_SEG_MIN_AREA = 35
NODE_SEG_CLOSE_KERNEL = 3

ENABLE_NODE_REFINEMENT = True
NODE_REFINEMENT_IOU_ACCEPT = 0.10
NODE_REFINEMENT_CENTER_TOLERANCE_RATIO = 0.02
NODE_REFINEMENT_KEEP_UNMATCHED_GRAPH_NODES = True
NODE_REFINEMENT_REPAIR_EDGE_IDS = False

# Evaluation/matching definitions
METRIC_TOLERANCE_RATIO = 0.02
NODE_IOU_ACCEPT = 0.25
PATH_BASE_TOLERANCE_PX = 3
NSEE_NORMALIZER = "max_pred_gt_nodes_plus_max_pred_gt_edges"

# Output behavior
SAVE_ALL_PREDICTION_JSON = True
COMPRESS_PREDICTIONS_AT_END = True
DELETE_UNCOMPRESSED_PREDICTIONS_AFTER_ZIP = True
SAVE_DENSE_MAPS = False
SAVE_DEBUG_EXAMPLES = True
MAX_DEBUG_EXAMPLES = 3
SAVE_DEBUG_OVERLAYS = True
SAVE_DEBUG_ASSEMBLER_JSON = True
DEBUG_EXAMPLE_POLICY = "first"
CLEAR_OUTPUT_BEFORE_RUN = False

# Manuscript-reported values are stored separately from computed run outputs.
MANUSCRIPT_RESULTS_FILE = PROJECT_ROOT / "MANUSCRIPT_REPORTED_RESULTS.json"


# -----------------------------
# Parameter sensitivity experiment
# -----------------------------
RUN_PARAMETER_SENSITIVITY = True
SENSITIVITY_STRENGTH = 0.20
SENSITIVITY_METRIC_PROFILE = "revision"
SENSITIVITY_STAGE2_TRIGGER_PP = 1.50
SENSITIVITY_SAVE_VARIANT_JSON = False
SENSITIVITY_PRINT_PER_VARIANT = False
SENSITIVITY_OUTPUT_SUBDIR = "parameter_sensitivity"

# "stage1" = five functional groups (13 configurations).
# "stage2" = individual parameters in the Stage-1-identified group.
SENSITIVITY_STAGE = "stage2"
SENSITIVITY_STAGE2_GROUP = "G1"
SENSITIVITY_STAGE2_MODES = ("relaxed", "strict")


## 1. Environment, dataset discovery, and checkpoint discovery

This cell performs **read-only discovery first**. The configured dataset is used if valid; only if it is absent do optional discovery/download paths become relevant.

In [ ]:
import os, sys, json, math, random, shutil, subprocess, time, zipfile, hashlib
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

try:
    import torch
except Exception as exc:
    raise RuntimeError("PyTorch is required for this notebook.") from exc

try:
    import cv2
except Exception as exc:
    raise RuntimeError("OpenCV is required for this notebook.") from exc

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
WORKING_ROOT = Path(WORKING_ROOT).expanduser().resolve()
WORKING_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = str(Path(OUTPUT_ROOT).expanduser())
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
cv2.setNumThreads(1)

GB = 1024 ** 3
SEARCH_ROOTS = [
    PROJECT_ROOT,
    WORKING_ROOT,
    PROJECT_ROOT / "external_data",
    PROJECT_ROOT / "trained_models",
]


def expected_splits():
    if USE_EXPLICIT_DATASET_SPLITS:
        return tuple(dict.fromkeys(tuple(TRAIN_SPLITS) + tuple(VAL_SPLITS) + tuple(TEST_SPLITS)))
    return None

def validate_converted_dataset_root(root, verbose=False):
    root = Path(root)
    report = {"root": str(root), "exists": root.is_dir(), "valid": False, "families": {}, "errors": []}
    if not root.is_dir():
        report["errors"].append("root_not_found")
        return report
    total_pairs = 0
    requested = expected_splits()
    for family in DATASET_FAMILIES:
        fam = root / family
        frep = {"exists": fam.is_dir(), "splits": {}}
        report["families"][family] = frep
        if not fam.is_dir():
            report["errors"].append(f"missing_family:{family}")
            continue
        split_names = requested or tuple(p.name for p in fam.iterdir() if p.is_dir())
        for split in split_names:
            sd = fam / split
            idir, jdir = sd / "images", sd / "json"
            imgs = [] if not idir.is_dir() else [
                p for p in idir.iterdir()
                if p.is_file() and p.suffix.lower() in {".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"}
            ]
            js = [] if not jdir.is_dir() else list(jdir.glob("*.json"))
            stems_i = {p.stem for p in imgs}
            stems_j = {p.stem for p in js}
            matched = len(stems_i & stems_j)
            frep["splits"][split] = {
                "images": len(imgs), "json": len(js), "matched": matched, "path": str(sd)
            }
            total_pairs += matched
            if sd.exists() and matched == 0:
                report["errors"].append(f"empty_or_malformed:{family}/{split}")
    report["total_matched_pairs"] = total_pairs
    report["valid"] = (
        total_pairs > 0
        and all((root/f).is_dir() for f in DATASET_FAMILIES)
        and not any(e.startswith("empty_or_malformed") for e in report["errors"])
    )
    if verbose:
        print(json.dumps(report, indent=2))
    return report

def search_valid_converted_roots(search_roots=None, max_results=20):
    found = []
    roots = search_roots or SEARCH_ROOTS
    candidates = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        if root.name in {"datasets_converted", "converted_annotations"}:
            candidates.append(root)
        for name in ("datasets_converted", "converted_annotations"):
            candidates.extend(p for p in root.rglob(name) if p.is_dir())
    seen = set()
    for p in candidates:
        try:
            key = str(p.resolve())
        except Exception:
            key = str(p)
        if key in seen:
            continue
        seen.add(key)
        rep = validate_converted_dataset_root(p)
        if rep["valid"]:
            found.append((p.resolve(), rep["total_matched_pairs"]))
    return sorted(found, key=lambda x: (-x[1], str(x[0])))[:max_results]

def resolve_dataset_root():
    preferred = Path(DATASET_ROOT).expanduser()
    rep = validate_converted_dataset_root(preferred)
    if rep["valid"]:
        print(f"Dataset: FOUND -> {preferred}")
        return preferred.resolve(), rep
    matches = search_valid_converted_roots()
    if matches:
        path, _ = matches[0]
        print(f"Dataset: auto-discovered -> {path}")
        return path, validate_converted_dataset_root(path)
    print("Dataset: no complete image+JSON dataset currently available.")
    return None, rep

def discover_checkpoint(explicit_path, candidate_names, label):
    raw = str(explicit_path).strip()
    if raw:
        p = Path(raw).expanduser()
        if p.is_file():
            print(f"{label}: FOUND -> {p}")
            return p.resolve()
    matches = []
    names = set(candidate_names)
    for root in SEARCH_ROOTS:
        root = Path(root)
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if p.is_file() and p.name in names:
                matches.append(p.resolve())
    matches = sorted(set(matches), key=lambda p: (
        0 if p.name == candidate_names[0] else 1, len(str(p)), str(p)
    ))
    if matches:
        print(f"{label}: auto-discovered -> {matches[0]}")
        return matches[0]
    print(f"{label}: NOT FOUND")
    return None

def resolve_node_mask_root():
    raw = str(NODE_MASK_DATASET_ROOT).strip()
    if raw:
        p = Path(raw).expanduser()
        if (p/"masks").is_dir() and (p/"splits").is_dir():
            return p.resolve()
    for root in SEARCH_ROOTS:
        root = Path(root)
        if not root.exists():
            continue
        for p in root.rglob("node_mask_dataset"):
            if (p/"masks").is_dir() and (p/"splits").is_dir():
                return p.resolve()
    return None

def run_command(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=None if cwd is None else str(cwd), check=True)

def source_family_root_ok(path):
    p = Path(path)
    return all((p/f).is_dir() for f in DATASET_FAMILIES)

def find_source_dataset_root(root):
    root = Path(root)
    for p in (root/"datasets", root):
        if source_family_root_ok(p):
            return p.resolve()
    if root.exists():
        for p in root.rglob("*"):
            if p.is_dir() and len(p.relative_to(root).parts) <= 4 and source_family_root_ok(p):
                return p.resolve()
    return None

def resolve_or_download_source_dataset():
    raw = str(SOURCE_DATASETS_ROOT).strip()
    if raw and source_family_root_ok(raw):
        return Path(raw).resolve()
    for root in SEARCH_ROOTS:
        src = find_source_dataset_root(root) if Path(root).exists() else None
        if src is not None:
            print(f"Original source dataset auto-discovered -> {src}")
            return src
    if not ALLOW_SOURCE_DOWNLOAD:
        raise FileNotFoundError(
            "Converted dataset is missing and no original source dataset was found. "
            "Set SOURCE_DATASETS_ROOT or enable ALLOW_SOURCE_DOWNLOAD."
        )
    clone_dir = WORKING_ROOT / "external_data" / "handwritten-diagram-datasets"
    if not clone_dir.exists():
        run_command(["git","clone","--depth","1",SOURCE_DATASET_GITHUB_URL,clone_dir])
    src = find_source_dataset_root(clone_dir)
    if src is None:
        raise RuntimeError("Downloaded repository did not contain recognizable fa/fca/fcb source structure.")
    return src

RESOLVED_DATASET_ROOT, DATASET_REPORT = resolve_dataset_root()
RESOLVED_GRAPH_CHECKPOINT = discover_checkpoint(
    GRAPH_MODEL_CHECKPOINT, GRAPH_CHECKPOINT_FILENAMES, "Graph checkpoint"
)
RESOLVED_NODE_CHECKPOINT = discover_checkpoint(
    NODE_MODEL_CHECKPOINT, NODE_CHECKPOINT_FILENAMES, "Node checkpoint"
)
RESOLVED_NODE_MASK_ROOT = resolve_node_mask_root()

SHOULD_PREPARE_DATASET = RESOLVED_DATASET_ROOT is None and PREPARE_DATASET_IF_MISSING
SHOULD_TRAIN_GRAPH = FORCE_TRAIN_GRAPH_MODEL or (
    RESOLVED_GRAPH_CHECKPOINT is None and TRAIN_GRAPH_MODEL_IF_MISSING
)
SHOULD_TRAIN_NODE = FORCE_TRAIN_NODE_MODEL or (
    RESOLVED_NODE_CHECKPOINT is None and TRAIN_NODE_MODEL_IF_MISSING
)

print("\nEnvironment")
print("-"*72)
print("Python         :", sys.version.split()[0])
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
print("GPU count      :", torch.cuda.device_count() if torch.cuda.is_available() else 0)
for i in range(torch.cuda.device_count() if torch.cuda.is_available() else 0):
    print(f"GPU {i:<10}: {torch.cuda.get_device_name(i)}")
print("CPU cores      :", os.cpu_count())
print("Dataset ready  :", RESOLVED_DATASET_ROOT is not None)
print("Train graph    :", SHOULD_TRAIN_GRAPH)
print("Train node     :", SHOULD_TRAIN_NODE)


## 2. Optional dataset reconstruction

Skipped when the configured/auto-discovered converted dataset is already valid. If enabled and required, this section reproduces the publication conversion and 1024×1024 reconstruction stages. Original images are never downloaded when a usable dataset is already mounted.

In [ ]:
if SHOULD_PREPARE_DATASET:

    import json
    import math
    import heapq
    import os
    from pathlib import Path
    from collections import defaultdict, deque
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from typing import Dict, Iterable, List, Optional, Sequence, Tuple

    import cv2
    import numpy as np

    try:
        from skimage.morphology import skeletonize
        HAS_SKIMAGE = True
    except Exception:
        HAS_SKIMAGE = False

    print(f"OpenCV version: {cv2.__version__}")
    print(f"scikit-image skeletonization available: {HAS_SKIMAGE}")


In [ ]:
if SHOULD_PREPARE_DATASET:
    # ============================================================
    # Geometry helpers
    # ============================================================

    def pt(x, y):
        return {"x": float(x), "y": float(y)}


    def bbox_to_xyxy(bbox):
        x, y, w, h = bbox
        return int(x), int(y), int(x + w), int(y + h)


    def clamp(v, lo, hi):
        return max(lo, min(hi, v))


    def scale_point(p, sx, sy):
        return {"x": float(p["x"]) * sx, "y": float(p["y"]) * sy}


    def scale_bbox(bbox, sx, sy):
        x, y, w, h = bbox
        return [
            int(round(x * sx)),
            int(round(y * sy)),
            int(round(w * sx)),
            int(round(h * sy)),
        ]


    def scale_path(path, sx, sy):
        return [scale_point(p, sx, sy) for p in path]


    def straight_line_path(p0, p1):
        x0, y0 = p0["x"], p0["y"]
        x1, y1 = p1["x"], p1["y"]

        n = int(max(abs(x1 - x0), abs(y1 - y0))) + 1
        if n <= 1:
            return [pt(x0, y0)]

        xs = np.linspace(x0, x1, n)
        ys = np.linspace(y0, y1, n)

        return [pt(x, y) for x, y in zip(xs, ys)]


    def polyline_path(points):
        if not points:
            return []

        out = []
        for i in range(len(points) - 1):
            p0 = pt(points[i][0], points[i][1])
            p1 = pt(points[i + 1][0], points[i + 1][1])

            seg = straight_line_path(p0, p1)

            if i > 0 and seg:
                seg = seg[1:]

            out.extend(seg)

        return out


    def waypoint_path(arrow_ann, tail, head):
        waypoints = arrow_ann.get("waypoints", None)

        if waypoints and len(waypoints) >= 2:
            return polyline_path(waypoints)

        return straight_line_path(tail, head)


    def sample_equidistant_points(path, count=40):
        if not path or len(path) < 2:
            return []

        coords = np.array([[p["x"], p["y"]] for p in path], dtype=np.float32)
        seg = np.diff(coords, axis=0)
        seg_len = np.sqrt((seg ** 2).sum(axis=1))
        cum = np.concatenate([[0], np.cumsum(seg_len)])

        total = cum[-1]
        if total <= 1e-6:
            return []

        samples = []
        targets = np.linspace(0, total, count + 2)[1:-1]

        for i, t in enumerate(targets, start=1):
            idx = np.searchsorted(cum, t) - 1
            idx = clamp(idx, 0, len(seg_len) - 1)

            alpha = 0 if seg_len[idx] <= 1e-6 else (t - cum[idx]) / seg_len[idx]
            xy = coords[idx] * (1 - alpha) + coords[idx + 1] * alpha

            samples.append({
                "x": int(round(xy[0])),
                "y": int(round(xy[1])),
                "label": f"E{i}",
            })

        return samples


    def bbox_center(bbox):
        x, y, w, h = bbox
        return np.array([x + w / 2.0, y + h / 2.0], dtype=np.float32)


In [ ]:
if SHOULD_PREPARE_DATASET:
    # ============================================================
    # Shared routing helpers
    # ============================================================

    def make_disk_mask(shape, cx, cy, r):
        h, w = shape
        yy, xx = np.ogrid[:h, :w]
        return (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2


    def draw_polyline_mask(shape, points, thickness=35):
        mask = np.zeros(shape, dtype=np.uint8)

        if len(points) < 2:
            return mask.astype(bool)

        pts = np.array(
            [[int(round(p[0])), int(round(p[1]))] for p in points],
            dtype=np.int32,
        )

        cv2.polylines(mask, [pts], False, 255, thickness=thickness)
        return mask > 0


    def nearest_allowed_pixel(allowed, target_xy, max_radius=70):
        tx, ty = target_xy
        h, w = allowed.shape

        tx = clamp(int(round(tx)), 0, w - 1)
        ty = clamp(int(round(ty)), 0, h - 1)

        if allowed[ty, tx]:
            return tx, ty

        ys, xs = np.where(allowed)
        if len(xs) == 0:
            return None

        d2 = (xs - tx) ** 2 + (ys - ty) ** 2
        idx = int(np.argmin(d2))

        if max_radius is not None and math.sqrt(float(d2[idx])) > max_radius:
            return None

        return int(xs[idx]), int(ys[idx])


    def nearest_mask_pixel(mask, target_xy, max_radius=None):
        return nearest_allowed_pixel(mask, target_xy, max_radius=max_radius)


    def weighted_dijkstra_path(cost, allowed, start, goal):
        h, w = cost.shape
        sx, sy = start
        gx, gy = goal

        dist = np.full((h, w), np.inf, dtype=np.float32)
        visited = np.zeros((h, w), dtype=bool)
        parent = {}

        dist[sy, sx] = 0.0
        heap = [(0.0, sx, sy)]

        nbrs = [
            (-1, -1, 1.414), (0, -1, 1.0), (1, -1, 1.414),
            (-1,  0, 1.0),                  (1,  0, 1.0),
            (-1,  1, 1.414), (0,  1, 1.0),  (1,  1, 1.414),
        ]

        while heap:
            d, x, y = heapq.heappop(heap)

            if visited[y, x]:
                continue

            visited[y, x] = True

            if (x, y) == (gx, gy):
                break

            for dx, dy, step_cost in nbrs:
                nx, ny = x + dx, y + dy

                if nx < 0 or ny < 0 or nx >= w or ny >= h:
                    continue

                if not allowed[ny, nx]:
                    continue

                nd = d + step_cost * float(cost[ny, nx])

                if nd < dist[ny, nx]:
                    dist[ny, nx] = nd
                    parent[(nx, ny)] = (x, y)
                    heapq.heappush(heap, (nd, nx, ny))

        if not np.isfinite(dist[gy, gx]):
            return None

        path = []
        cur = (gx, gy)

        while cur != (sx, sy):
            path.append(cur)
            cur = parent.get(cur)

            if cur is None:
                return None

        path.append((sx, sy))
        path.reverse()

        return path


    def shortest_path_on_binary_mask(mask, start, goal):
        h, w = mask.shape
        sx, sy = start
        gx, gy = goal

        if sx < 0 or sy < 0 or gx < 0 or gy < 0 or sx >= w or gx >= w or sy >= h or gy >= h:
            return []

        if not mask[sy, sx] or not mask[gy, gx]:
            return []

        q = deque([(sx, sy)])
        visited = np.zeros((h, w), dtype=np.uint8)
        parent = {}

        visited[sy, sx] = 1

        nbrs = [
            (-1, -1), (0, -1), (1, -1),
            (-1,  0),          (1,  0),
            (-1,  1), (0,  1), (1,  1),
        ]

        found = False

        while q:
            x, y = q.popleft()

            if (x, y) == (gx, gy):
                found = True
                break

            for dx, dy in nbrs:
                nx, ny = x + dx, y + dy

                if nx < 0 or ny < 0 or nx >= w or ny >= h:
                    continue

                if not mask[ny, nx]:
                    continue

                if visited[ny, nx]:
                    continue

                visited[ny, nx] = 1
                parent[(nx, ny)] = (x, y)
                q.append((nx, ny))

        if not found:
            return []

        path = []
        cur = (gx, gy)

        while True:
            path.append(cur)

            if cur == (sx, sy):
                break

            cur = parent[cur]

        path.reverse()
        return path


    def dilate_bool_mask(mask, k=3, iterations=1):
        kernel = np.ones((k, k), np.uint8)
        out = cv2.dilate(mask.astype(np.uint8), kernel, iterations=iterations)
        return out > 0


    def localize_points(points, ox, oy):
        return [[p[0] - ox, p[1] - oy] for p in points]


    def bbox_intersects(b1, b2):
        x1, y1, w1, h1 = b1
        x2, y2, w2, h2 = b2

        return not (
            x1 + w1 < x2 or x2 + w2 < x1 or
            y1 + h1 < y2 or y2 + h2 < y1
        )


    def binarize_and_skeletonize_for_routing(gray):
        _, th = cv2.threshold(
            gray,
            0,
            255,
            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU,
        )

        kernel = np.ones((3, 3), np.uint8)
        th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=1)

        if HAS_SKIMAGE:
            skel = skeletonize(th > 0)
            return skel.astype(bool)

        return th > 0


In [ ]:
if SHOULD_PREPARE_DATASET:
    # ============================================================
    # Flowchart routing: uses explicit waypoints
    # ============================================================

    def extract_flowchart_path(
        image,
        arrow_ann,
        blocking_annotations,
        tail,
        head,
        pad=12,
        ink_threshold=180,
        endpoint_keep_radius=45,
        node_forbid_margin=10,
        corridor_thickness=45,
        allow_white_bridge=True,
    ):
        h, w = image.shape[:2]

        waypoints = arrow_ann.get("waypoints", None)
        guide_points = (
            waypoints
            if waypoints and len(waypoints) >= 2
            else [[tail["x"], tail["y"]], [head["x"], head["y"]]]
        )

        xs = [p[0] for p in guide_points] + [tail["x"], head["x"]]
        ys = [p[1] for p in guide_points] + [tail["y"], head["y"]]

        x1p = clamp(int(min(xs)) - corridor_thickness - pad, 0, w - 1)
        y1p = clamp(int(min(ys)) - corridor_thickness - pad, 0, h - 1)
        x2p = clamp(int(max(xs)) + corridor_thickness + pad, 0, w - 1)
        y2p = clamp(int(max(ys)) + corridor_thickness + pad, 0, h - 1)

        crop = image[y1p:y2p + 1, x1p:x2p + 1]
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

        ch, cw = gray.shape
        local_waypoints = localize_points(guide_points, x1p, y1p)

        corridor = draw_polyline_mask(
            (ch, cw),
            local_waypoints,
            thickness=corridor_thickness,
        )

        ink = gray < ink_threshold
        ink_u8 = ink.astype(np.uint8) * 255
        kernel = np.ones((3, 3), np.uint8)
        ink_u8 = cv2.morphologyEx(ink_u8, cv2.MORPH_CLOSE, kernel, iterations=1)
        ink = ink_u8 > 0

        allowed = corridor.copy()

        for ann in blocking_annotations:
            bx, by, bw, bh = ann["bbox"]

            lx1 = clamp(bx - x1p - node_forbid_margin, 0, cw - 1)
            ly1 = clamp(by - y1p - node_forbid_margin, 0, ch - 1)
            lx2 = clamp(bx + bw - x1p + node_forbid_margin, 0, cw - 1)
            ly2 = clamp(by + bh - y1p + node_forbid_margin, 0, ch - 1)

            region = np.zeros((ch, cw), dtype=bool)
            region[ly1:ly2 + 1, lx1:lx2 + 1] = True

            allowed[region & corridor] = False

        local_tail = (
            int(round(tail["x"] - x1p)),
            int(round(tail["y"] - y1p)),
        )

        local_head = (
            int(round(head["x"] - x1p)),
            int(round(head["y"] - y1p)),
        )

        for cx, cy in [local_tail, local_head]:
            cx = clamp(cx, 0, cw - 1)
            cy = clamp(cy, 0, ch - 1)

            disk = make_disk_mask((ch, cw), cx, cy, endpoint_keep_radius)
            allowed[disk & corridor] = True

        if allow_white_bridge:
            ink_u8 = ink.astype(np.uint8)

            dist_to_ink = cv2.distanceTransform(
                (1 - ink_u8).astype(np.uint8),
                cv2.DIST_L2,
                3,
            )

            cost = 1.0 + 8.0 * dist_to_ink
            cost[ink] = 1.0
        else:
            allowed &= ink
            cost = np.ones((ch, cw), dtype=np.float32)

        guide_line = draw_polyline_mask(
            (ch, cw),
            local_waypoints,
            thickness=3,
        ).astype(np.uint8)

        dist_to_guide = cv2.distanceTransform(
            (1 - guide_line).astype(np.uint8),
            cv2.DIST_L2,
            3,
        )

        cost += 0.35 * dist_to_guide
        cost[~allowed] = 1e9

        start = nearest_allowed_pixel(allowed, local_tail, max_radius=70)
        goal = nearest_allowed_pixel(allowed, local_head, max_radius=70)

        if start is None or goal is None:
            return waypoint_path(arrow_ann, tail, head)

        path_xy = weighted_dijkstra_path(cost, allowed, start, goal)

        if not path_xy or len(path_xy) < 2:
            return waypoint_path(arrow_ann, tail, head)

        return [pt(x + x1p, y + y1p) for x, y in path_xy]


In [ ]:
if SHOULD_PREPARE_DATASET:
    # ============================================================
    # FA routing: ellipse-aware node blocking + self-loop midpoint router
    # ============================================================

    FA_RETRY_CONFIGS = [
        {"node_ellipse_margin": 10, "endpoint_keep_radius": 38, "corridor_thickness": 65},
        {"node_ellipse_margin": 14, "endpoint_keep_radius": 30, "corridor_thickness": 75},
        {"node_ellipse_margin": 18, "endpoint_keep_radius": 24, "corridor_thickness": 90},
        {"node_ellipse_margin": 22, "endpoint_keep_radius": 20, "corridor_thickness": 105},
        {"node_ellipse_margin": 26, "endpoint_keep_radius": 16, "corridor_thickness": 120},
    ]


    def ellipse_mask_for_bbox(shape, bbox, ox, oy, margin=8):
        h, w = shape
        x, y, bw, bh = bbox

        cx = x + bw / 2.0 - ox
        cy = y + bh / 2.0 - oy
        rx = max(1.0, bw / 2.0 + margin)
        ry = max(1.0, bh / 2.0 + margin)

        yy, xx = np.ogrid[:h, :w]
        return ((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2 <= 1.0


    def path_inside_node_ratio(path, node_bbox, margin=4):
        if not path:
            return 1.0

        x, y, w, h = node_bbox
        cx = x + w / 2.0
        cy = y + h / 2.0
        rx = max(1.0, w / 2.0 + margin)
        ry = max(1.0, h / 2.0 + margin)

        inside = 0
        for p in path:
            val = ((p["x"] - cx) / rx) ** 2 + ((p["y"] - cy) / ry) ** 2
            if val <= 1.0:
                inside += 1

        return inside / max(len(path), 1)


    def path_inside_node_ratio_strict(path, node_bbox, margin=10):
        return path_inside_node_ratio(path, node_bbox, margin=margin)


    def find_node_by_source_id(node_anns, source_id):
        if source_id is None:
            return None

        for ann in node_anns:
            if ann.get("id") == source_id or ann.get("aid") == source_id:
                return ann

        return None


    def is_same_node(prev_node, next_node):
        if prev_node is None or next_node is None:
            return False

        return (
            prev_node.get("id") == next_node.get("id") or
            prev_node.get("aid") == next_node.get("aid")
        )


    def is_self_loop_arrow(arrow_ann, node_annotations):
        prev_node = find_node_by_source_id(node_annotations, arrow_ann.get("arrow_prev"))
        next_node = find_node_by_source_id(node_annotations, arrow_ann.get("arrow_next"))
        return is_same_node(prev_node, next_node), prev_node, next_node


    def adaptive_endpoint_radius_for_loop(tail, head, default_radius):
        dx = tail["x"] - head["x"]
        dy = tail["y"] - head["y"]
        d = math.sqrt(dx * dx + dy * dy)

        # Prevent tail/head restore disks from merging into one tunnel.
        return int(max(10, min(default_radius, 0.32 * d)))


    def synthetic_fa_guide_points(arrow_ann, tail, head, prev_node=None, next_node=None):
        same_node = is_same_node(prev_node, next_node)

        ax, ay, aw, ah = arrow_ann["bbox"]

        if not same_node or prev_node is None:
            return [[tail["x"], tail["y"]], [head["x"], head["y"]]]

        nc = bbox_center(prev_node["bbox"])
        ac = bbox_center(arrow_ann["bbox"])
        direction = ac - nc

        if abs(direction[0]) >= abs(direction[1]):
            # Left/right self-loop.
            side_x = ax if direction[0] < 0 else ax + aw
            mid1 = [side_x, ay + ah * 0.15]
            mid2 = [side_x, ay + ah * 0.85]
        else:
            # Top/bottom self-loop.
            side_y = ay if direction[1] < 0 else ay + ah
            mid1 = [ax + aw * 0.15, side_y]
            mid2 = [ax + aw * 0.85, side_y]

        return [
            [tail["x"], tail["y"]],
            mid1,
            mid2,
            [head["x"], head["y"]],
        ]



    def path_length_pixels(path):
        if not path or len(path) < 2:
            return 0.0

        total = 0.0
        for p0, p1 in zip(path[:-1], path[1:]):
            total += math.hypot(p1["x"] - p0["x"], p1["y"] - p0["y"])

        return total


    def synthetic_fa_guide_points_for_side(arrow_ann, tail, head, side):
        """
        Explicit self-loop guide for one chosen side of the arrow bbox.
        This is only a probe, not the final path.
        """

        ax, ay, aw, ah = arrow_ann["bbox"]

        if side == "left":
            mid1 = [ax, ay + ah * 0.15]
            mid2 = [ax, ay + ah * 0.85]
        elif side == "right":
            mid1 = [ax + aw, ay + ah * 0.15]
            mid2 = [ax + aw, ay + ah * 0.85]
        elif side == "top":
            mid1 = [ax + aw * 0.15, ay]
            mid2 = [ax + aw * 0.85, ay]
        elif side == "bottom":
            mid1 = [ax + aw * 0.15, ay + ah]
            mid2 = [ax + aw * 0.85, ay + ah]
        else:
            raise ValueError(f"Unknown loop side: {side}")

        return [
            [tail["x"], tail["y"]],
            mid1,
            mid2,
            [head["x"], head["y"]],
        ]


    def ordered_loop_sides_from_geometry(arrow_ann, loop_node):
        """
        Returns likely loop sides in priority order.
        The first one is based on arrow bbox center relative to the node center,
        but we still try all sides because some FA annotations are tight/ugly.
        """

        ax, ay, aw, ah = arrow_ann["bbox"]
        ac = bbox_center(arrow_ann["bbox"])
        nc = bbox_center(loop_node["bbox"])
        direction = ac - nc

        if abs(direction[0]) >= abs(direction[1]):
            primary = "left" if direction[0] < 0 else "right"
        else:
            primary = "top" if direction[1] < 0 else "bottom"

        all_sides = ["top", "right", "bottom", "left"]
        ordered = [primary] + [s for s in all_sides if s != primary]
        return ordered


    def choose_midpoint_on_real_loop_ink(
        route_mask,
        guide_band,
        arrow_region,
        start,
        goal,
        synthetic_mid_target,
        node_bbox,
        crop_origin,
        endpoint_exclusion_radius=18,
    ):
        """
        Selects a real skeleton pixel on the loop. The synthetic guide is used
        only to search the right region. The actual selected point must be ink.

        Scoring strongly prefers points far away from the node center. This avoids
        choosing a short bridge near the node boundary.
        """

        ch, cw = route_mask.shape
        x1p, y1p = crop_origin

        endpoint_exclusion = (
            make_disk_mask((ch, cw), start[0], start[1], endpoint_exclusion_radius) |
            make_disk_mask((ch, cw), goal[0], goal[1], endpoint_exclusion_radius)
        )

        candidates = route_mask & guide_band & arrow_region & (~endpoint_exclusion)

        # If arrow_region is too strict, relax to guide_band.
        if not candidates.any():
            candidates = route_mask & guide_band & (~endpoint_exclusion)

        if not candidates.any():
            return None

        ys, xs = np.where(candidates)
        pts = np.stack([xs, ys], axis=1).astype(np.float32)

        node_center_global = bbox_center(node_bbox)
        node_center = np.array(
            [node_center_global[0] - x1p, node_center_global[1] - y1p],
            dtype=np.float32,
        )

        mid_target = np.array(synthetic_mid_target, dtype=np.float32)

        dist_from_node = np.linalg.norm(pts - node_center[None, :], axis=1)
        dist_to_mid = np.linalg.norm(pts - mid_target[None, :], axis=1)

        # Normalize to avoid one term numerically bullying the other.
        dfn = dist_from_node / (dist_from_node.max() + 1e-6)
        dtm = dist_to_mid / (dist_to_mid.max() + 1e-6)

        # Main rule: pick real ink farthest from node, but still near synthetic side.
        score = 2.0 * dfn - 0.45 * dtm

        best_idx = int(np.argmax(score))
        return int(pts[best_idx, 0]), int(pts[best_idx, 1])




    def make_ink_mask_for_weighted_loop(gray):
        """
        Robust black-ink mask. Otsu handles writer/style variation better than
        one fixed threshold for these FA images.
        """
        _, th = cv2.threshold(
            gray,
            0,
            255,
            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU,
        )

        kernel = np.ones((3, 3), np.uint8)
        th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=1)
        return th > 0


    def weighted_route_forced_through_midpoint(
        gray,
        allowed_region,
        node_core,
        local_guide,
        start,
        mid,
        goal,
        extra_block_mask=None,
    ):
        """
        Fallback for stubborn FA self-loops.

        The midpoint router first tries pure skeleton BFS. That is clean but brittle:
        if a loop skeleton breaks near the state boundary, BFS fails.

        This fallback still forces tail -> midpoint -> head, but uses weighted routing:
        - black ink is cheap
        - near-ink white pixels are allowed but expensive
        - node core is forbidden
        - corridor/arrow region constrains the search
        """

        ch, cw = gray.shape

        ink = make_ink_mask_for_weighted_loop(gray)

        allowed = allowed_region.copy()
        allowed &= ~node_core

        if extra_block_mask is not None:
            allowed &= ~extra_block_mask

        # Make sure selected points are legal. They are already near real ink, but
        # tiny masking artifacts can otherwise invalidate them.
        for p in [start, mid, goal]:
            if p is None:
                return None, {
                    "weighted_midpoint_used": False,
                    "reason": "missing_start_mid_or_goal",
                }

            x, y = p
            if 0 <= x < cw and 0 <= y < ch:
                allowed[y, x] = True

        ink_u8 = ink.astype(np.uint8)
        dist_to_ink = cv2.distanceTransform(
            (1 - ink_u8).astype(np.uint8),
            cv2.DIST_L2,
            3,
        )

        # Cheap on ink, expensive away from ink.
        cost = 1.0 + 9.0 * dist_to_ink
        cost[ink] = 1.0

        # Prefer the synthetic side, but do not force it blindly.
        guide_line = draw_polyline_mask(
            (ch, cw),
            local_guide,
            thickness=3,
        ).astype(np.uint8)

        dist_to_guide = cv2.distanceTransform(
            (1 - guide_line).astype(np.uint8),
            cv2.DIST_L2,
            3,
        )

        cost += 0.22 * dist_to_guide

        # Repel from node boundary so the route does not scrape through the state.
        if node_core.any():
            dist_from_node = cv2.distanceTransform(
                (1 - node_core.astype(np.uint8)).astype(np.uint8),
                cv2.DIST_L2,
                3,
            )
            near_node = dist_from_node < 14
            cost[near_node] *= 3.5

        cost[~allowed] = 1e9

        p1 = weighted_dijkstra_path(cost, allowed, start, mid)
        p2 = weighted_dijkstra_path(cost, allowed, mid, goal)

        if not p1 or not p2:
            return None, {
                "weighted_midpoint_used": False,
                "reason": "weighted_tail_mid_or_mid_head_failed",
            }

        return p1 + p2[1:], {
            "weighted_midpoint_used": True,
            "reason": "passed",
        }


    def route_self_loop_with_one_guide(
        image,
        arrow_ann,
        text_annotations,
        node_annotations,
        tail,
        head,
        loop_node,
        guide_points,
        side_name,
        node_ellipse_margin=10,
        corridor_thickness=75,
        pad=12,
        midpoint_exclusion_radius=18,
    ):
        h, w = image.shape[:2]

        ax, ay, aw, ah = arrow_ann["bbox"]

        xs = [ax, ax + aw, tail["x"], head["x"]] + [p[0] for p in guide_points]
        ys = [ay, ay + ah, tail["y"], head["y"]] + [p[1] for p in guide_points]

        x1p = clamp(int(min(xs)) - corridor_thickness - pad, 0, w - 1)
        y1p = clamp(int(min(ys)) - corridor_thickness - pad, 0, h - 1)
        x2p = clamp(int(max(xs)) + corridor_thickness + pad, 0, w - 1)
        y2p = clamp(int(max(ys)) + corridor_thickness + pad, 0, h - 1)

        crop = image[y1p:y2p + 1, x1p:x2p + 1]
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

        ch, cw = gray.shape

        skel = binarize_and_skeletonize_for_routing(gray)

        local_guide = localize_points(guide_points, x1p, y1p)

        guide_band = draw_polyline_mask(
            (ch, cw),
            local_guide,
            thickness=corridor_thickness,
        )

        arrow_region = np.zeros((ch, cw), dtype=bool)

        lx1 = clamp(ax - x1p, 0, cw - 1)
        ly1 = clamp(ay - y1p, 0, ch - 1)
        lx2 = clamp(ax + aw - x1p, 0, cw - 1)
        ly2 = clamp(ay + ah - y1p, 0, ch - 1)

        arrow_region[ly1:ly2 + 1, lx1:lx2 + 1] = True

        allowed_region = guide_band | arrow_region

        node_core = ellipse_mask_for_bbox(
            (ch, cw),
            loop_node["bbox"],
            x1p,
            y1p,
            margin=node_ellipse_margin,
        )

        text_block = np.zeros((ch, cw), dtype=bool)

        for ann in text_annotations:
            if not bbox_intersects(ann["bbox"], arrow_ann["bbox"]):
                continue

            bx, by, bw, bh = ann["bbox"]

            tx1 = clamp(bx - x1p - 3, 0, cw - 1)
            ty1 = clamp(by - y1p - 3, 0, ch - 1)
            tx2 = clamp(bx + bw - x1p + 3, 0, cw - 1)
            ty2 = clamp(by + bh - y1p + 3, 0, ch - 1)

            text_block[ty1:ty2 + 1, tx1:tx2 + 1] = True

        route_mask = skel & allowed_region & (~node_core) & (~text_block)

        local_tail = (
            int(round(tail["x"] - x1p)),
            int(round(tail["y"] - y1p)),
        )

        local_head = (
            int(round(head["x"] - x1p)),
            int(round(head["y"] - y1p)),
        )

        # Prefer skeleton points for start/goal, but allow weighted fallback later.
        start = nearest_mask_pixel(route_mask, local_tail, max_radius=85)
        goal = nearest_mask_pixel(route_mask, local_head, max_radius=85)

        weighted_allowed = allowed_region & (~node_core) & (~text_block)

        if start is None:
            start = nearest_mask_pixel(weighted_allowed, local_tail, max_radius=85)

        if goal is None:
            goal = nearest_mask_pixel(weighted_allowed, local_head, max_radius=85)

        if start is None or goal is None:
            return None, {
                "side": side_name,
                "reason": "could_not_snap_tail_or_head",
            }

        if len(local_guide) >= 4:
            synthetic_mid_target = np.mean(
                np.array(local_guide[1:-1], dtype=np.float32),
                axis=0,
            )
        else:
            synthetic_mid_target = np.mean(
                np.array(local_guide, dtype=np.float32),
                axis=0,
            )

        mid = choose_midpoint_on_real_loop_ink(
            route_mask=route_mask,
            guide_band=guide_band,
            arrow_region=arrow_region,
            start=start,
            goal=goal,
            synthetic_mid_target=synthetic_mid_target,
            node_bbox=loop_node["bbox"],
            crop_origin=(x1p, y1p),
            endpoint_exclusion_radius=midpoint_exclusion_radius,
        )

        if mid is None:
            # Relax midpoint candidate only, not the final logic.
            relaxed_for_mid = dilate_bool_mask(route_mask, k=3, iterations=1)
            relaxed_for_mid &= allowed_region
            relaxed_for_mid &= ~node_core
            relaxed_for_mid &= ~text_block

            mid = choose_midpoint_on_real_loop_ink(
                route_mask=relaxed_for_mid,
                guide_band=guide_band,
                arrow_region=arrow_region,
                start=start,
                goal=goal,
                synthetic_mid_target=synthetic_mid_target,
                node_bbox=loop_node["bbox"],
                crop_origin=(x1p, y1p),
                endpoint_exclusion_radius=midpoint_exclusion_radius,
            )

        if mid is None:
            return None, {
                "side": side_name,
                "reason": "could_not_find_midpoint_on_loop_ink",
            }

        # First try strict skeleton route, exactly like manual tail-mid-head annotation.
        path1 = shortest_path_on_binary_mask(route_mask, start, mid)
        path2 = shortest_path_on_binary_mask(route_mask, mid, goal)

        route_method = "skeleton_tail_mid_head"
        relaxed_used = False

        if not path1 or not path2:
            relaxed_mask = dilate_bool_mask(route_mask, k=3, iterations=1)
            relaxed_mask &= allowed_region
            relaxed_mask &= ~node_core
            relaxed_mask &= ~text_block

            start_r = nearest_mask_pixel(relaxed_mask, local_tail, max_radius=85)
            goal_r = nearest_mask_pixel(relaxed_mask, local_head, max_radius=85)
            mid_r = nearest_mask_pixel(relaxed_mask & guide_band, mid, max_radius=30)

            if start_r is not None and goal_r is not None and mid_r is not None:
                path1 = shortest_path_on_binary_mask(relaxed_mask, start_r, mid_r)
                path2 = shortest_path_on_binary_mask(relaxed_mask, mid_r, goal_r)

                if path1 and path2:
                    start, mid, goal = start_r, mid_r, goal_r
                    relaxed_used = True
                    route_method = "relaxed_skeleton_tail_mid_head"

        # New stubborn-loop fallback:
        # still forced through the real midpoint, but not limited to 1-pixel skeleton.
        if not path1 or not path2:
            weighted_path, weighted_debug = weighted_route_forced_through_midpoint(
                gray=gray,
                allowed_region=allowed_region,
                node_core=node_core,
                local_guide=local_guide,
                start=start,
                mid=mid,
                goal=goal,
                extra_block_mask=text_block,
            )

            if weighted_path is None:
                return None, {
                    "side": side_name,
                    "reason": "tail_mid_or_mid_head_path_failed",
                    "weighted_debug": weighted_debug,
                }

            full_path = weighted_path
            route_method = "weighted_tail_mid_head"
        else:
            full_path = path1 + path2[1:]

        out_path = [
            pt(x + x1p, y + y1p)
            for x, y in full_path
        ]

        ratio = path_inside_node_ratio_strict(
            out_path,
            loop_node["bbox"],
            margin=10,
        )

        chord = math.hypot(tail["x"] - head["x"], tail["y"] - head["y"])
        loop_len = path_length_pixels(out_path)
        min_reasonable_len = max(25.0, 1.25 * chord)

        if ratio > 0.03:
            return None, {
                "side": side_name,
                "reason": "path_still_enters_node",
                "inside_node_ratio": ratio,
                "path_length": loop_len,
                "min_reasonable_len": min_reasonable_len,
                "relaxed_used": relaxed_used,
                "route_method": route_method,
            }

        if loop_len < min_reasonable_len:
            return None, {
                "side": side_name,
                "reason": "path_too_short_for_loop",
                "inside_node_ratio": ratio,
                "path_length": loop_len,
                "min_reasonable_len": min_reasonable_len,
                "relaxed_used": relaxed_used,
                "route_method": route_method,
            }

        return out_path, {
            "side": side_name,
            "reason": "passed",
            "inside_node_ratio": ratio,
            "path_length": loop_len,
            "min_reasonable_len": min_reasonable_len,
            "relaxed_used": relaxed_used,
            "route_method": route_method,
            "midpoint": {
                "x": int(mid[0] + x1p),
                "y": int(mid[1] + y1p),
            },
        }



    def extract_fa_self_loop_path_via_midpoint(
        image,
        arrow_ann,
        text_annotations,
        node_annotations,
        tail,
        head,
        node_ellipse_margin=10,
        corridor_thickness=75,
        pad=12,
        midpoint_exclusion_radius=18,
    ):
        """
        Self-loop router inspired by the manual GUI.

        It is applied to EVERY FA self-loop, not just one loop:
        - try a primary synthetic side based on geometry;
        - if it fails, try top/right/bottom/left;
        - for each side, find a real midpoint on black ink;
        - route tail -> midpoint -> head on skeleton.
        """

        is_loop, prev_node, next_node = is_self_loop_arrow(arrow_ann, node_annotations)

        if not is_loop:
            return None, {
                "midpoint_router_used": False,
                "reason": "not_self_loop",
            }

        if prev_node is None:
            return None, {
                "midpoint_router_used": False,
                "reason": "missing_loop_node",
            }

        side_order = ordered_loop_sides_from_geometry(arrow_ann, prev_node)

        attempts = []
        best_path = None
        best_debug = None
        best_score = float("inf")

        for side in side_order:
            guide_points = synthetic_fa_guide_points_for_side(
                arrow_ann,
                tail,
                head,
                side=side,
            )

            path, debug = route_self_loop_with_one_guide(
                image=image,
                arrow_ann=arrow_ann,
                text_annotations=text_annotations,
                node_annotations=node_annotations,
                tail=tail,
                head=head,
                loop_node=prev_node,
                guide_points=guide_points,
                side_name=side,
                node_ellipse_margin=node_ellipse_margin,
                corridor_thickness=corridor_thickness,
                pad=pad,
                midpoint_exclusion_radius=midpoint_exclusion_radius,
            )

            attempts.append(debug)

            if path is not None:
                # Prefer low inside ratio and long actual loop path.
                ratio = debug.get("inside_node_ratio", 1.0)
                plen = debug.get("path_length", 0.0)
                score = ratio - 0.0001 * plen

                if score < best_score:
                    best_score = score
                    best_path = path
                    best_debug = debug

                # First successful side is usually correct because side_order is geometry-prioritized.
                return path, {
                    "midpoint_router_used": True,
                    "reason": "passed",
                    "selected_side": side,
                    "inside_node_ratio": ratio,
                    "path_length": plen,
                    "midpoint": debug.get("midpoint"),
                    "attempts": attempts,
                }

        if best_path is not None:
            return best_path, {
                "midpoint_router_used": True,
                "reason": "best_available_path_used",
                "selected_side": best_debug.get("side"),
                "inside_node_ratio": best_debug.get("inside_node_ratio"),
                "path_length": best_debug.get("path_length"),
                "midpoint": best_debug.get("midpoint"),
                "attempts": attempts,
            }

        return None, {
            "midpoint_router_used": False,
            "reason": "all_sides_failed",
            "attempts": attempts,
        }

    def extract_fa_path(
        image,
        arrow_ann,
        text_annotations,
        node_annotations,
        tail,
        head,
        pad=12,
        ink_threshold=180,
        endpoint_keep_radius=38,
        node_ellipse_margin=10,
        text_forbid_margin=4,
        corridor_thickness=65,
        allow_white_bridge=True,
    ):
        h, w = image.shape[:2]

        prev_node = find_node_by_source_id(node_annotations, arrow_ann.get("arrow_prev"))
        next_node = find_node_by_source_id(node_annotations, arrow_ann.get("arrow_next"))

        guide_points = synthetic_fa_guide_points(
            arrow_ann,
            tail,
            head,
            prev_node,
            next_node,
        )

        ax, ay, aw, ah = arrow_ann["bbox"]

        xs = [ax, ax + aw, tail["x"], head["x"]] + [p[0] for p in guide_points]
        ys = [ay, ay + ah, tail["y"], head["y"]] + [p[1] for p in guide_points]

        x1p = clamp(int(min(xs)) - pad - corridor_thickness, 0, w - 1)
        y1p = clamp(int(min(ys)) - pad - corridor_thickness, 0, h - 1)
        x2p = clamp(int(max(xs)) + pad + corridor_thickness, 0, w - 1)
        y2p = clamp(int(max(ys)) + pad + corridor_thickness, 0, h - 1)

        crop = image[y1p:y2p + 1, x1p:x2p + 1]
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

        ch, cw = gray.shape
        local_guide = localize_points(guide_points, x1p, y1p)

        corridor = draw_polyline_mask(
            (ch, cw),
            local_guide,
            thickness=corridor_thickness,
        )

        arrow_region = np.zeros((ch, cw), dtype=bool)

        lx1 = clamp(ax - x1p, 0, cw - 1)
        ly1 = clamp(ay - y1p, 0, ch - 1)
        lx2 = clamp(ax + aw - x1p, 0, cw - 1)
        ly2 = clamp(ay + ah - y1p, 0, ch - 1)

        arrow_region[ly1:ly2 + 1, lx1:lx2 + 1] = True

        allowed = corridor | arrow_region

        ink = gray < ink_threshold
        ink_u8 = ink.astype(np.uint8) * 255
        kernel = np.ones((3, 3), np.uint8)
        ink_u8 = cv2.morphologyEx(ink_u8, cv2.MORPH_CLOSE, kernel, iterations=1)
        ink = ink_u8 > 0

        for ann in text_annotations:
            if not bbox_intersects(ann["bbox"], arrow_ann["bbox"]):
                continue

            bx, by, bw, bh = ann["bbox"]

            lx1 = clamp(bx - x1p - text_forbid_margin, 0, cw - 1)
            ly1 = clamp(by - y1p - text_forbid_margin, 0, ch - 1)
            lx2 = clamp(bx + bw - x1p + text_forbid_margin, 0, cw - 1)
            ly2 = clamp(by + bh - y1p + text_forbid_margin, 0, ch - 1)

            allowed[ly1:ly2 + 1, lx1:lx2 + 1] = False

        node_forbidden = np.zeros((ch, cw), dtype=bool)

        for ann in node_annotations:
            if not bbox_intersects(ann["bbox"], arrow_ann["bbox"]):
                continue

            emask = ellipse_mask_for_bbox(
                (ch, cw),
                ann["bbox"],
                x1p,
                y1p,
                margin=node_ellipse_margin,
            )

            allowed[emask] = False
            node_forbidden |= emask

        local_tail = (
            int(round(tail["x"] - x1p)),
            int(round(tail["y"] - y1p)),
        )

        local_head = (
            int(round(head["x"] - x1p)),
            int(round(head["y"] - y1p)),
        )

        is_loop, loop_node, _ = is_self_loop_arrow(arrow_ann, node_annotations)

        effective_endpoint_keep_radius = endpoint_keep_radius

        if is_loop:
            effective_endpoint_keep_radius = adaptive_endpoint_radius_for_loop(
                tail,
                head,
                endpoint_keep_radius,
            )

        for cx, cy in [local_tail, local_head]:
            cx = clamp(cx, 0, cw - 1)
            cy = clamp(cy, 0, ch - 1)

            disk = make_disk_mask((ch, cw), cx, cy, effective_endpoint_keep_radius)
            allowed[disk] = True

        if is_loop and loop_node is not None:
            # Re-block deep node interior after endpoint restoration.
            # This prevents close tail/head restore disks from making a tunnel through the node.
            node_core = ellipse_mask_for_bbox(
                (ch, cw),
                loop_node["bbox"],
                x1p,
                y1p,
                margin=-6,
            )
            allowed[node_core] = False

        if allow_white_bridge:
            ink_u8 = ink.astype(np.uint8)

            dist_to_ink = cv2.distanceTransform(
                (1 - ink_u8).astype(np.uint8),
                cv2.DIST_L2,
                3,
            )

            cost = 1.0 + 7.5 * dist_to_ink
            cost[ink] = 1.0
        else:
            allowed &= ink
            cost = np.ones((ch, cw), dtype=np.float32)

        guide_line = draw_polyline_mask(
            (ch, cw),
            local_guide,
            thickness=3,
        ).astype(np.uint8)

        dist_to_guide = cv2.distanceTransform(
            (1 - guide_line).astype(np.uint8),
            cv2.DIST_L2,
            3,
        )

        cost += 0.55 * dist_to_guide

        if node_forbidden.any():
            dist_from_node = cv2.distanceTransform(
                (1 - node_forbidden.astype(np.uint8)).astype(np.uint8),
                cv2.DIST_L2,
                3,
            )

            near_node = dist_from_node < 20
            cost[near_node] *= 4.5

        cost[~allowed] = 1e9

        start = nearest_allowed_pixel(allowed, local_tail, max_radius=75)
        goal = nearest_allowed_pixel(allowed, local_head, max_radius=75)

        if start is None or goal is None:
            return straight_line_path(tail, head)

        path_xy = weighted_dijkstra_path(cost, allowed, start, goal)

        if not path_xy or len(path_xy) < 2:
            return straight_line_path(tail, head)

        return [pt(x + x1p, y + y1p) for x, y in path_xy]


    def extract_fa_path_adaptive(
        image,
        arrow_ann,
        text_annotations,
        node_annotations,
        tail,
        head,
        bad_ratio_threshold=0.03,
    ):
        prev_node = find_node_by_source_id(node_annotations, arrow_ann.get("arrow_prev"))
        next_node = find_node_by_source_id(node_annotations, arrow_ann.get("arrow_next"))

        is_self_loop = is_same_node(prev_node, next_node)

        if not is_self_loop:
            path = extract_fa_path(
                image=image,
                arrow_ann=arrow_ann,
                text_annotations=text_annotations,
                node_annotations=node_annotations,
                tail=tail,
                head=head,
            )

            return path, {
                "adaptive": False,
                "self_loop": False,
                "method": "standard_fa_router",
                "inside_node_ratio": None,
                "selected_config": None,
                "passed": None,
            }

        midpoint_path, midpoint_debug = extract_fa_self_loop_path_via_midpoint(
            image=image,
            arrow_ann=arrow_ann,
            text_annotations=text_annotations,
            node_annotations=node_annotations,
            tail=tail,
            head=head,
            node_ellipse_margin=10,
            corridor_thickness=75,
        )

        if midpoint_path is not None:
            return midpoint_path, {
                "adaptive": True,
                "self_loop": True,
                "method": "midpoint_skeleton_router",
                "inside_node_ratio": midpoint_debug.get("inside_node_ratio"),
                "selected_config": {
                    "method": "synthetic_guide_to_real_midpoint",
                    "node_ellipse_margin": 10,
                    "corridor_thickness": 75,
                },
                "passed": True,
                "midpoint_debug": midpoint_debug,
            }

        best_path = None
        best_ratio = float("inf")
        best_cfg = None

        tried = [{
            "method": "midpoint_skeleton_router",
            "debug": midpoint_debug,
        }]

        for cfg in FA_RETRY_CONFIGS:
            path = extract_fa_path(
                image=image,
                arrow_ann=arrow_ann,
                text_annotations=text_annotations,
                node_annotations=node_annotations,
                tail=tail,
                head=head,
                **cfg,
            )

            ratio = path_inside_node_ratio_strict(
                path=path,
                node_bbox=prev_node["bbox"],
                margin=10,
            )

            tried.append({
                "method": "weighted_retry_router",
                "config": cfg,
                "inside_node_ratio": ratio,
            })

            if ratio < best_ratio:
                best_ratio = ratio
                best_path = path
                best_cfg = cfg

            if ratio <= bad_ratio_threshold:
                return path, {
                    "adaptive": True,
                    "self_loop": True,
                    "method": "weighted_retry_router",
                    "inside_node_ratio": ratio,
                    "selected_config": cfg,
                    "passed": True,
                    "tried": tried,
                }

        return best_path, {
            "adaptive": True,
            "self_loop": True,
            "method": "best_weighted_retry_router",
            "inside_node_ratio": best_ratio,
            "selected_config": best_cfg,
            "passed": False,
            "tried": tried,
        }


    def extract_arrow_path(
        image,
        arrow_ann,
        text_annotations,
        node_annotations,
        tail,
        head,
        routing_mode="auto",
        return_debug=False,
    ):
        if routing_mode == "auto":
            routing_mode = (
                "flowchart"
                if arrow_ann.get("waypoints") and len(arrow_ann.get("waypoints", [])) >= 2
                else "fa"
            )

        if routing_mode == "flowchart":
            path = extract_flowchart_path(
                image,
                arrow_ann,
                node_annotations + text_annotations,
                tail,
                head,
            )
            debug = {"routing_mode_used": "flowchart"}

        elif routing_mode == "fa":
            path, fa_debug = extract_fa_path_adaptive(
                image=image,
                arrow_ann=arrow_ann,
                text_annotations=text_annotations,
                node_annotations=node_annotations,
                tail=tail,
                head=head,
            )
            debug = {"routing_mode_used": "fa", "fa_adaptive_debug": fa_debug}

        elif routing_mode == "straight":
            path = straight_line_path(tail, head)
            debug = {"routing_mode_used": "straight"}

        elif routing_mode == "waypoint":
            path = waypoint_path(arrow_ann, tail, head)
            debug = {"routing_mode_used": "waypoint"}

        else:
            raise ValueError(f"Unknown routing mode: {routing_mode}")

        if return_debug:
            return path, debug

        return path


In [ ]:
if SHOULD_PREPARE_DATASET:
    # ============================================================
    # Overlay
    # ============================================================

    def draw_overlay(image, nodes, arrows):
        overlay = image.copy()

        for node in nodes:
            x, y, w, h = node["bbox"]

            cv2.rectangle(
                overlay,
                (x, y),
                (x + w, y + h),
                (0, 180, 0),
                2,
            )

            label = node.get("type", "node")
            cv2.putText(
                overlay,
                f'{label}:{node["id"]}',
                (x, max(15, y - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 150, 0),
                1,
                cv2.LINE_AA,
            )

        for arrow in arrows:
            path = arrow.get("path", [])

            if len(path) >= 2:
                pts = np.array(
                    [[int(round(p["x"])), int(round(p["y"]))] for p in path],
                    dtype=np.int32,
                )

                cv2.polylines(
                    overlay,
                    [pts],
                    False,
                    (0, 0, 255),
                    2,
                )

            x, y, w, h = arrow["bbox"]

            cv2.rectangle(
                overlay,
                (x, y),
                (x + w, y + h),
                (255, 0, 0),
                1,
            )

            tail = arrow["tail"]
            head = arrow["head"]

            cv2.circle(
                overlay,
                (int(round(tail["x"])), int(round(tail["y"]))),
                5,
                (255, 0, 255),
                -1,
            )

            cv2.circle(
                overlay,
                (int(round(head["x"])), int(round(head["y"]))),
                5,
                (0, 255, 0),
                -1,
            )

            cv2.putText(
                overlay,
                f'Tail{arrow["id"]}',
                (int(round(tail["x"])) + 5, int(round(tail["y"])) - 5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (255, 0, 255),
                1,
                cv2.LINE_AA,
            )

            cv2.putText(
                overlay,
                f'Head{arrow["id"]}',
                (int(round(head["x"])) + 5, int(round(head["y"])) - 5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (0, 140, 0),
                1,
                cv2.LINE_AA,
            )

            for p in arrow.get("equidistant_points", []):
                cv2.putText(
                    overlay,
                    p["label"],
                    (int(p["x"]), int(p["y"])),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.32,
                    (30, 30, 30),
                    1,
                    cv2.LINE_AA,
                )

        return overlay


In [ ]:
if SHOULD_PREPARE_DATASET:
    # ============================================================
    # Node-linking fallback
    # ============================================================

    def point_to_bbox_distance(p, bbox):
        """
        Distance from point to an axis-aligned bbox.
        Returns 0 if point lies inside the bbox.
        Coordinates must be in the same space.
        """
        x, y, w, h = bbox
        px = float(p["x"])
        py = float(p["y"])

        dx = max(x - px, 0.0, px - (x + w))
        dy = max(y - py, 0.0, py - (y + h))

        return math.hypot(dx, dy)


    def node_snap_threshold(node_anns):
        """
        Adaptive threshold in original image/JSON coordinate space.
        This avoids hardcoding one threshold for all datasets/resolutions.
        """
        if not node_anns:
            return 0.0

        diags = []
        for ann in node_anns:
            x, y, w, h = ann["bbox"]
            diags.append(math.hypot(w, h))

        med = float(np.median(diags)) if diags else 0.0
        return max(12.0, min(70.0, 0.35 * med))


    def infer_node_id_from_candidates(candidates, node_records, max_dist):
        """
        Geometric fallback for missing tail_node_id/head_node_id.

        It snaps an endpoint to the nearest node only if the endpoint is close
        enough to that node bbox. This preserves legitimate start arrows, whose
        free tail is usually far from every node.
        """
        best = None

        for cand_name, p in candidates:
            if p is None:
                continue

            for rec in node_records:
                d = point_to_bbox_distance(p, rec["raw_bbox"])

                if best is None or d < best["distance"]:
                    best = {
                        "node_id": rec["new_id"],
                        "distance": float(d),
                        "candidate": cand_name,
                        "point": {"x": float(p["x"]), "y": float(p["y"])},
                    }

        if best is None:
            return None, {
                "used": False,
                "reason": "no_candidate_or_node",
                "max_dist": max_dist,
            }

        if best["distance"] <= max_dist:
            return best["node_id"], {
                "used": True,
                "reason": "snapped_to_nearest_node",
                "max_dist": max_dist,
                **best,
            }

        return None, {
            "used": False,
            "reason": "nearest_node_too_far",
            "max_dist": max_dist,
            "nearest": best,
        }


    def resolve_arrow_node_ids(
        ann,
        tail,
        head,
        raw_path,
        node_id_map,
        node_records,
        snap_dist,
    ):
        """
        First trust the original annotation ids. If they fail, use geometric
        endpoint snapping. This fixes cases where train/test had matching ids
        but val used ids that did not map into node_id_map.
        """
        arrow_prev = ann.get("arrow_prev", None)
        arrow_next = ann.get("arrow_next", None)

        tail_node_id = node_id_map.get(arrow_prev, None)
        head_node_id = node_id_map.get(arrow_next, None)

        debug = {
            "arrow_prev": arrow_prev,
            "arrow_next": arrow_next,
            "tail_from_annotation": tail_node_id is not None,
            "head_from_annotation": head_node_id is not None,
            "snap_dist": snap_dist,
        }

        first_path = raw_path[0] if raw_path else None
        last_path = raw_path[-1] if raw_path else None

        if tail_node_id is None:
            tail_candidates = [
                ("tail_hint", tail),
                ("path_start", first_path),
            ]

            tail_node_id, tail_debug = infer_node_id_from_candidates(
                tail_candidates,
                node_records,
                snap_dist,
            )
            debug["tail_fallback"] = tail_debug
        else:
            debug["tail_fallback"] = {"used": False, "reason": "annotation_mapping_success"}

        if head_node_id is None:
            head_candidates = [
                ("head_hint", head),
                ("path_end", last_path),
            ]

            head_node_id, head_debug = infer_node_id_from_candidates(
                head_candidates,
                node_records,
                snap_dist,
            )
            debug["head_fallback"] = head_debug
        else:
            debug["head_fallback"] = {"used": False, "reason": "annotation_mapping_success"}

        return tail_node_id, head_node_id, debug


In [ ]:
if SHOULD_PREPARE_DATASET:
    # ============================================================
    # Per-image conversion
    # ============================================================

    def convert_one_image(
        image_info,
        annotations,
        image_dir,
        out_json_dir,
        out_overlay_dir,
        save_overlay=True,
        trace_from_image=True,
        mask_w=1024,
        mask_h=1024,
        routing_mode="auto",
        save_debug=True,
    ):
        file_name = image_info["file_name"]
        image_path = image_dir / file_name

        if not image_path.exists():
            return {"image": file_name, "status": "missing_image", "arrows": 0, "nodes": 0}

        image = cv2.imread(str(image_path))

        if image is None:
            return {"image": file_name, "status": "unreadable_image", "arrows": 0, "nodes": 0}

        json_w = int(image_info.get("width", image.shape[1]))
        json_h = int(image_info.get("height", image.shape[0]))

        sx = mask_w / json_w
        sy = mask_h / json_h

        arrow_anns = [a for a in annotations if a.get("category") == "arrow"]
        text_anns = [a for a in annotations if a.get("category") == "text"]
        node_anns = [a for a in annotations if a.get("category") not in {"arrow", "text"}]

        node_id_map = {}
        nodes = []
        overlay_nodes = []
        node_records = []

        for idx, ann in enumerate(node_anns, start=1):
            new_id = f"N{idx}"

            source_id = ann.get("id")
            source_aid = ann.get("aid")

            if source_id is not None:
                node_id_map[source_id] = new_id

            if source_aid is not None:
                node_id_map[source_aid] = new_id

            nodes.append({"id": new_id, "bbox": scale_bbox(ann["bbox"], sx, sy)})

            overlay_nodes.append({
                "id": new_id,
                "type": ann.get("category", "node"),
                "bbox": [int(round(v)) for v in ann["bbox"]],
            })

            node_records.append({
                "new_id": new_id,
                "raw_bbox": ann["bbox"],
                "source_id": source_id,
                "source_aid": source_aid,
                "type": ann.get("category", "node"),
            })

        snap_dist = node_snap_threshold(node_anns)

        arrows = []
        overlay_arrows = []

        for idx, ann in enumerate(arrow_anns, start=1):
            k = ann.get("keypoints", [])

            if len(k) < 6:
                continue

            tail = pt(k[0], k[1])
            head = pt(k[3], k[4])

            if trace_from_image:
                raw_path, routing_debug = extract_arrow_path(
                    image=image,
                    arrow_ann=ann,
                    text_annotations=text_anns,
                    node_annotations=node_anns,
                    tail=tail,
                    head=head,
                    routing_mode=routing_mode,
                    return_debug=True,
                )
            else:
                raw_path = waypoint_path(ann, tail, head)
                routing_debug = {"routing_mode_used": "waypoint_or_straight_no_trace"}

            tail_scaled = scale_point(tail, sx, sy)
            head_scaled = scale_point(head, sx, sy)
            path_scaled = scale_path(raw_path, sx, sy)

            tail_node_id, head_node_id, node_link_debug = resolve_arrow_node_ids(
                ann=ann,
                tail=tail,
                head=head,
                raw_path=raw_path,
                node_id_map=node_id_map,
                node_records=node_records,
                snap_dist=snap_dist,
            )

            arrow_json = {
                "id": idx,
                "tail_hint": tail_scaled,
                "head_hint": head_scaled,
                "path": path_scaled,
                "tail_node_id": tail_node_id,
                "head_node_id": head_node_id,
                "scale_debug": {
                    "json_space": {"w": json_w, "h": json_h},
                    "mask_space": {"w": mask_w, "h": mask_h},
                    "sx": sx,
                    "sy": sy,
                    "sx_arrow": sx,
                    "sy_arrow": sy,
                    "routing_mode_requested": routing_mode,
                },
                "control_points": [tail_scaled, head_scaled],
            }

            if save_debug:
                arrow_json["routing_debug"] = routing_debug

            arrows.append(arrow_json)

            overlay_arrows.append({
                "id": idx,
                "bbox": [int(round(v)) for v in ann["bbox"]],
                "tail": tail,
                "head": head,
                "path": raw_path,
                "equidistant_points": sample_equidistant_points(raw_path, count=40),
            })

        out = {
            "image": file_name,
            "size": {"w": mask_w, "h": mask_h},
            "nodes": nodes,
            "arrows": arrows,
        }

        out_json_dir.mkdir(parents=True, exist_ok=True)
        out_json_path = out_json_dir / f"{Path(file_name).stem}.json"

        with open(out_json_path, "w", encoding="utf-8") as f:
            json.dump(out, f, indent=2)

        if save_overlay:
            out_overlay_dir.mkdir(parents=True, exist_ok=True)
            overlay = draw_overlay(image.copy(), overlay_nodes, overlay_arrows)
            overlay_path = out_overlay_dir / f"{Path(file_name).stem}_overlay.png"
            cv2.imwrite(str(overlay_path), overlay)

        return {"image": file_name, "status": "ok", "arrows": len(arrows), "nodes": len(nodes)}


In [ ]:
if SHOULD_PREPARE_DATASET:

    # ============================================================
    # Repository-level validation
    # ============================================================

    def require_nonempty_path(value: str, variable_name: str) -> Path:
        value = str(value).strip()
        if not value:
            raise ValueError(
                f"{variable_name} is blank. Fill it in the USER SETTINGS cell before running."
            )
        return Path(value).expanduser().resolve()


    def annotation_bounds_report(json_path: Path) -> Dict[str, int]:
        """Validate one converted annotation without changing it."""
        with json_path.open("r", encoding="utf-8") as f:
            record = json.load(f)

        width = int(record.get("size", {}).get("w", 0))
        height = int(record.get("size", {}).get("h", 0))

        report = {
            "files": 1,
            "nodes": 0,
            "arrows": 0,
            "invalid_canvas": 0,
            "out_of_bounds_nodes": 0,
            "out_of_bounds_endpoints": 0,
            "out_of_bounds_path_points": 0,
            "unlinked_tail_ids": 0,
            "unlinked_head_ids": 0,
        }

        if width != TARGET_WIDTH or height != TARGET_HEIGHT:
            report["invalid_canvas"] += 1

        for node in record.get("nodes", []):
            report["nodes"] += 1
            x, y, w, h = node["bbox"]
            if x < 0 or y < 0 or w < 0 or h < 0 or x + w > width or y + h > height:
                report["out_of_bounds_nodes"] += 1

        for arrow in record.get("arrows", []):
            report["arrows"] += 1

            for key in ("tail_hint", "head_hint"):
                point = arrow.get(key)
                if point is None:
                    report["out_of_bounds_endpoints"] += 1
                    continue
                x, y = float(point["x"]), float(point["y"])
                if x < 0 or y < 0 or x >= width or y >= height:
                    report["out_of_bounds_endpoints"] += 1

            for point in arrow.get("path", []):
                x, y = float(point["x"]), float(point["y"])
                if x < 0 or y < 0 or x >= width or y >= height:
                    report["out_of_bounds_path_points"] += 1

            if arrow.get("tail_node_id") is None:
                report["unlinked_tail_ids"] += 1
            if arrow.get("head_node_id") is None:
                report["unlinked_head_ids"] += 1

        return report


    def merge_count_reports(reports: Iterable[Dict[str, int]]) -> Dict[str, int]:
        merged: Dict[str, int] = defaultdict(int)
        for report in reports:
            for key, value in report.items():
                merged[key] += int(value)
        return dict(merged)


    def validate_converted_repository(output_root: Path) -> Dict[str, object]:
        json_files = sorted(output_root.glob("*/*/json/*.json"))
        count_report = merge_count_reports(
            annotation_bounds_report(path) for path in json_files
        )

        split_counts: Dict[str, Dict[str, int]] = defaultdict(dict)
        for family_dir in sorted(p for p in output_root.iterdir() if p.is_dir()):
            for split_dir in sorted(p for p in family_dir.iterdir() if p.is_dir()):
                json_dir = split_dir / "json"
                if json_dir.exists():
                    split_counts[family_dir.name][split_dir.name] = len(
                        list(json_dir.glob("*.json"))
                    )

        return {
            "target_size": {"w": TARGET_WIDTH, "h": TARGET_HEIGHT},
            "families": dict(split_counts),
            "counts": count_report,
        }


In [ ]:
if SHOULD_PREPARE_DATASET:
    RESOLVED_SOURCE_DATASETS_ROOT = resolve_or_download_source_dataset()
    SOURCE_DATASETS_ROOT = str(RESOLVED_SOURCE_DATASETS_ROOT)
    OUTPUT_ANNOTATIONS_ROOT = str(WORKING_CONVERTED_DATASET_ROOT)
    TARGET_WIDTH = 1024
    TARGET_HEIGHT = 1024
    ROUTING_MODE = "auto"
    TRACE_FROM_IMAGE = True
    SAVE_OVERLAYS = False
    SAVE_ROUTING_DEBUG = False
    WORKERS = min(4, os.cpu_count() or 1)
    # ============================================================
    # RUN THE CONVERSION
    # ============================================================

    source_root = Path(RESOLVED_SOURCE_DATASETS_ROOT)
    output_root = Path(WORKING_CONVERTED_DATASET_ROOT); output_root.mkdir(parents=True, exist_ok=True)

    if source_root == output_root:
        raise ValueError("SOURCE_DATASETS_ROOT and OUTPUT_ANNOTATIONS_ROOT must be different.")

    if ROUTING_MODE not in {"auto", "flowchart", "fa", "straight", "waypoint"}:
        raise ValueError(f"Unsupported ROUTING_MODE: {ROUTING_MODE}")

    if TARGET_WIDTH <= 0 or TARGET_HEIGHT <= 0:
        raise ValueError("TARGET_WIDTH and TARGET_HEIGHT must be positive.")

    workers = max(1, int(WORKERS))
    output_root.mkdir(parents=True, exist_ok=True)

    repository_results: Dict[str, object] = {}

    for family in list(DATASET_FAMILIES):
        family_source = source_root / family
        family_output = output_root / family

        if not family_source.exists():
            raise FileNotFoundError(
                f"Dataset family '{family}' was not found at: {family_source}"
            )

        if list(expected_splits()) if USE_EXPLICIT_DATASET_SPLITS else None is None:
            split_pairs = discover_json_splits(family_source)
        else:
            split_pairs = []
            for split_name in list(expected_splits()) if USE_EXPLICIT_DATASET_SPLITS else None:
                image_dir = family_source / split_name
                json_path = family_source / f"{split_name}.json"

                if not image_dir.exists():
                    print(f"Skipping {family}/{split_name}: missing image folder {image_dir}")
                    continue
                if not json_path.exists():
                    print(f"Skipping {family}/{split_name}: missing source JSON {json_path}")
                    continue

                split_pairs.append((split_name, image_dir, json_path))

        if not split_pairs:
            raise RuntimeError(
                f"No valid split was found for '{family}'. Expected train.json with "
                "a matching train/ image folder, and similarly for other splits."
            )

        print("\n" + "=" * 72)
        print(f"DATASET FAMILY: {family}")
        print("=" * 72)
        for split_name, image_dir, json_path in split_pairs:
            print(f"  {split_name}: {json_path.name} -> {image_dir.name}/")

        family_results: Dict[str, object] = {}

        for split_name, image_dir, json_path in split_pairs:
            family_results[split_name] = process_split(
                split_name=split_name,
                image_dir=image_dir,
                json_path=json_path,
                out_root=family_output,
                workers=workers,
                save_overlay=SAVE_OVERLAYS,
                trace_from_image=TRACE_FROM_IMAGE,
                mask_w=TARGET_WIDTH,
                mask_h=TARGET_HEIGHT,
                routing_mode=ROUTING_MODE,
                save_debug=SAVE_ROUTING_DEBUG,
            )

        family_output.mkdir(parents=True, exist_ok=True)
        with (family_output / "all_splits_summary.json").open("w", encoding="utf-8") as f:
            json.dump(family_results, f, indent=2)

        repository_results[family] = family_results

    with (output_root / "all_families_summary.json").open("w", encoding="utf-8") as f:
        json.dump(repository_results, f, indent=2)

    validation_report = validate_converted_repository(output_root)
    with (output_root / "conversion_validation_report.json").open("w", encoding="utf-8") as f:
        json.dump(validation_report, f, indent=2)

    print("\n" + "=" * 72)
    print("CONVERSION COMPLETE")
    print("=" * 72)
    print(json.dumps(validation_report, indent=2))
    print(
        "\nNext step: run the image-resizing notebook to create 1024 × 1024 "
        "images matching these annotation coordinates."
    )

In [ ]:
if SHOULD_PREPARE_DATASET:

    import json
    import os
    from collections import defaultdict
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from pathlib import Path
    from typing import Dict, Iterable, List, Optional, Sequence, Tuple

    from PIL import Image, ImageOps

    try:
        from tqdm.auto import tqdm
    except Exception:
        tqdm = None

    print(f"Pillow version: {Image.__version__}")


In [ ]:
if SHOULD_PREPARE_DATASET:
    SUPPORTED_IMAGE_EXTENSIONS = {
        ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"
    }


    def require_nonempty_path(value: str, variable_name: str) -> Path:
        value = str(value).strip()
        if not value:
            raise ValueError(
                f"{variable_name} is blank. Fill it in the USER SETTINGS cell before running."
            )
        path = Path(value).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f"{variable_name} does not exist: {path}")
        return path


    def discover_converted_splits(
        converted_root: Path,
        families: Sequence[str],
        requested_splits: Optional[Sequence[str]] = None,
    ) -> List[Tuple[str, str, Path]]:
        """Return (family, split, json_dir) entries that actually exist."""
        discovered: List[Tuple[str, str, Path]] = []

        for family in families:
            family_dir = converted_root / family
            if not family_dir.exists():
                print(f"Skipping family '{family}': not found at {family_dir}")
                continue

            if requested_splits is None:
                candidate_split_dirs = sorted(
                    path for path in family_dir.iterdir() if path.is_dir()
                )
            else:
                candidate_split_dirs = [family_dir / split for split in requested_splits]

            for split_dir in candidate_split_dirs:
                json_dir = split_dir / "json"
                if json_dir.exists() and any(json_dir.glob("*.json")):
                    discovered.append((family, split_dir.name, json_dir))
                elif requested_splits is not None:
                    print(
                        f"Skipping {family}/{split_dir.name}: no converted JSON files at {json_dir}"
                    )

        return discovered


    def build_source_image_index(search_root: Path) -> Dict[str, List[Path]]:
        """
        Index source images by both exact filename and lowercase stem.

        Lists are retained so duplicate names can be detected safely.
        """
        index: Dict[str, List[Path]] = defaultdict(list)

        for path in search_root.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in SUPPORTED_IMAGE_EXTENSIONS:
                continue

            index[path.name.lower()].append(path)
            index[path.stem.lower()].append(path)

        return dict(index)


    def resolve_source_image(
        original_root: Path,
        family: str,
        split: str,
        image_name: str,
        split_index: Dict[str, List[Path]],
        family_index: Dict[str, List[Path]],
    ) -> Path:
        """Resolve one source image without silently choosing an ambiguous duplicate."""
        expected = original_root / family / split / image_name
        if expected.exists():
            return expected

        filename_key = Path(image_name).name.lower()
        stem_key = Path(image_name).stem.lower()

        for index, scope_name in (
            (split_index, f"{family}/{split}"),
            (family_index, family),
        ):
            candidates = index.get(filename_key, [])
            if not candidates:
                candidates = index.get(stem_key, [])

            unique_candidates = sorted(set(candidates))
            if len(unique_candidates) == 1:
                return unique_candidates[0]
            if len(unique_candidates) > 1:
                formatted = "\n".join(f"  - {path}" for path in unique_candidates)
                raise RuntimeError(
                    f"Ambiguous source image '{image_name}' inside {scope_name}:\n{formatted}"
                )

        raise FileNotFoundError(
            f"Could not find source image '{image_name}' for {family}/{split}."
        )


In [ ]:
if SHOULD_PREPARE_DATASET:
    def choose_resampling(width: int, height: int, target_width: int, target_height: int):
        """Use Lanczos when reducing an image and bicubic interpolation otherwise."""
        if width > target_width or height > target_height:
            return Image.Resampling.LANCZOS
        return Image.Resampling.BICUBIC


    def save_resized_image(
        source_path: Path,
        destination_path: Path,
        target_width: int,
        target_height: int,
        overwrite: bool,
    ) -> Dict[str, object]:
        """
        Resize and save one image.

        Returns a serializable status dictionary for the final report.
        """
        if destination_path.exists() and not overwrite:
            try:
                with Image.open(destination_path) as existing:
                    if existing.size == (target_width, target_height):
                        return {
                            "status": "skipped_existing",
                            "source": str(source_path),
                            "destination": str(destination_path),
                            "original_size": list(existing.size),
                            "output_size": list(existing.size),
                        }
            except Exception:
                # A corrupt or unreadable destination should be regenerated.
                pass

        destination_path.parent.mkdir(parents=True, exist_ok=True)

        with Image.open(source_path) as image:
            image = ImageOps.exif_transpose(image)
            original_size = image.size

            resample = choose_resampling(
                width=original_size[0],
                height=original_size[1],
                target_width=target_width,
                target_height=target_height,
            )
            resized = image.resize((target_width, target_height), resample=resample)

            suffix = destination_path.suffix.lower()
            save_kwargs: Dict[str, object] = {}

            # JPEG does not support alpha channels or palette transparency.
            if suffix in {".jpg", ".jpeg"} and resized.mode not in {"L", "RGB"}:
                resized = resized.convert("RGB")
                save_kwargs = {"quality": 95, "subsampling": 0}

            # Preserve binary/grayscale/RGB modes for the diagram images whenever possible.
            resized.save(destination_path, **save_kwargs)

        return {
            "status": "resized",
            "source": str(source_path),
            "destination": str(destination_path),
            "original_size": list(original_size),
            "output_size": [target_width, target_height],
        }


    def read_annotation_image_name(json_path: Path) -> str:
        with json_path.open("r", encoding="utf-8") as handle:
            record = json.load(handle)

        image_name = str(record.get("image", "")).strip()
        if not image_name:
            raise ValueError(f"Missing or empty 'image' field in {json_path}")

        declared_size = record.get("size", {})
        declared_width = int(declared_size.get("w", 0))
        declared_height = int(declared_size.get("h", 0))

        if (declared_width, declared_height) != (TARGET_WIDTH, TARGET_HEIGHT):
            raise ValueError(
                f"Annotation canvas mismatch in {json_path}: "
                f"{declared_width} × {declared_height}, expected "
                f"{TARGET_WIDTH} × {TARGET_HEIGHT}."
            )

        return image_name


In [ ]:
if SHOULD_PREPARE_DATASET:
    def process_converted_split(
        original_root: Path,
        converted_root: Path,
        family: str,
        split: str,
        json_dir: Path,
        target_width: int,
        target_height: int,
        overwrite: bool,
        workers: int,
    ) -> Dict[str, object]:
        json_files = sorted(json_dir.glob("*.json"))
        output_image_dir = converted_root / family / split / "images"
        output_image_dir.mkdir(parents=True, exist_ok=True)

        normal_split_root = original_root / family / split
        split_search_root = (
            normal_split_root if normal_split_root.exists() else original_root / family
        )
        family_search_root = original_root / family

        if not family_search_root.exists():
            raise FileNotFoundError(
                f"Original dataset family was not found: {family_search_root}"
            )

        split_index = build_source_image_index(split_search_root)
        if split_search_root == family_search_root:
            family_index = split_index
        else:
            family_index = build_source_image_index(family_search_root)

        tasks: List[Tuple[Path, Path, Path]] = []
        preparation_errors: List[Dict[str, str]] = []

        for json_path in json_files:
            try:
                image_name = read_annotation_image_name(json_path)
                source_path = resolve_source_image(
                    original_root=original_root,
                    family=family,
                    split=split,
                    image_name=image_name,
                    split_index=split_index,
                    family_index=family_index,
                )
                destination_path = output_image_dir / image_name
                tasks.append((json_path, source_path, destination_path))
            except Exception as exc:
                preparation_errors.append(
                    {
                        "json": str(json_path),
                        "error": f"{type(exc).__name__}: {exc}",
                    }
                )

        results: List[Dict[str, object]] = []
        execution_errors: List[Dict[str, str]] = []

        def run_one(task: Tuple[Path, Path, Path]) -> Dict[str, object]:
            json_path, source_path, destination_path = task
            result = save_resized_image(
                source_path=source_path,
                destination_path=destination_path,
                target_width=target_width,
                target_height=target_height,
                overwrite=overwrite,
            )
            result["json"] = str(json_path)
            return result

        if workers <= 1:
            iterator = tasks
            if tqdm is not None:
                iterator = tqdm(tasks, desc=f"{family}/{split}", unit="image")

            for task in iterator:
                try:
                    results.append(run_one(task))
                except Exception as exc:
                    execution_errors.append(
                        {
                            "json": str(task[0]),
                            "source": str(task[1]),
                            "error": f"{type(exc).__name__}: {exc}",
                        }
                    )
        else:
            with ThreadPoolExecutor(max_workers=workers) as executor:
                future_to_task = {
                    executor.submit(run_one, task): task for task in tasks
                }
                completed = as_completed(future_to_task)
                if tqdm is not None:
                    completed = tqdm(
                        completed,
                        total=len(future_to_task),
                        desc=f"{family}/{split}",
                        unit="image",
                    )

                for future in completed:
                    task = future_to_task[future]
                    try:
                        results.append(future.result())
                    except Exception as exc:
                        execution_errors.append(
                            {
                                "json": str(task[0]),
                                "source": str(task[1]),
                                "error": f"{type(exc).__name__}: {exc}",
                            }
                        )

        status_counts: Dict[str, int] = defaultdict(int)
        for result in results:
            status_counts[str(result["status"])] += 1

        summary = {
            "family": family,
            "split": split,
            "json_files": len(json_files),
            "prepared_tasks": len(tasks),
            "status_counts": dict(status_counts),
            "preparation_errors": preparation_errors,
            "execution_errors": execution_errors,
            "output_image_dir": str(output_image_dir),
        }

        with (converted_root / family / split / "image_resize_summary.json").open(
            "w", encoding="utf-8"
        ) as handle:
            json.dump(summary, handle, indent=2)

        return summary


In [ ]:
if SHOULD_PREPARE_DATASET:
    def verify_reconstructed_dataset(
        converted_root: Path,
        discovered_splits: Sequence[Tuple[str, str, Path]],
        target_width: int,
        target_height: int,
    ) -> Dict[str, object]:
        split_reports: List[Dict[str, object]] = []
        total_json = 0
        total_images = 0
        missing_images: List[str] = []
        invalid_dimensions: List[Dict[str, object]] = []
        unreadable_images: List[Dict[str, str]] = []
        unexpected_images: List[str] = []

        for family, split, json_dir in discovered_splits:
            image_dir = converted_root / family / split / "images"
            json_files = sorted(json_dir.glob("*.json"))
            expected_names: set[str] = set()

            for json_path in json_files:
                try:
                    expected_names.add(read_annotation_image_name(json_path))
                except Exception as exc:
                    unreadable_images.append(
                        {
                            "path": str(json_path),
                            "error": f"Annotation error: {type(exc).__name__}: {exc}",
                        }
                    )

            existing_image_paths = sorted(
                path
                for path in image_dir.glob("*")
                if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
            )
            existing_names = {path.name for path in existing_image_paths}

            for expected_name in sorted(expected_names):
                image_path = image_dir / expected_name
                if not image_path.exists():
                    missing_images.append(str(image_path))
                    continue

                try:
                    with Image.open(image_path) as image:
                        if image.size != (target_width, target_height):
                            invalid_dimensions.append(
                                {
                                    "path": str(image_path),
                                    "actual": list(image.size),
                                    "expected": [target_width, target_height],
                                }
                            )
                except Exception as exc:
                    unreadable_images.append(
                        {
                            "path": str(image_path),
                            "error": f"{type(exc).__name__}: {exc}",
                        }
                    )

            for extra_name in sorted(existing_names - expected_names):
                unexpected_images.append(str(image_dir / extra_name))

            split_reports.append(
                {
                    "family": family,
                    "split": split,
                    "annotations": len(json_files),
                    "expected_images": len(expected_names),
                    "existing_images": len(existing_image_paths),
                }
            )
            total_json += len(json_files)
            total_images += len(existing_image_paths)

        return {
            "target_size": [target_width, target_height],
            "total_annotations": total_json,
            "total_output_images": total_images,
            "missing_images": missing_images,
            "invalid_dimensions": invalid_dimensions,
            "unreadable_files": unreadable_images,
            "unexpected_images": unexpected_images,
            "splits": split_reports,
            "is_complete": not (
                missing_images
                or invalid_dimensions
                or unreadable_images
                or unexpected_images
            ),
        }


In [ ]:
if SHOULD_PREPARE_DATASET:
    ORIGINAL_DATASET_ROOT = str(RESOLVED_SOURCE_DATASETS_ROOT)
    CONVERTED_DATASET_ROOT = str(WORKING_CONVERTED_DATASET_ROOT)
    FAMILIES_TO_PROCESS = list(DATASET_FAMILIES)
    SPLITS_TO_PROCESS = list(expected_splits()) if USE_EXPLICIT_DATASET_SPLITS else None
    TARGET_WIDTH = 1024
    TARGET_HEIGHT = 1024
    OVERWRITE_EXISTING = False
    WORKERS = min(4, os.cpu_count() or 1)
    # ============================================================
    # RUN THE IMAGE RESIZING
    # ============================================================

    original_root = require_nonempty_path(
        str(RESOLVED_SOURCE_DATASETS_ROOT), "ORIGINAL_DATASET_ROOT"
    )
    converted_root = require_nonempty_path(
        str(WORKING_CONVERTED_DATASET_ROOT), "CONVERTED_DATASET_ROOT"
    )

    if original_root == converted_root:
        raise ValueError(
            "ORIGINAL_DATASET_ROOT and CONVERTED_DATASET_ROOT must be different."
        )

    if TARGET_WIDTH <= 0 or TARGET_HEIGHT <= 0:
        raise ValueError("TARGET_WIDTH and TARGET_HEIGHT must be positive integers.")

    workers = max(1, int(WORKERS))

    discovered_splits = discover_converted_splits(
        converted_root=converted_root,
        families=list(DATASET_FAMILIES),
        requested_splits=list(expected_splits()) if USE_EXPLICIT_DATASET_SPLITS else None,
    )

    if not discovered_splits:
        raise RuntimeError(
            "No converted split containing JSON annotations was found. "
            "Run 00_Convert_Original_Annotations.ipynb first."
        )

    print("Discovered converted splits:")
    for family, split, json_dir in discovered_splits:
        print(f"  - {family}/{split}: {len(list(json_dir.glob('*.json')))} annotations")

    all_summaries: List[Dict[str, object]] = []

    for family, split, json_dir in discovered_splits:
        summary = process_converted_split(
            original_root=original_root,
            converted_root=converted_root,
            family=family,
            split=split,
            json_dir=json_dir,
            target_width=TARGET_WIDTH,
            target_height=TARGET_HEIGHT,
            overwrite=OVERWRITE_EXISTING,
            workers=workers,
        )
        all_summaries.append(summary)

    validation_report = verify_reconstructed_dataset(
        converted_root=converted_root,
        discovered_splits=discovered_splits,
        target_width=TARGET_WIDTH,
        target_height=TARGET_HEIGHT,
    )

    final_report = {
        "settings": {
            "target_width": TARGET_WIDTH,
            "target_height": TARGET_HEIGHT,
            "overwrite_existing": OVERWRITE_EXISTING,
            "workers": workers,
        },
        "resize_summaries": all_summaries,
        "validation": validation_report,
    }

    report_path = converted_root / "image_resize_validation_report.json"
    with report_path.open("w", encoding="utf-8") as handle:
        json.dump(final_report, handle, indent=2)

    print("\n" + "=" * 72)
    print("IMAGE RESIZING COMPLETE")
    print("=" * 72)
    print(f"Annotations checked : {validation_report['total_annotations']}")
    print(f"Output images       : {validation_report['total_output_images']}")
    print(f"Missing images      : {len(validation_report['missing_images'])}")
    print(f"Invalid dimensions  : {len(validation_report['invalid_dimensions'])}")
    print(f"Unreadable files    : {len(validation_report['unreadable_files'])}")
    print(f"Unexpected images   : {len(validation_report['unexpected_images'])}")
    print(f"Dataset complete    : {validation_report['is_complete']}")
    print(f"Validation report   : {report_path}")

    if not validation_report["is_complete"]:
        print(
            "\nThe reconstructed dataset is not yet complete. "
            "Inspect the validation report before training."
        )
    else:
        print(
            "\nThe images and converted annotations now share the same "
            "1024 × 1024 coordinate system."
        )

In [ ]:
if SHOULD_PREPARE_DATASET:
    RESOLVED_DATASET_ROOT = Path(WORKING_CONVERTED_DATASET_ROOT).resolve()
    DATASET_REPORT = validate_converted_dataset_root(RESOLVED_DATASET_ROOT, verbose=True)
    if not DATASET_REPORT['valid']:
        raise RuntimeError('Dataset reconstruction completed but the resulting structure failed validation.')
    DATASET_ROOT = str(RESOLVED_DATASET_ROOT)
elif RESOLVED_DATASET_ROOT is not None:
    DATASET_ROOT = str(RESOLVED_DATASET_ROOT)


## 3. Optional plain ResNet34 node-model training

This is the publication training path. It is skipped when a node checkpoint is available unless `FORCE_TRAIN_NODE_MODEL=True`. On a two-GPU hosted-runtime runtime, training uses `DataParallel`; saved weights are unwrapped so the inference loader remains compatible.

In [ ]:
if SHOULD_TRAIN_NODE:
    if RESOLVED_DATASET_ROOT is None:
        raise RuntimeError('Node training requested but DATASET_ROOT is unavailable.')
    if RESOLVED_NODE_MASK_ROOT is None:
        raise FileNotFoundError('Node training requested but node_mask_dataset could not be located. Set NODE_MASK_DATASET_ROOT.')
    IMAGE_DIR = str(RESOLVED_DATASET_ROOT)
    MASK_DIR = str(RESOLVED_NODE_MASK_ROOT / 'masks')
    TRAIN_SPLIT_FILE = str(RESOLVED_NODE_MASK_ROOT / 'splits' / 'train.txt')
    VAL_SPLIT_FILE = str(RESOLVED_NODE_MASK_ROOT / 'splits' / 'val.txt')
    TEST_SPLIT_FILE = str(RESOLVED_NODE_MASK_ROOT / 'splits' / 'test.txt')
    OUTPUT_DIR = str(WORKING_ROOT / 'trained_models' / 'node_resnet34')
    RESUME_CHECKPOINT = ''
    IMAGE_SIZE = 256
    BATCH_SIZE = 16
    NUM_WORKERS = min(NUM_DATALOADER_WORKERS, os.cpu_count() or 1)
    EPOCHS = 60
    SEED = 42
    LEARNING_RATE = 3e-4
    WEIGHT_DECAY = 1e-4
    USE_AMP = True
    USE_IMAGENET_PRETRAINED = True
    HORIZONTAL_FLIP_PROBABILITY = 0.5
    MAX_ROTATION_DEGREES = 3.0
    METRIC_THRESHOLD = 0.50
    NORMALIZATION_MEAN = (0.485, 0.456, 0.406)
    NORMALIZATION_STD = (0.229, 0.224, 0.225)
    NUM_VISUAL_EXAMPLES = 3


In [ ]:
if SHOULD_TRAIN_NODE:


    import csv
    import json
    import math
    import os
    import random
    import time
    from dataclasses import dataclass
    from pathlib import Path
    from typing import Dict, Iterable, List, Optional, Sequence, Tuple

    import matplotlib.pyplot as plt
    import numpy as np
    from PIL import Image

    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torchvision.models as tvm
    import torchvision.transforms.functional as TF
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import CosineAnnealingLR
    from torch.utils.data import DataLoader, Dataset
    from torchvision.transforms import InterpolationMode

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
if SHOULD_TRAIN_NODE:

    def set_global_seed(seed: int) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # Deterministic settings make reruns more comparable.
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


    def seed_worker(worker_id: int) -> None:
        worker_seed = torch.initial_seed() % (2**32)
        random.seed(worker_seed)
        np.random.seed(worker_seed)


    set_global_seed(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    AMP_ENABLED = bool(USE_AMP and DEVICE.type == "cuda")
    print("Device:", DEVICE)
    print("Automatic mixed precision:", AMP_ENABLED)


In [ ]:
if SHOULD_TRAIN_NODE:

    IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}


    def require_path(value: str, label: str, *, directory: bool) -> Path:
        if not str(value).strip():
            raise ValueError(f"{label} is blank. Fill it in the USER PATHS cell.")
        path = Path(value).expanduser()
        if directory and not path.is_dir():
            raise NotADirectoryError(f"{label} is not a directory: {path}")
        if not directory and not path.is_file():
            raise FileNotFoundError(f"{label} is not a file: {path}")
        return path.resolve()


    image_dir = require_path(IMAGE_DIR, "IMAGE_DIR", directory=True)
    mask_dir = require_path(MASK_DIR, "MASK_DIR", directory=True)
    train_split_path = require_path(TRAIN_SPLIT_FILE, "TRAIN_SPLIT_FILE", directory=False)
    val_split_path = require_path(VAL_SPLIT_FILE, "VAL_SPLIT_FILE", directory=False)
    test_split_path = require_path(TEST_SPLIT_FILE, "TEST_SPLIT_FILE", directory=False)

    if not str(OUTPUT_DIR).strip():
        raise ValueError("OUTPUT_DIR is blank. Fill it in the USER PATHS cell.")
    output_dir = Path(OUTPUT_DIR).expanduser().resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    resume_checkpoint_path: Optional[Path] = None
    if str(RESUME_CHECKPOINT).strip():
        resume_checkpoint_path = require_path(
            RESUME_CHECKPOINT, "RESUME_CHECKPOINT", directory=False
        )

    print("Images:", image_dir)
    print("Masks:", mask_dir)
    print("Output:", output_dir)


In [ ]:
if SHOULD_TRAIN_NODE:

    def read_manifest(path: Path) -> List[str]:
        entries = [line.strip() for line in path.read_text(encoding="utf-8").splitlines()]
        entries = [entry for entry in entries if entry and not entry.startswith("#")]
        if not entries:
            raise ValueError(f"Manifest is empty: {path}")
        if len(entries) != len(set(entries)):
            duplicates = sorted({x for x in entries if entries.count(x) > 1})
            raise ValueError(f"Duplicate entries in {path.name}: {duplicates[:10]}")
        return entries


    train_entries = read_manifest(train_split_path)
    val_entries = read_manifest(val_split_path)
    test_entries = read_manifest(test_split_path)

    train_set, val_set, test_set = set(train_entries), set(val_entries), set(test_entries)
    if train_set & val_set:
        raise ValueError("The training and validation manifests overlap.")
    if train_set & test_set:
        raise ValueError("The training and test manifests overlap.")
    if val_set & test_set:
        raise ValueError("The validation and test manifests overlap.")

    print(f"Training samples:   {len(train_entries)}")
    print(f"Validation samples: {len(val_entries)}")
    print(f"Test samples:       {len(test_entries)}")
    print(f"Total samples:      {len(train_entries) + len(val_entries) + len(test_entries)}")


In [ ]:
if SHOULD_TRAIN_NODE:

    def build_image_index(root: Path) -> Tuple[Dict[str, Path], Dict[str, Path]]:
        by_name: Dict[str, Path] = {}
        by_stem: Dict[str, Path] = {}

        for path in sorted(root.rglob("*")):
            if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            if path.name in by_name and by_name[path.name] != path:
                raise ValueError(f"Duplicate image filename found: {path.name}")
            by_name[path.name] = path

            if path.stem in by_stem and by_stem[path.stem] != path:
                raise ValueError(
                    f"Duplicate image stem found: {path.stem}. "
                    "Use unique filenames or include exact filenames in the manifests."
                )
            by_stem[path.stem] = path

        if not by_name:
            raise FileNotFoundError(f"No supported images found under {root}")
        return by_name, by_stem


    IMAGE_BY_NAME, IMAGE_BY_STEM = build_image_index(image_dir)
    print(f"Indexed {len(IMAGE_BY_NAME)} image files.")


In [ ]:
if SHOULD_TRAIN_NODE:

    def resolve_image_path(entry: str) -> Path:
        candidate = Path(entry)
        if candidate.suffix:
            if candidate.name in IMAGE_BY_NAME:
                return IMAGE_BY_NAME[candidate.name]
            direct = image_dir / candidate
            if direct.is_file():
                return direct
            raise FileNotFoundError(f"Image listed in manifest was not found: {entry}")

        if entry in IMAGE_BY_STEM:
            return IMAGE_BY_STEM[entry]
        raise FileNotFoundError(f"No image found for manifest stem: {entry}")


    def resolve_mask_source(image_id: str, image_path: Path) -> Tuple[str, Path]:
        # Preferred layout: masks/<image_id>/mask_*.png
        folder = mask_dir / image_id
        if folder.is_dir():
            return "folder", folder

        # Allow nested mask folders while requiring unambiguous names.
        nested_folders = [p for p in mask_dir.rglob(image_id) if p.is_dir()]
        if len(nested_folders) == 1:
            return "folder", nested_folders[0]
        if len(nested_folders) > 1:
            raise ValueError(f"Multiple mask folders found for {image_id}: {nested_folders}")

        # Fallback: one binary mask file per image.
        candidates = []
        for extension in IMAGE_EXTENSIONS:
            direct = mask_dir / f"{image_id}{extension}"
            if direct.is_file():
                candidates.append(direct)
        if image_path.name:
            same_name = mask_dir / image_path.name
            if same_name.is_file() and same_name not in candidates:
                candidates.append(same_name)

        if len(candidates) == 1:
            return "file", candidates[0]
        if len(candidates) > 1:
            raise ValueError(f"Multiple single-mask files found for {image_id}: {candidates}")
        raise FileNotFoundError(f"No mask folder or mask file found for {image_id}")


    def load_combined_mask(source_type: str, source_path: Path) -> Image.Image:
        if source_type == "file":
            return Image.open(source_path).convert("L")

        mask_paths = [
            p for p in sorted(source_path.iterdir())
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        ]
        if not mask_paths:
            raise FileNotFoundError(f"No valid mask files inside {source_path}")

        combined: Optional[np.ndarray] = None
        expected_size: Optional[Tuple[int, int]] = None
        for mask_path in mask_paths:
            with Image.open(mask_path) as mask_image:
                mask_image = mask_image.convert("L")
                if expected_size is None:
                    expected_size = mask_image.size
                elif mask_image.size != expected_size:
                    raise ValueError(
                        f"Mask-size mismatch inside {source_path}: "
                        f"expected {expected_size}, found {mask_image.size} in {mask_path.name}"
                    )
                array = np.asarray(mask_image, dtype=np.uint8)
                binary = (array > 127).astype(np.uint8) * 255
                combined = binary if combined is None else np.maximum(combined, binary)

        return Image.fromarray(combined, mode="L")


In [ ]:
if SHOULD_TRAIN_NODE:

    @dataclass(frozen=True)
    class PairedTransform:
        image_size: int
        mean: Sequence[float]
        std: Sequence[float]
        train: bool
        horizontal_flip_probability: float = 0.5
        max_rotation_degrees: float = 3.0

        def __call__(self, image: Image.Image, mask: Image.Image) -> Tuple[torch.Tensor, torch.Tensor]:
            image = image.convert("L")
            mask = mask.convert("L")

            image = TF.resize(
                image,
                [self.image_size, self.image_size],
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            )
            mask = TF.resize(
                mask,
                [self.image_size, self.image_size],
                interpolation=InterpolationMode.NEAREST,
            )

            if self.train:
                if random.random() < self.horizontal_flip_probability:
                    image = TF.hflip(image)
                    mask = TF.hflip(mask)

                if self.max_rotation_degrees > 0:
                    angle = random.uniform(-self.max_rotation_degrees, self.max_rotation_degrees)
                    image = TF.rotate(
                        image,
                        angle,
                        interpolation=InterpolationMode.BILINEAR,
                        fill=255,
                    )
                    mask = TF.rotate(
                        mask,
                        angle,
                        interpolation=InterpolationMode.NEAREST,
                        fill=0,
                    )

            # Repeat the grayscale channel three times for the pretrained ResNet34 encoder.
            image_tensor = TF.to_tensor(image).repeat(3, 1, 1)
            image_tensor = TF.normalize(image_tensor, mean=self.mean, std=self.std)

            mask_tensor = (TF.pil_to_tensor(mask).float() > 127.5).float()
            return image_tensor, mask_tensor


    class NodeMaskDataset(Dataset):
        def __init__(self, entries: Sequence[str], transform: PairedTransform):
            self.entries = list(entries)
            self.transform = transform

            self.records = []
            for entry in self.entries:
                image_path = resolve_image_path(entry)
                image_id = Path(entry).stem if Path(entry).suffix else entry
                source_type, source_path = resolve_mask_source(image_id, image_path)
                self.records.append(
                    {
                        "image_id": image_id,
                        "image_path": image_path,
                        "mask_source_type": source_type,
                        "mask_source_path": source_path,
                    }
                )

        def __len__(self) -> int:
            return len(self.records)

        def __getitem__(self, index: int) -> Dict[str, object]:
            record = self.records[index]
            with Image.open(record["image_path"]) as image_file:
                image = image_file.convert("L")
            mask = load_combined_mask(
                record["mask_source_type"], record["mask_source_path"]
            )

            if image.size != mask.size:
                raise ValueError(
                    f"Image/mask size mismatch for {record['image_id']}: "
                    f"image={image.size}, mask={mask.size}"
                )

            image_tensor, mask_tensor = self.transform(image, mask)
            return {
                "image": image_tensor,
                "mask": mask_tensor,
                "image_id": record["image_id"],
                "image_path": str(record["image_path"]),
            }


In [ ]:
if SHOULD_TRAIN_NODE:

    train_transform = PairedTransform(
        image_size=IMAGE_SIZE,
        mean=NORMALIZATION_MEAN,
        std=NORMALIZATION_STD,
        train=True,
        horizontal_flip_probability=HORIZONTAL_FLIP_PROBABILITY,
        max_rotation_degrees=MAX_ROTATION_DEGREES,
    )

    eval_transform = PairedTransform(
        image_size=IMAGE_SIZE,
        mean=NORMALIZATION_MEAN,
        std=NORMALIZATION_STD,
        train=False,
        horizontal_flip_probability=0.0,
        max_rotation_degrees=0.0,
    )

    train_dataset = NodeMaskDataset(train_entries, train_transform)
    val_dataset = NodeMaskDataset(val_entries, eval_transform)
    test_dataset = NodeMaskDataset(test_entries, eval_transform)

    print("Dataset audit passed.")
    print("Train / validation / test:", len(train_dataset), len(val_dataset), len(test_dataset))

    loader_generator = torch.Generator()
    loader_generator.manual_seed(SEED)

    common_loader_arguments = {
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE.type == "cuda",
        "persistent_workers": NUM_WORKERS > 0,
        "worker_init_fn": seed_worker,
    }

    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        generator=loader_generator,
        **common_loader_arguments,
    )
    val_loader = DataLoader(val_dataset, shuffle=False, **common_loader_arguments)
    test_loader = DataLoader(test_dataset, shuffle=False, **common_loader_arguments)


In [ ]:
if SHOULD_TRAIN_NODE:

    def denormalize_image(image_tensor: torch.Tensor) -> np.ndarray:
        mean = torch.tensor(NORMALIZATION_MEAN, dtype=image_tensor.dtype).view(3, 1, 1)
        std = torch.tensor(NORMALIZATION_STD, dtype=image_tensor.dtype).view(3, 1, 1)
        image = image_tensor.cpu() * std + mean
        return image.clamp(0, 1).mean(dim=0).numpy()


    def show_dataset_examples(dataset: Dataset, count: int = 4) -> None:
        count = min(count, len(dataset))
        indices = np.linspace(0, len(dataset) - 1, num=count, dtype=int)
        fig, axes = plt.subplots(count, 2, figsize=(8, 3 * count))
        if count == 1:
            axes = np.asarray([axes])

        for row, index in enumerate(indices):
            sample = dataset[int(index)]
            axes[row, 0].imshow(denormalize_image(sample["image"]), cmap="gray")
            axes[row, 0].set_title(str(sample["image_id"]))
            axes[row, 1].imshow(sample["mask"][0].numpy(), cmap="gray")
            axes[row, 1].set_title("Combined node mask")
            axes[row, 0].axis("off")
            axes[row, 1].axis("off")

        plt.tight_layout()
        plt.show()


    show_dataset_examples(train_dataset, count=4)


In [ ]:
if SHOULD_TRAIN_NODE:

    def conv_bn_relu(
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
    ) -> nn.Sequential:
        return nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )


    class UpBlock(nn.Module):
        def __init__(self, in_channels: int, skip_channels: int, out_channels: int):
            super().__init__()
            self.up = nn.ConvTranspose2d(
                in_channels, out_channels, kernel_size=2, stride=2
            )
            self.conv1 = conv_bn_relu(out_channels + skip_channels, out_channels)
            self.conv2 = conv_bn_relu(out_channels, out_channels)

        def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
            x = self.up(x)
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(
                    x, size=skip.shape[-2:], mode="bilinear", align_corners=False
                )
            x = torch.cat([x, skip], dim=1)
            return self.conv2(self.conv1(x))


    class UNetResNet34(nn.Module):
        def __init__(self, out_channels: int = 1, pretrained: bool = True):
            super().__init__()
            weights = (
                tvm.ResNet34_Weights.IMAGENET1K_V1
                if pretrained
                else None
            )
            encoder = tvm.resnet34(weights=weights)

            self.conv1 = encoder.conv1
            self.bn1 = encoder.bn1
            self.relu = encoder.relu
            self.maxpool = encoder.maxpool
            self.layer1 = encoder.layer1
            self.layer2 = encoder.layer2
            self.layer3 = encoder.layer3
            self.layer4 = encoder.layer4

            self.bot = conv_bn_relu(512, 512)
            self.up4 = UpBlock(512, 256, 256)
            self.up3 = UpBlock(256, 128, 128)
            self.up2 = UpBlock(128, 64, 64)
            self.up1 = UpBlock(64, 64, 64)
            self.head = nn.Conv2d(64, out_channels, kernel_size=1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x0 = self.relu(self.bn1(self.conv1(x)))
            x1 = self.layer1(self.maxpool(x0))
            x2 = self.layer2(x1)
            x3 = self.layer3(x2)
            x4 = self.layer4(x3)

            bottleneck = self.bot(x4)
            d4 = self.up4(bottleneck, x3)
            d3 = self.up3(d4, x2)
            d2 = self.up2(d3, x1)
            d1 = self.up1(d2, x0)

            logits = self.head(d1)
            return F.interpolate(
                logits,
                size=x.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )


    model = UNetResNet34(
        out_channels=1,
        pretrained=USE_IMAGENET_PRETRAINED,
    ).to(DEVICE)

    def _node_state_model(m):
        return m.module if isinstance(m, nn.DataParallel) else m

    if DEVICE.type == "cuda" and torch.cuda.device_count() > 1:
        print(f"Using DataParallel for node training on {torch.cuda.device_count()} GPUs.")
        model = nn.DataParallel(model)

    trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable_parameters:,}")


In [ ]:
if SHOULD_TRAIN_NODE:

    class BCEDiceLoss(nn.Module):
        def __init__(self, smooth: float = 1.0):
            super().__init__()
            self.binary_cross_entropy = nn.BCEWithLogitsLoss()
            self.smooth = float(smooth)

        def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
            bce = self.binary_cross_entropy(logits, targets)
            probabilities = torch.sigmoid(logits)
            dimensions = (1, 2, 3)
            intersection = (probabilities * targets).sum(dim=dimensions)
            denominator = probabilities.sum(dim=dimensions) + targets.sum(dim=dimensions)
            dice = (2.0 * intersection + self.smooth) / (
                denominator + self.smooth
            )
            return bce + (1.0 - dice.mean())


    @torch.no_grad()
    def per_image_binary_metrics(
        logits: torch.Tensor,
        targets: torch.Tensor,
        threshold: float,
        epsilon: float = 1e-7,
    ) -> Dict[str, torch.Tensor]:
        predictions = (torch.sigmoid(logits) > threshold).float()

        true_positive = (predictions * targets).sum(dim=(1, 2, 3))
        false_positive = (predictions * (1.0 - targets)).sum(dim=(1, 2, 3))
        false_negative = ((1.0 - predictions) * targets).sum(dim=(1, 2, 3))

        precision = (true_positive + epsilon) / (
            true_positive + false_positive + epsilon
        )
        recall = (true_positive + epsilon) / (
            true_positive + false_negative + epsilon
        )
        f1 = (2.0 * precision * recall + epsilon) / (
            precision + recall + epsilon
        )
        iou = (true_positive + epsilon) / (
            true_positive + false_positive + false_negative + epsilon
        )

        return {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "iou": iou,
        }


In [ ]:
if SHOULD_TRAIN_NODE:

    loss_function = BCEDiceLoss()
    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

    try:
        scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


    def safe_torch_load(path: Path, map_location: torch.device):
        try:
            return torch.load(path, map_location=map_location, weights_only=False)
        except TypeError:
            return torch.load(path, map_location=map_location)


    @torch.no_grad()
    def evaluate_model(
        model: nn.Module,
        data_loader: DataLoader,
        loss_function: nn.Module,
        threshold: float,
    ) -> Dict[str, float]:
        model.eval()
        loss_sum = 0.0
        sample_count = 0
        metric_values = {"precision": [], "recall": [], "f1": [], "iou": []}

        for batch in data_loader:
            images = batch["image"].to(DEVICE, non_blocking=True)
            masks = batch["mask"].to(DEVICE, non_blocking=True)
            logits = model(images)
            loss = loss_function(logits, masks)

            batch_size = images.shape[0]
            loss_sum += float(loss.item()) * batch_size
            sample_count += batch_size

            batch_metrics = per_image_binary_metrics(logits, masks, threshold)
            for name, values in batch_metrics.items():
                metric_values[name].extend(values.detach().cpu().tolist())

        results = {"loss": loss_sum / max(sample_count, 1)}
        results.update(
            {
                name: float(np.mean(values))
                for name, values in metric_values.items()
            }
        )
        return results


In [ ]:
if SHOULD_TRAIN_NODE:

    CHECKPOINT_METADATA = {
        "architecture": "UNetResNet34",
        "encoder": "torchvision ResNet34",
        "pretrained_encoder": bool(USE_IMAGENET_PRETRAINED),
        "image_size": int(IMAGE_SIZE),
        "normalization_mean": list(NORMALIZATION_MEAN),
        "normalization_std": list(NORMALIZATION_STD),
        "loss": "BCEWithLogitsLoss + soft Dice loss",
        "optimizer": "AdamW",
        "scheduler": "CosineAnnealingLR",
        "learning_rate": float(LEARNING_RATE),
        "weight_decay": float(WEIGHT_DECAY),
        "batch_size": int(BATCH_SIZE),
        "epochs": int(EPOCHS),
        "seed": int(SEED),
        "metric_threshold": float(METRIC_THRESHOLD),
        "horizontal_flip_probability": float(HORIZONTAL_FLIP_PROBABILITY),
        "max_rotation_degrees": float(MAX_ROTATION_DEGREES),
        "split_counts": {
            "train": len(train_dataset),
            "validation": len(val_dataset),
            "test": len(test_dataset),
        },
    }

    best_checkpoint_path = output_dir / "best.pth"
    last_checkpoint_path = output_dir / "last.pth"
    training_log_path = output_dir / "training_log.csv"

    start_epoch = 1
    best_validation_iou = -math.inf

    if resume_checkpoint_path is not None:
        resume_state = safe_torch_load(resume_checkpoint_path, DEVICE)
        _node_state_model(model).load_state_dict(resume_state["model"])
        if "optimizer" in resume_state:
            optimizer.load_state_dict(resume_state["optimizer"])
        elif "opt" in resume_state:
            optimizer.load_state_dict(resume_state["opt"])
        if "scheduler" in resume_state:
            scheduler.load_state_dict(resume_state["scheduler"])
        elif "sched" in resume_state:
            scheduler.load_state_dict(resume_state["sched"])

        start_epoch = int(resume_state.get("epoch", 0)) + 1
        validation_metrics = resume_state.get("validation_metrics", resume_state.get("val_metrics", {}))
        best_validation_iou = float(validation_metrics.get("iou", -math.inf))
        print(f"Resuming from epoch {start_epoch}.")

    if start_epoch == 1 or not training_log_path.exists():
        with training_log_path.open("w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(
                [
                    "epoch",
                    "train_loss",
                    "validation_loss",
                    "validation_iou",
                    "validation_f1",
                    "validation_precision",
                    "validation_recall",
                    "learning_rate",
                    "elapsed_seconds",
                ]
            )

    for epoch in range(start_epoch, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        running_loss = 0.0
        samples_seen = 0

        for batch in train_loader:
            images = batch["image"].to(DEVICE, non_blocking=True)
            masks = batch["mask"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                logits = model(images)
                loss = loss_function(logits, masks)

            if AMP_ENABLED:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            batch_size = images.shape[0]
            running_loss += float(loss.item()) * batch_size
            samples_seen += batch_size

        train_loss = running_loss / max(samples_seen, 1)
        validation_metrics = evaluate_model(
            model,
            val_loader,
            loss_function,
            threshold=METRIC_THRESHOLD,
        )
        scheduler.step()

        elapsed_seconds = time.time() - epoch_start
        current_learning_rate = optimizer.param_groups[0]["lr"]
        is_best = validation_metrics["iou"] > best_validation_iou
        if is_best:
            best_validation_iou = validation_metrics["iou"]

        checkpoint = {
            "epoch": epoch,
            "model": _node_state_model(model).state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "metadata": CHECKPOINT_METADATA,
            "validation_metrics": validation_metrics,
        }
        torch.save(checkpoint, last_checkpoint_path)
        if is_best:
            torch.save(checkpoint, best_checkpoint_path)

        with training_log_path.open("a", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(
                [
                    epoch,
                    f"{train_loss:.8f}",
                    f"{validation_metrics['loss']:.8f}",
                    f"{validation_metrics['iou']:.8f}",
                    f"{validation_metrics['f1']:.8f}",
                    f"{validation_metrics['precision']:.8f}",
                    f"{validation_metrics['recall']:.8f}",
                    f"{current_learning_rate:.10f}",
                    f"{elapsed_seconds:.2f}",
                ]
            )

        marker = "  [BEST]" if is_best else ""
        print(
            f"Epoch {epoch:03d}/{EPOCHS} | "
            f"train loss {train_loss:.4f} | "
            f"val loss {validation_metrics['loss']:.4f} | "
            f"IoU {validation_metrics['iou']:.4f} | "
            f"F1 {validation_metrics['f1']:.4f} | "
            f"P {validation_metrics['precision']:.4f} | "
            f"R {validation_metrics['recall']:.4f} | "
            f"{elapsed_seconds:.1f}s{marker}"
        )

    print("Best checkpoint:", best_checkpoint_path)
    print("Final checkpoint:", last_checkpoint_path)


In [ ]:
if SHOULD_TRAIN_NODE:

    if not best_checkpoint_path.is_file():
        raise FileNotFoundError(f"Best checkpoint was not created: {best_checkpoint_path}")

    best_state = safe_torch_load(best_checkpoint_path, DEVICE)
    _node_state_model(model).load_state_dict(best_state["model"])
    model.eval()

    test_metrics = evaluate_model(
        model,
        test_loader,
        loss_function,
        threshold=METRIC_THRESHOLD,
    )

    with (output_dir / "test_metrics.json").open("w", encoding="utf-8") as file:
        json.dump(test_metrics, file, indent=2)

    print("Held-out test metrics")
    for name, value in test_metrics.items():
        print(f"  {name:>10s}: {value:.6f}")


In [ ]:
if SHOULD_TRAIN_NODE:

    @torch.no_grad()
    def export_prediction_examples(
        model: nn.Module,
        dataset: Dataset,
        output_directory: Path,
        count: int,
        threshold: float,
    ) -> None:
        output_directory.mkdir(parents=True, exist_ok=True)
        count = min(count, len(dataset))
        indices = np.linspace(0, len(dataset) - 1, num=count, dtype=int)

        fig, axes = plt.subplots(count, 3, figsize=(11, 3.2 * count))
        if count == 1:
            axes = np.asarray([axes])

        model.eval()
        for row, index in enumerate(indices):
            sample = dataset[int(index)]
            image = sample["image"].unsqueeze(0).to(DEVICE)
            probability = torch.sigmoid(model(image))[0, 0].cpu().numpy()
            prediction = probability > threshold
            ground_truth = sample["mask"][0].numpy() > 0.5
            display_image = denormalize_image(sample["image"])

            axes[row, 0].imshow(display_image, cmap="gray")
            axes[row, 0].set_title(str(sample["image_id"]))
            axes[row, 1].imshow(ground_truth, cmap="gray")
            axes[row, 1].set_title("Ground truth")
            axes[row, 2].imshow(prediction, cmap="gray")
            axes[row, 2].set_title("Prediction")
            for axis in axes[row]:
                axis.axis("off")

            Image.fromarray((prediction.astype(np.uint8) * 255), mode="L").save(
                output_directory / f"{sample['image_id']}_prediction.png"
            )

        plt.tight_layout()
        plt.savefig(output_directory / "prediction_examples.png", dpi=160, bbox_inches="tight")
        plt.show()


    export_prediction_examples(
        model,
        test_dataset,
        output_dir / "prediction_examples",
        count=NUM_VISUAL_EXAMPLES,
        threshold=METRIC_THRESHOLD,
    )


In [ ]:
if SHOULD_TRAIN_NODE:
    RESOLVED_NODE_CHECKPOINT = Path(best_checkpoint_path).resolve()
    NODE_MODEL_CHECKPOINT = str(RESOLVED_NODE_CHECKPOINT)
    print('Node checkpoint ready:', NODE_MODEL_CHECKPOINT)
elif RESOLVED_NODE_CHECKPOINT is not None:
    NODE_MODEL_CHECKPOINT = str(RESOLVED_NODE_CHECKPOINT)


## 4. Optional graph-evidence model training

Skipped when the publication checkpoint is available unless explicitly forced. The original compressed `.npz` precompute path is retained, but its sampled storage estimate is checked against the notebook-wide **12 GB soft budget** before the full cache is written.

In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # USER PATHS — FILL THESE BEFORE RUNNING
    # ============================================================

    # Reconstructed dataset root containing:
    #   fa|fca|fcb / train|val|test / images + json
    CONVERTED_DATASET_ROOT = str(RESOLVED_DATASET_ROOT) if RESOLVED_DATASET_ROOT is not None else ""

    # Folder used for the compressed supervision cache
    PRECOMPUTE_CACHE_ROOT = str(WORKING_ROOT / "graph_precompute")

    # Folder in which the trained model and training history will be saved
    OUTPUT_MODEL_ROOT = str(WORKING_ROOT / "trained_models" / "graph_evidence")


    # ============================================================
    # PAPER TRAINING CONFIGURATION
    # ============================================================

    DATASET_FAMILIES = ("fa", "fca", "fcb")
    LEARNING_SOURCE_SPLITS = ("train", "val")

    # Model input resolution. The source images and annotations remain 1024 × 1024;
    # they are resized jointly to 512 × 512 for graph-evidence learning.
    TARGET_W = 512
    TARGET_H = 512
    IMAGE_MODE = "grayscale"
    NORMALIZE_IMAGE = True
    IMAGE_INTERPOLATION = cv2.INTER_LINEAR if "cv2" in globals() else 1

    # Reproducible family-stratified learning split
    LEARNING_VAL_RATIO = 0.20
    LEARNING_SPLIT_SEED = 39
    SPLIT_SEED = LEARNING_SPLIT_SEED
    USE_STRATIFIED_LEARNING_SPLIT = True

    # Model and optimization
    BASE_CH = 16
    NUM_EPOCHS = 50
    TRAIN_BATCH_SIZE = 4
    TRAIN_NUM_WORKERS = min(NUM_DATALOADER_WORKERS, os.cpu_count() or 1)
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-4
    USE_AMP = True
    GRAD_CLIP_NORM = None
    EARLY_STOPPING_PATIENCE = 8
    EARLY_STOPPING_MIN_DELTA = 1e-3
    USE_POS_WEIGHT = False

    # Precompute
    PRECOMPUTE_PARALLEL = True
    PRECOMPUTE_MAX_WORKERS = min(NUM_PREPROCESS_WORKERS, os.cpu_count() or 1)
    OVERWRITE_CACHE = False
    PRECOMPUTE_MEMORY_SAMPLE_N = 5
    PRECOMPUTE_MEMORY_LIMIT_GB = float("inf")
    SAVE_RESIZED_IMAGE_IN_CACHE = True

    # Input preparation
    APPLY_OTSU_THRESHOLD = True
    OTSU_FOREGROUND_WHITE = True

    # Dense target generation
    NODE_CENTER_ALPHA = 0.18
    NODE_EXTENT_ALPHA = 0.32
    NODE_EXTENT_BETA = 6.0
    SHAFT_THICKNESS = 2
    SHAFT_BLUR_SIGMA = 1.5
    POINT_SIGMA = 2.0
    DIR_RADIUS = 2.0
    DIR_FROM_SHAFT_MASK = True
    SHAFT_MASK_THRESHOLD = 0.05

    # Skeleton and topology supervision
    USE_SKELETON_HEAD = True
    SKELETON_MASK_THRESHOLD = 0.05
    SKELETON_DILATE_KERNEL = 0
    SKELETON_POS_WEIGHT = None
    W_SKELETON = 1.0
    W_CLDICE = 1.3
    CLDICE_ITERS = 15

    # Self-loop continuity
    USE_LOOP_PATH_LOSS = True
    W_LOOP_PATH = 1.0
    LOOP_PATH_THICKNESS = 3
    LOOP_PATH_BLUR_SIGMA = 1.0

    # Endpoint structure
    USE_ENDPOINT_STRUCTURED_HEADS = True
    ENDPOINT_HEATMAP_SIGMA = 2.0
    ENDPOINT_RADIUS = 8
    PROGRESS_THICKNESS = 3
    PROGRESS_BLUR_SIGMA = 1.0
    PROGRESS_USE_FULL_SHAFT_MASK = False
    W_PROGRESS = 1.0
    W_HEAD_OFFSET = 0.75
    W_TAIL_OFFSET = 0.75
    W_HEAD_TANGENT = 0.50
    W_TAIL_TANGENT = 0.50
    OFFSET_LOSS_RADIUS = 8
    TANGENT_LOSS_RADIUS = 8

    # General connector continuity
    USE_GENERAL_PATH_LOSS = True
    W_GENERAL_PATH = 1.3
    ALL_PATH_THICKNESS = 3
    ALL_PATH_BLUR_SIGMA = 1.0

    # Long-arrow-aware supervision
    LONG_ARROW_MIN_NORM = 0.20
    VERY_LONG_ARROW_MIN_NORM = 0.35
    LONG_ARROW_WEIGHT_ALPHA = 2.0
    LONG_ARROW_WEIGHT_CLIP = 4.0
    W_LONG_SHAFT = 1.25
    W_LONG_SKELETON = 1.00
    W_LONG_PATH_RECALL = 1.00
    W_VERY_LONG_PATH_RECALL = 1.25
    W_LONG_CLDICE = 0.75

    # Learning-side metric thresholds
    METRIC_PROB_THRESH = 0.25
    METRIC_TARGET_THRESH = 0.05

    # Small, bounded visual checks
    SAVE_TARGET_PREVIEWS = True
    MAX_TARGET_PREVIEWS = min(3, MAX_DEBUG_EXAMPLES)
    SAVE_PREDICTION_PREVIEWS = True
    MAX_PREDICTION_PREVIEWS = min(3, MAX_DEBUG_EXAMPLES)


In [ ]:
if SHOULD_TRAIN_GRAPH:

    import json
    import math
    import os
    import random
    import shutil
    import tempfile
    import time
    from collections import Counter, defaultdict
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from copy import deepcopy
    from pathlib import Path
    from typing import Dict, Iterable, List, Optional, Sequence, Tuple

    import cv2
    import matplotlib.pyplot as plt
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    import torchvision.transforms.functional as TF
    from skimage.morphology import skeletonize
    from torch.utils.data import DataLoader, Dataset
    from torchvision.transforms import InterpolationMode

    try:
        import pandas as pd
    except Exception:
        pd = None

    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA devices: {torch.cuda.device_count()}")


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # VALIDATE PATHS AND FREEZE THE RUN CONFIGURATION
    # ============================================================

    def require_nonempty_directory(value: str, variable_name: str, create: bool = False) -> Path:
        value = str(value).strip()
        if not value:
            raise ValueError(
                f"{variable_name} is blank. Fill it in the USER PATHS cell before running."
            )

        path = Path(value).expanduser().resolve()
        if create:
            path.mkdir(parents=True, exist_ok=True)
        elif not path.exists():
            raise FileNotFoundError(f"{variable_name} does not exist: {path}")
        return path


    def seed_everything(seed: int) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.benchmark = False


    DATASET_ROOT = require_nonempty_directory(
        CONVERTED_DATASET_ROOT, "CONVERTED_DATASET_ROOT"
    )
    CACHE_ROOT = require_nonempty_directory(
        PRECOMPUTE_CACHE_ROOT, "PRECOMPUTE_CACHE_ROOT", create=True
    )
    MODEL_ROOT = require_nonempty_directory(
        OUTPUT_MODEL_ROOT, "OUTPUT_MODEL_ROOT", create=True
    )

    RUN_NAME = "GraphEvidence_DiagramUNet_base16_paper"
    TARGET_VERSION = "graph_evidence_targets_512_long_arrow_v1"
    CACHE_DIR = CACHE_ROOT / TARGET_VERSION
    RUN_DIR = MODEL_ROOT / RUN_NAME
    BEST_CKPT_PATH = RUN_DIR / "best.pt"
    LAST_CKPT_PATH = RUN_DIR / "last.pt"

    PIN_MEMORY = torch.cuda.is_available()
    USE_AMP = bool(USE_AMP and torch.cuda.is_available())

    seed_everything(LEARNING_SPLIT_SEED)

    PAPER_RUN_CONFIG = {
        "run_name": RUN_NAME,
        "dataset_families": list(DATASET_FAMILIES),
        "learning_source_splits": list(LEARNING_SOURCE_SPLITS),
        "learning_val_ratio": LEARNING_VAL_RATIO,
        "learning_split_seed": LEARNING_SPLIT_SEED,
        "target_w": TARGET_W,
        "target_h": TARGET_H,
        "image_mode": IMAGE_MODE,
        "base_ch": BASE_CH,
        "num_epochs": NUM_EPOCHS,
        "batch_size": TRAIN_BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "long_arrow_min_norm": LONG_ARROW_MIN_NORM,
        "very_long_arrow_min_norm": VERY_LONG_ARROW_MIN_NORM,
        "long_arrow_weight_alpha": LONG_ARROW_WEIGHT_ALPHA,
        "long_arrow_weight_clip": LONG_ARROW_WEIGHT_CLIP,
        "loss_weights": {
            "skeleton": W_SKELETON,
            "cldice": W_CLDICE,
            "loop_path": W_LOOP_PATH,
            "progress": W_PROGRESS,
            "head_offset": W_HEAD_OFFSET,
            "tail_offset": W_TAIL_OFFSET,
            "head_tangent": W_HEAD_TANGENT,
            "tail_tangent": W_TAIL_TANGENT,
            "general_path": W_GENERAL_PATH,
            "long_shaft": W_LONG_SHAFT,
            "long_skeleton": W_LONG_SKELETON,
            "long_path_recall": W_LONG_PATH_RECALL,
            "very_long_path_recall": W_VERY_LONG_PATH_RECALL,
            "long_cldice": W_LONG_CLDICE,
        },
    }

    RUN_DIR.mkdir(parents=True, exist_ok=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    with (RUN_DIR / "paper_training_config.json").open("w", encoding="utf-8") as handle:
        json.dump(PAPER_RUN_CONFIG, handle, indent=2)

    print(f"Dataset root : {DATASET_ROOT}")
    print(f"Cache dir    : {CACHE_DIR}")
    print(f"Run dir      : {RUN_DIR}")
    print(f"BASE_CH      : {BASE_CH}")


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # SAMPLE DISCOVERY
    # ============================================================

    IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp")


    def find_image_for_annotation(json_path: Path, image_dir: Path, annotation: Dict) -> Optional[Path]:
        candidate_names: List[str] = []

        declared_name = str(annotation.get("image", "")).strip()
        if declared_name:
            candidate_names.append(declared_name)

        candidate_names.extend(json_path.stem + ext for ext in IMAGE_EXTENSIONS)

        for name in candidate_names:
            path = image_dir / name
            if path.exists():
                return path
        return None


    def build_samples_from_converted_root(dataset_root: Path) -> List[Dict[str, str]]:
        requested_splits = tuple(LEARNING_SOURCE_SPLITS)
        samples: List[Dict[str, str]] = []

        for family in DATASET_FAMILIES:
            for split in requested_splits:
                split_dir = dataset_root / family / split
                json_dir = split_dir / "json"
                image_dir = split_dir / "images"

                if not json_dir.exists() or not image_dir.exists():
                    print(f"Skipping unavailable source split: {family}/{split}")
                    continue

                for json_path in sorted(json_dir.glob("*.json")):
                    with json_path.open("r", encoding="utf-8") as handle:
                        annotation = json.load(handle)

                    image_path = find_image_for_annotation(
                        json_path=json_path,
                        image_dir=image_dir,
                        annotation=annotation,
                    )
                    if image_path is None:
                        raise FileNotFoundError(
                            f"No image was found for annotation: {json_path}"
                        )

                    samples.append(
                        {
                            "sample_id": json_path.stem,
                            "json_path": str(json_path),
                            "image_path": str(image_path),
                            "family": family,
                            "split": split,
                        }
                    )

        if not samples:
            raise RuntimeError(
                "No learning samples were discovered. Check the reconstructed dataset path."
            )

        sample_ids = [sample["sample_id"] for sample in samples]
        duplicate_ids = sorted(
            sample_id for sample_id, count in Counter(sample_ids).items() if count > 1
        )
        if duplicate_ids:
            raise RuntimeError(
                "Duplicate sample stems would collide in the target cache: "
                + ", ".join(duplicate_ids[:20])
            )

        print(f"Discovered {len(samples)} learning-source samples.")
        for key, count in sorted(
            Counter((sample["family"], sample["split"]) for sample in samples).items()
        ):
            print(f"  {key[0]}/{key[1]}: {count}")

        return samples


    def save_json(path: Path, value) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("w", encoding="utf-8") as handle:
            json.dump(value, handle, indent=2, ensure_ascii=False)


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # PRECOMPUTE TARGET GENERATOR
    # ============================================================

    class PrecomputeTargetGenerator:
        def __init__(
            self,
            target_w,
            target_h,
            image_mode="grayscale",
            normalize_image=True,
            image_interpolation=cv2.INTER_LINEAR,
            apply_otsu_threshold=True,
            otsu_foreground_white=True,
            node_center_alpha=0.18,
            node_extent_alpha=0.22,
            node_extent_beta=6.0,
            shaft_thickness=3,
            shaft_blur_sigma=2.5,
            point_sigma=8.0,
            dir_radius=5.0,
            dir_from_shaft_mask=True,
            shaft_mask_threshold=0.05,
            save_resized_image_in_cache=False,
            use_skeleton_head=True,
            skeleton_mask_threshold=0.05,
            skeleton_dilate_kernel=0,
            use_loop_path_loss=True,
            loop_path_thickness=3,
            loop_path_blur_sigma=1.0,
            endpoint_heatmap_sigma=2.0,
            endpoint_radius=8,
            progress_thickness=3,
            progress_blur_sigma=1.0,
            progress_use_full_shaft_mask=False,
            use_endpoint_structured_heads=True,
            use_general_path_loss=True,
            all_path_thickness=3,
            all_path_blur_sigma=1.0,
            target_version="",
        ):
            self.target_w = int(target_w)
            self.target_h = int(target_h)

            self.image_mode = image_mode
            self.normalize_image = normalize_image
            self.image_interpolation = image_interpolation
            self.apply_otsu_threshold = apply_otsu_threshold
            self.otsu_foreground_white = otsu_foreground_white

            self.node_center_alpha = node_center_alpha
            self.node_extent_alpha = node_extent_alpha
            self.node_extent_beta = node_extent_beta

            self.shaft_thickness = shaft_thickness
            self.shaft_blur_sigma = shaft_blur_sigma

            self.point_sigma = point_sigma

            self.dir_radius = dir_radius
            self.dir_from_shaft_mask = dir_from_shaft_mask
            self.shaft_mask_threshold = shaft_mask_threshold

            self.save_resized_image_in_cache = save_resized_image_in_cache
            self.use_skeleton_head = use_skeleton_head
            self.skeleton_mask_threshold = skeleton_mask_threshold
            self.skeleton_dilate_kernel = skeleton_dilate_kernel

            self.use_loop_path_loss = use_loop_path_loss
            self.loop_path_thickness = loop_path_thickness
            self.loop_path_blur_sigma = loop_path_blur_sigma    

            self.endpoint_heatmap_sigma = endpoint_heatmap_sigma
            self.endpoint_radius = endpoint_radius
            self.progress_thickness = progress_thickness
            self.progress_blur_sigma = progress_blur_sigma
            self.progress_use_full_shaft_mask = progress_use_full_shaft_mask
            self.use_endpoint_structured_heads = use_endpoint_structured_heads
        
            self.use_general_path_loss = use_general_path_loss
            self.all_path_thickness = all_path_thickness
            self.all_path_blur_sigma = all_path_blur_sigma
        
            self.target_version = target_version

            assert self.image_mode in {"grayscale", "rgb"}

            self.yy, self.xx = np.mgrid[0:self.target_h, 0:self.target_w].astype(np.float32)

        @staticmethod
        def _is_self_loop_arrow(arrow):
            tail_id = arrow.get("tail_node_id", None)
            head_id = arrow.get("head_node_id", None)
            return (tail_id is not None) and (head_id is not None) and (tail_id == head_id)
    
        @staticmethod
        def _load_json(json_path):
            with open(json_path, "r", encoding="utf-8") as f:
                return json.load(f)

        @staticmethod
        def _clip01(x):
            return np.clip(x, 0.0, 1.0)

        @staticmethod
        def _path_to_xy_array(path):
            return np.array([[float(p["x"]), float(p["y"])] for p in path], dtype=np.float32)

        @staticmethod
        def _extract_scale_debug(ann):
            arrows = ann.get("arrows", [])
            for arrow in arrows:
                scale_debug = arrow.get("scale_debug", None)
                if scale_debug is not None:
                    sx = float(scale_debug.get("sx", 1.0))
                    sy = float(scale_debug.get("sy", 1.0))
                    return sx, sy
            return None, None

        @staticmethod
        def _scale_point(pt, sx, sy):
            return {
                "x": float(pt["x"]) * sx,
                "y": float(pt["y"]) * sy,
            }

        @staticmethod
        def _scale_bbox(bbox, sx, sy):
            x, y, w, h = bbox
            return [float(x) * sx, float(y) * sy, float(w) * sx, float(h) * sy]

        @staticmethod
        def _apply_otsu_threshold(img, foreground_white=True):
            """
            Expects a single-channel uint8 image.
            Returns a binarized uint8 image with values {0, 255}.
            """
            if img.ndim != 2:
                raise ValueError("Otsu thresholding expects a grayscale image.")

            thresh_type = cv2.THRESH_BINARY if foreground_white else cv2.THRESH_BINARY_INV
            _, bin_img = cv2.threshold(img, 0, 255, thresh_type + cv2.THRESH_OTSU)
            return bin_img

        def _gaussian_2d(self, cx, cy, sigma_x, sigma_y=None):
            if sigma_y is None:
                sigma_y = sigma_x

            g = np.exp(
                -(
                    ((self.xx - cx) ** 2) / (2.0 * sigma_x ** 2 + 1e-12)
                    + ((self.yy - cy) ** 2) / (2.0 * sigma_y ** 2 + 1e-12)
                )
            )
            return g.astype(np.float32)

        def _load_image(self, image_path):
            if self.image_mode == "grayscale":
                img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
                if img is None:
                    raise FileNotFoundError(f"Could not read image: {image_path}")
                return img

            img_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if img_bgr is None:
                raise FileNotFoundError(f"Could not read image: {image_path}")
            return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        def _resize_into_annotation_space(self, img, ann):
            if img.ndim == 2:
                H_orig, W_orig = img.shape
            else:
                H_orig, W_orig = img.shape[:2]

            sx, sy = self._extract_scale_debug(ann)

            if sx is not None and sy is not None:
                W_new = int(round(W_orig * sx))
                H_new = int(round(H_orig * sy))
                img_resized = cv2.resize(img, (W_new, H_new), interpolation=self.image_interpolation)
                return img_resized, H_new, W_new

            size_info = ann.get("size", None)
            if size_info is not None and "w" in size_info and "h" in size_info:
                W_new = int(size_info["w"])
                H_new = int(size_info["h"])
                if (W_new, H_new) != (W_orig, H_orig):
                    img_resized = cv2.resize(img, (W_new, H_new), interpolation=self.image_interpolation)
                else:
                    img_resized = img
                return img_resized, H_new, W_new

            return img, H_orig, W_orig

        def _rescale_everything(self, ann, img_resized, H_new, W_new, target_w, target_h):
            if target_w == W_new and target_h == H_new:
                ann_new = deepcopy(ann)
                ann_new.setdefault("size", {})
                ann_new["size"]["w"] = target_w
                ann_new["size"]["h"] = target_h
                return ann_new, img_resized, target_h, target_w

            sx = target_w / W_new
            sy = target_h / H_new

            ann_new = deepcopy(ann)
            ann_new.setdefault("size", {})
            ann_new["size"]["w"] = target_w
            ann_new["size"]["h"] = target_h

            for node in ann_new.get("nodes", []):
                if "bbox" in node:
                    node["bbox"] = self._scale_bbox(node["bbox"], sx, sy)

            for arrow in ann_new.get("arrows", []):
                if "tail_hint" in arrow:
                    arrow["tail_hint"] = self._scale_point(arrow["tail_hint"], sx, sy)
                if "head_hint" in arrow:
                    arrow["head_hint"] = self._scale_point(arrow["head_hint"], sx, sy)
                if "path" in arrow:
                    arrow["path"] = [self._scale_point(p, sx, sy) for p in arrow["path"]]

                arrow["scale_debug"] = {
                    "prev_space": {"w": W_new, "h": H_new},
                    "target_space": {"w": target_w, "h": target_h},
                    "sx": sx,
                    "sy": sy,
                }

            img_new = cv2.resize(img_resized, (target_w, target_h), interpolation=self.image_interpolation)
            return ann_new, img_new, target_h, target_w

        def _image_to_chw_numpy(self, img):
            if self.image_mode == "grayscale":
                if img.ndim == 2:
                    arr = img[None, :, :]
                else:
                    arr = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)[None, :, :]
                return arr.astype(np.uint8)

            return np.transpose(img, (2, 0, 1)).astype(np.uint8)

        def _make_node_center_map(self, nodes):
            out = np.zeros((self.target_h, self.target_w), dtype=np.float32)
            for node in nodes:
                x, y, w_box, h_box = node["bbox"]
                cx = x + w_box / 2.0
                cy = y + h_box / 2.0
                sigma = max(4.0, self.node_center_alpha * min(w_box, h_box))
                g = self._gaussian_2d(cx, cy, sigma)
                out = np.maximum(out, g)
            return self._clip01(out)

        def _make_node_extent_map_soft_bbox(self, nodes):
            out = np.zeros((self.target_h, self.target_w), dtype=np.float32)

            for node in nodes:
                x, y, w_box, h_box = node["bbox"]
                cx = x + w_box / 2.0
                cy = y + h_box / 2.0

                sigma_x = max(4.0, self.node_extent_alpha * w_box)
                sigma_y = max(4.0, self.node_extent_alpha * h_box)

                g = np.exp(
                    -(
                        ((self.xx - cx) ** 2) / (2.0 * sigma_x ** 2 + 1e-12)
                        + ((self.yy - cy) ** 2) / (2.0 * sigma_y ** 2 + 1e-12)
                    )
                ).astype(np.float32)

                u = (self.xx - cx) / (w_box / 2.0 + 1e-6)
                v = (self.yy - cy) / (h_box / 2.0 + 1e-6)

                r = (np.abs(u) ** 4 + np.abs(v) ** 4) ** 0.25
                s = np.exp(-self.node_extent_beta * (r ** 4)).astype(np.float32)

                out = np.maximum(out, g * s)

            return self._clip01(out)

        def _make_shaft_map(self, arrows):
            shaft = np.zeros((self.target_h, self.target_w), dtype=np.uint8)

            for arrow in arrows:
                pts = self._path_to_xy_array(arrow["path"])
                if len(pts) < 2:
                    continue

                pts_int = np.round(pts).astype(np.int32).reshape(-1, 1, 2)
                cv2.polylines(
                    shaft,
                    [pts_int],
                    isClosed=False,
                    color=255,
                    thickness=self.shaft_thickness,
                )

            shaft = shaft.astype(np.float32) / 255.0

            if self.shaft_blur_sigma > 0:
                k = max(3, int(round(6 * self.shaft_blur_sigma)) | 1)
                shaft = cv2.GaussianBlur(shaft, (k, k), self.shaft_blur_sigma)

            if shaft.max() > 0:
                shaft = shaft / shaft.max()

            return shaft.astype(np.float32)

        def _make_loop_supervision_maps(self, arrows):
            """
            Build loop-specific supervision using only semantic self-loop arrows
            identified from GT JSON by tail_node_id == head_node_id.

            Returns
            -------
            loop_shaft_map : float32 [H, W]
                Rasterized support for self-loop shafts only.
            loop_path_map : float32 [H, W]
                Stronger path-support map for self-loop continuity supervision.
            """
            loop_shaft = np.zeros((self.target_h, self.target_w), dtype=np.uint8)
            loop_path = np.zeros((self.target_h, self.target_w), dtype=np.uint8)

            for arrow in arrows:
                if not self._is_self_loop_arrow(arrow):
                    continue

                pts = self._path_to_xy_array(arrow["path"])
                if len(pts) < 2:
                    continue

                pts_int = np.round(pts).astype(np.int32).reshape(-1, 1, 2)

                # loop shaft support
                cv2.polylines(
                    loop_shaft,
                    [pts_int],
                    isClosed=False,
                    color=255,
                    thickness=self.shaft_thickness,
                )

                # stronger path-specific supervision
                cv2.polylines(
                    loop_path,
                    [pts_int],
                    isClosed=False,
                    color=255,
                    thickness=self.loop_path_thickness,
                )

            loop_shaft = loop_shaft.astype(np.float32) / 255.0
            loop_path = loop_path.astype(np.float32) / 255.0

            if self.shaft_blur_sigma > 0:
                k = max(3, int(round(6 * self.shaft_blur_sigma)) | 1)
                loop_shaft = cv2.GaussianBlur(loop_shaft, (k, k), self.shaft_blur_sigma)

            if self.loop_path_blur_sigma > 0:
                k = max(3, int(round(6 * self.loop_path_blur_sigma)) | 1)
                loop_path = cv2.GaussianBlur(loop_path, (k, k), self.loop_path_blur_sigma)

            if loop_shaft.max() > 0:
                loop_shaft = loop_shaft / loop_shaft.max()

            if loop_path.max() > 0:
                loop_path = loop_path / loop_path.max()

            return loop_shaft.astype(np.float32), loop_path.astype(np.float32)    

        def _make_skeleton_map(self, shaft_map):
            """
            Build a GT skeleton target from the shaft map.
            Uses the same function for:
              - precompute caching
              - notebook preview
            """
            shaft_bin = (shaft_map > self.skeleton_mask_threshold).astype(np.uint8)

            skel = skeletonize(shaft_bin > 0).astype(np.float32)

            if self.skeleton_dilate_kernel is not None and self.skeleton_dilate_kernel > 0:
                k = int(self.skeleton_dilate_kernel)
                kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
                skel = cv2.dilate((skel * 255).astype(np.uint8), kernel).astype(np.float32) / 255.0
                skel = (skel > 0.0).astype(np.float32)

            return skel.astype(np.float32)

        def _make_point_heatmap(self, arrows, key="head_hint"):
            hm = np.zeros((self.target_h, self.target_w), dtype=np.float32)
            for arrow in arrows:
                if key not in arrow:
                    continue
                p = arrow[key]
                x = float(p["x"])
                y = float(p["y"])
                sigma = self.endpoint_heatmap_sigma if key in {"head_hint", "tail_hint"} else self.point_sigma
                g = self._gaussian_2d(x, y, sigma)
                hm = np.maximum(hm, g)
            return self._clip01(hm)

        def _make_direction_field(self, arrows, shaft_map=None):
            vx = np.zeros((self.target_h, self.target_w), dtype=np.float32)
            vy = np.zeros((self.target_h, self.target_w), dtype=np.float32)
            wt = np.zeros((self.target_h, self.target_w), dtype=np.float32)

            for arrow in arrows:
                pts = self._path_to_xy_array(arrow["path"])
                if len(pts) < 2:
                    continue

                for i in range(len(pts) - 1):
                    p1 = pts[i]
                    p2 = pts[i + 1]

                    d = p2 - p1
                    norm = np.linalg.norm(d)
                    if norm < 1e-6:
                        continue

                    t = d / norm
                    mx, my = (p1 + p2) / 2.0

                    dist2 = (self.xx - mx) ** 2 + (self.yy - my) ** 2
                    local_w = np.exp(-dist2 / (2.0 * self.dir_radius ** 2 + 1e-12)).astype(np.float32)

                    vx += local_w * t[0]
                    vy += local_w * t[1]
                    wt += local_w

            mask = wt > 1e-6
            vx[mask] /= wt[mask]
            vy[mask] /= wt[mask]

            mag = np.sqrt(vx ** 2 + vy ** 2)
            nz = mag > 1e-6
            vx[nz] /= mag[nz]
            vy[nz] /= mag[nz]

            if self.dir_from_shaft_mask and shaft_map is not None:
                shaft_mask = shaft_map > self.shaft_mask_threshold
                vx = vx * shaft_mask.astype(np.float32)
                vy = vy * shaft_mask.astype(np.float32)
                mask = mask & shaft_mask

            return vx.astype(np.float32), vy.astype(np.float32), mask.astype(np.uint8)
        
        def _draw_disk_mask(self, cx, cy, radius):
            mask = np.zeros((self.target_h, self.target_w), dtype=np.uint8)
            cv2.circle(mask, (int(round(cx)), int(round(cy))), int(radius), 1, thickness=-1)
            return mask.astype(np.float32)

        def _unit_vector_from_points(self, p1, p2):
            d = np.array(p2, dtype=np.float32) - np.array(p1, dtype=np.float32)
            n = np.linalg.norm(d)
            if n < 1e-6:
                return np.array([0.0, 0.0], dtype=np.float32)
            return d / n

        def _make_progress_and_endpoint_targets(self, arrows, shaft_map):
            H, W = self.target_h, self.target_w

            progress_sum = np.zeros((H, W), dtype=np.float32)
            progress_count = np.zeros((H, W), dtype=np.float32)

            head_off_x = np.zeros((H, W), dtype=np.float32)
            head_off_y = np.zeros((H, W), dtype=np.float32)
            head_off_mask = np.zeros((H, W), dtype=np.float32)

            tail_off_x = np.zeros((H, W), dtype=np.float32)
            tail_off_y = np.zeros((H, W), dtype=np.float32)
            tail_off_mask = np.zeros((H, W), dtype=np.float32)

            head_tan_x = np.zeros((H, W), dtype=np.float32)
            head_tan_y = np.zeros((H, W), dtype=np.float32)
            head_tan_mask = np.zeros((H, W), dtype=np.float32)

            tail_tan_x = np.zeros((H, W), dtype=np.float32)
            tail_tan_y = np.zeros((H, W), dtype=np.float32)
            tail_tan_mask = np.zeros((H, W), dtype=np.float32)

            for arrow in arrows:
                path = self._path_to_xy_array(arrow["path"])
                if len(path) < 2:
                    continue

                # ---- progress along ordered path ----
                seg_lengths = np.linalg.norm(path[1:] - path[:-1], axis=1)
                total_len = float(seg_lengths.sum())
                if total_len < 1e-6:
                    continue

                cum_len = np.concatenate([[0.0], np.cumsum(seg_lengths)], axis=0)
                node_progress = cum_len / total_len

                # draw progress on segments
                for i in range(len(path) - 1):
                    p1 = path[i]
                    p2 = path[i + 1]
                    t1 = node_progress[i]
                    t2 = node_progress[i + 1]

                    x1, y1 = p1
                    x2, y2 = p2

                    seg_mask = np.zeros((H, W), dtype=np.uint8)
                    cv2.line(
                        seg_mask,
                        (int(round(x1)), int(round(y1))),
                        (int(round(x2)), int(round(y2))),
                        color=1,
                        thickness=self.progress_thickness,
                    )

                    ys, xs = np.where(seg_mask > 0)
                    if len(xs) == 0:
                        continue

                    # interpolate progress locally by projection
                    d = np.array([x2 - x1, y2 - y1], dtype=np.float32)
                    denom = float(np.dot(d, d)) + 1e-6

                    px = xs.astype(np.float32) - x1
                    py = ys.astype(np.float32) - y1
                    alpha = (px * d[0] + py * d[1]) / denom
                    alpha = np.clip(alpha, 0.0, 1.0)

                    vals = (1.0 - alpha) * t1 + alpha * t2
                    progress_sum[ys, xs] += vals.astype(np.float32)
                    progress_count[ys, xs] += 1.0

                # ---- exact endpoint offsets + tangents ----
                tail = arrow.get("tail_hint", None)
                head = arrow.get("head_hint", None)
                if tail is None or head is None:
                    continue

                tx, ty = float(tail["x"]), float(tail["y"])
                hx, hy = float(head["x"]), float(head["y"])

                tail_mask = self._draw_disk_mask(tx, ty, self.endpoint_radius)
                head_mask = self._draw_disk_mask(hx, hy, self.endpoint_radius)

                tail_ys, tail_xs = np.where(tail_mask > 0)
                head_ys, head_xs = np.where(head_mask > 0)

                # offsets point from local pixel to exact endpoint
                r = float(self.endpoint_radius)

                tail_off_x[tail_ys, tail_xs] = (tx - tail_xs.astype(np.float32)) / r
                tail_off_y[tail_ys, tail_xs] = (ty - tail_ys.astype(np.float32)) / r
                tail_off_mask[tail_ys, tail_xs] = 1.0

                head_off_x[head_ys, head_xs] = (hx - head_xs.astype(np.float32)) / r
                head_off_y[head_ys, head_xs] = (hy - head_ys.astype(np.float32)) / r
                head_off_mask[head_ys, head_xs] = 1.0

                # local endpoint tangents from ordered path
                if len(path) >= 2:
                    tail_k = min(9, len(path) - 1)
                    head_k = min(12, len(path) - 1)
                
                    tail_t = self._unit_vector_from_points(path[0], path[tail_k])
                    head_t = self._unit_vector_from_points(path[-head_k-1], path[-1])

                    tail_tan_x[tail_ys, tail_xs] = tail_t[0]
                    tail_tan_y[tail_ys, tail_xs] = tail_t[1]
                    tail_tan_mask[tail_ys, tail_xs] = 1.0

                    head_tan_x[head_ys, head_xs] = head_t[0]
                    head_tan_y[head_ys, head_xs] = head_t[1]
                    head_tan_mask[head_ys, head_xs] = 1.0

            progress_map = np.zeros((H, W), dtype=np.float32)
            valid = progress_count > 0
            progress_map[valid] = progress_sum[valid] / progress_count[valid]

            if self.progress_blur_sigma > 0:
                k = max(3, int(round(6 * self.progress_blur_sigma)) | 1)
                progress_map = cv2.GaussianBlur(progress_map, (k, k), self.progress_blur_sigma)

            if self.progress_use_full_shaft_mask:
                shaft_mask = (shaft_map > self.shaft_mask_threshold).astype(np.float32)
                progress_mask = shaft_mask
            else:
                progress_mask = (progress_count > 0).astype(np.float32)

            progress_map = progress_map * progress_mask

            return {
                "progress_map": progress_map.astype(np.float32),
                "progress_mask": progress_mask.astype(np.float32),

                "head_off_x": head_off_x.astype(np.float32),
                "head_off_y": head_off_y.astype(np.float32),
                "head_off_mask": head_off_mask.astype(np.float32),

                "tail_off_x": tail_off_x.astype(np.float32),
                "tail_off_y": tail_off_y.astype(np.float32),
                "tail_off_mask": tail_off_mask.astype(np.float32),

                "head_tan_x": head_tan_x.astype(np.float32),
                "head_tan_y": head_tan_y.astype(np.float32),
                "head_tan_mask": head_tan_mask.astype(np.float32),

                "tail_tan_x": tail_tan_x.astype(np.float32),
                "tail_tan_y": tail_tan_y.astype(np.float32),
                "tail_tan_mask": tail_tan_mask.astype(np.float32),
            }
    
        def _make_all_path_map(self, arrows):
            """
            Build a GT path-support map for ALL arrows.
            This is different from loop_path_map:
            - loop_path_map = only semantic self-loops
            - all_path_map  = every arrow path in the JSON
            """
            all_path = np.zeros((self.target_h, self.target_w), dtype=np.uint8)

            for arrow in arrows:
                pts = self._path_to_xy_array(arrow["path"])
                if len(pts) < 2:
                    continue

                pts_int = np.round(pts).astype(np.int32).reshape(-1, 1, 2)

                cv2.polylines(
                    all_path,
                    [pts_int],
                    isClosed=False,
                    color=255,
                    thickness=self.all_path_thickness,
                )

            all_path = all_path.astype(np.float32) / 255.0

            if self.all_path_blur_sigma > 0:
                k = max(3, int(round(6 * self.all_path_blur_sigma)) | 1)
                all_path = cv2.GaussianBlur(all_path, (k, k), self.all_path_blur_sigma)

            if all_path.max() > 0:
                all_path = all_path / all_path.max()

            return all_path.astype(np.float32)    
    
        def prepare_payload(self, sample_spec):
            sample_id = sample_spec["sample_id"]
            json_path = Path(sample_spec["json_path"])
            image_path = Path(sample_spec["image_path"])

            ann = self._load_json(json_path)
            img = self._load_image(image_path)

            img_resized, H_new, W_new = self._resize_into_annotation_space(img, ann)
            ann_final, img_final, H_final, W_final = self._rescale_everything(
                ann=ann,
                img_resized=img_resized,
                H_new=H_new,
                W_new=W_new,
                target_w=self.target_w,
                target_h=self.target_h,
            )

            if self.apply_otsu_threshold:
                if self.image_mode != "grayscale":
                    raise ValueError("Otsu thresholding is currently implemented only for grayscale mode.")
                img_final = self._apply_otsu_threshold(
                    img_final,
                    foreground_white=self.otsu_foreground_white,
                )

            assert H_final == self.target_h and W_final == self.target_w, (
                f"Final size mismatch: got {(W_final, H_final)} expected {(self.target_w, self.target_h)}"
            )

            nodes = ann_final.get("nodes", [])
            arrows = ann_final.get("arrows", [])

            node_center_map = self._make_node_center_map(nodes)
            node_extent_map = self._make_node_extent_map_soft_bbox(nodes)
            shaft_map = self._make_shaft_map(arrows)
            skeleton_map = self._make_skeleton_map(shaft_map) if self.use_skeleton_head else np.zeros_like(shaft_map, dtype=np.float32)
            head_map = self._make_point_heatmap(arrows, key="head_hint")
            tail_map = self._make_point_heatmap(arrows, key="tail_hint")
            dir_x, dir_y, dir_mask = self._make_direction_field(arrows, shaft_map=shaft_map)
            endpoint_struct = self._make_progress_and_endpoint_targets(arrows, shaft_map)
            if self.use_loop_path_loss:
                loop_shaft_map, loop_path_map = self._make_loop_supervision_maps(arrows)
            else:
                loop_shaft_map = np.zeros_like(shaft_map, dtype=np.float32)
                loop_path_map = np.zeros_like(shaft_map, dtype=np.float32)
        
            if self.use_general_path_loss:
                all_path_map = self._make_all_path_map(arrows)
            else:
                all_path_map = np.zeros_like(shaft_map, dtype=np.float32)
        
            target = np.stack(
                [
                    node_center_map,                      # 0
                    node_extent_map,                      # 1
                    shaft_map,                            # 2
                    skeleton_map,                         # 3
                    head_map,                             # 4
                    tail_map,                             # 5
                    dir_x,                                # 6
                    dir_y,                                # 7
                    loop_shaft_map,                       # 8
                    loop_path_map,                        # 9

                    endpoint_struct["progress_map"],      # 10
                    endpoint_struct["progress_mask"],     # 11

                    endpoint_struct["head_off_x"],        # 12
                    endpoint_struct["head_off_y"],        # 13
                    endpoint_struct["head_off_mask"],     # 14

                    endpoint_struct["tail_off_x"],        # 15
                    endpoint_struct["tail_off_y"],        # 16
                    endpoint_struct["tail_off_mask"],     # 17

                    endpoint_struct["head_tan_x"],        # 18
                    endpoint_struct["head_tan_y"],        # 19
                    endpoint_struct["head_tan_mask"],     # 20

                    endpoint_struct["tail_tan_x"],        # 21
                    endpoint_struct["tail_tan_y"],        # 22
                    endpoint_struct["tail_tan_mask"],     # 23
                    all_path_map,                         # 24
                ],
                axis=0,
            ).astype(np.float16)

            payload = {
                "target": torch.from_numpy(target),
                "dir_mask": torch.from_numpy(dir_mask[None, ...].astype(np.uint8)),
                "meta": {
                    "sample_id": sample_id,
                    "json_path": str(json_path),
                    "image_path": str(image_path),
                    "image_name": ann_final.get("image", image_path.name),
                    "num_nodes": len(nodes),
                    "num_arrows": len(arrows),
                    "target_size": (self.target_w, self.target_h),
                    "target_version": self.target_version,
                },
            }

            if self.save_resized_image_in_cache:
                payload["image"] = torch.from_numpy(self._image_to_chw_numpy(img_final))

            return payload

        def process_sample(self, sample_spec, cache_dir, overwrite=False):
            sample_id = sample_spec["sample_id"]
            cache_path = Path(cache_dir) / f"{sample_id}{PRECOMPUTE_CACHE_EXT}"

            if cache_path.exists() and not overwrite:
                return {"sample_id": sample_id, "cache_path": str(cache_path), "status": "skipped_exists"}

            payload = self.prepare_payload(sample_spec)
            save_precompute_payload(payload, cache_path)

            return {"sample_id": sample_id, "cache_path": str(cache_path), "status": "written"}


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # LONG-ARROW TARGET EXTENSION
    # ============================================================

    BasePrecomputeTargetGenerator = PrecomputeTargetGenerator

    class LongArrowPrecomputeTargetGenerator(BasePrecomputeTargetGenerator):
        """
        Adds extra target channels used only by the learning-side loss/metrics.

        Base channels remain unchanged:
            0..24 = original target channels

        Extra channels:
            25 = length_weight_map
            26 = long_path_map
            27 = very_long_path_map

        The model output heads are NOT changed.
        """

        def _polyline_length(self, pts):
            if pts is None or len(pts) < 2:
                return 0.0
            pts = np.asarray(pts, dtype=np.float32)
            diff = pts[1:] - pts[:-1]
            return float(np.sqrt((diff ** 2).sum(axis=1)).sum())

        def _draw_path_mask(self, pts, thickness):
            mask = np.zeros((self.target_h, self.target_w), dtype=np.uint8)
            if pts is None or len(pts) < 2:
                return mask.astype(np.float32)
            pts_int = np.round(pts).astype(np.int32).reshape(-1, 1, 2)
            cv2.polylines(mask, [pts_int], isClosed=False, color=255, thickness=int(thickness))
            return (mask.astype(np.float32) / 255.0)

        def _make_long_arrow_maps(self, arrows):
            diag = float(math.hypot(self.target_w, self.target_h))
            length_weight = np.ones((self.target_h, self.target_w), dtype=np.float32)
            long_path = np.zeros((self.target_h, self.target_w), dtype=np.float32)
            very_long_path = np.zeros((self.target_h, self.target_w), dtype=np.float32)

            lengths = []
            norm_lengths = []
            long_count = 0
            very_long_count = 0

            for arrow in arrows:
                pts = self._path_to_xy_array(arrow.get("path", []))
                if len(pts) < 2:
                    continue
                length = self._polyline_length(pts)
                norm_len = float(length / max(diag, 1e-6))
                lengths.append(length)
                norm_lengths.append(norm_len)

                # Smooth ramp after LONG_ARROW_MIN_NORM.
                ramp = (norm_len - float(LONG_ARROW_MIN_NORM)) / max(
                    1e-6, float(VERY_LONG_ARROW_MIN_NORM) - float(LONG_ARROW_MIN_NORM)
                )
                ramp = float(np.clip(ramp, 0.0, 1.0))
                w = 1.0 + float(LONG_ARROW_WEIGHT_ALPHA) * ramp
                w = float(min(w, LONG_ARROW_WEIGHT_CLIP))

                path_mask = self._draw_path_mask(pts, thickness=max(1, int(self.all_path_thickness)))
                if self.all_path_blur_sigma > 0:
                    k = max(3, int(round(6 * self.all_path_blur_sigma)) | 1)
                    soft = cv2.GaussianBlur(path_mask, (k, k), self.all_path_blur_sigma)
                    if soft.max() > 0:
                        soft = soft / soft.max()
                else:
                    soft = path_mask

                length_weight = np.maximum(length_weight, 1.0 + (w - 1.0) * soft)

                if norm_len >= float(LONG_ARROW_MIN_NORM):
                    long_count += 1
                    long_path = np.maximum(long_path, soft)
                if norm_len >= float(VERY_LONG_ARROW_MIN_NORM):
                    very_long_count += 1
                    very_long_path = np.maximum(very_long_path, soft)

            if len(lengths):
                arr = np.asarray(lengths, dtype=np.float32)
                narr = np.asarray(norm_lengths, dtype=np.float32)
                stats = {
                    "arrow_length_mean": float(np.mean(arr)),
                    "arrow_length_p50": float(np.percentile(arr, 50)),
                    "arrow_length_p90": float(np.percentile(arr, 90)),
                    "arrow_length_max": float(np.max(arr)),
                    "arrow_norm_length_mean": float(np.mean(narr)),
                    "arrow_norm_length_p90": float(np.percentile(narr, 90)),
                    "arrow_norm_length_max": float(np.max(narr)),
                    "long_arrow_count": int(long_count),
                    "very_long_arrow_count": int(very_long_count),
                }
            else:
                stats = {
                    "arrow_length_mean": 0.0,
                    "arrow_length_p50": 0.0,
                    "arrow_length_p90": 0.0,
                    "arrow_length_max": 0.0,
                    "arrow_norm_length_mean": 0.0,
                    "arrow_norm_length_p90": 0.0,
                    "arrow_norm_length_max": 0.0,
                    "long_arrow_count": 0,
                    "very_long_arrow_count": 0,
                }

            return (
                np.clip(length_weight, 1.0, float(LONG_ARROW_WEIGHT_CLIP)).astype(np.float32),
                self._clip01(long_path).astype(np.float32),
                self._clip01(very_long_path).astype(np.float32),
                stats,
            )

        def prepare_payload(self, sample_spec):
            sample_id = sample_spec["sample_id"]
            json_path = Path(sample_spec["json_path"])
            image_path = Path(sample_spec["image_path"])

            ann = self._load_json(json_path)
            img = self._load_image(image_path)

            img_resized, H_new, W_new = self._resize_into_annotation_space(img, ann)
            ann_final, img_final, H_final, W_final = self._rescale_everything(
                ann=ann,
                img_resized=img_resized,
                H_new=H_new,
                W_new=W_new,
                target_w=self.target_w,
                target_h=self.target_h,
            )

            if self.apply_otsu_threshold:
                if self.image_mode != "grayscale":
                    raise ValueError("Otsu thresholding is currently implemented only for grayscale mode.")
                img_final = self._apply_otsu_threshold(
                    img_final,
                    foreground_white=self.otsu_foreground_white,
                )

            assert H_final == self.target_h and W_final == self.target_w, (
                f"Final size mismatch: got {(W_final, H_final)} expected {(self.target_w, self.target_h)}"
            )

            nodes = ann_final.get("nodes", [])
            arrows = ann_final.get("arrows", [])

            node_center_map = self._make_node_center_map(nodes)
            node_extent_map = self._make_node_extent_map_soft_bbox(nodes)
            shaft_map = self._make_shaft_map(arrows)
            skeleton_map = self._make_skeleton_map(shaft_map) if self.use_skeleton_head else np.zeros_like(shaft_map, dtype=np.float32)
            head_map = self._make_point_heatmap(arrows, key="head_hint")
            tail_map = self._make_point_heatmap(arrows, key="tail_hint")
            dir_x, dir_y, dir_mask = self._make_direction_field(arrows, shaft_map=shaft_map)
            endpoint_struct = self._make_progress_and_endpoint_targets(arrows, shaft_map)

            if self.use_loop_path_loss:
                loop_shaft_map, loop_path_map = self._make_loop_supervision_maps(arrows)
            else:
                loop_shaft_map = np.zeros_like(shaft_map, dtype=np.float32)
                loop_path_map = np.zeros_like(shaft_map, dtype=np.float32)

            if self.use_general_path_loss:
                all_path_map = self._make_all_path_map(arrows)
            else:
                all_path_map = np.zeros_like(shaft_map, dtype=np.float32)

            length_weight_map, long_path_map, very_long_path_map, length_stats = self._make_long_arrow_maps(arrows)

            target = np.stack(
                [
                    node_center_map,                      # 0
                    node_extent_map,                      # 1
                    shaft_map,                            # 2
                    skeleton_map,                         # 3
                    head_map,                             # 4
                    tail_map,                             # 5
                    dir_x,                                # 6
                    dir_y,                                # 7
                    loop_shaft_map,                       # 8
                    loop_path_map,                        # 9
                    endpoint_struct["progress_map"],      # 10
                    endpoint_struct["progress_mask"],     # 11
                    endpoint_struct["head_off_x"],        # 12
                    endpoint_struct["head_off_y"],        # 13
                    endpoint_struct["head_off_mask"],     # 14
                    endpoint_struct["tail_off_x"],        # 15
                    endpoint_struct["tail_off_y"],        # 16
                    endpoint_struct["tail_off_mask"],     # 17
                    endpoint_struct["head_tan_x"],        # 18
                    endpoint_struct["head_tan_y"],        # 19
                    endpoint_struct["head_tan_mask"],     # 20
                    endpoint_struct["tail_tan_x"],        # 21
                    endpoint_struct["tail_tan_y"],        # 22
                    endpoint_struct["tail_tan_mask"],     # 23
                    all_path_map,                         # 24
                    length_weight_map,                    # 25
                    long_path_map,                        # 26
                    very_long_path_map,                   # 27
                ],
                axis=0,
            ).astype(np.float16)

            payload = {
                "target": torch.from_numpy(target),
                "dir_mask": torch.from_numpy(dir_mask[None, ...].astype(np.uint8)),
                "meta": {
                    "sample_id": sample_id,
                    "json_path": str(json_path),
                    "image_path": str(image_path),
                    "image_name": ann_final.get("image", image_path.name),
                    "family": sample_spec.get("family", ""),
                    "split": sample_spec.get("split", ""),
                    "num_nodes": len(nodes),
                    "num_arrows": len(arrows),
                    "target_size": (self.target_w, self.target_h),
                    "target_version": self.target_version,
                    **length_stats,
                },
            }

            if self.save_resized_image_in_cache:
                payload["image"] = torch.from_numpy(self._image_to_chw_numpy(img_final))

            return payload


    # Use the long-arrow-aware generator for all subsequent precompute operations.
    PrecomputeTargetGenerator = LongArrowPrecomputeTargetGenerator
    print("Using long-arrow-aware graph-evidence target generation.")


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # PARALLEL / SERIAL PRECOMPUTE
    # ============================================================

    def _worker_process_sample(sample_spec, generator_kwargs, cache_dir, overwrite):
        generator = PrecomputeTargetGenerator(**generator_kwargs)
        return generator.process_sample(sample_spec, cache_dir=cache_dir, overwrite=overwrite)


    def precompute_all_targets_serial(samples, cache_dir, generator_kwargs, overwrite=False):
        cache_dir = Path(cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)

        print(f"Starting SERIAL preprocessing for {len(samples)} samples...")
        print(f"Cache directory: {cache_dir}")

        generator = PrecomputeTargetGenerator(**generator_kwargs)

        results = []
        num_written = 0
        num_skipped = 0
        num_failed = 0

        for i, sample_spec in enumerate(samples, start=1):
            try:
                res = generator.process_sample(sample_spec, cache_dir=cache_dir, overwrite=overwrite)
                results.append(res)

                if res["status"] == "written":
                    num_written += 1
                elif res["status"] == "skipped_exists":
                    num_skipped += 1

                if i % 25 == 0 or i == len(samples):
                    print(f"[{i}/{len(samples)}] written={num_written}, skipped={num_skipped}, failed={num_failed}")

            except Exception as e:
                num_failed += 1
                print(f"❌ Failed on sample {sample_spec['sample_id']}: {repr(e)}")

        print("\nSerial precompute finished.")
        print(f"Written: {num_written}")
        print(f"Skipped: {num_skipped}")
        print(f"Failed : {num_failed}")

        return results


    def precompute_all_targets_parallel(samples, cache_dir, generator_kwargs, num_workers=2, overwrite=False):
        cache_dir = Path(cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)

        print(f"Starting PARALLEL preprocessing for {len(samples)} samples...")
        print(f"Cache directory: {cache_dir}")
        print(f"Workers: {num_workers}")

        results = []
        num_written = 0
        num_skipped = 0
        num_failed = 0

        with ThreadPoolExecutor(max_workers=num_workers) as ex:
            futures = [
                ex.submit(
                    _worker_process_sample,
                    sample_spec,
                    generator_kwargs,
                    str(cache_dir),
                    overwrite,
                )
                for sample_spec in samples
            ]

            for i, fut in enumerate(as_completed(futures), start=1):
                try:
                    res = fut.result()
                    results.append(res)

                    if res["status"] == "written":
                        num_written += 1
                    elif res["status"] == "skipped_exists":
                        num_skipped += 1

                    if i % 25 == 0 or i == len(futures):
                        print(f"[{i}/{len(futures)}] written={num_written}, skipped={num_skipped}, failed={num_failed}")

                except Exception as e:
                    num_failed += 1
                    print(f"❌ Worker failed: {repr(e)}")

        print("\nParallel precompute finished.")
        print(f"Written: {num_written}")
        print(f"Skipped: {num_skipped}")
        print(f"Failed : {num_failed}")

        return results


    def precompute_all_targets(samples, cache_dir, generator_kwargs, num_workers=2, overwrite=False):
        if num_workers is None or num_workers <= 1:
            print("Working Serially")
            return precompute_all_targets_serial(samples, cache_dir, generator_kwargs, overwrite=overwrite)

        return precompute_all_targets_parallel(
            samples=samples,
            cache_dir=cache_dir,
            generator_kwargs=generator_kwargs,
            num_workers=num_workers,
            overwrite=overwrite,
        )


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # COMPRESSED PRECOMPUTE CACHE HELPERS
    # ============================================================
    import shutil
    import tempfile

    PRECOMPUTE_CACHE_EXT = ".npz"


    def _tensor_or_array_to_numpy(x):
        if torch.is_tensor(x):
            return x.detach().cpu().numpy()
        return np.asarray(x)


    def save_precompute_payload(payload, cache_path):
        cache_path = Path(cache_path)
        cache_path.parent.mkdir(parents=True, exist_ok=True)

        arrays = {
            "target": _tensor_or_array_to_numpy(payload["target"]).astype(np.float16),
            "dir_mask": _tensor_or_array_to_numpy(payload["dir_mask"]).astype(np.uint8),
            "meta_json": np.array(json.dumps(payload.get("meta", {}), ensure_ascii=False)),
        }
        if "image" in payload:
            arrays["image"] = _tensor_or_array_to_numpy(payload["image"]).astype(np.float32)

        np.savez_compressed(str(cache_path), **arrays)


    def load_precompute_payload(cache_path):
        cache_path = Path(cache_path)
        with np.load(str(cache_path), allow_pickle=False) as data:
            payload = {
                "target": torch.from_numpy(data["target"]),
                "dir_mask": torch.from_numpy(data["dir_mask"]),
                "meta": json.loads(str(data["meta_json"].item())) if "meta_json" in data else {},
            }
            if "image" in data:
                payload["image"] = torch.from_numpy(data["image"])
        return payload


    def estimate_precompute_cache_usage(samples, generator_kwargs, sample_n=5, reference_total_n=None, random_seed=39):
        if not samples:
            return {
                "sample_count_used": 0,
                "reference_total_n": int(reference_total_n or 0),
                "sample_total_bytes": 0,
                "sample_total_gb": 0.0,
                "average_bytes_per_sample": 0.0,
                "estimated_total_bytes": 0.0,
                "estimated_total_gb": 0.0,
                "samples": [],
                "temp_dir": "",
            }

        sample_n = min(int(sample_n), len(samples))
        rng = random.Random(random_seed)
        if len(samples) <= sample_n:
            picked = list(samples)
        else:
            picked = rng.sample(list(samples), sample_n)

        temp_base = "/hosted_runtime/working" if Path("/hosted_runtime/working").exists() else None
        temp_dir = Path(tempfile.mkdtemp(prefix="precompute_estimate_", dir=temp_base))
        generator = PrecomputeTargetGenerator(**generator_kwargs)

        rows = []
        total_bytes = 0
        try:
            for s in picked:
                res = generator.process_sample(s, cache_dir=temp_dir, overwrite=True)
                cpath = Path(res["cache_path"])
                size_bytes = cpath.stat().st_size if cpath.exists() else 0
                rows.append({
                    "sample_id": s.get("sample_id", ""),
                    "family": s.get("family", ""),
                    "split": s.get("split", ""),
                    "cache_path": str(cpath),
                    "size_bytes": int(size_bytes),
                    "size_mb": float(size_bytes / (1024 ** 2)),
                })
                total_bytes += size_bytes
        finally:
            shutil.rmtree(temp_dir, ignore_errors=True)

        ref_total_n = int(reference_total_n if reference_total_n is not None else len(samples))
        avg_bytes = float(total_bytes) / max(1, len(rows))
        est_total_bytes = avg_bytes * ref_total_n
        return {
            "sample_count_used": len(rows),
            "reference_total_n": ref_total_n,
            "sample_total_bytes": int(total_bytes),
            "sample_total_gb": float(total_bytes / (1024 ** 3)),
            "average_bytes_per_sample": float(avg_bytes),
            "estimated_total_bytes": float(est_total_bytes),
            "estimated_total_gb": float(est_total_bytes / (1024 ** 3)),
            "samples": rows,
            "temp_dir": str(temp_dir),
        }


    def run_precompute_memory_trial(samples, generator_kwargs, sample_n=5, limit_gb=18.0, reference_total_n=None, save_path=None):
        estimate = estimate_precompute_cache_usage(
            samples=samples,
            generator_kwargs=generator_kwargs,
            sample_n=sample_n,
            reference_total_n=reference_total_n,
            random_seed=SPLIT_SEED,
        )
        estimate["limit_gb"] = float(limit_gb)
        estimate["within_limit"] = bool(estimate["estimated_total_gb"] <= float(limit_gb))
        if save_path is not None:
            save_json(Path(save_path), estimate)
        print(json.dumps(estimate, indent=2))
        return estimate


In [ ]:
if SHOULD_TRAIN_GRAPH:
    class DiagramJointAugment:
        def __init__(
            self,
            p=0.8,
            degrees=8,
            translate_frac=0.04,
            scale_range=(0.90, 1.10),
            brightness=0.15,
            contrast=0.15,
        ):
            self.p = p
            self.degrees = degrees
            self.translate_frac = translate_frac
            self.scale_range = scale_range
            self.brightness = brightness
            self.contrast = contrast

        def _rotate_vectors(self, target, angle_deg, scale):
            """
            Rotate vector-valued target channels after image affine.
            Vector-valued target channel layout:
            dir:          6,7
            head offset:  12,13
            tail offset:  15,16
            head tangent: 18,19
            tail tangent: 21,22
            """

            theta = math.radians(angle_deg)
            c = math.cos(theta)
            s = math.sin(theta)

            def rotate_pair(x_ch, y_ch, apply_scale=False, normalize=False):
                x = target[x_ch].clone()
                y = target[y_ch].clone()

                xr = c * x - s * y
                yr = s * x + c * y

                if apply_scale:
                    xr = xr * scale
                    yr = yr * scale

                if normalize:
                    mag = torch.sqrt(xr ** 2 + yr ** 2).clamp_min(1e-6)
                    xr = xr / mag
                    yr = yr / mag

                target[x_ch] = xr
                target[y_ch] = yr

            # Direction vectors: rotate only, keep unit-ish magnitude
            rotate_pair(6, 7, apply_scale=False, normalize=False)

            # Offset vectors: rotate and scale because pixel distances change
            rotate_pair(12, 13, apply_scale=False, normalize=False)
            rotate_pair(15, 16, apply_scale=False, normalize=False)

            # Tangents: rotate and normalize
            rotate_pair(18, 19, apply_scale=False, normalize=True)
            rotate_pair(21, 22, apply_scale=False, normalize=True)

            return target

        def __call__(self, image, target, dir_mask):
            if random.random() > self.p:
                return image, target, dir_mask

            _, H, W = image.shape

            angle = random.uniform(-self.degrees, self.degrees)
            max_dx = int(self.translate_frac * W)
            max_dy = int(self.translate_frac * H)
            translate = (
                random.randint(-max_dx, max_dx),
                random.randint(-max_dy, max_dy),
            )
            scale = random.uniform(*self.scale_range)
            shear = [0.0, 0.0]

            # Image: use bilinear, fill with white background
            image = TF.affine(
                image,
                angle=angle,
                translate=translate,
                scale=scale,
                shear=shear,
                interpolation=InterpolationMode.BILINEAR,
                fill=1.0,
            )

            # Target maps: same exact geometric transform
            target = TF.affine(
                target,
                angle=angle,
                translate=translate,
                scale=scale,
                shear=shear,
                interpolation=InterpolationMode.BILINEAR,
                fill=0.0,
            )

            # Binary direction mask: same transform, nearest interpolation
            dir_mask = TF.affine(
                dir_mask,
                angle=angle,
                translate=translate,
                scale=scale,
                shear=shear,
                interpolation=InterpolationMode.NEAREST,
                fill=0.0,
            )

            # Correct vector-valued annotations
            target = self._rotate_vectors(target, angle, scale)

            # Keep masks in sane range
            target = torch.clamp(target, min=-10.0, max=10.0)
            target[[0,1,2,3,4,5,8,9,10,11,14,17,20,23,24]] = torch.clamp(
                target[[0,1,2,3,4,5,8,9,10,11,14,17,20,23,24]],
                0.0,
                1.0
            )
            dir_mask = (dir_mask > 0.5).float()

            # Image-only augmentation
            if random.random() < 0.5:
                image = TF.adjust_brightness(
                    image,
                    brightness_factor=random.uniform(1 - self.brightness, 1 + self.brightness)
                )

            if random.random() < 0.5:
                image = TF.adjust_contrast(
                    image,
                    contrast_factor=random.uniform(1 - self.contrast, 1 + self.contrast)
                )

            image = torch.clamp(image, 0.0, 1.0)

            return image, target, dir_mask


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # FAST TRAINING DATASET
    # ============================================================

    class PrecomputedDiagramDataset(Dataset):
        def __init__(
            self,
            samples,
            cache_dir,
            image_mode="grayscale",
            normalize_image=True,
            image_interpolation=cv2.INTER_LINEAR,
            target_w=1024,
            target_h=1024,
            return_metadata=True,
            prefer_cached_image=False,
            apply_otsu_threshold=True,
            otsu_foreground_white=True,
            augment=None,
        ):
            self.samples = samples
            self.cache_dir = Path(cache_dir)
            self.image_mode = image_mode
            self.normalize_image = normalize_image
            self.image_interpolation = image_interpolation
            self.target_w = int(target_w)
            self.target_h = int(target_h)
            self.return_metadata = return_metadata
            self.prefer_cached_image = prefer_cached_image
            self.apply_otsu_threshold = apply_otsu_threshold
            self.otsu_foreground_white = otsu_foreground_white
            self.augment = augment

            assert self.image_mode in {"grayscale", "rgb"}

            self.sample_map = {s["sample_id"]: s for s in samples}
            self.cached_ids = []

            for s in samples:
                sample_id = s["sample_id"]
                cache_path = self.cache_dir / f"{sample_id}{PRECOMPUTE_CACHE_EXT}"
                if cache_path.exists():
                    self.cached_ids.append(sample_id)

            if len(self.cached_ids) == 0:
                raise RuntimeError(f"No cached {PRECOMPUTE_CACHE_EXT} files found in {self.cache_dir}")

            print(f"Dataset initialized with {len(self.cached_ids)} cached samples.")

        def __len__(self):
            return len(self.cached_ids)

        @staticmethod
        def _apply_otsu_threshold(img, foreground_white=True):
            if img.ndim != 2:
                raise ValueError("Otsu thresholding expects a grayscale image.")

            thresh_type = cv2.THRESH_BINARY if foreground_white else cv2.THRESH_BINARY_INV
            _, bin_img = cv2.threshold(img, 0, 255, thresh_type + cv2.THRESH_OTSU)
            return bin_img

        def _load_image(self, image_path):
            if self.image_mode == "grayscale":
                img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
                if img is None:
                    raise FileNotFoundError(f"Could not read image: {image_path}")
                img = cv2.resize(img, (self.target_w, self.target_h), interpolation=self.image_interpolation)
                if self.apply_otsu_threshold:
                    img = self._apply_otsu_threshold(
                        img,
                        foreground_white=self.otsu_foreground_white,
                    )
            
                arr = img.astype(np.float32)
                if self.normalize_image:
                    arr /= 255.0
                return torch.from_numpy(arr[None, :, :].astype(np.float32))

            img_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if img_bgr is None:
                raise FileNotFoundError(f"Could not read image: {image_path}")
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            img_rgb = cv2.resize(img_rgb, (self.target_w, self.target_h), interpolation=self.image_interpolation)

            arr = img_rgb.astype(np.float32)
            if self.normalize_image:
                arr /= 255.0
            return torch.from_numpy(np.transpose(arr, (2, 0, 1)).astype(np.float32))

        def __getitem__(self, idx):
            sample_id = self.cached_ids[idx]
            sample_spec = self.sample_map[sample_id]

            cache_path = self.cache_dir / f"{sample_id}{PRECOMPUTE_CACHE_EXT}"
            payload = load_precompute_payload(cache_path)

            if self.prefer_cached_image and "image" in payload:
                image = payload["image"].float()
                if self.normalize_image:
                    image = image / 255.0
            else:
                image = self._load_image(sample_spec["image_path"])

            target = payload["target"].float()
            dir_mask = payload["dir_mask"].float()

            if self.augment is not None:
                image, target, dir_mask = self.augment(image, target, dir_mask)

            out = {
                "image": image,
                "target": target,
                "dir_mask": dir_mask,
            }

        


            if self.return_metadata:
                out["meta"] = payload.get("meta", {})

            return out


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # DATASET, CHECKPOINT, AND VISUALIZATION HELPERS
    # ============================================================

    def build_full_dataset(samples, augment=None):
        return PrecomputedDiagramDataset(
            samples=samples,
            cache_dir=CACHE_DIR,
            image_mode=IMAGE_MODE,
            normalize_image=NORMALIZE_IMAGE,
            image_interpolation=IMAGE_INTERPOLATION,
            target_w=TARGET_W,
            target_h=TARGET_H,
            return_metadata=True,
            prefer_cached_image=SAVE_RESIZED_IMAGE_IN_CACHE,
            apply_otsu_threshold=APPLY_OTSU_THRESHOLD,
            otsu_foreground_white=OTSU_FOREGROUND_WHITE,
            augment=augment,
        )


    def to_uint8_vis(array) -> np.ndarray:
        array = np.asarray(array, dtype=np.float32)
        finite = np.isfinite(array)
        if not finite.any():
            return np.zeros(array.shape, dtype=np.uint8)

        valid = array[finite]
        lo = float(valid.min())
        hi = float(valid.max())
        if hi <= lo:
            return np.zeros(array.shape, dtype=np.uint8)

        normalized = (array - lo) / (hi - lo)
        normalized[~finite] = 0.0
        return np.clip(normalized * 255.0, 0, 255).astype(np.uint8)


    def get_checkpoint_state(
        epoch,
        model,
        optimizer,
        scheduler,
        best_val_loss,
        train_loss,
        val_loss,
        run_name,
    ):
        return {
            "epoch": int(epoch),
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": None if scheduler is None else scheduler.state_dict(),
            "best_val_loss": float(best_val_loss),
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "run_name": run_name,
            "target_version": TARGET_VERSION,
            "config": PAPER_RUN_CONFIG,
        }


    def save_checkpoint(state, path: Path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(state, path)


    def save_split_manifest(path: Path, samples: Sequence[Dict[str, str]]) -> None:
        rows = [
            {
                "sample_id": sample["sample_id"],
                "family": sample["family"],
                "source_split": sample["split"],
                "image_path": sample["image_path"],
                "json_path": sample["json_path"],
            }
            for sample in samples
        ]
        save_json(path, rows)


In [ ]:
if SHOULD_TRAIN_GRAPH:
    def make_generator_kwargs():
        return {
            "target_w": TARGET_W,
            "target_h": TARGET_H,
            "image_mode": IMAGE_MODE,
            "normalize_image": NORMALIZE_IMAGE,
            "image_interpolation": IMAGE_INTERPOLATION,
            "apply_otsu_threshold": APPLY_OTSU_THRESHOLD,
            "otsu_foreground_white": OTSU_FOREGROUND_WHITE,
            "node_center_alpha": NODE_CENTER_ALPHA,
            "node_extent_alpha": NODE_EXTENT_ALPHA,
            "node_extent_beta": NODE_EXTENT_BETA,
            "shaft_thickness": SHAFT_THICKNESS,
            "shaft_blur_sigma": SHAFT_BLUR_SIGMA,
            "point_sigma": POINT_SIGMA,
            "dir_radius": DIR_RADIUS,
            "dir_from_shaft_mask": DIR_FROM_SHAFT_MASK,
            "shaft_mask_threshold": SHAFT_MASK_THRESHOLD,
            "save_resized_image_in_cache": SAVE_RESIZED_IMAGE_IN_CACHE,
            "use_skeleton_head": USE_SKELETON_HEAD,
            "skeleton_mask_threshold": SKELETON_MASK_THRESHOLD,
            "skeleton_dilate_kernel": SKELETON_DILATE_KERNEL,
            "use_loop_path_loss": USE_LOOP_PATH_LOSS,
            "loop_path_thickness": LOOP_PATH_THICKNESS,
            "loop_path_blur_sigma": LOOP_PATH_BLUR_SIGMA,
            "endpoint_heatmap_sigma": ENDPOINT_HEATMAP_SIGMA,
            "endpoint_radius": ENDPOINT_RADIUS,
            "progress_thickness": PROGRESS_THICKNESS,
            "progress_blur_sigma": PROGRESS_BLUR_SIGMA,
            "progress_use_full_shaft_mask": PROGRESS_USE_FULL_SHAFT_MASK,
            "use_endpoint_structured_heads": USE_ENDPOINT_STRUCTURED_HEADS,
            "use_general_path_loss": USE_GENERAL_PATH_LOSS,
            "all_path_thickness": ALL_PATH_THICKNESS,
            "all_path_blur_sigma": ALL_PATH_BLUR_SIGMA,
            "target_version": TARGET_VERSION,
        }


In [ ]:
if SHOULD_TRAIN_GRAPH:
    def save_target_previews(samples, generator_kwargs, output_dir, max_items=3):
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        generator = PrecomputeTargetGenerator(**generator_kwargs)

        for sample in list(samples)[:max_items]:
            payload = generator.prepare_payload(sample)
            target = payload["target"].numpy()
            image_tensor = payload.get("image")
            if image_tensor is not None:
                image = image_tensor[0].numpy()
            else:
                image = cv2.imread(sample["image_path"], cv2.IMREAD_GRAYSCALE)
                image = cv2.resize(image, (TARGET_W, TARGET_H))

            def tile(array, title):
                visual = to_uint8_vis(array)
                visual = cv2.applyColorMap(visual, cv2.COLORMAP_TURBO)
                cv2.putText(
                    visual, title, (5, 18), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (255, 255, 255), 2
                )
                cv2.putText(
                    visual, title, (5, 18), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (0, 0, 0), 1
                )
                return visual

            tiles = [
                tile(image, "image"),
                tile(target[0], "node_center"),
                tile(target[1], "node_extent"),
                tile(target[2], "shaft"),
                tile(target[3], "skeleton"),
                tile(target[4], "head"),
                tile(target[5], "tail"),
                tile(target[24], "all_path"),
                tile(target[25], "length_weight"),
                tile(target[26], "long_path"),
                tile(target[27], "very_long_path"),
                np.zeros((TARGET_H, TARGET_W, 3), dtype=np.uint8),
            ]
            rows = [
                np.concatenate(tiles[0:4], axis=1),
                np.concatenate(tiles[4:8], axis=1),
                np.concatenate(tiles[8:12], axis=1),
            ]
            panel = np.concatenate(rows, axis=0)
            cv2.imwrite(
                str(output_dir / f"{sample['sample_id']}_target_preview.png"),
                panel,
            )


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # MODEL
    # ============================================================

    def make_group_norm(num_channels, max_groups=8):
        """
        Pick a valid number of groups for GroupNorm.
        num_channels must be divisible by num_groups.
        """
        for g in [max_groups, 4, 2, 1]:
            if num_channels % g == 0:
                return nn.GroupNorm(num_groups=g, num_channels=num_channels)

        return nn.GroupNorm(num_groups=1, num_channels=num_channels)

    class DoubleConv(nn.Module):
        def __init__(self, in_ch, out_ch):
            super().__init__()
            self.block = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                # nn.BatchNorm2d(out_ch),
                make_group_norm(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
                # nn.BatchNorm2d(out_ch),
                make_group_norm(out_ch),
                nn.ReLU(inplace=True),
            )

        def forward(self, x):
            return self.block(x)


    class Down(nn.Module):
        def __init__(self, in_ch, out_ch):
            super().__init__()
            self.block = nn.Sequential(
                nn.MaxPool2d(2),
                DoubleConv(in_ch, out_ch),
            )

        def forward(self, x):
            return self.block(x)


    class Up(nn.Module):
        def __init__(self, in_ch, skip_ch, out_ch):
            super().__init__()
            self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv((in_ch // 2) + skip_ch, out_ch)

        def forward(self, x, skip):
            x = self.up(x)

            diff_y = skip.size(2) - x.size(2)
            diff_x = skip.size(3) - x.size(3)
            if diff_x != 0 or diff_y != 0:
                x = F.pad(
                    x,
                    [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2],
                )

            x = torch.cat([skip, x], dim=1)
            return self.conv(x)


    class DiagramUNet(nn.Module):
        def __init__(self, in_channels=1, base_ch=32):
            super().__init__()

            self.inc = DoubleConv(in_channels, base_ch)
            self.down1 = Down(base_ch, base_ch * 2)
            self.down2 = Down(base_ch * 2, base_ch * 4)
            self.down3 = Down(base_ch * 4, base_ch * 8)
            self.down4 = Down(base_ch * 8, base_ch * 16)

            self.bottleneck_dropout = nn.Dropout2d(0.25)

            self.up1 = Up(base_ch * 16, base_ch * 8, base_ch * 8)
            self.up2 = Up(base_ch * 8, base_ch * 4, base_ch * 4)
            self.up3 = Up(base_ch * 4, base_ch * 2, base_ch * 2)
            self.up4 = Up(base_ch * 2, base_ch, base_ch)

            self.node_center_head = nn.Conv2d(base_ch, 1, kernel_size=1)
            self.node_extent_head = nn.Conv2d(base_ch, 1, kernel_size=1)
            self.shaft_head = nn.Conv2d(base_ch, 1, kernel_size=1)
            self.skeleton_head = nn.Conv2d(base_ch, 1, kernel_size=1)
            self.head_head = nn.Conv2d(base_ch, 1, kernel_size=1)
            self.tail_head = nn.Conv2d(base_ch, 1, kernel_size=1)
            self.dir_head = nn.Conv2d(base_ch, 2, kernel_size=1)
        
            self.progress_head = nn.Conv2d(base_ch, 1, kernel_size=1)

            self.head_offset_head = nn.Conv2d(base_ch, 2, kernel_size=1)
            self.tail_offset_head = nn.Conv2d(base_ch, 2, kernel_size=1)

            self.head_tangent_head = nn.Conv2d(base_ch, 2, kernel_size=1)
            self.tail_tangent_head = nn.Conv2d(base_ch, 2, kernel_size=1)
        

        def forward(self, x):
            x1 = self.inc(x)
            x2 = self.down1(x1)
            x3 = self.down2(x2)
            x4 = self.down3(x3)
            x5 = self.down4(x4)
        
            x5 = self.bottleneck_dropout(x5) 

            x = self.up1(x5, x4)
            x = self.up2(x, x3)
            x = self.up3(x, x2)
            x = self.up4(x, x1)

            return {
                "node_center": self.node_center_head(x),
                "node_extent": self.node_extent_head(x),
                "shaft": self.shaft_head(x),
                "skeleton": self.skeleton_head(x),
                "head": self.head_head(x),
                "tail": self.tail_head(x),
                "dir": self.dir_head(x),
                "progress": self.progress_head(x),
                "head_offset": self.head_offset_head(x),
                "tail_offset": self.tail_offset_head(x),
                "head_tangent": self.head_tangent_head(x),
                "tail_tangent": self.tail_tangent_head(x),
            }


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # LOSSES
    # ============================================================

    class SoftDiceLoss(nn.Module):
        def __init__(self, smooth=1.0):
            super().__init__()
            self.smooth = smooth

        def forward(self, pred_logits, target):
            pred = torch.sigmoid(pred_logits)

            pred = pred.contiguous().view(pred.size(0), -1)
            target = target.contiguous().view(target.size(0), -1)

            intersection = (pred * target).sum(dim=1)
            denom = pred.sum(dim=1) + target.sum(dim=1)

            dice = (2.0 * intersection + self.smooth) / (denom + self.smooth)
            return (1.0 - dice).mean()


    class BCEPlusDiceLoss(nn.Module):
        def __init__(self, bce_weight=1.0, dice_weight=1.0, pos_weight=None):
            super().__init__()
            self.register_buffer(
                "pos_weight_tensor",
                None if pos_weight is None else torch.tensor([pos_weight], dtype=torch.float32),
                persistent=False,
            )
            self.bce_weight = bce_weight
            self.dice_weight = dice_weight
            self.dice = SoftDiceLoss()

        def forward(self, pred_logits, target):
            if self.pos_weight_tensor is not None:
                bce = F.binary_cross_entropy_with_logits(
                    pred_logits, target, pos_weight=self.pos_weight_tensor
                )
            else:
                bce = F.binary_cross_entropy_with_logits(pred_logits, target)

            dice = self.dice(pred_logits, target)
            return self.bce_weight * bce + self.dice_weight * dice


    class DirectionLoss(nn.Module):
        def __init__(self, cosine_weight=1.0, l2_weight=1.0, eps=1e-8):
            super().__init__()
            self.cosine_weight = cosine_weight
            self.l2_weight = l2_weight
            self.eps = eps

        def forward(self, pred_dir, target_dir, dir_mask):
            mask = dir_mask.expand_as(pred_dir)

            diff2 = (pred_dir - target_dir) ** 2
            l2 = (diff2 * mask).sum() / (mask.sum() + self.eps)

            pred_norm = torch.sqrt((pred_dir ** 2).sum(dim=1, keepdim=True) + self.eps)
            targ_norm = torch.sqrt((target_dir ** 2).sum(dim=1, keepdim=True) + self.eps)

            pred_unit = pred_dir / pred_norm
            targ_unit = target_dir / targ_norm

            cos = (pred_unit * targ_unit).sum(dim=1, keepdim=True)
            cos_loss = ((1.0 - cos) * dir_mask).sum() / (dir_mask.sum() + self.eps)

            return self.cosine_weight * cos_loss + self.l2_weight * l2

    def soft_erode(img):
        if img.dim() != 4:
            raise ValueError("soft_erode expects BxCxHxW")
        p1 = -F.max_pool2d(-img, kernel_size=(3, 1), stride=1, padding=(1, 0))
        p2 = -F.max_pool2d(-img, kernel_size=(1, 3), stride=1, padding=(0, 1))
        return torch.min(p1, p2)


    def soft_dilate(img):
        return F.max_pool2d(img, kernel_size=3, stride=1, padding=1)


    def soft_open(img):
        return soft_dilate(soft_erode(img))


    def soft_skeletonize(img, iters=15):
        img = img.clamp(0, 1)
        skel = torch.relu(img - soft_open(img))
        for _ in range(iters - 1):
            img = soft_erode(img)
            delta = torch.relu(img - soft_open(img))
            skel = skel + torch.relu(delta - skel * delta)
        return skel


    class SoftClDiceLoss(nn.Module):
        def __init__(self, iters=15, smooth=1.0):
            super().__init__()
            self.iters = iters
            self.smooth = smooth

        def forward(self, pred_prob, target):
            """
            pred_prob: sigmoid(shaft_logits), shape [B,1,H,W]
            target   : GT shaft mask, shape [B,1,H,W]
            """
            pred_prob = pred_prob.clamp(0, 1)
            target = target.clamp(0, 1)

            skel_pred = soft_skeletonize(pred_prob, self.iters)
            skel_true = soft_skeletonize(target, self.iters)

            tprec = (skel_pred * target).sum(dim=(1, 2, 3)) / (skel_pred.sum(dim=(1, 2, 3)) + self.smooth)
            tsens = (skel_true * pred_prob).sum(dim=(1, 2, 3)) / (skel_true.sum(dim=(1, 2, 3)) + self.smooth)

            cl_dice = (2.0 * tprec * tsens + self.smooth) / (tprec + tsens + self.smooth)
            return (1.0 - cl_dice).mean()

    class LoopPathCoverageLoss(nn.Module):
        """
        Strong loop-specific continuity loss.

        Uses GT loop_path map to require predicted shaft probability
        to stay high along semantic self-loop paths.

        If no loop exists in a sample, loss becomes 0 for that sample.
        """
        def __init__(self, smooth=1.0):
            super().__init__()
            self.smooth = smooth

        def forward(self, pred_shaft_prob, loop_path_target):
            """
            pred_shaft_prob : [B,1,H,W] after sigmoid
            loop_path_target: [B,1,H,W], float target in [0,1]
            """
            pred_shaft_prob = pred_shaft_prob.clamp(0, 1)
            loop_path_target = loop_path_target.clamp(0, 1)

            # numerator = predicted support along loop path
            numer = (pred_shaft_prob * loop_path_target).sum(dim=(1, 2, 3))

            # denominator = amount of loop path supervision
            denom = loop_path_target.sum(dim=(1, 2, 3))

            # if no loop in sample, set loss to 0
            valid = denom > 1e-6

            loss_per_sample = torch.zeros_like(numer)
            loss_per_sample[valid] = 1.0 - (
                (numer[valid] + self.smooth) / (denom[valid] + self.smooth)
            )

            return loss_per_sample.mean()


    class GeneralPathCoverageLoss(nn.Module):
        """
        General path continuity loss for ALL arrows.

        Requires predicted shaft probability to stay high along the GT path map.
        """
        def __init__(self, smooth=1.0):
            super().__init__()
            self.smooth = smooth

        def forward(self, pred_shaft_prob, all_path_target):
            pred_shaft_prob = pred_shaft_prob.clamp(0, 1)
            all_path_target = all_path_target.clamp(0, 1)

            numer = (pred_shaft_prob * all_path_target).sum(dim=(1, 2, 3))
            denom = all_path_target.sum(dim=(1, 2, 3))

            valid = denom > 1e-6
            loss_per_sample = torch.zeros_like(numer)

            loss_per_sample[valid] = 1.0 - (
                (numer[valid] + self.smooth) / (denom[valid] + self.smooth)
            )

            return loss_per_sample.mean()

        
    class MaskedSmoothL1Loss(nn.Module):
        def __init__(self, beta=1.0, eps=1e-8):
            super().__init__()
            self.beta = beta
            self.eps = eps

        def forward(self, pred, target, mask):
            # pred, target: [B,C,H,W]
            # mask: [B,1,H,W]
            diff = F.smooth_l1_loss(pred, target, reduction="none", beta=self.beta)
            diff = diff * mask.expand_as(diff)
            return diff.sum() / (mask.sum() * pred.shape[1] + self.eps)


    def masked_angle_error_deg(pred, target, mask, eps=1e-6):
        pred = torch.nn.functional.normalize(pred, dim=1, eps=eps)
        target = torch.nn.functional.normalize(target, dim=1, eps=eps)

        cos = (pred * target).sum(dim=1).clamp(-1 + eps, 1 - eps)
        angle = torch.acos(cos) * 180.0 / torch.pi

        m = mask.squeeze(1) > 0.5
        if m.sum() == 0:
            return torch.tensor(0.0, device=pred.device)

        return angle[m].mean()

    class MaskedCosineDirectionLoss(nn.Module):
        def __init__(self, eps=1e-8):
            super().__init__()
            self.eps = eps

        def forward(self, pred_vec, target_vec, mask):
            pred_norm = torch.sqrt((pred_vec ** 2).sum(dim=1, keepdim=True) + self.eps)
            targ_norm = torch.sqrt((target_vec ** 2).sum(dim=1, keepdim=True) + self.eps)

            pred_unit = pred_vec / pred_norm
            targ_unit = target_vec / targ_norm

            cos = (pred_unit * targ_unit).sum(dim=1, keepdim=True)
            loss = (1.0 - cos) * mask
            return loss.sum() / (mask.sum() + self.eps)


    class MaskedL1Loss(nn.Module):
        def __init__(self, eps=1e-8):
            super().__init__()
            self.eps = eps

        def forward(self, pred, target, mask):
            diff = torch.abs(pred - target) * mask
            return diff.sum() / (mask.sum() + self.eps)

    def diagram_outputs_to_probs(outputs):
        return {
            "node_center": torch.sigmoid(outputs["node_center"]),
            "node_extent": torch.sigmoid(outputs["node_extent"]),
            "shaft": torch.sigmoid(outputs["shaft"]),
            "skeleton": torch.sigmoid(outputs["skeleton"]),
            "head": torch.sigmoid(outputs["head"]),
            "tail": torch.sigmoid(outputs["tail"]),
            "progress": torch.sigmoid(outputs["progress"]),
            "head_offset": outputs["head_offset"],
            "tail_offset": outputs["tail_offset"],
            "head_tangent": outputs["head_tangent"],
            "tail_tangent": outputs["tail_tangent"],
            "dir": outputs["dir"],
        }


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # LONG-ARROW-AWARE LOSS, METRICS, AND TRAINING RUNNER
    # ============================================================
    from collections import defaultdict, Counter

    class WeightedSoftDiceLoss(nn.Module):
        def __init__(self, smooth=1.0):
            super().__init__()
            self.smooth = smooth

        def forward(self, logits, target, weight=None):
            pred = torch.sigmoid(logits)
            if weight is None:
                weight = torch.ones_like(target)
            pred = pred.contiguous().view(pred.size(0), -1)
            target = target.contiguous().view(target.size(0), -1)
            weight = weight.contiguous().view(weight.size(0), -1)

            inter = (weight * pred * target).sum(dim=1)
            denom = (weight * pred).sum(dim=1) + (weight * target).sum(dim=1)
            dice = (2.0 * inter + self.smooth) / (denom + self.smooth)
            return (1.0 - dice).mean()


    class WeightedBCEPlusDiceLoss(nn.Module):
        def __init__(self, bce_weight=1.0, dice_weight=1.0, pos_weight=None):
            super().__init__()
            self.bce_weight = bce_weight
            self.dice_weight = dice_weight
            self.dice = WeightedSoftDiceLoss()
            self.register_buffer(
                "pos_weight_tensor",
                None if pos_weight is None else torch.tensor([pos_weight], dtype=torch.float32),
                persistent=False,
            )

        def forward(self, logits, target, weight=None):
            if weight is None:
                weight = torch.ones_like(target)

            if self.pos_weight_tensor is not None:
                bce = F.binary_cross_entropy_with_logits(
                    logits, target, pos_weight=self.pos_weight_tensor.to(logits.device), reduction="none"
                )
            else:
                bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")

            bce = (bce * weight).sum() / (weight.sum() + 1e-8)
            dice = self.dice(logits, target, weight)
            return self.bce_weight * bce + self.dice_weight * dice


    class PathRecallLoss(nn.Module):
        """
        Encourages predicted probability mass on target path pixels.
        This is useful for long arrows where missing a small segment breaks continuity.
        """
        def __init__(self, eps=1e-8):
            super().__init__()
            self.eps = eps

        def forward(self, logits, target, weight=None):
            prob = torch.sigmoid(logits)
            if weight is None:
                weight = torch.ones_like(target)
            numer = (prob * target * weight).sum(dim=(1, 2, 3))
            denom = (target * weight).sum(dim=(1, 2, 3))
            valid = denom > self.eps
            if valid.sum() == 0:
                return torch.zeros((), device=logits.device, dtype=logits.dtype)
            recall = numer[valid] / (denom[valid] + self.eps)
            return (1.0 - recall).mean()


    def get_extra_target_channel(target, idx, fallback_like):
        if target.size(1) > idx:
            return target[:, idx:idx+1]
        return fallback_like


    class DiagramLoss(nn.Module):
        """
        Combines the dense graph-evidence objectives with long-arrow-aware continuity supervision.
        """
        def __init__(
            self,
            w_node_center=1.0,
            w_node_extent=1.0,
            w_shaft=1.0,
            w_skeleton=1.0,
            w_head=1.0,
            w_tail=1.0,
            w_dir=1.0,
            w_cldice=1.0,
            w_loop_path=1.0,
            w_progress=1.0,
            w_head_offset=1.0,
            w_tail_offset=1.0,
            w_head_tangent=0.5,
            w_tail_tangent=0.5,
            w_general_path=1.0,
            cldice_iters=15,
            pos_weight_head=None,
            pos_weight_tail=None,
            pos_weight_shaft=None,
            pos_weight_skeleton=None,
            pos_weight_node_center=None,
            pos_weight_node_extent=None,
            w_long_shaft=None,
            w_long_skeleton=None,
            w_long_path_recall=None,
            w_very_long_path_recall=None,
            w_long_cldice=None,
        ):
            super().__init__()
            self.node_center_loss = BCEPlusDiceLoss(pos_weight=pos_weight_node_center)
            self.node_extent_loss = BCEPlusDiceLoss(pos_weight=pos_weight_node_extent)
            self.shaft_loss = WeightedBCEPlusDiceLoss(pos_weight=pos_weight_shaft)
            self.skeleton_loss = WeightedBCEPlusDiceLoss(pos_weight=pos_weight_skeleton)
            self.head_loss = BCEPlusDiceLoss(pos_weight=pos_weight_head)
            self.tail_loss = BCEPlusDiceLoss(pos_weight=pos_weight_tail)

            self.dir_loss = DirectionLoss(cosine_weight=1.0, l2_weight=1.0)
            self.cldice_loss = SoftClDiceLoss(iters=cldice_iters)
            self.loop_path_loss = LoopPathCoverageLoss()
            self.general_path_loss = GeneralPathCoverageLoss()

            self.progress_loss = MaskedL1Loss()
            self.head_offset_loss = MaskedSmoothL1Loss(beta=1.0)
            self.tail_offset_loss = MaskedSmoothL1Loss(beta=1.0)
            self.head_tangent_loss = MaskedCosineDirectionLoss()
            self.tail_tangent_loss = MaskedCosineDirectionLoss()

            self.long_path_loss = WeightedBCEPlusDiceLoss(pos_weight=pos_weight_shaft)
            self.long_skeleton_loss = WeightedBCEPlusDiceLoss(pos_weight=pos_weight_skeleton)
            self.path_recall_loss = PathRecallLoss()

            self.w_node_center = w_node_center
            self.w_node_extent = w_node_extent
            self.w_shaft = w_shaft
            self.w_skeleton = w_skeleton
            self.w_head = w_head
            self.w_tail = w_tail
            self.w_dir = w_dir
            self.w_cldice = w_cldice
            self.w_loop_path = w_loop_path
            self.w_progress = w_progress
            self.w_head_offset = w_head_offset
            self.w_tail_offset = w_tail_offset
            self.w_head_tangent = w_head_tangent
            self.w_tail_tangent = w_tail_tangent
            self.w_general_path = w_general_path

            self.w_long_shaft = W_LONG_SHAFT if w_long_shaft is None else w_long_shaft
            self.w_long_skeleton = W_LONG_SKELETON if w_long_skeleton is None else w_long_skeleton
            self.w_long_path_recall = W_LONG_PATH_RECALL if w_long_path_recall is None else w_long_path_recall
            self.w_very_long_path_recall = W_VERY_LONG_PATH_RECALL if w_very_long_path_recall is None else w_very_long_path_recall
            self.w_long_cldice = W_LONG_CLDICE if w_long_cldice is None else w_long_cldice

        def forward(self, outputs, batch):
            target = batch["target"]
            dir_mask = batch["dir_mask"]

            t_node_center = target[:, 0:1]
            t_node_extent = target[:, 1:2]
            t_shaft = target[:, 2:3]
            t_skeleton = target[:, 3:4]
            t_head = target[:, 4:5]
            t_tail = target[:, 5:6]
            t_dir = target[:, 6:8]
            t_loop_path = target[:, 9:10]
            t_progress = target[:, 10:11]
            t_progress_mask = target[:, 11:12]
            t_head_offset = target[:, 12:14]
            t_head_offset_mask = target[:, 14:15]
            t_tail_offset = target[:, 15:17]
            t_tail_offset_mask = target[:, 17:18]
            t_head_tangent = target[:, 18:20]
            t_head_tangent_mask = target[:, 20:21]
            t_tail_tangent = target[:, 21:23]
            t_tail_tangent_mask = target[:, 23:24]
            t_all_path = target[:, 24:25]

            length_weight = get_extra_target_channel(target, 25, torch.ones_like(t_shaft)).clamp(1.0, LONG_ARROW_WEIGHT_CLIP)
            t_long_path = get_extra_target_channel(target, 26, torch.zeros_like(t_shaft))
            t_very_long_path = get_extra_target_channel(target, 27, torch.zeros_like(t_shaft))

            l_node_center = self.node_center_loss(outputs["node_center"], t_node_center)
            l_node_extent = self.node_extent_loss(outputs["node_extent"], t_node_extent)
            l_shaft = self.shaft_loss(outputs["shaft"], t_shaft, weight=length_weight)
            l_skeleton = self.skeleton_loss(outputs["skeleton"], t_skeleton, weight=length_weight)
            l_head = self.head_loss(outputs["head"], t_head)
            l_tail = self.tail_loss(outputs["tail"], t_tail)
            l_dir = self.dir_loss(outputs["dir"], t_dir, dir_mask)

            shaft_prob = torch.sigmoid(outputs["shaft"])
            skel_prob = torch.sigmoid(outputs["skeleton"])

            l_cldice = self.cldice_loss(shaft_prob, t_shaft)
            l_loop_path = self.loop_path_loss(shaft_prob, t_loop_path)
            l_general_path = self.general_path_loss(shaft_prob, t_all_path)

            l_long_shaft = self.long_path_loss(outputs["shaft"], t_long_path, weight=length_weight)
            l_long_skeleton = self.long_skeleton_loss(outputs["skeleton"], t_long_path, weight=length_weight)
            l_long_path_recall = self.path_recall_loss(outputs["shaft"], t_long_path, weight=length_weight)
            l_very_long_path_recall = self.path_recall_loss(outputs["shaft"], t_very_long_path, weight=length_weight)
            l_long_cldice = self.cldice_loss(shaft_prob, t_long_path)

            progress_prob = torch.sigmoid(outputs["progress"])
            l_progress = self.progress_loss(progress_prob, t_progress, t_progress_mask)

            l_head_offset = self.head_offset_loss(outputs["head_offset"], t_head_offset, t_head_offset_mask)
            l_tail_offset = self.tail_offset_loss(outputs["tail_offset"], t_tail_offset, t_tail_offset_mask)
            l_head_tangent = self.head_tangent_loss(outputs["head_tangent"], t_head_tangent, t_head_tangent_mask)
            l_tail_tangent = self.tail_tangent_loss(outputs["tail_tangent"], t_tail_tangent, t_tail_tangent_mask)

            total = (
                self.w_node_center * l_node_center
                + self.w_node_extent * l_node_extent
                + self.w_shaft * l_shaft
                + self.w_skeleton * l_skeleton
                + self.w_head * l_head
                + self.w_tail * l_tail
                + self.w_dir * l_dir
                + self.w_cldice * l_cldice
                + self.w_loop_path * l_loop_path
                + self.w_progress * l_progress
                + self.w_head_offset * l_head_offset
                + self.w_tail_offset * l_tail_offset
                + self.w_head_tangent * l_head_tangent
                + self.w_tail_tangent * l_tail_tangent
                + self.w_general_path * l_general_path
                + self.w_long_shaft * l_long_shaft
                + self.w_long_skeleton * l_long_skeleton
                + self.w_long_path_recall * l_long_path_recall
                + self.w_very_long_path_recall * l_very_long_path_recall
                + self.w_long_cldice * l_long_cldice
            )

            return total, {
                "loss_total": total.detach(),
                "loss_node_center": l_node_center.detach(),
                "loss_node_extent": l_node_extent.detach(),
                "loss_shaft": l_shaft.detach(),
                "loss_skeleton": l_skeleton.detach(),
                "loss_head": l_head.detach(),
                "loss_tail": l_tail.detach(),
                "loss_dir": l_dir.detach(),
                "loss_cldice": l_cldice.detach(),
                "loss_loop_path": l_loop_path.detach(),
                "loss_progress": l_progress.detach(),
                "loss_head_offset": l_head_offset.detach(),
                "loss_tail_offset": l_tail_offset.detach(),
                "loss_head_tangent": l_head_tangent.detach(),
                "loss_tail_tangent": l_tail_tangent.detach(),
                "loss_general_path": l_general_path.detach(),
                "loss_long_shaft": l_long_shaft.detach(),
                "loss_long_skeleton": l_long_skeleton.detach(),
                "loss_long_path_recall": l_long_path_recall.detach(),
                "loss_very_long_path_recall": l_very_long_path_recall.detach(),
                "loss_long_cldice": l_long_cldice.detach(),
                "head_tangent_angle": torch.zeros((), device=target.device),
                "tail_tangent_angle": torch.zeros((), device=target.device),
            }


    def diagram_collate_fn(batch):
        return {
            "image": torch.stack([b["image"] for b in batch], dim=0),
            "target": torch.stack([b["target"] for b in batch], dim=0),
            "dir_mask": torch.stack([b["dir_mask"] for b in batch], dim=0),
            "meta": [b.get("meta", {}) for b in batch],
        }


    def select_learning_samples(samples):
        selected = [s for s in samples if s.get("split") in set(LEARNING_SOURCE_SPLITS)]
        return selected


    def stratified_learning_split(samples):
        samples = list(samples)
        if not USE_STRATIFIED_LEARNING_SPLIT:
            return _split_sample_list(samples)[:2]

        by_family = defaultdict(list)
        for s in samples:
            by_family[s.get("family", "unknown")].append(s)

        train_samples, val_samples = [], []
        rng = random.Random(LEARNING_SPLIT_SEED)
        for fam, items in sorted(by_family.items()):
            items = list(items)
            rng.shuffle(items)
            n_val = max(1, int(round(len(items) * LEARNING_VAL_RATIO))) if len(items) > 1 else 0
            val_samples.extend(items[:n_val])
            train_samples.extend(items[n_val:])

        rng.shuffle(train_samples)
        rng.shuffle(val_samples)
        print("Stratified learning split:")
        print("  train:", Counter([s.get("family") for s in train_samples]))
        print("  val  :", Counter([s.get("family") for s in val_samples]))
        return train_samples, val_samples


    def build_learning_loaders(samples):
        learning_samples = select_learning_samples(samples)
        train_samples, val_samples = stratified_learning_split(learning_samples)

        train_aug = DiagramJointAugment(
            p=0.85,
            degrees=8,
            translate_frac=0.04,
            scale_range=(0.90, 1.10),
            brightness=0.15,
            contrast=0.15,
        )

        train_dataset = build_full_dataset(train_samples, augment=train_aug)
        val_dataset = build_full_dataset(val_samples, augment=None)

        train_loader = DataLoader(
            train_dataset,
            batch_size=TRAIN_BATCH_SIZE,
            shuffle=True,
            num_workers=TRAIN_NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            persistent_workers=(TRAIN_NUM_WORKERS > 0),
            collate_fn=diagram_collate_fn,
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=TRAIN_BATCH_SIZE,
            shuffle=False,
            num_workers=TRAIN_NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            persistent_workers=(TRAIN_NUM_WORKERS > 0),
            collate_fn=diagram_collate_fn,
        )
        print(f"Train cached samples: {len(train_dataset)}")
        print(f"Val cached samples  : {len(val_dataset)}")
        return train_loader, val_loader, train_samples, val_samples


    def binary_map_metrics_from_logits(logits, target, prob_thresh=METRIC_PROB_THRESH, target_thresh=METRIC_TARGET_THRESH, mask=None):
        prob = torch.sigmoid(logits)
        pred = prob >= prob_thresh
        gt = target >= target_thresh
        if mask is not None:
            m = mask > 0.5
            pred = pred & m
            gt = gt & m
        tp = (pred & gt).sum().float()
        fp = (pred & ~gt).sum().float()
        fn = (~pred & gt).sum().float()
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2.0 * precision * recall / (precision + recall + 1e-8)
        iou = tp / (tp + fp + fn + 1e-8)
        return {
            "precision": precision.detach(),
            "recall": recall.detach(),
            "f1": f1.detach(),
            "iou": iou.detach(),
        }


    def recall_on_target_region(logits, region, prob_thresh=METRIC_PROB_THRESH, region_thresh=METRIC_TARGET_THRESH):
        prob = torch.sigmoid(logits)
        pred = prob >= prob_thresh
        gt = region >= region_thresh
        denom = gt.sum().float()
        if denom.item() <= 0:
            return torch.zeros((), device=logits.device)
        return ((pred & gt).sum().float() / (denom + 1e-8)).detach()


    def map_mae_on_mask(pred, target, mask):
        valid = mask > 0.5
        if valid.sum().item() == 0:
            return torch.zeros((), device=pred.device)
        return torch.abs(pred - target)[valid.expand_as(pred)].mean().detach()


    def batch_breakage_metrics(outputs, target):
        """
        CPU metric. Higher breakage index = worse fragmentation.
        It compares predicted support on long GT path regions.
        """
        shaft_prob = torch.sigmoid(outputs["shaft"]).detach().cpu().numpy()
        skel_prob = torch.sigmoid(outputs["skeleton"]).detach().cpu().numpy()
        long_path = get_extra_target_channel(target, 26, torch.zeros_like(target[:, 2:3])).detach().cpu().numpy()
        very_long_path = get_extra_target_channel(target, 27, torch.zeros_like(target[:, 2:3])).detach().cpu().numpy()

        rows = []
        for b in range(shaft_prob.shape[0]):
            for name, prob, region in [
                ("long", shaft_prob[b, 0], long_path[b, 0]),
                ("very_long", shaft_prob[b, 0], very_long_path[b, 0]),
                ("long_skeleton", skel_prob[b, 0], long_path[b, 0]),
            ]:
                gt = (region >= METRIC_TARGET_THRESH).astype(np.uint8)
                if gt.sum() == 0:
                    rows.append((name, 0.0, 0.0, 0.0, 0.0))
                    continue
                pred = ((prob >= METRIC_PROB_THRESH).astype(np.uint8) & gt).astype(np.uint8)
                gt_cc = max(1, cv2.connectedComponents(gt, connectivity=8)[0] - 1)
                pred_cc = cv2.connectedComponents(pred, connectivity=8)[0] - 1 if pred.sum() > 0 else 0
                recall = float(pred.sum() / max(1, gt.sum()))
                gap_fraction = float(1.0 - recall)
                frag_ratio = float(pred_cc / max(1, gt_cc))
                breakage_index = float(gap_fraction + max(0.0, frag_ratio - 1.0))
                rows.append((name, recall, gap_fraction, frag_ratio, breakage_index))

        out = {}
        for name in ["long", "very_long", "long_skeleton"]:
            vals = [r for r in rows if r[0] == name]
            if vals:
                out[f"metric_{name}_path_recall"] = float(np.mean([v[1] for v in vals]))
                out[f"metric_{name}_gap_fraction"] = float(np.mean([v[2] for v in vals]))
                out[f"metric_{name}_fragmentation_ratio"] = float(np.mean([v[3] for v in vals]))
                out[f"metric_{name}_breakage_index"] = float(np.mean([v[4] for v in vals]))
        return out


    def learning_side_metrics(outputs, batch):
        target = batch["target"]
        metrics = {}

        mapping = {
            "node_extent": ("node_extent", target[:, 1:2]),
            "shaft": ("shaft", target[:, 2:3]),
            "skeleton": ("skeleton", target[:, 3:4]),
            "head": ("head", target[:, 4:5]),
            "tail": ("tail", target[:, 5:6]),
            "all_path": ("shaft", target[:, 24:25]),
        }

        for name, (out_name, tgt) in mapping.items():
            m = binary_map_metrics_from_logits(outputs[out_name], tgt)
            for k, v in m.items():
                metrics[f"metric_{name}_{k}"] = v

        long_path = get_extra_target_channel(target, 26, torch.zeros_like(target[:, 2:3]))
        very_long_path = get_extra_target_channel(target, 27, torch.zeros_like(target[:, 2:3]))

        metrics["metric_long_shaft_recall"] = recall_on_target_region(outputs["shaft"], long_path)
        metrics["metric_very_long_shaft_recall"] = recall_on_target_region(outputs["shaft"], very_long_path)
        metrics["metric_long_skeleton_recall"] = recall_on_target_region(outputs["skeleton"], long_path)
        metrics["metric_very_long_skeleton_recall"] = recall_on_target_region(outputs["skeleton"], very_long_path)

        # short region = shaft target excluding long path
        short_region = (target[:, 2:3] - long_path).clamp(0, 1)
        metrics["metric_short_shaft_recall"] = recall_on_target_region(outputs["shaft"], short_region)

        t_progress = target[:, 10:11]
        t_progress_mask = target[:, 11:12]
        metrics["metric_progress_mae"] = map_mae_on_mask(torch.sigmoid(outputs["progress"]), t_progress, t_progress_mask)

        metrics.update({k: torch.tensor(v, device=target.device) for k, v in batch_breakage_metrics(outputs, target).items()})
        return metrics


    def aggregate_family_metrics(metric_rows):
        groups = defaultdict(list)
        for row in metric_rows:
            groups[row.get("family", "unknown")].append(row)
        out = {}
        for fam, rows in sorted(groups.items()):
            fam_metrics = {}
            keys = sorted({k for r in rows for k in r if k.startswith("metric_") or k.startswith("loss_")})
            for k in keys:
                vals = [float(r[k]) for r in rows if k in r]
                if vals:
                    fam_metrics[k] = float(np.mean(vals))
            out[fam] = fam_metrics
        return out


    def move_batch_to_device_keep_meta(batch, device):
        return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}


    def train_one_epoch(model, loader, criterion, optimizer, device, scaler=None, use_amp=False, grad_clip_norm=None):
        model.train()
        running = defaultdict(float)
        running_count = 0

        for batch in loader:
            batch = move_batch_to_device_keep_meta(batch, device)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=(use_amp and torch.cuda.is_available())):
                outputs = model(batch["image"])
                loss, loss_dict = criterion(outputs, batch)

            if scaler is not None and scaler.is_enabled():
                scaler.scale(loss).backward()
                if grad_clip_norm is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                if grad_clip_norm is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
                optimizer.step()

            bs = batch["image"].size(0)
            running_count += bs
            for k, v in loss_dict.items():
                if torch.is_tensor(v):
                    running[k] += float(v.detach().cpu()) * bs
                else:
                    running[k] += float(v) * bs

        return {k: v / max(1, running_count) for k, v in running.items()}


    @torch.no_grad()
    def validate_one_epoch(model, loader, criterion, device, use_amp=False):
        model.eval()
        running = defaultdict(float)
        running_count = 0
        family_rows = []

        for batch in loader:
            batch = move_batch_to_device_keep_meta(batch, device)
            with torch.amp.autocast("cuda", enabled=(use_amp and torch.cuda.is_available())):
                outputs = model(batch["image"])
                loss, loss_dict = criterion(outputs, batch)
                metric_dict = learning_side_metrics(outputs, batch)

            bs = batch["image"].size(0)
            running_count += bs
            for k, v in {**loss_dict, **metric_dict}.items():
                if torch.is_tensor(v):
                    running[k] += float(v.detach().cpu()) * bs
                else:
                    running[k] += float(v) * bs

            # Per-family rows. We store batch-level metrics repeated by family weights for readability.
            metas = batch.get("meta", [{} for _ in range(bs)])
            fams = [m.get("family", "unknown") if isinstance(m, dict) else "unknown" for m in metas]
            for fam in sorted(set(fams)):
                idxs = [i for i, f in enumerate(fams) if f == fam]
                if not idxs:
                    continue
                sub = {
                    "image": batch["image"][idxs],
                    "target": batch["target"][idxs],
                    "dir_mask": batch["dir_mask"][idxs],
                    "meta": [metas[i] for i in idxs],
                }
                sub_outputs = {k: v[idxs] for k, v in outputs.items()}
                sub_metrics = learning_side_metrics(sub_outputs, sub)
                row = {"family": fam, "n": len(idxs)}
                for k, v in sub_metrics.items():
                    row[k] = float(v.detach().cpu()) if torch.is_tensor(v) else float(v)
                family_rows.append(row)

        out = {k: v / max(1, running_count) for k, v in running.items()}
        fam_summary = aggregate_family_metrics(family_rows)
        for fam, fm in fam_summary.items():
            for k, v in fm.items():
                out[f"family_{fam}_{k}"] = v
        return out


    def save_prediction_previews(loader, model, device, out_dir, max_items=3):
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        if max_items <= 0:
            return
        model.eval()
        saved = 0
        with torch.no_grad():
            for batch in loader:
                batch_gpu = move_batch_to_device_keep_meta(batch, device)
                outputs = model(batch_gpu["image"])
                for i in range(batch["image"].size(0)):
                    if saved >= max_items:
                        return
                    meta = batch["meta"][i] if isinstance(batch.get("meta"), list) else {}
                    sample_id = meta.get("sample_id", f"sample_{saved}")
                    target = batch["target"][i].numpy()
                    image = batch["image"][i, 0].numpy()
                    pred_shaft = torch.sigmoid(outputs["shaft"][i, 0]).detach().cpu().numpy()
                    pred_skel = torch.sigmoid(outputs["skeleton"][i, 0]).detach().cpu().numpy()

                    tiles = []
                    def to_bgr(a, title):
                        vis = to_uint8_vis(a)
                        vis = cv2.applyColorMap(vis, cv2.COLORMAP_TURBO)
                        cv2.putText(vis, title, (5, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)
                        cv2.putText(vis, title, (5, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1)
                        return vis
                    tiles.append(to_bgr(image, "image"))
                    tiles.append(to_bgr(target[2], "gt_shaft"))
                    tiles.append(to_bgr(pred_shaft, "pred_shaft"))
                    tiles.append(to_bgr(target[26] if target.shape[0] > 26 else np.zeros_like(target[2]), "gt_long"))
                    tiles.append(to_bgr(pred_skel, "pred_skeleton"))
                    tiles.append(to_bgr(target[25] if target.shape[0] > 25 else np.ones_like(target[2]), "length_weight"))
                    panel = np.concatenate([np.concatenate(tiles[:3], axis=1), np.concatenate(tiles[3:], axis=1)], axis=0)
                    cv2.imwrite(str(out_dir / f"{sample_id}_graph_evidence_preview.png"), panel)
                    saved += 1


    def run_long_arrow_learning(samples):
        RUN_DIR.mkdir(parents=True, exist_ok=True)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        in_channels = 1 if IMAGE_MODE == "grayscale" else 3

        seed_everything(LEARNING_SPLIT_SEED)
        train_loader, val_loader, train_samples, val_samples = build_learning_loaders(samples)
        save_split_manifest(RUN_DIR / "train_manifest.json", train_samples)
        save_split_manifest(RUN_DIR / "validation_manifest.json", val_samples)

        model = DiagramUNet(in_channels=in_channels, base_ch=BASE_CH).to(device)

        if torch.cuda.device_count() > 1:
            print(f"Using DataParallel on {torch.cuda.device_count()} GPUs.")
            model = nn.DataParallel(model)

        criterion = DiagramLoss(
            w_skeleton=W_SKELETON,
            w_cldice=W_CLDICE,
            w_loop_path=W_LOOP_PATH,
            w_progress=W_PROGRESS,
            w_head_offset=W_HEAD_OFFSET,
            w_tail_offset=W_TAIL_OFFSET,
            w_head_tangent=W_HEAD_TANGENT,
            w_tail_tangent=W_TAIL_TANGENT,
            w_general_path=W_GENERAL_PATH,
            cldice_iters=CLDICE_ITERS,
            pos_weight_skeleton=SKELETON_POS_WEIGHT,
        ).to(device)

        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
        scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and torch.cuda.is_available()))

        print(f"Run name : {RUN_NAME}")
        print(f"Run dir  : {RUN_DIR}")
        print(f"Device   : {device}")
        print(f"Epochs   : {NUM_EPOCHS}")
        print(f"Long loss weights: shaft={W_LONG_SHAFT}, skeleton={W_LONG_SKELETON}, path={W_LONG_PATH_RECALL}, very_long={W_VERY_LONG_PATH_RECALL}, cldice={W_LONG_CLDICE}")

        history = []
        best_val = float("inf")
        best_epoch = -1
        epochs_without_improvement = 0
        early_stop_patience = EARLY_STOPPING_PATIENCE
        min_delta = EARLY_STOPPING_MIN_DELTA
        t_start = time.time()

        if SAVE_PREDICTION_PREVIEWS:
            # Save before-training previews to show initial fragmentation.
            try:
                save_prediction_previews(
                    val_loader,
                    model,
                    device,
                    RUN_DIR / "prediction_previews_epoch000",
                    MAX_PREDICTION_PREVIEWS,
                )
            except Exception as e:
                print("Prediction preview generation was skipped:", e)

        for epoch in range(1, NUM_EPOCHS + 1):
            t0 = time.time()
            train_metrics = train_one_epoch(
                model=model,
                loader=train_loader,
                criterion=criterion,
                optimizer=optimizer,
                device=device,
                scaler=scaler,
                use_amp=USE_AMP,
                grad_clip_norm=GRAD_CLIP_NORM,
            )
            val_metrics = validate_one_epoch(
                model=model,
                loader=val_loader,
                criterion=criterion,
                device=device,
                use_amp=USE_AMP,
            )

            val_loss = float(val_metrics["loss_total"])
            scheduler.step(val_loss)
            epoch_time = time.time() - t0
            lr = optimizer.param_groups[0]["lr"]

            row = {
                "epoch": epoch,
                "lr": float(lr),
                "epoch_time_sec": float(epoch_time),
                **{f"train_{k}": float(v) for k, v in train_metrics.items()},
                **{f"val_{k}": float(v) for k, v in val_metrics.items()},
            }
            history.append(row)

            print(
                f"Epoch {epoch:03d}/{NUM_EPOCHS:03d} | "
                f"train={train_metrics['loss_total']:.4f} | val={val_loss:.4f} | "
                f"longRecall={val_metrics.get('metric_long_shaft_recall', 0):.3f} | "
                f"veryLongRecall={val_metrics.get('metric_very_long_shaft_recall', 0):.3f} | "
                f"longBreak={val_metrics.get('metric_long_breakage_index', 0):.3f} | "
                f"shaftF1={val_metrics.get('metric_shaft_f1', 0):.3f} | "
                f"skelF1={val_metrics.get('metric_skeleton_f1', 0):.3f} | "
                f"time={epoch_time:.1f}s"
            )

            # Family-wise concise print
            for fam in ["fa", "fca", "fcb"]:
                key = f"family_{fam}_metric_long_breakage_index"
                if key in val_metrics:
                    print(
                        f"    {fam}: "
                        f"longRec={val_metrics.get(f'family_{fam}_metric_long_shaft_recall', 0):.3f}, "
                        f"vLongRec={val_metrics.get(f'family_{fam}_metric_very_long_shaft_recall', 0):.3f}, "
                        f"break={val_metrics.get(f'family_{fam}_metric_long_breakage_index', 0):.3f}, "
                        f"shaftF1={val_metrics.get(f'family_{fam}_metric_shaft_f1', 0):.3f}, "
                        f"skelF1={val_metrics.get(f'family_{fam}_metric_skeleton_f1', 0):.3f}"
                    )

            state = get_checkpoint_state(
                epoch=epoch,
                model=model.module if isinstance(model, nn.DataParallel) else model,
                optimizer=optimizer,
                scheduler=scheduler,
                best_val_loss=best_val,
                train_loss=float(train_metrics["loss_total"]),
                val_loss=val_loss,
                run_name=RUN_NAME,
            )
            save_checkpoint(state, LAST_CKPT_PATH)

            if val_loss < best_val - min_delta:
                best_val = val_loss
                best_epoch = epoch
                epochs_without_improvement = 0
                state = get_checkpoint_state(
                    epoch=epoch,
                    model=model.module if isinstance(model, nn.DataParallel) else model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    best_val_loss=best_val,
                    train_loss=float(train_metrics["loss_total"]),
                    val_loss=val_loss,
                    run_name=RUN_NAME,
                )
                save_checkpoint(state, BEST_CKPT_PATH)
                print(f"  ✓ Saved best: {BEST_CKPT_PATH}")
                if SAVE_PREDICTION_PREVIEWS:
                    try:
                        save_prediction_previews(
                            val_loader,
                            model,
                            device,
                            RUN_DIR / f"prediction_previews_epoch{epoch:03d}",
                            MAX_PREDICTION_PREVIEWS,
                        )
                    except Exception as e:
                        print("Prediction preview generation was skipped:", e)
            else:
                epochs_without_improvement += 1
                print(f"  No improvement {epochs_without_improvement}/{early_stop_patience}; best epoch={best_epoch}, best val={best_val:.4f}")

            if epochs_without_improvement >= early_stop_patience:
                print(f"Early stopping at epoch {epoch}.")
                break

        history_path = RUN_DIR / "learning_history.json"
        save_json(history_path, history)

        # Save CSV too for plotting.
        try:
            import pandas as pd
            pd.DataFrame(history).to_csv(RUN_DIR / "learning_history.csv", index=False)
        except Exception as e:
            print("CSV history save skipped:", e)

        print("\nTraining finished.")
        print(f"Best val loss: {best_val:.6f}")
        print(f"Best epoch   : {best_epoch}")
        print(f"Best ckpt    : {BEST_CKPT_PATH}")
        print(f"History      : {history_path}")
        print(f"Total time   : {(time.time() - t_start) / 60.0:.2f} min")
        return history


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # DISCOVER SAMPLES AND PRECOMPUTE DENSE TARGETS
    # ============================================================

    ALL_SAMPLES = build_samples_from_converted_root(DATASET_ROOT)
    LEARNING_SAMPLES = select_learning_samples(ALL_SAMPLES)

    print("Learning pool by family:", Counter(s["family"] for s in LEARNING_SAMPLES))
    print("Learning pool by source split:", Counter(s["split"] for s in LEARNING_SAMPLES))

    GENERATOR_KWARGS = make_generator_kwargs()

    memory_estimate = run_precompute_memory_trial(
        samples=LEARNING_SAMPLES,
        generator_kwargs=GENERATOR_KWARGS,
        sample_n=PRECOMPUTE_MEMORY_SAMPLE_N,
        limit_gb=PRECOMPUTE_MEMORY_LIMIT_GB,
        reference_total_n=len(LEARNING_SAMPLES),
        save_path=RUN_DIR / "precompute_memory_estimate.json",
    )

    if not memory_estimate["within_limit"]:
        raise RuntimeError(
            f"Projected target cache is {memory_estimate['estimated_total_gb']:.2f} GB, "
            f"above the configured {PRECOMPUTE_MEMORY_LIMIT_GB:.2f} GB limit."
        )

    if PRECOMPUTE_PARALLEL:
        precompute_all_targets_parallel(
            samples=LEARNING_SAMPLES,
            cache_dir=CACHE_DIR,
            generator_kwargs=GENERATOR_KWARGS,
            overwrite=OVERWRITE_CACHE,
            num_workers=PRECOMPUTE_MAX_WORKERS,
        )
    else:
        precompute_all_targets_serial(
            samples=LEARNING_SAMPLES,
            cache_dir=CACHE_DIR,
            generator_kwargs=GENERATOR_KWARGS,
            overwrite=OVERWRITE_CACHE,
        )

    if SAVE_TARGET_PREVIEWS:
        save_target_previews(
            samples=LEARNING_SAMPLES,
            generator_kwargs=GENERATOR_KWARGS,
            output_dir=RUN_DIR / "target_previews",
            max_items=MAX_TARGET_PREVIEWS,
        )

    print(f"Target cache ready: {CACHE_DIR}")


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # ONE-BATCH SANITY CHECK
    # ============================================================

    sanity_train, sanity_val = stratified_learning_split(LEARNING_SAMPLES)
    sanity_dataset = build_full_dataset(sanity_train[: max(1, TRAIN_BATCH_SIZE)], augment=None)
    sanity_loader = DataLoader(
        sanity_dataset,
        batch_size=min(TRAIN_BATCH_SIZE, len(sanity_dataset)),
        shuffle=False,
        num_workers=0,
        collate_fn=diagram_collate_fn,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DiagramUNet(
        in_channels=1 if IMAGE_MODE == "grayscale" else 3,
        base_ch=BASE_CH,
    ).to(device)
    criterion = DiagramLoss(
        w_skeleton=W_SKELETON,
        w_cldice=W_CLDICE,
        w_loop_path=W_LOOP_PATH,
        w_progress=W_PROGRESS,
        w_head_offset=W_HEAD_OFFSET,
        w_tail_offset=W_TAIL_OFFSET,
        w_head_tangent=W_HEAD_TANGENT,
        w_tail_tangent=W_TAIL_TANGENT,
        w_general_path=W_GENERAL_PATH,
        cldice_iters=CLDICE_ITERS,
        pos_weight_skeleton=SKELETON_POS_WEIGHT,
    ).to(device)

    batch = next(iter(sanity_loader))
    batch = move_batch_to_device_keep_meta(batch, device)

    with torch.no_grad():
        outputs = model(batch["image"])
        total_loss, component_losses = criterion(outputs, batch)

    print("Input:", tuple(batch["image"].shape))
    for name, tensor in outputs.items():
        print(f"{name:16s}: {tuple(tensor.shape)}")
    print(f"Total sanity loss: {float(total_loss):.6f}")


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # TRAIN THE PAPER MODEL
    # ============================================================

    HISTORY = run_long_arrow_learning(ALL_SAMPLES)


In [ ]:
if SHOULD_TRAIN_GRAPH:
    # ============================================================
    # REVIEW THE SAVED RUN
    # ============================================================

    print(f"Best checkpoint : {BEST_CKPT_PATH}")
    print(f"Latest checkpoint: {LAST_CKPT_PATH}")
    print(f"Run configuration: {RUN_DIR / 'paper_training_config.json'}")
    print(f"Training history : {RUN_DIR / 'learning_history.json'}")

    history_path = RUN_DIR / "learning_history.csv"
    if pd is not None and history_path.exists():
        history = pd.read_csv(history_path)
        display(history.tail())

        if {"epoch", "train_loss_total", "val_loss_total"}.issubset(history.columns):
            plt.figure(figsize=(8, 4))
            plt.plot(history["epoch"], history["train_loss_total"], label="Train")
            plt.plot(history["epoch"], history["val_loss_total"], label="Validation")
            plt.xlabel("Epoch")
            plt.ylabel("Total loss")
            plt.title("Graph-evidence training history")
            plt.legend()
            plt.tight_layout()
            plt.show()


In [ ]:
if SHOULD_TRAIN_GRAPH:
    RESOLVED_GRAPH_CHECKPOINT = Path(BEST_CKPT_PATH).resolve()
    GRAPH_MODEL_CHECKPOINT = str(RESOLVED_GRAPH_CHECKPOINT)
    print('Graph checkpoint ready:', GRAPH_MODEL_CHECKPOINT)
elif RESOLVED_GRAPH_CHECKPOINT is not None:
    GRAPH_MODEL_CHECKPOINT = str(RESOLVED_GRAPH_CHECKPOINT)


## 5. Frozen graph model, Pass-I assembler, node model, and Pass-II finalizer

The following cells come from the canonical publication inference notebook. They define inference only; no evaluation is triggered here.

In [ ]:
import random
import shutil
import time
import traceback
from copy import deepcopy


In [ ]:
"""
Graph-evidence model and first-pass physical graph assembler.

This notebook cell contains the model architecture and the deterministic first-pass
decoder used by the paper. It consumes every output produced by the graph-evidence
network:

    node_center, node_extent, shaft, skeleton, head, tail, dir,
    progress, head_offset, tail_offset, head_tangent, tail_tangent

Main idea
---------
1. Detect nodes from node_extent + node_center.
2. Detect coarse head/tail endpoints from heatmaps.
3. Refine endpoints using the learned offset heads. The offsets are interpreted
   exactly as trained in the notebook: normalized displacement from local pixel
   to exact endpoint, multiplied by endpoint_radius.
4. Attach endpoints to nodes using nearest bbox plus the endpoint tangent.
5. Trace tail -> head paths using a deterministic A* over a fused support field.
   The path cost uses shaft, skeleton, direction, and progress monotonicity.
6. Select non-conflicting arrows.
7. Repair crowded ports by creating virtual opposite-role endpoints when shaft evidence remains.
8. Export graph JSON.

Coordinates
-----------
By default, writer JSON is in model space, e.g. 512x512. The script also saves an
optional original-space writer JSON by scaling coordinates back to the source image.
"""

from __future__ import annotations

import csv
import heapq
import json
import math
import os
import sys
import traceback
from types import SimpleNamespace
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import networkx as nx
except Exception:  # networkx is useful but not mandatory for JSON export
    nx = None


# Notebook paths are supplied only in the USER SETTINGS cell.


IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
NEIGHBORS_8 = [
    (-1, -1), (0, -1), (1, -1),
    (-1,  0),          (1,  0),
    (-1,  1), (0,  1), (1,  1),
]


# -----------------------------------------------------------------------------
# Model definition: matches the current notebook model, including GroupNorm,
# dropout, and all upgraded heads.
# -----------------------------------------------------------------------------

def make_group_norm(num_channels: int, max_groups: int = 8) -> nn.GroupNorm:
    for g in [max_groups, 4, 2, 1]:
        if num_channels % g == 0:
            return nn.GroupNorm(num_groups=g, num_channels=num_channels)
    return nn.GroupNorm(num_groups=1, num_channels=num_channels)


def make_norm_layer(num_channels: int, norm_type: str) -> nn.Module:
    norm_type = norm_type.lower()
    if norm_type == "group":
        return make_group_norm(num_channels)
    if norm_type == "batch":
        return nn.BatchNorm2d(num_channels)
    raise ValueError(f"Unsupported norm_type={norm_type!r}; use 'group' or 'batch'.")


class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, norm_type: str = "group"):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            make_norm_layer(out_ch, norm_type),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            make_norm_layer(out_ch, norm_type),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class Down(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, norm_type: str = "group"):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_ch, out_ch, norm_type=norm_type),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class Up(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, norm_type: str = "group"):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv((in_ch // 2) + skip_ch, out_ch, norm_type=norm_type)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        diff_y = skip.size(2) - x.size(2)
        diff_x = skip.size(3) - x.size(3)
        if diff_x != 0 or diff_y != 0:
            x = F.pad(
                x,
                [diff_x // 2, diff_x - diff_x // 2,
                 diff_y // 2, diff_y - diff_y // 2],
            )
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class DiagramUNet(nn.Module):
    def __init__(self, in_channels: int = 1, base_ch: int = 32, norm_type: str = "group"):
        super().__init__()
        self.inc = DoubleConv(in_channels, base_ch, norm_type=norm_type)
        self.down1 = Down(base_ch, base_ch * 2, norm_type=norm_type)
        self.down2 = Down(base_ch * 2, base_ch * 4, norm_type=norm_type)
        self.down3 = Down(base_ch * 4, base_ch * 8, norm_type=norm_type)
        self.down4 = Down(base_ch * 8, base_ch * 16, norm_type=norm_type)

        self.bottleneck_dropout = nn.Dropout2d(0.25)

        self.up1 = Up(base_ch * 16, base_ch * 8, base_ch * 8, norm_type=norm_type)
        self.up2 = Up(base_ch * 8, base_ch * 4, base_ch * 4, norm_type=norm_type)
        self.up3 = Up(base_ch * 4, base_ch * 2, base_ch * 2, norm_type=norm_type)
        self.up4 = Up(base_ch * 2, base_ch, base_ch, norm_type=norm_type)

        self.node_center_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.node_extent_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.shaft_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.skeleton_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.head_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.tail_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.dir_head = nn.Conv2d(base_ch, 2, kernel_size=1)

        self.progress_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.head_offset_head = nn.Conv2d(base_ch, 2, kernel_size=1)
        self.tail_offset_head = nn.Conv2d(base_ch, 2, kernel_size=1)
        self.head_tangent_head = nn.Conv2d(base_ch, 2, kernel_size=1)
        self.tail_tangent_head = nn.Conv2d(base_ch, 2, kernel_size=1)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x5 = self.bottleneck_dropout(x5)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        return {
            "node_center": self.node_center_head(x),
            "node_extent": self.node_extent_head(x),
            "shaft": self.shaft_head(x),
            "skeleton": self.skeleton_head(x),
            "head": self.head_head(x),
            "tail": self.tail_head(x),
            "dir": self.dir_head(x),
            "progress": self.progress_head(x),
            "head_offset": self.head_offset_head(x),
            "tail_offset": self.tail_offset_head(x),
            "head_tangent": self.head_tangent_head(x),
            "tail_tangent": self.tail_tangent_head(x),
        }


# -----------------------------------------------------------------------------
# Data containers
# -----------------------------------------------------------------------------

@dataclass
class Node:
    node_id: str
    bbox_xyxy: Tuple[int, int, int, int]
    bbox_xywh: Tuple[int, int, int, int]
    center_xy: Tuple[float, float]
    score: float
    area: int


@dataclass
class Endpoint:
    endpoint_id: str
    kind: str  # "head" or "tail"
    coarse_xy: Tuple[float, float]
    refined_xy: Tuple[float, float]
    snapped_xy: Tuple[int, int]
    score: float
    progress_value: float
    tangent_xy: Tuple[float, float]
    offset_xy: Tuple[float, float]
    attached_node_id: Optional[str] = None
    attach_score: Optional[float] = None
    component_id: Optional[int] = None


@dataclass
class Arrow:
    arrow_id: int
    tail_endpoint_id: str
    head_endpoint_id: str
    tail_xy: Tuple[float, float]
    head_xy: Tuple[float, float]
    tail_hint: Dict[str, float]
    head_hint: Dict[str, float]
    path_xy: List[Tuple[int, int]]
    tail_node_id: Optional[str]
    head_node_id: Optional[str]
    path_cost: float
    confidence: float
    support_score: float
    shaft_score: float
    skeleton_score: float
    direction_score: float
    progress_score: float
    endpoint_score: float
    tangent_score: float
    self_loop: bool
    meta: Dict[str, Any] = field(default_factory=dict)


@dataclass
class DecodeResult:
    nodes: List[Node]
    heads: List[Endpoint]
    tails: List[Endpoint]
    candidate_arrows: List[Arrow]
    arrows: List[Arrow]
    writer_style_json: Dict[str, Any]
    writer_style_original_json: Optional[Dict[str, Any]]
    decoder_debug_json: Dict[str, Any]
    debug_maps: Dict[str, np.ndarray]
    nx_graph: Optional[Any] = None


@dataclass
class LADDecoderConfig:
    # Node extraction
    node_extent_thresh: float = 0.20
    node_center_thresh: float = 0.25
    min_node_area: int = 35
    node_merge_iou: float = 0.45
    node_close_kernel: int = 3

    # Endpoint extraction
    head_thresh: float = 0.25
    tail_thresh: float = 0.25
    min_endpoint_area: int = 2
    endpoint_nms_dist: float = 5.0
    endpoint_radius: float = 8.0  # must match training offset normalization radius
    endpoint_patch_radius: int = 2

    # Support construction
    shaft_thresh: float = 0.20
    skeleton_thresh: float = 0.20
    progress_support_thresh: float = 0.02
    support_thresh: float = 0.20
    support_close_kernel: int = 3
    support_dilate_kernel: int = 1
    node_core_shrink_px: int = 5
    endpoint_force_radius: int = 5
    use_image_ink_support: bool = False
    ink_weight: float = 0.20
    shaft_weight: float = 0.45
    skeleton_weight: float = 0.30
    progress_weight: float = 0.15
    endpoint_weight: float = 0.10

    # Attachment
    node_attach_max_dist: float = 28.0
    tangent_probe_dist: float = 14.0

    # A* tracing
    max_pair_dist: float = 1e9
    max_expansions: int = 90000
    goal_radius: int = 4
    nearest_walkable_radius: int = 12
    min_path_len: int = 4
    support_cost_weight: float = 3.00
    direction_cost_weight: float = 1.00
    progress_cost_weight: float = 0.85
    curvature_cost_weight: float = 0.25
    node_core_cost_weight: float = 2.50
    step_cost_weight: float = 0.15
    heuristic_weight: float = 0.08
    progress_margin: float = 0.015

    # Candidate acceptance / selection
    min_arrow_confidence: float = 0.32

    # Endpoint reuse / branch handling
    # allow_endpoint_reuse=True is the broad switch: all endpoint reuse is allowed
    # after duplicate filtering. Useful for debugging, but it can overgenerate.
    allow_endpoint_reuse: bool = False

    # Safer default: allow reuse only when it looks like a real fan-out/fan-in branch.
    # This is needed because crowded node boundaries can collapse two real arrows into
    # one heatmap endpoint. The decoder should not throw away the second arrow merely
    # because the tail/head seed is shared.
    branch_aware_endpoint_reuse: bool = True
    branch_min_opposite_endpoint_dist: float = 16.0
    branch_max_path_iou: float = 0.62
    branch_require_different_node_pair: bool = True

    duplicate_endpoint_dist: float = 8.0
    duplicate_path_iou: float = 0.82
    keep_unattached_arrows: bool = True

    # Port-role repair
    # Some DFA nodes reuse the same tiny boundary port as the head of one edge and
    # the tail of another. In those cases the heatmap may keep only one role. The
    # normal decoder then has shaft evidence but no legal tail/head seed. This
    # repair pass creates low-priority virtual opposite-role endpoints at already
    # accepted node ports and tries to connect them to otherwise unused endpoints.
    enable_port_role_repair: bool = True
    port_repair_score_scale: float = 0.62
    port_repair_confidence_scale: float = 0.82
    port_repair_min_confidence: float = 0.46
    port_repair_progress_cost_weight: float = 0.20
    port_repair_max_path_iou: float = 0.52
    port_repair_min_support_score: float = 0.72
    port_repair_min_shaft_score: float = 0.72
    port_repair_max_candidates: int = 12

    # Adaptive decoding
    # These values are not meant to be hand-tuned per image. When auto_adapt=True,
    # the decoder derives thresholds/distances from the current prediction maps and
    # node geometry. A few guard rails remain because fully unconstrained decoding
    # is just hallucination wearing a lab coat.
    auto_adapt: bool = True
    adaptive_min_prob_floor: float = 0.05
    adaptive_endpoint_quantile: float = 99.75
    adaptive_node_quantile: float = 95.0
    adaptive_support_quantile: float = 88.0

    # Reviewer-sensitivity hook. Empty during ordinary paper reproduction.
    # Factors are applied only after adaptive thresholds/distances are computed,
    # so Stage-1 perturbations change the effective runtime behaviour rather than
    # dead/default numbers that may be overwritten by auto_adapt.
    sensitivity_effective_factors: Dict[str, float] = field(default_factory=dict)

    # Stroke-graph candidate generation
    # This is the core V5 change: generate extra edge candidates from continuous
    # stroke topology/support, not only from predicted tail/head endpoint pairs.
    enable_stroke_graph_candidates: bool = True
    stroke_graph_min_component_area: int = 18
    stroke_graph_min_path_len: int = 12
    stroke_graph_node_contact_factor: float = 0.42
    stroke_graph_node_contact_min: float = 10.0
    stroke_graph_node_contact_max: float = 42.0
    stroke_graph_min_confidence: float = 0.34
    stroke_graph_max_path_iou: float = 0.78
    stroke_graph_max_components: int = 96
    stroke_graph_max_candidates: int = 80
    # In V6 this is False by default. DFA self-loops are not exotic; they are normal edges.
    # We still validate them through path length/excursion instead of blindly accepting tiny node-border strokes.
    stroke_graph_skip_self_loops: bool = False
    stroke_graph_endpoint_score: float = 0.82
    stroke_graph_confidence_scale: float = 0.92

    # V6 ink-bridge candidate generation
    # Endpoint/stroke candidates can fail when the learned shaft support is broken, even
    # though raw ink visibly connects a tail/head pair. This fallback builds a narrow
    # endpoint-to-endpoint corridor and allows A* to use raw ink only inside that corridor.
    enable_ink_bridge_candidates: bool = True
    ink_bridge_min_confidence: float = 0.40
    ink_bridge_confidence_scale: float = 0.88
    ink_bridge_min_support_score: float = 0.42
    ink_bridge_min_width: int = 10
    ink_bridge_max_width: int = 32
    ink_bridge_width_factor: float = 0.10
    ink_bridge_threshold: float = 0.18
    ink_bridge_max_candidates: int = 80
    ink_bridge_progress_cost_weight: float = 0.28
    ink_bridge_support_cost_weight: float = 1.35
    ink_bridge_allow_self_loops: bool = True

    # Runtime
    verbose: bool = False


# -----------------------------------------------------------------------------
# Numeric / image utilities
# -----------------------------------------------------------------------------

def normalize01(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    mn = float(np.nanmin(x))
    mx = float(np.nanmax(x))
    if mx - mn < eps:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn)).astype(np.float32)


def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return (1.0 / (1.0 + np.exp(-np.clip(x, -60, 60)))).astype(np.float32)


def to_uint8_vis(x: np.ndarray) -> np.ndarray:
    return np.clip(normalize01(x) * 255.0, 0, 255).astype(np.uint8)


def ensure_rgb_uint8(img: np.ndarray) -> np.ndarray:
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    if img.dtype != np.uint8:
        if float(np.nanmax(img)) <= 1.0:
            img = (img * 255.0).clip(0, 255).astype(np.uint8)
        else:
            img = img.clip(0, 255).astype(np.uint8)
    return img


def safe_json(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): safe_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [safe_json(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if hasattr(obj, "__dataclass_fields__"):
        return safe_json(asdict(obj))
    return obj


def inside(x: int, y: int, w: int, h: int) -> bool:
    return 0 <= x < w and 0 <= y < h


def l2(p: Sequence[float], q: Sequence[float]) -> float:
    return float(math.hypot(float(p[0]) - float(q[0]), float(p[1]) - float(q[1])))


def unit(vx: float, vy: float, eps: float = 1e-8) -> Tuple[float, float, float]:
    mag = math.hypot(float(vx), float(vy))
    if mag < eps:
        return 0.0, 0.0, 0.0
    return float(vx) / mag, float(vy) / mag, mag


def cosine2(ax: float, ay: float, bx: float, by: float, eps: float = 1e-8) -> float:
    an = math.hypot(ax, ay)
    bn = math.hypot(bx, by)
    if an < eps or bn < eps:
        return 0.0
    return float((ax * bx + ay * by) / (an * bn))


def sample_map_bilinear(arr: np.ndarray, xy: Sequence[float]) -> float:
    h, w = arr.shape[:2]
    x = float(np.clip(xy[0], 0, w - 1))
    y = float(np.clip(xy[1], 0, h - 1))
    x0 = int(math.floor(x)); y0 = int(math.floor(y))
    x1 = min(w - 1, x0 + 1); y1 = min(h - 1, y0 + 1)
    wx = x - x0; wy = y - y0
    v00 = float(arr[y0, x0]); v10 = float(arr[y0, x1])
    v01 = float(arr[y1, x0]); v11 = float(arr[y1, x1])
    return (1 - wx) * (1 - wy) * v00 + wx * (1 - wy) * v10 + (1 - wx) * wy * v01 + wx * wy * v11


def sample_pair_bilinear(xmap: np.ndarray, ymap: np.ndarray, xy: Sequence[float]) -> Tuple[float, float]:
    return sample_map_bilinear(xmap, xy), sample_map_bilinear(ymap, xy)


def step_length(dx: int, dy: int) -> float:
    return math.sqrt(2.0) if dx != 0 and dy != 0 else 1.0


def bbox_iou(a: Sequence[int], b: Sequence[int]) -> float:
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1 + 1), max(0.0, iy2 - iy1 + 1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1 + 1) * max(0.0, ay2 - ay1 + 1)
    area_b = max(0.0, bx2 - bx1 + 1) * max(0.0, by2 - by1 + 1)
    return float(inter / max(area_a + area_b - inter, 1e-6))


def point_to_bbox_distance(pt: Sequence[float], box_xyxy: Sequence[int]) -> float:
    x, y = float(pt[0]), float(pt[1])
    x1, y1, x2, y2 = map(float, box_xyxy)
    dx = max(x1 - x, 0.0, x - x2)
    dy = max(y1 - y, 0.0, y - y2)
    return float(math.hypot(dx, dy))


def point_inside_bbox(pt: Sequence[float], box_xyxy: Sequence[int]) -> bool:
    x, y = float(pt[0]), float(pt[1])
    x1, y1, x2, y2 = map(float, box_xyxy)
    return x1 <= x <= x2 and y1 <= y <= y2


def draw_disk(mask: np.ndarray, xy: Sequence[float], radius: int, value: int = 1) -> None:
    h, w = mask.shape[:2]
    x = int(round(float(xy[0]))); y = int(round(float(xy[1])))
    if inside(x, y, w, h):
        cv2.circle(mask, (x, y), int(radius), int(value), -1)


def image_ink_support(decoder_img: np.ndarray) -> np.ndarray:
    """Auto-detect whether foreground ink is dark-on-light or light-on-dark."""
    rgb = ensure_rgb_uint8(decoder_img)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    # If mean is bright, assume black ink on white background. If dark, assume white ink on black.
    if float(gray.mean()) > 127.0:
        ink = 1.0 - gray.astype(np.float32) / 255.0
    else:
        ink = gray.astype(np.float32) / 255.0
    return normalize01(ink)


# -----------------------------------------------------------------------------
# Checkpoint loading and model output conversion
# -----------------------------------------------------------------------------

def extract_state_dict(ckpt: Any) -> Dict[str, torch.Tensor]:
    if isinstance(ckpt, dict):
        for key in ["model_state_dict", "state_dict", "model"]:
            if key in ckpt and isinstance(ckpt[key], dict):
                return ckpt[key]
        return ckpt
    raise ValueError("Unsupported checkpoint object; expected a dict or a dict containing model_state_dict.")


def strip_module_prefix(state: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k[7:] if k.startswith("module.") else k: v for k, v in state.items()}


def infer_arch_from_state(state: Dict[str, torch.Tensor]) -> Dict[str, Any]:
    clean = strip_module_prefix(state)
    base_ch = None
    in_channels = None
    norm_type = "group"

    for k, v in clean.items():
        if k.endswith("inc.block.0.weight") and hasattr(v, "shape"):
            base_ch = int(v.shape[0])
            in_channels = int(v.shape[1])
            break

    # BatchNorm checkpoints have running stats. GroupNorm checkpoints do not.
    if any("running_mean" in k or "running_var" in k for k in clean.keys()):
        norm_type = "batch"

    return {"base_ch": base_ch, "in_channels": in_channels, "norm_type": norm_type}


def load_checkpoint_model(
    checkpoint_path: Path,
    device: torch.device,
    image_mode: str,
    base_ch: int = 0,
    norm_type: str = "auto",
    strict: bool = True,
) -> Tuple[DiagramUNet, Dict[str, Any]]:
    ckpt = torch.load(str(checkpoint_path), map_location=device)
    state = strip_module_prefix(extract_state_dict(ckpt))
    inferred = infer_arch_from_state(state)

    ckpt_base_ch = inferred.get("base_ch")
    ckpt_in_channels = inferred.get("in_channels")
    ckpt_norm_type = inferred.get("norm_type") or "group"

    if base_ch <= 0:
        if ckpt_base_ch is None:
            raise ValueError("Could not infer base_ch from checkpoint. Pass --base-ch explicitly.")
        base_ch = ckpt_base_ch

    requested_in_channels = 1 if image_mode.lower() == "grayscale" else 3
    in_channels = ckpt_in_channels if ckpt_in_channels is not None else requested_in_channels
    if in_channels != requested_in_channels:
        raise ValueError(
            f"Checkpoint expects in_channels={in_channels}, but --image-mode={image_mode!r} gives "
            f"{requested_in_channels}. Use the same image mode used during training."
        )

    if norm_type == "auto":
        norm_type = ckpt_norm_type

    model = DiagramUNet(in_channels=in_channels, base_ch=base_ch, norm_type=norm_type).to(device)
    load_info: Dict[str, Any] = {
        "checkpoint_path": str(checkpoint_path),
        "base_ch": base_ch,
        "in_channels": in_channels,
        "norm_type": norm_type,
        "strict": strict,
        "inferred": inferred,
        "checkpoint_top_level_keys": list(ckpt.keys()) if isinstance(ckpt, dict) else [],
    }

    if strict:
        model.load_state_dict(state, strict=True)
        load_info["missing_keys"] = []
        load_info["unexpected_keys"] = []
    else:
        missing, unexpected = model.load_state_dict(state, strict=False)
        load_info["missing_keys"] = list(missing)
        load_info["unexpected_keys"] = list(unexpected)

    model.eval()
    return model, load_info


@torch.no_grad()
def outputs_to_pred_maps(outputs: Dict[str, torch.Tensor]) -> Dict[str, np.ndarray]:
    # Semantic / probability heads
    pred: Dict[str, np.ndarray] = {
        "node_center": torch.sigmoid(outputs["node_center"])[0, 0].detach().cpu().numpy().astype(np.float32),
        "node_extent": torch.sigmoid(outputs["node_extent"])[0, 0].detach().cpu().numpy().astype(np.float32),
        "shaft": torch.sigmoid(outputs["shaft"])[0, 0].detach().cpu().numpy().astype(np.float32),
        "skeleton": torch.sigmoid(outputs["skeleton"])[0, 0].detach().cpu().numpy().astype(np.float32),
        "head": torch.sigmoid(outputs["head"])[0, 0].detach().cpu().numpy().astype(np.float32),
        "tail": torch.sigmoid(outputs["tail"])[0, 0].detach().cpu().numpy().astype(np.float32),
        "progress": torch.sigmoid(outputs["progress"])[0, 0].detach().cpu().numpy().astype(np.float32),
    }

    # Raw regression/vector heads
    pred["dir_x"] = outputs["dir"][0, 0].detach().cpu().numpy().astype(np.float32)
    pred["dir_y"] = outputs["dir"][0, 1].detach().cpu().numpy().astype(np.float32)

    pred["head_offset_x"] = outputs["head_offset"][0, 0].detach().cpu().numpy().astype(np.float32)
    pred["head_offset_y"] = outputs["head_offset"][0, 1].detach().cpu().numpy().astype(np.float32)
    pred["tail_offset_x"] = outputs["tail_offset"][0, 0].detach().cpu().numpy().astype(np.float32)
    pred["tail_offset_y"] = outputs["tail_offset"][0, 1].detach().cpu().numpy().astype(np.float32)

    pred["head_tangent_x"] = outputs["head_tangent"][0, 0].detach().cpu().numpy().astype(np.float32)
    pred["head_tangent_y"] = outputs["head_tangent"][0, 1].detach().cpu().numpy().astype(np.float32)
    pred["tail_tangent_x"] = outputs["tail_tangent"][0, 0].detach().cpu().numpy().astype(np.float32)
    pred["tail_tangent_y"] = outputs["tail_tangent"][0, 1].detach().cpu().numpy().astype(np.float32)

    pred["dir_mag"] = np.sqrt(pred["dir_x"] ** 2 + pred["dir_y"] ** 2).astype(np.float32)
    pred["head_offset_mag"] = np.sqrt(pred["head_offset_x"] ** 2 + pred["head_offset_y"] ** 2).astype(np.float32)
    pred["tail_offset_mag"] = np.sqrt(pred["tail_offset_x"] ** 2 + pred["tail_offset_y"] ** 2).astype(np.float32)
    return pred


# -----------------------------------------------------------------------------
# Decoder internals
# -----------------------------------------------------------------------------

def connected_components_from_prob(prob: np.ndarray, thresh: float, close_kernel: int = 0) -> Tuple[int, np.ndarray, np.ndarray, np.ndarray]:
    mask = (prob >= thresh).astype(np.uint8)
    if close_kernel and close_kernel > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel, close_kernel))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    return cv2.connectedComponentsWithStats(mask, connectivity=8)


def weighted_centroid_from_mask(weight_map: np.ndarray, comp_mask: np.ndarray, fallback_xy: Tuple[float, float]) -> Tuple[float, float]:
    ys, xs = np.where(comp_mask > 0)
    if len(xs) == 0:
        return fallback_xy
    weights = weight_map[ys, xs].astype(np.float64)
    s = float(weights.sum())
    if s <= 1e-8:
        return float(xs.mean()), float(ys.mean())
    return float((xs * weights).sum() / s), float((ys * weights).sum() / s)


def nms_nodes(nodes: List[Node], iou_thresh: float) -> List[Node]:
    kept: List[Node] = []
    for n in sorted(nodes, key=lambda x: x.score, reverse=True):
        if any(bbox_iou(n.bbox_xyxy, k.bbox_xyxy) >= iou_thresh for k in kept):
            continue
        kept.append(n)
    kept.sort(key=lambda n: (n.center_xy[1], n.center_xy[0]))
    for i, n in enumerate(kept):
        n.node_id = f"N{i}"
    return kept


def extract_nodes(pred: Dict[str, np.ndarray], cfg: LADDecoderConfig) -> List[Node]:
    extent = pred["node_extent"]
    center = pred["node_center"]
    h, w = extent.shape

    num, labels, stats, cents = connected_components_from_prob(
        extent, cfg.node_extent_thresh, close_kernel=cfg.node_close_kernel
    )

    nodes: List[Node] = []
    for cid in range(1, num):
        x, y, bw, bh, area = stats[cid].tolist()
        if int(area) < cfg.min_node_area:
            continue
        x1, y1 = int(x), int(y)
        x2, y2 = int(x + bw - 1), int(y + bh - 1)
        comp_mask = (labels == cid).astype(np.uint8)
        cx, cy = weighted_centroid_from_mask(center, comp_mask, (float(cents[cid][0]), float(cents[cid][1])))
        score = float(max(center[labels == cid].max(initial=0), extent[labels == cid].mean()))
        nodes.append(
            Node(
                node_id=f"N{len(nodes)}",
                bbox_xyxy=(x1, y1, x2, y2),
                bbox_xywh=(x1, y1, int(bw), int(bh)),
                center_xy=(cx, cy),
                score=score,
                area=int(area),
            )
        )

    # Fallback: if extents are weak, use center blobs as tiny boxes.
    if not nodes:
        num_c, lab_c, stats_c, cents_c = connected_components_from_prob(center, cfg.node_center_thresh, close_kernel=3)
        for cid in range(1, num_c):
            x, y, bw, bh, area = stats_c[cid].tolist()
            if int(area) < max(2, cfg.min_node_area // 8):
                continue
            cx, cy = float(cents_c[cid][0]), float(cents_c[cid][1])
            pad = 12
            x1 = max(0, int(cx - pad)); y1 = max(0, int(cy - pad))
            x2 = min(w - 1, int(cx + pad)); y2 = min(h - 1, int(cy + pad))
            nodes.append(
                Node(
                    node_id=f"N{len(nodes)}",
                    bbox_xyxy=(x1, y1, x2, y2),
                    bbox_xywh=(x1, y1, x2 - x1 + 1, y2 - y1 + 1),
                    center_xy=(cx, cy),
                    score=float(center[lab_c == cid].max(initial=0)),
                    area=int(area),
                )
            )

    return nms_nodes(nodes, cfg.node_merge_iou)


def endpoint_patch_average(
    map_x: np.ndarray,
    map_y: np.ndarray,
    heat: np.ndarray,
    xy: Sequence[float],
    patch_radius: int,
) -> Tuple[float, float]:
    h, w = heat.shape
    x0 = int(round(float(xy[0]))); y0 = int(round(float(xy[1])))
    xs0 = max(0, x0 - patch_radius); xs1 = min(w - 1, x0 + patch_radius)
    ys0 = max(0, y0 - patch_radius); ys1 = min(h - 1, y0 + patch_radius)
    patch_w = heat[ys0:ys1 + 1, xs0:xs1 + 1].astype(np.float64)
    s = float(patch_w.sum())
    if s <= 1e-8:
        return sample_pair_bilinear(map_x, map_y, xy)
    mx = float((map_x[ys0:ys1 + 1, xs0:xs1 + 1] * patch_w).sum() / s)
    my = float((map_y[ys0:ys1 + 1, xs0:xs1 + 1] * patch_w).sum() / s)
    return mx, my


def nms_endpoints(endpoints: List[Endpoint], min_dist: float) -> List[Endpoint]:
    kept: List[Endpoint] = []
    for ep in sorted(endpoints, key=lambda e: e.score, reverse=True):
        if any(l2(ep.refined_xy, k.refined_xy) <= min_dist for k in kept):
            continue
        kept.append(ep)
    kept.sort(key=lambda e: (e.refined_xy[1], e.refined_xy[0]))
    for i, ep in enumerate(kept):
        ep.endpoint_id = f"{ep.kind[0].upper()}{i}"
    return kept


def extract_endpoints(pred: Dict[str, np.ndarray], kind: str, cfg: LADDecoderConfig) -> List[Endpoint]:
    assert kind in {"head", "tail"}
    heat = pred[kind]
    thresh = cfg.head_thresh if kind == "head" else cfg.tail_thresh
    num, labels, stats, cents = connected_components_from_prob(heat, thresh, close_kernel=0)
    h, w = heat.shape

    if kind == "head":
        off_x, off_y = pred["head_offset_x"], pred["head_offset_y"]
        tan_x, tan_y = pred["head_tangent_x"], pred["head_tangent_y"]
    else:
        off_x, off_y = pred["tail_offset_x"], pred["tail_offset_y"]
        tan_x, tan_y = pred["tail_tangent_x"], pred["tail_tangent_y"]

    endpoints: List[Endpoint] = []
    for cid in range(1, num):
        x, y, bw, bh, area = stats[cid].tolist()
        if int(area) < cfg.min_endpoint_area:
            continue
        comp = labels == cid
        ys, xs = np.where(comp)
        if len(xs) == 0:
            continue

        values = heat[ys, xs]
        imax = int(np.argmax(values))
        max_x = float(xs[imax]); max_y = float(ys[imax])
        wcx, wcy = weighted_centroid_from_mask(heat, comp.astype(np.uint8), (max_x, max_y))
        # Use weighted centroid as coarse point, but keep it pulled toward the max.
        coarse = (0.65 * wcx + 0.35 * max_x, 0.65 * wcy + 0.35 * max_y)

        ox_norm, oy_norm = endpoint_patch_average(off_x, off_y, heat, coarse, cfg.endpoint_patch_radius)
        ox = float(ox_norm) * cfg.endpoint_radius
        oy = float(oy_norm) * cfg.endpoint_radius
        refined_x = float(np.clip(coarse[0] + ox, 0, w - 1))
        refined_y = float(np.clip(coarse[1] + oy, 0, h - 1))

        tx, ty = endpoint_patch_average(tan_x, tan_y, heat, coarse, cfg.endpoint_patch_radius)
        tx, ty, _ = unit(tx, ty)

        progress_val = sample_map_bilinear(pred["progress"], (refined_x, refined_y))
        score = float(values.max())
        endpoints.append(
            Endpoint(
                endpoint_id=f"{kind[0].upper()}{len(endpoints)}",
                kind=kind,
                coarse_xy=(float(coarse[0]), float(coarse[1])),
                refined_xy=(refined_x, refined_y),
                snapped_xy=(int(round(refined_x)), int(round(refined_y))),
                score=score,
                progress_value=float(progress_val),
                tangent_xy=(float(tx), float(ty)),
                offset_xy=(float(ox), float(oy)),
                component_id=int(cid),
            )
        )

    return nms_endpoints(endpoints, cfg.endpoint_nms_dist)




def adaptive_prob_threshold(
    arr: np.ndarray,
    floor: float,
    quantile: float,
    cap: float = 0.92,
) -> float:
    """Map-wise robust threshold.

    This intentionally returns conservative values. For sparse heatmaps, high
    quantiles can collapse to background, so the training-time floor remains a
    lower bound. For very confident maps, the cap prevents the decoder from using
    only the single brightest pixel and fragmenting a real stroke.
    """
    x = np.asarray(arr, dtype=np.float32)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return float(floor)
    qv = float(np.percentile(x, quantile))
    mu = float(np.mean(x))
    sd = float(np.std(x))
    stat = mu + 2.5 * sd
    # Use the stronger of percentile/statistical threshold, but do not exceed cap.
    t = max(float(floor), min(max(qv, stat), float(cap)))
    return float(np.clip(t, floor, cap))


def median_node_diag(nodes: List[Node]) -> float:
    if not nodes:
        return 48.0
    vals = []
    for n in nodes:
        x1, y1, x2, y2 = n.bbox_xyxy
        vals.append(math.hypot(max(1, x2 - x1 + 1), max(1, y2 - y1 + 1)))
    return float(np.median(vals)) if vals else 48.0


def adapt_config_pre_nodes(pred: Dict[str, np.ndarray], cfg: LADDecoderConfig) -> LADDecoderConfig:
    """Adapt map thresholds before node/endpoint extraction."""
    if not cfg.auto_adapt:
        return cfg
    out = replace(cfg)
    out.node_extent_thresh = adaptive_prob_threshold(
        pred["node_extent"], cfg.node_extent_thresh, cfg.adaptive_node_quantile, cap=0.88
    )
    out.node_center_thresh = adaptive_prob_threshold(
        pred["node_center"], cfg.node_center_thresh, 99.2, cap=0.92
    )
    out.head_thresh = adaptive_prob_threshold(
        pred["head"], cfg.head_thresh, cfg.adaptive_endpoint_quantile, cap=0.90
    )
    out.tail_thresh = adaptive_prob_threshold(
        pred["tail"], cfg.tail_thresh, cfg.adaptive_endpoint_quantile, cap=0.90
    )
    out.shaft_thresh = adaptive_prob_threshold(
        pred["shaft"], cfg.shaft_thresh, cfg.adaptive_support_quantile, cap=0.75
    )
    out.skeleton_thresh = adaptive_prob_threshold(
        pred["skeleton"], cfg.skeleton_thresh, cfg.adaptive_support_quantile, cap=0.75
    )

    # Sensitivity factors are applied *after* adaptation so a ±20% experiment
    # actually changes the threshold seen by extraction/tracing.
    factors = getattr(cfg, "sensitivity_effective_factors", {}) or {}
    pre_caps = {
        "node_extent_thresh": 0.98,
        "node_center_thresh": 0.98,
        "head_thresh": 0.98,
        "tail_thresh": 0.98,
        "shaft_thresh": 0.98,
        "skeleton_thresh": 0.98,
    }
    for name, cap in pre_caps.items():
        factor = float(factors.get(name, 1.0))
        if abs(factor - 1.0) > 1e-12:
            value = float(getattr(out, name)) * factor
            setattr(out, name, float(np.clip(value, 0.01, cap)))
    return out


def adapt_config_post_nodes(pred: Dict[str, np.ndarray], cfg: LADDecoderConfig, nodes: List[Node]) -> LADDecoderConfig:
    """Adapt geometry-sensitive distances after node extraction."""
    if not cfg.auto_adapt:
        return cfg
    out = replace(cfg)
    med = median_node_diag(nodes)
    # These scale with actual diagram geometry instead of one 512x512-wide constant.
    out.endpoint_nms_dist = float(np.clip(0.075 * med, 4.0, 10.0))
    out.node_attach_max_dist = float(np.clip(0.42 * med, 16.0, 45.0))
    out.branch_min_opposite_endpoint_dist = float(np.clip(0.22 * med, 10.0, 28.0))
    out.duplicate_endpoint_dist = float(np.clip(0.12 * med, 6.0, 14.0))
    out.node_core_shrink_px = int(np.clip(round(0.08 * med), 4, 10))
    out.endpoint_force_radius = int(np.clip(round(0.08 * med), 4, 8))
    out.nearest_walkable_radius = int(np.clip(round(0.22 * med), 8, 18))
    # Support threshold adapts to the fused map but keeps a low floor, because
    # handwritten arrow strokes may be thin and broken.
    fused = (
        out.shaft_weight * pred["shaft"].astype(np.float32)
        + out.skeleton_weight * pred["skeleton"].astype(np.float32)
        + out.progress_weight * (pred["progress"] >= out.progress_support_thresh).astype(np.float32)
        + out.endpoint_weight * np.maximum(pred["head"], pred["tail"]).astype(np.float32)
    ) / max(out.shaft_weight + out.skeleton_weight + out.progress_weight + out.endpoint_weight, 1e-6)
    dyn_support = adaptive_prob_threshold(fused, cfg.support_thresh, cfg.adaptive_support_quantile, cap=0.55)
    out.support_thresh = float(np.clip(min(dyn_support, 0.30), 0.12, 0.30))

    # Apply runtime-behaviour perturbations after image-adaptive geometry is known.
    factors = getattr(cfg, "sensitivity_effective_factors", {}) or {}
    float_bounds = {
        "support_thresh": (0.01, 0.98),
        "endpoint_nms_dist": (1.0, 24.0),
        "node_attach_max_dist": (4.0, 90.0),
        "branch_min_opposite_endpoint_dist": (2.0, 60.0),
        "duplicate_endpoint_dist": (1.0, 30.0),
        "nearest_walkable_radius": (2.0, 36.0),
    }
    for name, (lo, hi) in float_bounds.items():
        factor = float(factors.get(name, 1.0))
        if abs(factor - 1.0) <= 1e-12:
            continue
        value = float(getattr(out, name)) * factor
        if isinstance(getattr(out, name), int):
            setattr(out, name, int(np.clip(round(value), lo, hi)))
        else:
            setattr(out, name, float(np.clip(value, lo, hi)))
    return out

def build_node_core_mask(nodes: List[Node], shape_hw: Tuple[int, int], shrink_px: int) -> np.ndarray:
    h, w = shape_hw
    mask = np.zeros((h, w), dtype=np.uint8)
    for n in nodes:
        x1, y1, x2, y2 = n.bbox_xyxy
        sx1 = int(np.clip(x1 + shrink_px, 0, w - 1))
        sy1 = int(np.clip(y1 + shrink_px, 0, h - 1))
        sx2 = int(np.clip(x2 - shrink_px, 0, w - 1))
        sy2 = int(np.clip(y2 - shrink_px, 0, h - 1))
        if sx2 < sx1 or sy2 < sy1:
            sx1, sy1, sx2, sy2 = x1, y1, x2, y2
        cv2.rectangle(mask, (sx1, sy1), (sx2, sy2), 1, -1)
    return mask


def build_support_field(
    pred: Dict[str, np.ndarray],
    decoder_img: np.ndarray,
    nodes: List[Node],
    heads: List[Endpoint],
    tails: List[Endpoint],
    cfg: LADDecoderConfig,
) -> Dict[str, np.ndarray]:
    shaft = pred["shaft"].astype(np.float32)
    skeleton = pred["skeleton"].astype(np.float32)
    progress_mask_like = (pred["progress"] >= cfg.progress_support_thresh).astype(np.float32)
    endpoints_union = np.maximum(pred["head"], pred["tail"]).astype(np.float32)

    support = (
        cfg.shaft_weight * shaft +
        cfg.skeleton_weight * skeleton +
        cfg.progress_weight * progress_mask_like +
        cfg.endpoint_weight * endpoints_union
    )
    total_weight = cfg.shaft_weight + cfg.skeleton_weight + cfg.progress_weight + cfg.endpoint_weight

    # Always compute raw ink. V5 only computed this when use_image_ink_support=True,
    # but V6 uses it as a local fallback for broken learned-support paths. It is not
    # mixed into the main support unless requested, so normal cases remain model-driven.
    ink = image_ink_support(decoder_img).astype(np.float32)
    if cfg.use_image_ink_support:
        support += cfg.ink_weight * ink
        total_weight += cfg.ink_weight

    support = np.clip(support / max(total_weight, 1e-6), 0.0, 1.0).astype(np.float32)

    walkable = (support >= cfg.support_thresh).astype(np.uint8)
    walkable = np.maximum(walkable, (shaft >= cfg.shaft_thresh).astype(np.uint8))
    walkable = np.maximum(walkable, (skeleton >= cfg.skeleton_thresh).astype(np.uint8))

    if cfg.support_close_kernel and cfg.support_close_kernel > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (cfg.support_close_kernel, cfg.support_close_kernel))
        walkable = cv2.morphologyEx(walkable, cv2.MORPH_CLOSE, k)
    if cfg.support_dilate_kernel and cfg.support_dilate_kernel > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (cfg.support_dilate_kernel, cfg.support_dilate_kernel))
        walkable = cv2.dilate(walkable, k, iterations=1)

    node_core = build_node_core_mask(nodes, shaft.shape, cfg.node_core_shrink_px)
    # Do not completely remove node interiors from walkable. Instead, A* gets a penalty.
    # For endpoints, force small disks walkable so snapped starts/goals do not disappear.
    endpoint_force = np.zeros_like(walkable, dtype=np.uint8)
    for ep in list(heads) + list(tails):
        draw_disk(endpoint_force, ep.refined_xy, cfg.endpoint_force_radius, value=1)
    walkable = np.maximum(walkable, endpoint_force)

    return {
        "support": support,
        "walkable": walkable.astype(np.uint8),
        "node_core": node_core.astype(np.uint8),
        "ink": ink.astype(np.float32),
        "endpoint_force": endpoint_force.astype(np.uint8),
    }


def attach_endpoint_to_node(ep: Endpoint, nodes: List[Node], cfg: LADDecoderConfig) -> Tuple[Optional[str], Optional[float]]:
    if not nodes:
        return None, None

    px, py = ep.refined_xy
    tx, ty = ep.tangent_xy
    # Tail tangent points away from source node, so source probe is behind.
    # Head tangent points into destination endpoint, so destination probe is ahead.
    if ep.kind == "tail":
        probe = (px - tx * cfg.tangent_probe_dist, py - ty * cfg.tangent_probe_dist)
    else:
        probe = (px + tx * cfg.tangent_probe_dist, py + ty * cfg.tangent_probe_dist)

    best_id = None
    best_score = float("inf")
    for n in nodes:
        d_point = point_to_bbox_distance(ep.refined_xy, n.bbox_xyxy)
        d_probe = point_to_bbox_distance(probe, n.bbox_xyxy)
        inside_bonus = -4.0 if point_inside_bbox(ep.refined_xy, n.bbox_xyxy) else 0.0
        score = 0.65 * d_point + 0.35 * d_probe + inside_bonus
        if score < best_score:
            best_score = float(score)
            best_id = n.node_id

    if best_score <= cfg.node_attach_max_dist:
        return best_id, best_score
    return None, best_score


def attach_all_endpoints(endpoints: List[Endpoint], nodes: List[Node], cfg: LADDecoderConfig) -> None:
    for ep in endpoints:
        nid, score = attach_endpoint_to_node(ep, nodes, cfg)
        ep.attached_node_id = nid
        ep.attach_score = score


def nearest_walkable(seed_xy: Sequence[float], walkable: np.ndarray, max_radius: int) -> Optional[Tuple[int, int]]:
    h, w = walkable.shape
    sx = int(round(float(seed_xy[0]))); sy = int(round(float(seed_xy[1])))
    if inside(sx, sy, w, h) and walkable[sy, sx] > 0:
        return sx, sy
    best = None
    best_d2 = float("inf")
    for r in range(1, max_radius + 1):
        x0 = max(0, sx - r); x1 = min(w - 1, sx + r)
        y0 = max(0, sy - r); y1 = min(h - 1, sy + r)
        ys, xs = np.where(walkable[y0:y1 + 1, x0:x1 + 1] > 0)
        if len(xs) == 0:
            continue
        xs = xs + x0; ys = ys + y0
        d2 = (xs - sx) ** 2 + (ys - sy) ** 2
        idx = int(np.argmin(d2))
        if float(d2[idx]) < best_d2:
            best_d2 = float(d2[idx])
            best = (int(xs[idx]), int(ys[idx]))
        if best is not None:
            return best
    return None


def reconstruct_path(came_from: Dict[Tuple[int, int], Tuple[int, int]], end_state: Tuple[int, int]) -> List[Tuple[int, int]]:
    path = [end_state]
    cur = end_state
    while cur in came_from:
        cur = came_from[cur]
        path.append(cur)
    path.reverse()
    return path


def astar_trace(
    tail: Endpoint,
    head: Endpoint,
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
    cfg: LADDecoderConfig,
) -> Optional[Dict[str, Any]]:
    support = support_maps["support"]
    walkable = support_maps["walkable"]
    node_core = support_maps["node_core"]
    h, w = support.shape

    start = nearest_walkable(tail.refined_xy, walkable, cfg.nearest_walkable_radius)
    goal = nearest_walkable(head.refined_xy, walkable, cfg.nearest_walkable_radius)
    if start is None or goal is None:
        return None

    dir_x = pred["dir_x"]
    dir_y = pred["dir_y"]
    progress = pred["progress"]

    pq: List[Tuple[float, float, Tuple[int, int], Optional[Tuple[int, int]]]] = []
    heapq.heappush(pq, (0.0, 0.0, start, None))
    came_from: Dict[Tuple[int, int], Tuple[int, int]] = {}
    best_g: Dict[Tuple[int, int], float] = {start: 0.0}
    expansions = 0
    gx, gy = goal

    while pq:
        _, g, cur, prev = heapq.heappop(pq)
        cx, cy = cur
        if (cx - gx) ** 2 + (cy - gy) ** 2 <= cfg.goal_radius ** 2:
            path = reconstruct_path(came_from, cur)
            if cur != goal:
                path.append(goal)
            return {
                "path": path,
                "cost": float(g),
                "start_snap": start,
                "goal_snap": goal,
                "expansions": expansions,
            }

        expansions += 1
        if expansions > cfg.max_expansions:
            return None

        cur_prog = float(progress[cy, cx])
        for ddx, ddy in NEIGHBORS_8:
            nx_, ny_ = cx + ddx, cy + ddy
            if not inside(nx_, ny_, w, h):
                continue
            if walkable[ny_, nx_] == 0:
                continue

            slen = step_length(ddx, ddy)
            svx, svy = ddx / slen, ddy / slen

            support_val = float(support[ny_, nx_])
            support_cost = cfg.support_cost_weight * (1.0 - support_val)

            local_dir_score = max(0.0, cosine2(svx, svy, float(dir_x[ny_, nx_]), float(dir_y[ny_, nx_])))
            direction_cost = cfg.direction_cost_weight * (1.0 - local_dir_score)

            next_prog = float(progress[ny_, nx_])
            backward = max(0.0, cur_prog - next_prog + cfg.progress_margin)
            progress_cost = cfg.progress_cost_weight * backward

            curvature_cost = 0.0
            if prev is not None:
                px, py = prev
                pvx, pvy = cx - px, cy - py
                pn = math.hypot(pvx, pvy)
                if pn > 1e-8:
                    pvx /= pn; pvy /= pn
                    turn_score = max(0.0, cosine2(pvx, pvy, svx, svy))
                    curvature_cost = cfg.curvature_cost_weight * (1.0 - turn_score)

            node_cost = cfg.node_core_cost_weight * float(node_core[ny_, nx_] > 0)
            edge_cost = (
                cfg.step_cost_weight * slen + support_cost + direction_cost +
                progress_cost + curvature_cost + node_cost
            )
            ng = g + edge_cost
            nxt = (nx_, ny_)
            if ng < best_g.get(nxt, float("inf")):
                best_g[nxt] = ng
                came_from[nxt] = cur
                heuristic = cfg.heuristic_weight * math.hypot(gx - nx_, gy - ny_)
                heapq.heappush(pq, (ng + heuristic, ng, nxt, cur))

    return None


def path_iou(path_a: Sequence[Tuple[int, int]], path_b: Sequence[Tuple[int, int]]) -> float:
    sa = set((int(x), int(y)) for x, y in path_a)
    sb = set((int(x), int(y)) for x, y in path_b)
    if not sa and not sb:
        return 1.0
    return float(len(sa & sb) / max(1, len(sa | sb)))


def score_path(
    path: List[Tuple[int, int]],
    tail: Endpoint,
    head: Endpoint,
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
) -> Dict[str, float]:
    if len(path) < 2:
        return {
            "support_score": 0.0, "shaft_score": 0.0, "skeleton_score": 0.0,
            "direction_score": 0.0, "progress_score": 0.0, "tangent_score": 0.0,
        }
    xs = np.array([p[0] for p in path], dtype=np.int32)
    ys = np.array([p[1] for p in path], dtype=np.int32)
    support_vals = support_maps["support"][ys, xs]
    shaft_vals = pred["shaft"][ys, xs]
    skel_vals = pred["skeleton"][ys, xs]
    prog_vals = pred["progress"][ys, xs]

    dir_scores = []
    good_progress = []
    for i in range(len(path) - 1):
        x1, y1 = path[i]
        x2, y2 = path[i + 1]
        dx = x2 - x1; dy = y2 - y1
        n = math.hypot(dx, dy)
        if n < 1e-8:
            continue
        sx, sy = dx / n, dy / n
        dir_scores.append(max(0.0, cosine2(sx, sy, float(pred["dir_x"][y2, x2]), float(pred["dir_y"][y2, x2]))))
        good_progress.append(1.0 if float(pred["progress"][y2, x2]) + 0.025 >= float(pred["progress"][y1, x1]) else 0.0)

    # Endpoint tangent score: first step should align with tail tangent; last step with head tangent.
    x0, y0 = path[0]
    x1, y1 = path[min(len(path) - 1, 2)]
    dx0, dy0 = x1 - x0, y1 - y0
    xn0, yn0 = path[max(0, len(path) - 3)]
    xn1, yn1 = path[-1]
    dxn, dyn = xn1 - xn0, yn1 - yn0
    tail_tan_score = max(0.0, cosine2(dx0, dy0, *tail.tangent_xy))
    head_tan_score = max(0.0, cosine2(dxn, dyn, *head.tangent_xy))
    tangent_score = 0.5 * tail_tan_score + 0.5 * head_tan_score

    progress_monotonic = float(np.mean(good_progress)) if good_progress else 0.0
    progress_span = float(np.clip(prog_vals[-1] - prog_vals[0], 0.0, 1.0)) if len(prog_vals) else 0.0
    progress_score = 0.75 * progress_monotonic + 0.25 * progress_span

    return {
        "support_score": float(np.mean(support_vals)) if len(support_vals) else 0.0,
        "shaft_score": float(np.mean(shaft_vals)) if len(shaft_vals) else 0.0,
        "skeleton_score": float(np.mean(skel_vals)) if len(skel_vals) else 0.0,
        "direction_score": float(np.mean(dir_scores)) if dir_scores else 0.0,
        "progress_score": float(progress_score),
        "tangent_score": float(tangent_score),
    }


def make_arrow_candidate(
    tail: Endpoint,
    head: Endpoint,
    trace: Dict[str, Any],
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
) -> Arrow:
    path = trace["path"]
    scores = score_path(path, tail, head, pred, support_maps)
    endpoint_score = 0.5 * float(tail.score) + 0.5 * float(head.score)
    attach_bonus = 1.0 if tail.attached_node_id is not None and head.attached_node_id is not None else 0.55
    confidence = (
        0.30 * scores["support_score"] +
        0.15 * scores["direction_score"] +
        0.15 * scores["progress_score"] +
        0.10 * scores["tangent_score"] +
        0.15 * endpoint_score +
        0.10 * scores["shaft_score"] +
        0.05 * scores["skeleton_score"]
    ) * attach_bonus
    return Arrow(
        arrow_id=-1,
        tail_endpoint_id=tail.endpoint_id,
        head_endpoint_id=head.endpoint_id,
        tail_xy=tail.refined_xy,
        head_xy=head.refined_xy,
        tail_hint={"x": float(tail.refined_xy[0]), "y": float(tail.refined_xy[1])},
        head_hint={"x": float(head.refined_xy[0]), "y": float(head.refined_xy[1])},
        path_xy=[(int(x), int(y)) for x, y in path],
        tail_node_id=tail.attached_node_id,
        head_node_id=head.attached_node_id,
        path_cost=float(trace["cost"]),
        confidence=float(confidence),
        support_score=scores["support_score"],
        shaft_score=scores["shaft_score"],
        skeleton_score=scores["skeleton_score"],
        direction_score=scores["direction_score"],
        progress_score=scores["progress_score"],
        endpoint_score=float(endpoint_score),
        tangent_score=scores["tangent_score"],
        self_loop=bool(tail.attached_node_id is not None and tail.attached_node_id == head.attached_node_id),
        meta={
            "tail_progress": float(tail.progress_value),
            "head_progress": float(head.progress_value),
            "start_snap": trace.get("start_snap"),
            "goal_snap": trace.get("goal_snap"),
            "expansions": trace.get("expansions"),
            "tail_attach_score": tail.attach_score,
            "head_attach_score": head.attach_score,
        },
    )


def generate_arrow_candidates(
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
    tails: List[Endpoint],
    heads: List[Endpoint],
    cfg: LADDecoderConfig,
) -> List[Arrow]:
    candidates: List[Arrow] = []
    for tail in tails:
        for head in heads:
            if l2(tail.refined_xy, head.refined_xy) > cfg.max_pair_dist:
                continue
            trace = astar_trace(tail, head, pred, support_maps, cfg)
            if trace is None:
                continue
            if len(trace["path"]) < cfg.min_path_len:
                continue
            arrow = make_arrow_candidate(tail, head, trace, pred, support_maps)
            if (not cfg.keep_unattached_arrows) and (arrow.tail_node_id is None or arrow.head_node_id is None):
                continue
            candidates.append(arrow)
    return candidates


def _same_node_pair(a: Arrow, b: Arrow) -> bool:
    return a.tail_node_id == b.tail_node_id and a.head_node_id == b.head_node_id


def _branch_endpoint_reuse_ok(cand: Arrow, selected: List[Arrow], cfg: LADDecoderConfig) -> bool:
    """Permit endpoint reuse only when it behaves like a genuine branch.

    Crowded DFA/flowchart nodes often produce one tail/head heatmap peak for two
    nearby physical arrows. A strict one-endpoint-one-arrow rule then deletes a
    correct edge even when the shaft/path evidence exists. This check keeps the
    duplicate guard but allows fan-out/fan-in when the opposite endpoint and traced
    path clearly diverge.
    """
    shared = [
        kept for kept in selected
        if cand.tail_endpoint_id == kept.tail_endpoint_id or cand.head_endpoint_id == kept.head_endpoint_id
    ]
    if not shared:
        return True

    for kept in shared:
        shared_tail = cand.tail_endpoint_id == kept.tail_endpoint_id
        shared_head = cand.head_endpoint_id == kept.head_endpoint_id

        if shared_tail and shared_head:
            return False

        if cfg.branch_require_different_node_pair and _same_node_pair(cand, kept):
            return False

        if shared_tail and l2(cand.head_xy, kept.head_xy) < cfg.branch_min_opposite_endpoint_dist:
            return False

        if shared_head and l2(cand.tail_xy, kept.tail_xy) < cfg.branch_min_opposite_endpoint_dist:
            return False

        if path_iou(cand.path_xy, kept.path_xy) > cfg.branch_max_path_iou:
            return False

    return True


def select_arrows(candidates: List[Arrow], cfg: LADDecoderConfig) -> List[Arrow]:
    selected: List[Arrow] = []
    used_tails = set()
    used_heads = set()
    for cand in sorted(candidates, key=lambda a: a.confidence, reverse=True):
        if cand.confidence < cfg.min_arrow_confidence:
            continue

        if not cfg.allow_endpoint_reuse:
            endpoint_already_used = cand.tail_endpoint_id in used_tails or cand.head_endpoint_id in used_heads
            if endpoint_already_used:
                if not cfg.branch_aware_endpoint_reuse:
                    continue
                if not _branch_endpoint_reuse_ok(cand, selected, cfg):
                    continue

        duplicate = False
        for kept in selected:
            same_ep = l2(cand.tail_xy, kept.tail_xy) <= cfg.duplicate_endpoint_dist and l2(cand.head_xy, kept.head_xy) <= cfg.duplicate_endpoint_dist
            same_path = path_iou(cand.path_xy, kept.path_xy) >= cfg.duplicate_path_iou
            if same_ep or (same_path and cand.tail_node_id == kept.tail_node_id and cand.head_node_id == kept.head_node_id):
                duplicate = True
                break
        if duplicate:
            continue
        selected.append(cand)
        used_tails.add(cand.tail_endpoint_id)
        used_heads.add(cand.head_endpoint_id)
    for idx, a in enumerate(selected, start=1):
        a.arrow_id = idx
    return selected


def _clone_endpoint_as_virtual(ep: Endpoint, new_kind: str, new_id: str, cfg: LADDecoderConfig) -> Endpoint:
    """Create a low-priority opposite-role endpoint at the same physical port."""
    tx, ty = ep.tangent_xy
    return Endpoint(
        endpoint_id=new_id,
        kind=new_kind,
        coarse_xy=ep.coarse_xy,
        refined_xy=ep.refined_xy,
        snapped_xy=ep.snapped_xy,
        score=float(ep.score) * cfg.port_repair_score_scale,
        progress_value=ep.progress_value,
        tangent_xy=(-float(tx), -float(ty)),
        offset_xy=ep.offset_xy,
        attached_node_id=ep.attached_node_id,
        attach_score=ep.attach_score,
        component_id=ep.component_id,
    )


def _max_iou_against_selected(path_xy: Sequence[Tuple[int, int]], selected: Sequence[Arrow]) -> float:
    if not selected:
        return 0.0
    return max(path_iou(path_xy, a.path_xy) for a in selected)


def generate_port_role_repair_candidates(
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
    heads: List[Endpoint],
    tails: List[Endpoint],
    selected: List[Arrow],
    cfg: LADDecoderConfig,
) -> List[Arrow]:
    """Second-pass candidate generation for missing role at crowded node ports.

    Normal decoding requires a tail heatmap peak and a head heatmap peak. In dense
    handwritten DFA nodes, a port can be both the head of one arrow and the tail of
    another. The heatmap may preserve only the stronger role. This pass uses the
    already selected edges to identify such ports and creates virtual opposite-role
    endpoints only at those selected ports. It then tries to connect them to unused
    endpoints.
    """
    if not cfg.enable_port_role_repair or not selected:
        return []

    selected_head_ids = {a.head_endpoint_id for a in selected}
    selected_tail_ids = {a.tail_endpoint_id for a in selected}
    unused_heads = [h for h in heads if h.endpoint_id not in selected_head_ids and h.attached_node_id is not None]
    unused_tails = [t for t in tails if t.endpoint_id not in selected_tail_ids and t.attached_node_id is not None]

    head_by_id = {h.endpoint_id: h for h in heads}
    tail_by_id = {t.endpoint_id: t for t in tails}

    repair_cfg = replace(cfg, progress_cost_weight=cfg.port_repair_progress_cost_weight)
    repair_candidates: List[Arrow] = []

    # Case A: accepted head also behaves as a missing outgoing tail.
    for kept in selected:
        src_head = head_by_id.get(kept.head_endpoint_id)
        if src_head is None or src_head.attached_node_id is None:
            continue
        vt = _clone_endpoint_as_virtual(src_head, "tail", f"VT_from_{src_head.endpoint_id}", cfg)
        for head in unused_heads:
            if vt.attached_node_id == head.attached_node_id:
                continue
            if l2(vt.refined_xy, head.refined_xy) > cfg.max_pair_dist:
                continue
            trace = astar_trace(vt, head, pred, support_maps, repair_cfg)
            if trace is None or len(trace["path"]) < cfg.min_path_len:
                continue
            arrow = make_arrow_candidate(vt, head, trace, pred, support_maps)
            arrow.confidence *= cfg.port_repair_confidence_scale
            arrow.meta["port_role_repair"] = True
            arrow.meta["virtual_tail_from_head"] = src_head.endpoint_id
            arrow.meta["selected_arrow_that_exposed_port"] = kept.arrow_id
            arrow.meta["max_iou_with_selected"] = _max_iou_against_selected(arrow.path_xy, selected)
            if arrow.confidence < cfg.port_repair_min_confidence:
                continue
            if arrow.support_score < cfg.port_repair_min_support_score or arrow.shaft_score < cfg.port_repair_min_shaft_score:
                continue
            if arrow.meta["max_iou_with_selected"] > cfg.port_repair_max_path_iou:
                continue
            repair_candidates.append(arrow)

    # Case B: accepted tail also behaves as a missing incoming head.
    for kept in selected:
        dst_tail = tail_by_id.get(kept.tail_endpoint_id)
        if dst_tail is None or dst_tail.attached_node_id is None:
            continue
        vh = _clone_endpoint_as_virtual(dst_tail, "head", f"VH_from_{dst_tail.endpoint_id}", cfg)
        for tail in unused_tails:
            if tail.attached_node_id == vh.attached_node_id:
                continue
            if l2(tail.refined_xy, vh.refined_xy) > cfg.max_pair_dist:
                continue
            trace = astar_trace(tail, vh, pred, support_maps, repair_cfg)
            if trace is None or len(trace["path"]) < cfg.min_path_len:
                continue
            arrow = make_arrow_candidate(tail, vh, trace, pred, support_maps)
            arrow.confidence *= cfg.port_repair_confidence_scale
            arrow.meta["port_role_repair"] = True
            arrow.meta["virtual_head_from_tail"] = dst_tail.endpoint_id
            arrow.meta["selected_arrow_that_exposed_port"] = kept.arrow_id
            arrow.meta["max_iou_with_selected"] = _max_iou_against_selected(arrow.path_xy, selected)
            if arrow.confidence < cfg.port_repair_min_confidence:
                continue
            if arrow.support_score < cfg.port_repair_min_support_score or arrow.shaft_score < cfg.port_repair_min_shaft_score:
                continue
            if arrow.meta["max_iou_with_selected"] > cfg.port_repair_max_path_iou:
                continue
            repair_candidates.append(arrow)

    repair_candidates = sorted(repair_candidates, key=lambda a: a.confidence, reverse=True)[:cfg.port_repair_max_candidates]
    return repair_candidates



# -----------------------------------------------------------------------------
# V5 stroke-graph fallback: topology-first candidate generation
# -----------------------------------------------------------------------------

def zhang_suen_thinning(binary: np.ndarray, max_iter: int = 128) -> np.ndarray:
    """Small dependency-free thinning fallback.

    Input/output are uint8 {0,1}. It is slower than skimage/cv2.ximgproc but fine
    for 512x512 inference maps.
    """
    img = (binary > 0).astype(np.uint8).copy()
    if img.size == 0 or int(img.sum()) == 0:
        return img
    h, w = img.shape
    for _ in range(max_iter):
        changed = False
        to_remove: List[Tuple[int, int]] = []
        for y in range(1, h - 1):
            for x in range(1, w - 1):
                if img[y, x] == 0:
                    continue
                p2 = img[y - 1, x];     p3 = img[y - 1, x + 1]
                p4 = img[y, x + 1];     p5 = img[y + 1, x + 1]
                p6 = img[y + 1, x];     p7 = img[y + 1, x - 1]
                p8 = img[y, x - 1];     p9 = img[y - 1, x - 1]
                nb = int(p2+p3+p4+p5+p6+p7+p8+p9)
                if nb < 2 or nb > 6:
                    continue
                seq = [p2,p3,p4,p5,p6,p7,p8,p9,p2]
                trans = sum(1 for i in range(8) if seq[i] == 0 and seq[i+1] == 1)
                if trans != 1:
                    continue
                if p2 * p4 * p6 != 0:
                    continue
                if p4 * p6 * p8 != 0:
                    continue
                to_remove.append((x, y))
        if to_remove:
            changed = True
            for x, y in to_remove:
                img[y, x] = 0

        to_remove = []
        for y in range(1, h - 1):
            for x in range(1, w - 1):
                if img[y, x] == 0:
                    continue
                p2 = img[y - 1, x];     p3 = img[y - 1, x + 1]
                p4 = img[y, x + 1];     p5 = img[y + 1, x + 1]
                p6 = img[y + 1, x];     p7 = img[y + 1, x - 1]
                p8 = img[y, x - 1];     p9 = img[y - 1, x - 1]
                nb = int(p2+p3+p4+p5+p6+p7+p8+p9)
                if nb < 2 or nb > 6:
                    continue
                seq = [p2,p3,p4,p5,p6,p7,p8,p9,p2]
                trans = sum(1 for i in range(8) if seq[i] == 0 and seq[i+1] == 1)
                if trans != 1:
                    continue
                if p2 * p4 * p8 != 0:
                    continue
                if p2 * p6 * p8 != 0:
                    continue
                to_remove.append((x, y))
        if to_remove:
            changed = True
            for x, y in to_remove:
                img[y, x] = 0
        if not changed:
            break
    return img.astype(np.uint8)


def skeletonize_mask(binary: np.ndarray) -> np.ndarray:
    binary = (binary > 0).astype(np.uint8)
    try:
        from skimage.morphology import skeletonize as _sk_skeletonize  # type: ignore
        return _sk_skeletonize(binary > 0).astype(np.uint8)
    except Exception:
        return zhang_suen_thinning(binary)


def component_endpoint_pixels(skel: np.ndarray, max_points: int = 10) -> List[Tuple[int, int]]:
    ys, xs = np.where(skel > 0)
    pts = [(int(x), int(y)) for x, y in zip(xs, ys)]
    if not pts:
        return []
    h, w = skel.shape
    endpoints: List[Tuple[int, int]] = []
    for x, y in pts:
        deg = 0
        for dx, dy in NEIGHBORS_8:
            nx_, ny_ = x + dx, y + dy
            if inside(nx_, ny_, w, h) and skel[ny_, nx_] > 0:
                deg += 1
        if deg <= 1:
            endpoints.append((x, y))
    if len(endpoints) >= 2:
        # Keep spatially diverse endpoints by greedily taking far-apart points.
        chosen: List[Tuple[int, int]] = []
        for pnt in sorted(endpoints, key=lambda p: (p[1], p[0])):
            if all(l2(pnt, c) > 4.0 for c in chosen):
                chosen.append(pnt)
            if len(chosen) >= max_points:
                break
        return chosen

    # Closed curves or over-thick components: use approximate diameter points.
    if len(pts) == 1:
        return pts
    sample = pts if len(pts) <= 500 else pts[::max(1, len(pts)//500)]
    a = sample[0]
    b = max(sample, key=lambda q: l2(a, q))
    c = max(sample, key=lambda q: l2(b, q))
    return [b, c]


def nearest_point_in_mask(seed: Sequence[float], mask: np.ndarray, max_radius: int = 12) -> Optional[Tuple[int, int]]:
    h, w = mask.shape
    sx = int(round(float(seed[0]))); sy = int(round(float(seed[1])))
    if inside(sx, sy, w, h) and mask[sy, sx] > 0:
        return (sx, sy)
    best = None
    best_d = float("inf")
    for r in range(1, max_radius + 1):
        x0 = max(0, sx - r); x1 = min(w - 1, sx + r)
        y0 = max(0, sy - r); y1 = min(h - 1, sy + r)
        ys, xs = np.where(mask[y0:y1+1, x0:x1+1] > 0)
        if len(xs) == 0:
            continue
        for xx, yy in zip(xs + x0, ys + y0):
            d = math.hypot(float(xx - sx), float(yy - sy))
            if d < best_d:
                best_d = d
                best = (int(xx), int(yy))
        if best is not None:
            return best
    return None


def shortest_path_on_component(
    mask: np.ndarray,
    support: np.ndarray,
    start_xy: Sequence[float],
    goal_xy: Sequence[float],
    max_expansions: int = 60000,
) -> Optional[List[Tuple[int, int]]]:
    """A* constrained to a single stroke component mask."""
    h, w = mask.shape
    start = nearest_point_in_mask(start_xy, mask, max_radius=14)
    goal = nearest_point_in_mask(goal_xy, mask, max_radius=14)
    if start is None or goal is None:
        return None
    if start == goal:
        return [start]
    gx, gy = goal
    pq: List[Tuple[float, float, Tuple[int, int], Optional[Tuple[int, int]]]] = []
    heapq.heappush(pq, (0.0, 0.0, start, None))
    came_from: Dict[Tuple[int, int], Optional[Tuple[int, int]]] = {}
    best_g: Dict[Tuple[int, int], float] = {start: 0.0}
    expansions = 0
    while pq:
        _, g, cur, prev = heapq.heappop(pq)
        if cur in came_from:
            continue
        came_from[cur] = prev
        cx, cy = cur
        if math.hypot(cx - gx, cy - gy) <= 1.5:
            path = [(cx, cy)]
            pcur = cur
            while came_from[pcur] is not None:
                pcur = came_from[pcur]  # type: ignore[index]
                path.append(pcur)
            path.reverse()
            if path[-1] != goal:
                path.append(goal)
            return path
        expansions += 1
        if expansions > max_expansions:
            return None
        for dx, dy in NEIGHBORS_8:
            nx_, ny_ = cx + dx, cy + dy
            if not inside(nx_, ny_, w, h) or mask[ny_, nx_] == 0:
                continue
            sl = step_length(dx, dy)
            # Stay inside high-support pixels but allow thin weak links in the same component.
            cost = sl + 1.25 * (1.0 - float(support[ny_, nx_]))
            nxt = (nx_, ny_)
            ng = g + cost
            if ng < best_g.get(nxt, float("inf")):
                best_g[nxt] = ng
                heuristic = 0.08 * math.hypot(gx - nx_, gy - ny_)
                heapq.heappush(pq, (ng + heuristic, ng, nxt, cur))
    return None


def nearest_node_to_point(pt: Sequence[float], nodes: List[Node], max_dist: float) -> Tuple[Optional[Node], float]:
    best: Optional[Node] = None
    best_d = float("inf")
    for n in nodes:
        d = point_to_bbox_distance(pt, n.bbox_xyxy)
        if d < best_d:
            best_d = d
            best = n
    if best is not None and best_d <= max_dist:
        return best, best_d
    return None, best_d


def orient_stroke_path(
    path: List[Tuple[int, int]],
    pred: Dict[str, np.ndarray],
) -> Tuple[List[Tuple[int, int]], float, float, float]:
    """Return path ordered tail->head using progress + direction-field votes."""
    if len(path) < 2:
        return path, 0.0, 0.0, 0.0
    def orient_score(pp: List[Tuple[int, int]]) -> Tuple[float, float, float]:
        dir_scores: List[float] = []
        prog_good: List[float] = []
        for i in range(len(pp) - 1):
            x1, y1 = pp[i]
            x2, y2 = pp[i + 1]
            dx, dy = x2 - x1, y2 - y1
            n = math.hypot(dx, dy)
            if n < 1e-8:
                continue
            sx, sy = dx / n, dy / n
            dir_scores.append(max(0.0, cosine2(sx, sy, float(pred["dir_x"][y2, x2]), float(pred["dir_y"][y2, x2]))))
            prog_good.append(1.0 if float(pred["progress"][y2, x2]) + 0.025 >= float(pred["progress"][y1, x1]) else 0.0)
        p0 = sample_map_bilinear(pred["progress"], pp[0])
        p1 = sample_map_bilinear(pred["progress"], pp[-1])
        prog_span = float(np.clip(p1 - p0, -1.0, 1.0))
        dir_mean = float(np.mean(dir_scores)) if dir_scores else 0.0
        prog_mean = float(np.mean(prog_good)) if prog_good else 0.0
        score = 0.45 * dir_mean + 0.35 * prog_mean + 0.20 * max(0.0, prog_span)
        return score, dir_mean, prog_span
    f_score, f_dir, f_prog = orient_score(path)
    rpath = list(reversed(path))
    r_score, r_dir, r_prog = orient_score(rpath)
    if r_score > f_score:
        return rpath, r_score, r_dir, r_prog
    return path, f_score, f_dir, f_prog


def virtual_endpoint_from_stroke(
    endpoint_id: str,
    kind: str,
    xy: Sequence[float],
    path: List[Tuple[int, int]],
    node: Optional[Node],
    score: float,
    pred: Dict[str, np.ndarray],
) -> Endpoint:
    if kind == "tail":
        if len(path) >= 3:
            dx = path[min(2, len(path)-1)][0] - path[0][0]
            dy = path[min(2, len(path)-1)][1] - path[0][1]
        else:
            dx, dy = 1.0, 0.0
    else:
        if len(path) >= 3:
            dx = path[-1][0] - path[max(0, len(path)-3)][0]
            dy = path[-1][1] - path[max(0, len(path)-3)][1]
        else:
            dx, dy = 1.0, 0.0
    tx, ty, _ = unit(dx, dy)
    x, y = float(xy[0]), float(xy[1])
    return Endpoint(
        endpoint_id=endpoint_id,
        kind=kind,
        coarse_xy=(x, y),
        refined_xy=(x, y),
        snapped_xy=(int(round(x)), int(round(y))),
        score=float(score),
        progress_value=float(sample_map_bilinear(pred["progress"], (x, y))),
        tangent_xy=(tx, ty),
        offset_xy=(0.0, 0.0),
        attached_node_id=node.node_id if node is not None else None,
        attach_score=point_to_bbox_distance((x, y), node.bbox_xyxy) if node is not None else None,
        component_id=None,
    )


def generate_stroke_graph_candidates(
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
    nodes: List[Node],
    cfg: LADDecoderConfig,
) -> List[Arrow]:
    """Generate edge candidates from stroke topology, not endpoint roles.

    This is designed for exactly the failure case where a visible arrow shaft exists
    but the endpoint heatmap gives the wrong role at a crowded node port. The stroke
    itself becomes the candidate; endpoints/progress/direction only score/orient it.
    """
    if not cfg.enable_stroke_graph_candidates or not nodes:
        return []

    support = support_maps["support"].astype(np.float32)
    walkable = support_maps["walkable"].astype(np.uint8).copy()
    node_core = support_maps["node_core"].astype(np.uint8)
    h, w = walkable.shape

    # Remove node interiors so components represent strokes between/around nodes.
    stroke_mask = walkable.copy()
    stroke_mask[node_core > 0] = 0

    # Slightly clean the topology while preserving thin hand-drawn links.
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    stroke_mask = cv2.morphologyEx(stroke_mask, cv2.MORPH_OPEN, k)
    stroke_mask = cv2.morphologyEx(stroke_mask, cv2.MORPH_CLOSE, k)

    num, labels, stats, _ = cv2.connectedComponentsWithStats(stroke_mask.astype(np.uint8), connectivity=8)
    med = median_node_diag(nodes)
    contact_max = float(np.clip(cfg.stroke_graph_node_contact_factor * med,
                                cfg.stroke_graph_node_contact_min,
                                cfg.stroke_graph_node_contact_max))
    min_area = max(int(cfg.stroke_graph_min_component_area), int(0.12 * med))

    comp_ids = list(range(1, num))
    comp_ids = sorted(comp_ids, key=lambda cid: int(stats[cid, cv2.CC_STAT_AREA]), reverse=True)[:cfg.stroke_graph_max_components]

    candidates: List[Arrow] = []
    seen_node_pairs: set = set()
    for cid in comp_ids:
        area = int(stats[cid, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        comp = (labels == cid).astype(np.uint8)
        skel = skeletonize_mask(comp)
        if int(skel.sum()) < cfg.stroke_graph_min_path_len:
            # Fall back to the component itself when thinning is too aggressive.
            skel = comp

        endpoints = component_endpoint_pixels(skel, max_points=12)
        if len(endpoints) < 2:
            continue

        # Attach component endpoints to nearby nodes.
        contacts: List[Tuple[Tuple[int, int], Node, float]] = []
        for ep_xy in endpoints:
            node, dist = nearest_node_to_point(ep_xy, nodes, contact_max)
            if node is not None:
                if all(not (node.node_id == old_node.node_id and l2(ep_xy, old_xy) < 6.0) for old_xy, old_node, _ in contacts):
                    contacts.append((ep_xy, node, dist))
        if len(contacts) < 2:
            continue

        # Try pairs of contacts on the same stroke component.
        for i in range(len(contacts)):
            for j in range(i + 1, len(contacts)):
                p0, n0, d0 = contacts[i]
                p1, n1, d1 = contacts[j]
                same_node_contact = n0.node_id == n1.node_id
                if cfg.stroke_graph_skip_self_loops and same_node_contact:
                    continue

                raw_path = shortest_path_on_component(skel, support, p0, p1, max_expansions=min(cfg.max_expansions, 60000))
                if raw_path is None or len(raw_path) < cfg.stroke_graph_min_path_len:
                    raw_path = shortest_path_on_component(comp, support, p0, p1, max_expansions=min(cfg.max_expansions, 60000))
                if raw_path is None or len(raw_path) < cfg.stroke_graph_min_path_len:
                    continue

                if same_node_contact:
                    # A real self-loop must leave the node region. Otherwise tiny border strokes
                    # and node outline fragments become fake loops.
                    bbox = n0.bbox_xyxy
                    max_excursion = max(point_to_bbox_distance((float(x), float(y)), bbox) for x, y in raw_path)
                    if len(raw_path) < max(cfg.stroke_graph_min_path_len, int(0.75 * med)):
                        continue
                    if max_excursion < max(8.0, 0.16 * med):
                        continue

                ordered_path, orient_score, orient_dir, orient_prog = orient_stroke_path(raw_path, pred)
                tail_node = n0 if ordered_path[0] == raw_path[0] else n1
                head_node = n1 if ordered_path[-1] == raw_path[-1] else n0

                pair_key = (cid, tail_node.node_id, head_node.node_id)
                if pair_key in seen_node_pairs:
                    continue
                seen_node_pairs.add(pair_key)

                tail_score = max(cfg.stroke_graph_endpoint_score, sample_map_bilinear(pred["tail"], ordered_path[0]))
                head_score = max(cfg.stroke_graph_endpoint_score, sample_map_bilinear(pred["head"], ordered_path[-1]))
                tail = virtual_endpoint_from_stroke(
                    f"SG{cid}_T_{tail_node.node_id}_{head_node.node_id}", "tail", ordered_path[0], ordered_path, tail_node, tail_score, pred
                )
                head = virtual_endpoint_from_stroke(
                    f"SG{cid}_H_{tail_node.node_id}_{head_node.node_id}", "head", ordered_path[-1], ordered_path, head_node, head_score, pred
                )
                arrow = make_arrow_candidate(
                    tail,
                    head,
                    {"path": ordered_path, "cost": float(len(ordered_path)), "start_snap": ordered_path[0], "goal_snap": ordered_path[-1], "expansions": None},
                    pred,
                    support_maps,
                )
                # Stroke topology is a fallback, not a licence to hallucinate. Scale it
                # slightly below direct endpoint candidates, then keep only strong paths.
                arrow.confidence *= cfg.stroke_graph_confidence_scale
                arrow.meta.update({
                    "source": "stroke_graph",
                    "stroke_component_id": int(cid),
                    "stroke_component_area": int(area),
                    "stroke_contact_dist_tail": float(d0 if tail_node.node_id == n0.node_id else d1),
                    "stroke_contact_dist_head": float(d1 if head_node.node_id == n1.node_id else d0),
                    "stroke_orient_score": float(orient_score),
                    "stroke_orient_dir_score": float(orient_dir),
                    "stroke_orient_progress_span": float(orient_prog),
                })
                if arrow.confidence < cfg.stroke_graph_min_confidence:
                    continue
                # Avoid exact duplicates inside stroke candidates.
                if any(path_iou(arrow.path_xy, old.path_xy) > cfg.stroke_graph_max_path_iou and _same_node_pair(arrow, old) for old in candidates):
                    continue
                candidates.append(arrow)
                if len(candidates) >= cfg.stroke_graph_max_candidates:
                    return sorted(candidates, key=lambda a: a.confidence, reverse=True)

    return sorted(candidates, key=lambda a: a.confidence, reverse=True)



# -----------------------------------------------------------------------------
# V6 ink-bridge fallback: connect strong endpoint pairs through raw ink corridors
# -----------------------------------------------------------------------------

def line_corridor_mask(shape_hw: Tuple[int, int], p0: Sequence[float], p1: Sequence[float], width: int) -> np.ndarray:
    h, w = shape_hw
    mask = np.zeros((h, w), dtype=np.uint8)
    a = (int(round(float(p0[0]))), int(round(float(p0[1]))))
    b = (int(round(float(p1[0]))), int(round(float(p1[1]))))
    width = max(3, int(width))
    cv2.line(mask, a, b, 1, thickness=width, lineType=cv2.LINE_AA)
    draw_disk(mask, a, max(3, width // 2), value=1)
    draw_disk(mask, b, max(3, width // 2), value=1)
    return mask.astype(np.uint8)


def build_ink_bridge_support_maps(
    tail: Endpoint,
    head: Endpoint,
    support_maps: Dict[str, np.ndarray],
    cfg: LADDecoderConfig,
) -> Optional[Dict[str, np.ndarray]]:
    ink = support_maps.get("ink")
    if ink is None or float(np.max(ink)) <= 1e-6:
        return None
    base_support = support_maps["support"].astype(np.float32)
    base_walkable = support_maps["walkable"].astype(np.uint8)
    node_core = support_maps["node_core"].astype(np.uint8)
    endpoint_force = support_maps.get("endpoint_force", np.zeros_like(base_walkable, dtype=np.uint8)).astype(np.uint8)
    h, w = base_support.shape

    dist = l2(tail.refined_xy, head.refined_xy)
    width = int(np.clip(cfg.ink_bridge_width_factor * dist, cfg.ink_bridge_min_width, cfg.ink_bridge_max_width))
    corridor = line_corridor_mask((h, w), tail.refined_xy, head.refined_xy, width)

    vals = ink[corridor > 0]
    if vals.size == 0:
        return None
    # Local threshold: dark/white stroke ink becomes high after image_ink_support().
    local_thr = float(np.clip(max(cfg.ink_bridge_threshold, 0.55 * float(np.quantile(vals, 0.82))), 0.12, 0.48))
    ink_walk = ((ink >= local_thr) & (corridor > 0)).astype(np.uint8)

    # Bridge small handwriting gaps inside the corridor only. This is the key safety valve:
    # raw ink helps the decoder, but it cannot freely wander over the whole page.
    ksz = int(np.clip(round(width / 3), 3, 11))
    if ksz % 2 == 0:
        ksz += 1
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksz, ksz))
    ink_walk = cv2.morphologyEx(ink_walk, cv2.MORPH_CLOSE, k)
    ink_walk = cv2.dilate(ink_walk, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)), iterations=1)
    ink_walk = (ink_walk & corridor).astype(np.uint8)

    bridge_walkable = ((base_walkable > 0) & (corridor > 0)).astype(np.uint8)
    bridge_walkable = np.maximum(bridge_walkable, ink_walk)
    bridge_walkable = np.maximum(bridge_walkable, endpoint_force)

    bridge_support = np.maximum(base_support, 0.86 * ink * corridor.astype(np.float32)).astype(np.float32)
    bridge_support = np.clip(bridge_support, 0.0, 1.0)

    return {
        "support": bridge_support,
        "walkable": bridge_walkable.astype(np.uint8),
        "node_core": node_core,
        "ink": ink.astype(np.float32),
        "endpoint_force": endpoint_force,
        "bridge_corridor": corridor,
    }


def generate_ink_bridge_candidates(
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
    tails: List[Endpoint],
    heads: List[Endpoint],
    existing_candidates: List[Arrow],
    cfg: LADDecoderConfig,
) -> List[Arrow]:
    """Recover endpoint pairs that have strong raw-ink evidence but broken learned support.

    This specifically targets cases such as a right-side DFA edge where tail/head
    heatmaps are present, but the learned shaft/skeleton support is discontinuous.
    We do not globally add raw ink to the decoder. We create a narrow corridor per
    endpoint pair and allow raw ink only inside that corridor.
    """
    if not cfg.enable_ink_bridge_candidates:
        return []

    existing_ep_pairs = {(a.tail_endpoint_id, a.head_endpoint_id) for a in existing_candidates}
    existing_node_pairs = {(a.tail_node_id, a.head_node_id) for a in existing_candidates}
    repair_cfg = replace(
        cfg,
        progress_cost_weight=cfg.ink_bridge_progress_cost_weight,
        support_cost_weight=cfg.ink_bridge_support_cost_weight,
        nearest_walkable_radius=max(cfg.nearest_walkable_radius, cfg.ink_bridge_max_width),
    )

    out: List[Arrow] = []
    for tail in tails:
        if tail.attached_node_id is None:
            continue
        for head in heads:
            if head.attached_node_id is None:
                continue
            if (tail.endpoint_id, head.endpoint_id) in existing_ep_pairs:
                continue
            same_node = tail.attached_node_id == head.attached_node_id
            if same_node and not cfg.ink_bridge_allow_self_loops:
                continue

            # Avoid trying every boring pair when the same node pair is already well represented,
            # except for self-loops and multi-edge cases where another geometric route may exist.
            if (tail.attached_node_id, head.attached_node_id) in existing_node_pairs and not same_node:
                continue

            # If the learned progress is strongly contradictory, raw ink alone is not enough.
            if not same_node and (head.progress_value + 0.08 < tail.progress_value):
                continue

            bridge_maps = build_ink_bridge_support_maps(tail, head, support_maps, cfg)
            if bridge_maps is None:
                continue
            trace = astar_trace(tail, head, pred, bridge_maps, repair_cfg)
            if trace is None or len(trace["path"]) < cfg.min_path_len:
                continue

            arrow = make_arrow_candidate(tail, head, trace, pred, bridge_maps)
            arrow.confidence *= cfg.ink_bridge_confidence_scale
            arrow.meta["source"] = "ink_bridge"
            arrow.meta["bridge_pair_already_present"] = (tail.attached_node_id, head.attached_node_id) in existing_node_pairs
            arrow.meta["bridge_corridor_width"] = int(np.clip(cfg.ink_bridge_width_factor * l2(tail.refined_xy, head.refined_xy), cfg.ink_bridge_min_width, cfg.ink_bridge_max_width))

            if arrow.confidence < cfg.ink_bridge_min_confidence:
                continue
            if arrow.support_score < cfg.ink_bridge_min_support_score:
                continue
            # Same-node candidates must actually leave the node area.
            if same_node:
                # We do not have the node bbox here, so use a conservative path length gate.
                if len(arrow.path_xy) < max(cfg.stroke_graph_min_path_len, 32):
                    continue
            out.append(arrow)
            if len(out) >= cfg.ink_bridge_max_candidates:
                return sorted(out, key=lambda a: a.confidence, reverse=True)
    return sorted(out, key=lambda a: a.confidence, reverse=True)

def build_writer_json(image_name: str, width: int, height: int, nodes: List[Node], arrows: List[Arrow]) -> Dict[str, Any]:
    return {
        "image": image_name,
        "size": {"w": int(width), "h": int(height)},
        "nodes": [
            {
                "id": n.node_id,
                "bbox": [int(v) for v in n.bbox_xywh],
                "center": {"x": float(n.center_xy[0]), "y": float(n.center_xy[1])},
                "score": float(n.score),
            }
            for n in nodes
        ],
        "arrows": [
            {
                "id": int(a.arrow_id),
                "tail_hint": {"x": float(a.tail_hint["x"]), "y": float(a.tail_hint["y"])},
                "head_hint": {"x": float(a.head_hint["x"]), "y": float(a.head_hint["y"])},
                "path": [{"x": float(x), "y": float(y)} for x, y in a.path_xy],
                "tail_node_id": a.tail_node_id,
                "head_node_id": a.head_node_id,
                "confidence": float(a.confidence),
            }
            for a in arrows
        ],
    }


def rescale_writer_json(writer: Dict[str, Any], orig_w: int, orig_h: int) -> Dict[str, Any]:
    w0 = float(writer["size"]["w"]); h0 = float(writer["size"]["h"])
    sx = float(orig_w) / max(w0, 1.0); sy = float(orig_h) / max(h0, 1.0)
    out = json.loads(json.dumps(writer))
    out["size"] = {"w": int(orig_w), "h": int(orig_h)}
    out["coordinate_space"] = "original_image"
    for n in out.get("nodes", []):
        x, y, bw, bh = n["bbox"]
        n["bbox"] = [float(x) * sx, float(y) * sy, float(bw) * sx, float(bh) * sy]
        if "center" in n:
            n["center"]["x"] *= sx
            n["center"]["y"] *= sy
    for a in out.get("arrows", []):
        for key in ["tail_hint", "head_hint"]:
            a[key]["x"] *= sx
            a[key]["y"] *= sy
        for p in a.get("path", []):
            p["x"] *= sx
            p["y"] *= sy
    return out


def build_debug_json(
    image_name: str,
    nodes: List[Node],
    heads: List[Endpoint],
    tails: List[Endpoint],
    candidates: List[Arrow],
    arrows: List[Arrow],
    cfg: LADDecoderConfig,
) -> Dict[str, Any]:
    return {
        "image": image_name,
        "decoder": "AdaptiveStrokeGraphDecoderV6",
        "config": asdict(cfg),
        "nodes": [asdict(n) for n in nodes],
        "heads": [asdict(e) for e in heads],
        "tails": [asdict(e) for e in tails],
        "candidate_arrows": [asdict(a) for a in candidates],
        "selected_arrows": [asdict(a) for a in arrows],
        "summary": {
            "num_nodes": len(nodes),
            "num_heads": len(heads),
            "num_tails": len(tails),
            "num_candidates": len(candidates),
            "num_endpoint_candidates": sum(1 for a in candidates if a.meta.get("source") not in {"stroke_graph"} and not a.meta.get("port_role_repair", False)),
            "num_port_repair_candidates": sum(1 for a in candidates if a.meta.get("port_role_repair", False)),
            "num_stroke_graph_candidates": sum(1 for a in candidates if a.meta.get("source") == "stroke_graph"),
            "num_ink_bridge_candidates": sum(1 for a in candidates if a.meta.get("source") == "ink_bridge"),
            "num_arrows": len(arrows),
            "num_stroke_graph_arrows_selected": sum(1 for a in arrows if a.meta.get("source") == "stroke_graph"),
            "num_ink_bridge_arrows_selected": sum(1 for a in arrows if a.meta.get("source") == "ink_bridge"),
        },
    }


def build_networkx_graph(nodes: List[Node], arrows: List[Arrow]) -> Optional[Any]:
    if nx is None:
        return None
    g = nx.MultiDiGraph()
    for n in nodes:
        g.add_node(n.node_id, bbox_xyxy=n.bbox_xyxy, bbox_xywh=n.bbox_xywh, center_xy=n.center_xy, score=n.score)
    for a in arrows:
        src = a.tail_node_id if a.tail_node_id is not None else f"EP::{a.tail_endpoint_id}"
        dst = a.head_node_id if a.head_node_id is not None else f"EP::{a.head_endpoint_id}"
        if src not in g:
            g.add_node(src, kind="endpoint_stub")
        if dst not in g:
            g.add_node(dst, kind="endpoint_stub")
        g.add_edge(src, dst, key=f"A{a.arrow_id}", **asdict(a))
    return g


def decode_graph_learning_aligned(
    pred: Dict[str, np.ndarray],
    decoder_img: np.ndarray,
    image_name: str,
    cfg: LADDecoderConfig,
    orig_size: Optional[Tuple[int, int]] = None,
) -> DecodeResult:
    h, w = pred["shaft"].shape

    # V5 adaptation happens inside the decoder, image by image. The original cfg
    # is still kept in the run config; the debug JSON records the adapted cfg that
    # was actually used for this image.
    cfg = adapt_config_pre_nodes(pred, cfg)
    nodes = extract_nodes(pred, cfg)
    cfg = adapt_config_post_nodes(pred, cfg, nodes)

    heads = extract_endpoints(pred, "head", cfg)
    tails = extract_endpoints(pred, "tail", cfg)
    attach_all_endpoints(heads, nodes, cfg)
    attach_all_endpoints(tails, nodes, cfg)

    support_maps = build_support_field(pred, decoder_img, nodes, heads, tails, cfg)

    endpoint_candidates = generate_arrow_candidates(pred, support_maps, tails, heads, cfg)
    first_pass_arrows = select_arrows(endpoint_candidates, cfg)

    repair_candidates = generate_port_role_repair_candidates(
        pred=pred,
        support_maps=support_maps,
        heads=heads,
        tails=tails,
        selected=first_pass_arrows,
        cfg=cfg,
    )

    # V5: add topology-first candidates from the visible/learned stroke graph.
    # These can recover edges even when no clean role-specific tail/head seed exists.
    stroke_candidates = generate_stroke_graph_candidates(
        pred=pred,
        support_maps=support_maps,
        nodes=nodes,
        cfg=cfg,
    )

    # V6: endpoint-to-endpoint raw-ink bridge fallback. This is deliberately after
    # normal/repair/stroke candidates so it only fills holes, not dominate the decoder.
    ink_bridge_candidates = generate_ink_bridge_candidates(
        pred=pred,
        support_maps=support_maps,
        tails=tails,
        heads=heads,
        existing_candidates=endpoint_candidates + repair_candidates + stroke_candidates,
        cfg=cfg,
    )

    candidates = endpoint_candidates + repair_candidates + stroke_candidates + ink_bridge_candidates
    arrows = select_arrows(candidates, cfg)

    writer = build_writer_json(image_name, w, h, nodes, arrows)
    writer["coordinate_space"] = "model"
    writer_orig = None
    if orig_size is not None:
        orig_w, orig_h = orig_size
        writer_orig = rescale_writer_json(writer, orig_w, orig_h)

    debug_json = build_debug_json(image_name, nodes, heads, tails, candidates, arrows, cfg)
    debug_maps = {
        "support": support_maps["support"],
        "walkable": support_maps["walkable"],
        "node_core": support_maps["node_core"],
        "ink": support_maps["ink"],
        "endpoint_force": support_maps["endpoint_force"],
    }
    graph = build_networkx_graph(nodes, arrows)

    if cfg.verbose:
        print(f"  decoded nodes={len(nodes)}, heads={len(heads)}, tails={len(tails)}, candidates={len(candidates)}, arrows={len(arrows)}")

    return DecodeResult(
        nodes=nodes,
        heads=heads,
        tails=tails,
        candidate_arrows=candidates,
        arrows=arrows,
        writer_style_json=writer,
        writer_style_original_json=writer_orig,
        decoder_debug_json=debug_json,
        debug_maps=debug_maps,
        nx_graph=graph,
    )


# -----------------------------------------------------------------------------
# File IO / visualization
# -----------------------------------------------------------------------------

def save_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_json(obj), f, indent=2)


def save_gray(path: Path, arr: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(path), to_uint8_vis(arr))


def save_heat(path: Path, arr: np.ndarray, cmap: int = cv2.COLORMAP_TURBO) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    vis = to_uint8_vis(arr)
    color = cv2.applyColorMap(vis, cmap)
    cv2.imwrite(str(path), color)


def save_rgb(path: Path, img: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    rgb = ensure_rgb_uint8(img)
    cv2.imwrite(str(path), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))


def draw_cross(img: np.ndarray, x: int, y: int, color: Tuple[int, int, int], size: int = 5, thickness: int = 1) -> None:
    cv2.line(img, (x - size, y - size), (x + size, y + size), color, thickness)
    cv2.line(img, (x - size, y + size), (x + size, y - size), color, thickness)


def draw_plus(img: np.ndarray, x: int, y: int, color: Tuple[int, int, int], size: int = 5, thickness: int = 1) -> None:
    cv2.line(img, (x - size, y), (x + size, y), color, thickness)
    cv2.line(img, (x, y - size), (x, y + size), color, thickness)


def overlay_nodes(img: np.ndarray, nodes: List[Node]) -> np.ndarray:
    vis = ensure_rgb_uint8(img).copy()
    for n in nodes:
        x1, y1, x2, y2 = n.bbox_xyxy
        cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cx, cy = map(int, map(round, n.center_xy))
        cv2.circle(vis, (cx, cy), 3, (255, 255, 0), -1)
        cv2.putText(vis, n.node_id, (x1, max(0, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 1, cv2.LINE_AA)
    return vis


def overlay_endpoints(img: np.ndarray, heads: List[Endpoint], tails: List[Endpoint]) -> np.ndarray:
    vis = ensure_rgb_uint8(img).copy()
    for ep in tails:
        x, y = map(int, map(round, ep.refined_xy))
        draw_plus(vis, x, y, (0, 255, 0), 6, 2)
        if ep.attached_node_id:
            cv2.putText(vis, f"{ep.endpoint_id}->{ep.attached_node_id}", (x + 5, y), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 255, 0), 1, cv2.LINE_AA)
    for ep in heads:
        x, y = map(int, map(round, ep.refined_xy))
        draw_cross(vis, x, y, (255, 0, 0), 6, 2)
        if ep.attached_node_id:
            cv2.putText(vis, f"{ep.endpoint_id}->{ep.attached_node_id}", (x + 5, y), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 0, 0), 1, cv2.LINE_AA)
    return vis


def overlay_final_graph(img: np.ndarray, result: DecodeResult) -> np.ndarray:
    vis = overlay_nodes(img, result.nodes)
    for a in result.arrows:
        if len(a.path_xy) >= 2:
            pts = np.array(a.path_xy, dtype=np.int32).reshape(-1, 1, 2)
            cv2.polylines(vis, [pts], isClosed=False, color=(0, 255, 0), thickness=2)
            mid = pts[len(pts) // 2, 0]
            label = f"A{a.arrow_id}: {a.tail_node_id}->{a.head_node_id} {a.confidence:.2f}"
            cv2.putText(vis, label, (int(mid[0]), int(mid[1])), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 255, 0), 1, cv2.LINE_AA)
        tx, ty = map(int, map(round, a.tail_xy))
        hx, hy = map(int, map(round, a.head_xy))
        cv2.circle(vis, (tx, ty), 4, (0, 255, 255), -1)
        cv2.circle(vis, (hx, hy), 4, (255, 255, 0), -1)
    return vis


def save_prediction_maps(output_dir: Path, stem: str, pred: Dict[str, np.ndarray]) -> None:
    # Probability heads
    for key in ["node_center", "node_extent", "shaft", "skeleton", "head", "tail", "progress"]:
        save_gray(output_dir / f"{stem}_pred_{key}.png", pred[key])
    # Vector/regression heads
    for key in [
        "dir_x", "dir_y", "dir_mag",
        "head_offset_x", "head_offset_y", "head_offset_mag",
        "tail_offset_x", "tail_offset_y", "tail_offset_mag",
        "head_tangent_x", "head_tangent_y",
        "tail_tangent_x", "tail_tangent_y",
    ]:
        save_heat(output_dir / f"{stem}_pred_{key}.png", pred[key], cmap=cv2.COLORMAP_TURBO)


def save_decode_artifacts(output_dir: Path, stem: str, decoder_img: np.ndarray, result: DecodeResult, save_maps: bool = True) -> None:
    save_json(output_dir / f"{stem}_writer_model.json", result.writer_style_json)
    if result.writer_style_original_json is not None:
        save_json(output_dir / f"{stem}_writer_original.json", result.writer_style_original_json)
    save_json(output_dir / f"{stem}_debug.json", result.decoder_debug_json)

    if save_maps:
        save_heat(output_dir / f"{stem}_stage_support.png", result.debug_maps["support"], cmap=cv2.COLORMAP_MAGMA)
        save_gray(output_dir / f"{stem}_stage_walkable.png", result.debug_maps["walkable"])
        save_gray(output_dir / f"{stem}_stage_node_core.png", result.debug_maps["node_core"])
        save_gray(output_dir / f"{stem}_stage_endpoint_force.png", result.debug_maps["endpoint_force"])
        if float(result.debug_maps["ink"].max()) > 0:
            save_gray(output_dir / f"{stem}_stage_ink_support.png", result.debug_maps["ink"])

    save_rgb(output_dir / f"{stem}_viz_input.png", ensure_rgb_uint8(decoder_img))
    save_rgb(output_dir / f"{stem}_viz_nodes.png", overlay_nodes(decoder_img, result.nodes))
    save_rgb(output_dir / f"{stem}_viz_endpoints_refined.png", overlay_endpoints(decoder_img, result.heads, result.tails))
    save_rgb(output_dir / f"{stem}_viz_final_graph.png", overlay_final_graph(decoder_img, result))


def append_csv_row(csv_path: Path, fieldnames: List[str], row: Dict[str, Any]) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    exists = csv_path.exists()
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not exists:
            writer.writeheader()
        writer.writerow(row)



@dataclass
class InferenceSample:
    sample_id: str
    image_path: Path
    json_path: Optional[Path] = None
    family: str = ""
    split: str = ""


def _parse_csv_list(value: Optional[str]) -> Optional[List[str]]:
    if value is None:
        return None
    value = str(value).strip()
    if not value or value.lower() in {"all", "*"}:
        return None
    return [x.strip() for x in value.split(",") if x.strip()]


def find_json_for_image(image_path: Path, json_dir: Optional[Path] = None) -> Optional[Path]:
    candidates: List[Path] = []
    if json_dir is not None:
        candidates.append(json_dir / f"{image_path.stem}.json")
    # New converted dataset convention: sibling json folder next to images.
    if image_path.parent.name == "images":
        candidates.append(image_path.parent.parent / "json" / f"{image_path.stem}.json")
    # Generic fallback: same directory.
    candidates.append(image_path.with_suffix(".json"))
    for p in candidates:
        if p.exists():
            return p
    return None


def discover_converted_dataset_samples(
    dataset_root: Path,
    families: Optional[Sequence[str]] = None,
    splits: Optional[Sequence[str]] = None,
    require_gt: bool = False,
) -> List[InferenceSample]:
    """Discover samples from <root>/<family>/<split>/{images,json}.

    The converted dataset also contains node_instance_masks, but this inference and
    graph-evaluation path intentionally ignores that folder. The JSON files are the
    annotation source used by the old ANN_DIR logic.
    """
    if not dataset_root.exists():
        raise FileNotFoundError(f"Dataset root not found: {dataset_root}")

    fam_filter = set(families) if families else None
    split_filter = set(splits) if splits else None
    samples: List[InferenceSample] = []

    for fam_dir in sorted([p for p in dataset_root.iterdir() if p.is_dir()]):
        family = fam_dir.name
        if fam_filter is not None and family not in fam_filter:
            continue
        for split_dir in sorted([p for p in fam_dir.iterdir() if p.is_dir()]):
            split = split_dir.name
            if split_filter is not None and split not in split_filter:
                continue
            image_dir = split_dir / "images"
            json_dir = split_dir / "json"
            if not image_dir.exists():
                continue
            for image_path in find_images(image_dir):
                json_path = find_json_for_image(image_path, json_dir=json_dir)
                if require_gt and json_path is None:
                    print(f"⚠️ Missing GT JSON for {family}/{split}/{image_path.name}")
                    continue
                samples.append(
                    InferenceSample(
                        sample_id=image_path.stem,
                        image_path=image_path,
                        json_path=json_path,
                        family=family,
                        split=split,
                    )
                )

    return samples


def discover_flat_samples(
    input_dir: Path,
    gt_json_dir: Optional[Path] = None,
    require_gt: bool = False,
) -> List[InferenceSample]:
    images = find_images(input_dir)
    samples: List[InferenceSample] = []
    for image_path in images:
        json_path = find_json_for_image(image_path, gt_json_dir)
        if require_gt and json_path is None:
            print(f"⚠️ Missing GT JSON for {image_path.name}")
            continue
        samples.append(InferenceSample(sample_id=image_path.stem, image_path=image_path, json_path=json_path))
    return samples


def sample_output_dir(base_output_dir: Path, sample: InferenceSample, preserve_structure: bool = True) -> Path:
    if preserve_structure and sample.family and sample.split:
        return base_output_dir / sample.family / sample.split
    return base_output_dir


# -----------------------------------------------------------------------------
# Graph/JSON evaluation metrics
# -----------------------------------------------------------------------------

def _bbox_xyxy_from_node(node: Dict[str, Any]) -> Tuple[float, float, float, float]:
    b = node.get("bbox", [0, 0, 0, 0])
    if len(b) < 4:
        return 0.0, 0.0, 0.0, 0.0
    x, y, w, h = map(float, b[:4])
    return x, y, x + max(0.0, w) - 1.0, y + max(0.0, h) - 1.0


def _bbox_center_xy(node: Dict[str, Any]) -> Tuple[float, float]:
    if "center" in node and isinstance(node["center"], dict):
        return float(node["center"].get("x", 0.0)), float(node["center"].get("y", 0.0))
    x1, y1, x2, y2 = _bbox_xyxy_from_node(node)
    return 0.5 * (x1 + x2), 0.5 * (y1 + y2)


def _bbox_iou_float(a: Sequence[float], b: Sequence[float]) -> float:
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1 + 1.0), max(0.0, iy2 - iy1 + 1.0)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1 + 1.0) * max(0.0, ay2 - ay1 + 1.0)
    area_b = max(0.0, bx2 - bx1 + 1.0) * max(0.0, by2 - by1 + 1.0)
    return float(inter / max(area_a + area_b - inter, 1e-8))


def _safe_div(num: float, den: float) -> float:
    return float(num) / float(den) if float(den) != 0.0 else 0.0


def _prf(tp: int, fp: int, fn: int) -> Tuple[float, float, float]:
    p = _safe_div(tp, tp + fp)
    r = _safe_div(tp, tp + fn)
    f = _safe_div(2.0 * p * r, p + r)
    return p, r, f


def match_nodes_by_iou(
    pred_nodes: Sequence[Dict[str, Any]],
    gt_nodes: Sequence[Dict[str, Any]],
    iou_thresh: float,
) -> Tuple[Dict[str, str], List[Tuple[str, str, float]]]:
    candidates: List[Tuple[float, str, str]] = []
    for pn in pred_nodes:
        pid = str(pn.get("id"))
        pb = _bbox_xyxy_from_node(pn)
        for gn in gt_nodes:
            gid = str(gn.get("id"))
            giou = _bbox_iou_float(pb, _bbox_xyxy_from_node(gn))
            if giou >= iou_thresh:
                candidates.append((giou, pid, gid))

    pred_used, gt_used = set(), set()
    pred_to_gt: Dict[str, str] = {}
    matches: List[Tuple[str, str, float]] = []
    for giou, pid, gid in sorted(candidates, reverse=True):
        if pid in pred_used or gid in gt_used:
            continue
        pred_used.add(pid)
        gt_used.add(gid)
        pred_to_gt[pid] = gid
        matches.append((pid, gid, float(giou)))
    return pred_to_gt, matches


def _edge_pair(edge: Dict[str, Any]) -> Tuple[Optional[str], Optional[str]]:
    t = edge.get("tail_node_id")
    h = edge.get("head_node_id")
    return (str(t) if t is not None else None, str(h) if h is not None else None)


def _map_edge_pair(edge: Dict[str, Any], pred_to_gt: Dict[str, str]) -> Tuple[Optional[str], Optional[str]]:
    t, h = _edge_pair(edge)
    return (pred_to_gt.get(t) if t is not None else None, pred_to_gt.get(h) if h is not None else None)


def _counter_from_pairs(pairs: Sequence[Tuple[Optional[str], Optional[str]]]) -> Dict[Tuple[Optional[str], Optional[str]], int]:
    out: Dict[Tuple[Optional[str], Optional[str]], int] = {}
    for p in pairs:
        out[p] = out.get(p, 0) + 1
    return out


def _undir(pair: Tuple[Optional[str], Optional[str]]) -> Tuple[str, str]:
    a = "__NONE__" if pair[0] is None else str(pair[0])
    b = "__NONE__" if pair[1] is None else str(pair[1])
    return tuple(sorted([a, b]))


def _counter_intersection_count(a: Dict[Any, int], b: Dict[Any, int]) -> int:
    return int(sum(min(v, b.get(k, 0)) for k, v in a.items()))


def _point_from_hint(edge: Dict[str, Any], key: str) -> Optional[Tuple[float, float]]:
    p = edge.get(key)
    if isinstance(p, dict) and "x" in p and "y" in p:
        return float(p["x"]), float(p["y"])
    return None


def _path_points(edge: Dict[str, Any]) -> List[Tuple[float, float]]:
    pts = []
    for p in edge.get("path", []) or []:
        if isinstance(p, dict) and "x" in p and "y" in p:
            pts.append((float(p["x"]), float(p["y"])))
    if len(pts) >= 2:
        return pts
    tail = _point_from_hint(edge, "tail_hint")
    head = _point_from_hint(edge, "head_hint")
    return [tail, head] if tail is not None and head is not None else []


def _rasterize_polyline(points: Sequence[Tuple[float, float]], w: int, h: int, tol_px: int) -> np.ndarray:
    mask = np.zeros((max(1, int(h)), max(1, int(w))), dtype=np.uint8)
    if len(points) == 0:
        return mask
    arr = np.array([[int(round(x)), int(round(y))] for x, y in points], dtype=np.int32)
    arr[:, 0] = np.clip(arr[:, 0], 0, mask.shape[1] - 1)
    arr[:, 1] = np.clip(arr[:, 1], 0, mask.shape[0] - 1)
    if len(arr) == 1:
        cv2.circle(mask, tuple(arr[0]), max(1, int(tol_px)), 1, -1)
    else:
        cv2.polylines(mask, [arr.reshape(-1, 1, 2)], False, 1, thickness=max(1, int(tol_px)))
    return mask


def _mask_prf_iou(pred_mask: np.ndarray, gt_mask: np.ndarray) -> Dict[str, float]:
    p = pred_mask.astype(bool)
    g = gt_mask.astype(bool)
    tp = int(np.logical_and(p, g).sum())
    fp = int(np.logical_and(p, np.logical_not(g)).sum())
    fn = int(np.logical_and(np.logical_not(p), g).sum())
    prec, rec, f1 = _prf(tp, fp, fn)
    iou = _safe_div(tp, tp + fp + fn)
    return {"precision": prec, "recall": rec, "f1": f1, "iou": iou}


def _rescale_writer_to_size(writer: Dict[str, Any], target_w: int, target_h: int) -> Dict[str, Any]:
    cur = writer.get("size", {}) or {}
    cur_w = float(cur.get("w", target_w))
    cur_h = float(cur.get("h", target_h))
    if abs(cur_w - target_w) < 1e-6 and abs(cur_h - target_h) < 1e-6:
        return writer
    sx = float(target_w) / max(cur_w, 1.0)
    sy = float(target_h) / max(cur_h, 1.0)
    out = json.loads(json.dumps(writer))
    out["size"] = {"w": int(target_w), "h": int(target_h)}
    for n in out.get("nodes", []):
        if "bbox" in n and len(n["bbox"]) >= 4:
            n["bbox"] = [float(n["bbox"][0]) * sx, float(n["bbox"][1]) * sy, float(n["bbox"][2]) * sx, float(n["bbox"][3]) * sy]
        if "center" in n:
            n["center"]["x"] *= sx
            n["center"]["y"] *= sy
    for e in out.get("arrows", []):
        for key in ["tail_hint", "head_hint"]:
            if isinstance(e.get(key), dict):
                e[key]["x"] *= sx
                e[key]["y"] *= sy
        for p in e.get("path", []) or []:
            p["x"] *= sx
            p["y"] *= sy
    return out


def _pair_matched_edges_for_geometry(
    pred_edges: Sequence[Dict[str, Any]],
    gt_edges: Sequence[Dict[str, Any]],
    pred_pairs: Sequence[Tuple[Optional[str], Optional[str]]],
    gt_pairs: Sequence[Tuple[Optional[str], Optional[str]]],
) -> List[Tuple[Dict[str, Any], Dict[str, Any]]]:
    gt_by_pair: Dict[Tuple[Optional[str], Optional[str]], List[int]] = {}
    for i, pair in enumerate(gt_pairs):
        gt_by_pair.setdefault(pair, []).append(i)

    matched: List[Tuple[Dict[str, Any], Dict[str, Any]]] = []
    used_gt = set()
    for pred_edge, pair in zip(pred_edges, pred_pairs):
        candidate_ids = [i for i in gt_by_pair.get(pair, []) if i not in used_gt]
        if not candidate_ids:
            continue
        pt = _point_from_hint(pred_edge, "tail_hint")
        ph = _point_from_hint(pred_edge, "head_hint")
        best_i, best_d = None, float("inf")
        for gi in candidate_ids:
            gt_edge = gt_edges[gi]
            gt = _point_from_hint(gt_edge, "tail_hint")
            gh = _point_from_hint(gt_edge, "head_hint")
            d = 0.0
            c = 0
            if pt is not None and gt is not None:
                d += l2(pt, gt); c += 1
            if ph is not None and gh is not None:
                d += l2(ph, gh); c += 1
            d = d / max(1, c)
            if d < best_d:
                best_d, best_i = d, gi
        if best_i is not None:
            used_gt.add(best_i)
            matched.append((pred_edge, gt_edges[best_i]))
    return matched


def evaluate_graph_prediction(
    pred_writer: Dict[str, Any],
    gt_writer: Dict[str, Any],
    node_iou_thresh: float = 0.50,
    path_tolerance_px: int = 3,
) -> Dict[str, Any]:
    gt_size = gt_writer.get("size", {}) or {}
    gt_w = int(gt_size.get("w", pred_writer.get("size", {}).get("w", 1)))
    gt_h = int(gt_size.get("h", pred_writer.get("size", {}).get("h", 1)))
    pred_writer = _rescale_writer_to_size(pred_writer, gt_w, gt_h)

    pred_nodes = pred_writer.get("nodes", []) or []
    gt_nodes = gt_writer.get("nodes", []) or []
    pred_edges = pred_writer.get("arrows", []) or []
    gt_edges = gt_writer.get("arrows", []) or []

    pred_to_gt, node_matches = match_nodes_by_iou(pred_nodes, gt_nodes, node_iou_thresh)
    node_tp = len(node_matches)
    node_fp = len(pred_nodes) - node_tp
    node_fn = len(gt_nodes) - node_tp
    node_precision, node_recall, node_f1 = _prf(node_tp, node_fp, node_fn)

    matched_ious = [m[2] for m in node_matches]
    center_errors: List[float] = []
    gt_by_id = {str(n.get("id")): n for n in gt_nodes}
    pred_by_id = {str(n.get("id")): n for n in pred_nodes}
    for pid, gid, _ in node_matches:
        pc = _bbox_center_xy(pred_by_id[pid])
        gc = _bbox_center_xy(gt_by_id[gid])
        center_errors.append(l2(pc, gc))

    gt_pairs = [_edge_pair(e) for e in gt_edges]
    pred_pairs_mapped = [_map_edge_pair(e, pred_to_gt) for e in pred_edges]
    gt_counter = _counter_from_pairs(gt_pairs)
    pred_counter = _counter_from_pairs(pred_pairs_mapped)
    edge_tp = _counter_intersection_count(pred_counter, gt_counter)
    edge_fp = len(pred_edges) - edge_tp
    edge_fn = len(gt_edges) - edge_tp
    edge_precision, edge_recall, edge_f1 = _prf(edge_tp, edge_fp, edge_fn)

    gt_undir_counter = _counter_from_pairs([_undir(p) for p in gt_pairs])
    pred_undir_counter = _counter_from_pairs([_undir(p) for p in pred_pairs_mapped])
    undir_tp = _counter_intersection_count(pred_undir_counter, gt_undir_counter)
    undir_fp = len(pred_edges) - undir_tp
    undir_fn = len(gt_edges) - undir_tp
    undir_precision, undir_recall, undir_f1 = _prf(undir_tp, undir_fp, undir_fn)
    direction_proxy = _safe_div(edge_tp, undir_tp)

    gt_loop_counter = {k: v for k, v in gt_counter.items() if k[0] is not None and k[0] == k[1]}
    pred_loop_counter = {k: v for k, v in pred_counter.items() if k[0] is not None and k[0] == k[1]}
    gt_loop_total = int(sum(gt_loop_counter.values()))
    matched_loop = _counter_intersection_count(pred_loop_counter, gt_loop_counter)
    loop_recall = _safe_div(matched_loop, gt_loop_total)

    out_degree: Dict[Optional[str], int] = {}
    for t, h in gt_pairs:
        if t is not None:
            out_degree[t] = out_degree.get(t, 0) + 1
    branch_sources = {nid for nid, deg in out_degree.items() if deg >= 2}
    branch_gt_pairs = [p for p in gt_pairs if p[0] in branch_sources]
    branch_counter = _counter_from_pairs(branch_gt_pairs)
    branch_total = int(sum(branch_counter.values()))
    branch_matched = _counter_intersection_count(pred_counter, branch_counter)
    decision_branch_recall = _safe_div(branch_matched, branch_total)

    matched_edge_pairs = _pair_matched_edges_for_geometry(pred_edges, gt_edges, pred_pairs_mapped, gt_pairs)
    path_ious: List[float] = []
    path_f1s: List[float] = []
    tail_errors: List[float] = []
    head_errors: List[float] = []
    for pred_edge, gt_edge in matched_edge_pairs:
        pred_path = _path_points(pred_edge)
        gt_path = _path_points(gt_edge)
        if pred_path and gt_path:
            pm = _rasterize_polyline(pred_path, gt_w, gt_h, path_tolerance_px)
            gm = _rasterize_polyline(gt_path, gt_w, gt_h, path_tolerance_px)
            mr = _mask_prf_iou(pm, gm)
            path_ious.append(mr["iou"])
            path_f1s.append(mr["f1"])
        pt = _point_from_hint(pred_edge, "tail_hint")
        gt = _point_from_hint(gt_edge, "tail_hint")
        ph = _point_from_hint(pred_edge, "head_hint")
        gh = _point_from_hint(gt_edge, "head_hint")
        if pt is not None and gt is not None:
            tail_errors.append(l2(pt, gt))
        if ph is not None and gh is not None:
            head_errors.append(l2(ph, gh))

    edit_cost = node_fp + node_fn + edge_fp + edge_fn
    approx_nsee = _safe_div(edit_cost, max(1, len(gt_nodes) + len(gt_edges)))

    return {
        "gt_nodes": len(gt_nodes),
        "pred_nodes": len(pred_nodes),
        "node_tp": node_tp,
        "node_fp": node_fp,
        "node_fn": node_fn,
        "node_precision": node_precision,
        "node_recall": node_recall,
        "node_f1": node_f1,
        "node_mean_iou": float(np.mean(matched_ious)) if matched_ious else 0.0,
        "node_center_mae_px": float(np.mean(center_errors)) if center_errors else 0.0,
        "gt_edges": len(gt_edges),
        "pred_edges": len(pred_edges),
        "edge_tp": edge_tp,
        "edge_fp": edge_fp,
        "edge_fn": edge_fn,
        "edge_precision": edge_precision,
        "edge_recall": edge_recall,
        "edge_f1": edge_f1,
        "undirected_edge_precision": undir_precision,
        "undirected_edge_recall": undir_recall,
        "undirected_edge_f1": undir_f1,
        "direction_proxy": direction_proxy,
        "gt_self_loops": gt_loop_total,
        "matched_self_loops": matched_loop,
        "loop_back_recall": loop_recall,
        "gt_branch_edges": branch_total,
        "matched_branch_edges": branch_matched,
        "decision_branch_recall": decision_branch_recall,
        "matched_edge_geometry_count": len(matched_edge_pairs),
        "path_mean_iou": float(np.mean(path_ious)) if path_ious else 0.0,
        "path_mean_f1": float(np.mean(path_f1s)) if path_f1s else 0.0,
        "tail_endpoint_mae_px": float(np.mean(tail_errors)) if tail_errors else 0.0,
        "head_endpoint_mae_px": float(np.mean(head_errors)) if head_errors else 0.0,
        "approx_nsee": approx_nsee,
        "structural_edit_cost": edit_cost,
    }


def aggregate_metric_rows(rows: Sequence[Dict[str, Any]], group_keys: Sequence[str] = ()) -> List[Dict[str, Any]]:
    groups: Dict[Tuple[Any, ...], List[Dict[str, Any]]] = {}
    for r in rows:
        key = tuple(r.get(k, "") for k in group_keys)
        groups.setdefault(key, []).append(r)

    out: List[Dict[str, Any]] = []
    for key, items in sorted(groups.items(), key=lambda kv: kv[0]):
        s = lambda name: sum(float(x.get(name, 0.0)) for x in items)
        node_tp, node_fp, node_fn = int(s("node_tp")), int(s("node_fp")), int(s("node_fn"))
        edge_tp, edge_fp, edge_fn = int(s("edge_tp")), int(s("edge_fp")), int(s("edge_fn"))
        node_p, node_r, node_f = _prf(node_tp, node_fp, node_fn)
        edge_p, edge_r, edge_f = _prf(edge_tp, edge_fp, edge_fn)
        undir_tp = sum(float(x.get("undirected_edge_recall", 0.0)) * float(x.get("gt_edges", 0.0)) for x in items)
        # For direction, the most stable aggregate is matched directed edges divided by matched undirected edges.
        # Reconstruct matched undirected count from direction proxy where available.
        undir_match = 0.0
        for x in items:
            dp = float(x.get("direction_proxy", 0.0))
            if dp > 0:
                undir_match += float(x.get("edge_tp", 0.0)) / dp
        row = {"num_images": len(items)}
        for k, v in zip(group_keys, key):
            row[k] = v
        row.update({
            "gt_nodes": int(s("gt_nodes")),
            "pred_nodes": int(s("pred_nodes")),
            "node_precision": node_p,
            "node_recall": node_r,
            "node_f1": node_f,
            "node_mean_iou": float(np.mean([float(x.get("node_mean_iou", 0.0)) for x in items])),
            "node_center_mae_px": float(np.mean([float(x.get("node_center_mae_px", 0.0)) for x in items])),
            "gt_edges": int(s("gt_edges")),
            "pred_edges": int(s("pred_edges")),
            "edge_precision": edge_p,
            "edge_recall": edge_r,
            "edge_f1": edge_f,
            "direction_proxy": _safe_div(edge_tp, undir_match),
            "gt_self_loops": int(s("gt_self_loops")),
            "matched_self_loops": int(s("matched_self_loops")),
            "loop_back_recall": _safe_div(s("matched_self_loops"), s("gt_self_loops")),
            "gt_branch_edges": int(s("gt_branch_edges")),
            "matched_branch_edges": int(s("matched_branch_edges")),
            "decision_branch_recall": _safe_div(s("matched_branch_edges"), s("gt_branch_edges")),
            "path_mean_iou": float(np.mean([float(x.get("path_mean_iou", 0.0)) for x in items])),
            "path_mean_f1": float(np.mean([float(x.get("path_mean_f1", 0.0)) for x in items])),
            "tail_endpoint_mae_px": float(np.mean([float(x.get("tail_endpoint_mae_px", 0.0)) for x in items])),
            "head_endpoint_mae_px": float(np.mean([float(x.get("head_endpoint_mae_px", 0.0)) for x in items])),
            "approx_nsee": _safe_div(s("structural_edit_cost"), max(1.0, s("gt_nodes") + s("gt_edges"))),
            "structural_edit_cost": int(s("structural_edit_cost")),
        })
        out.append(row)
    return out


def write_csv_rows(csv_path: Path, rows: Sequence[Dict[str, Any]]) -> None:
    if not rows:
        return
    fieldnames: List[str] = []
    for r in rows:
        for k in r.keys():
            if k not in fieldnames:
                fieldnames.append(k)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow(r)


# -----------------------------------------------------------------------------
# Image loading / batch runner
# -----------------------------------------------------------------------------

def find_images(input_dir: Path) -> List[Path]:
    return sorted([p for p in input_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])


def load_image_for_model(
    image_path: Path,
    target_w: int,
    target_h: int,
    image_mode: str,
    normalize_image: bool,
    use_otsu: bool,
) -> Tuple[np.ndarray, np.ndarray, torch.Tensor]:
    original_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if original_bgr is None:
        raise ValueError(f"Failed to read image: {image_path}")

    if image_mode.lower() == "grayscale":
        gray = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2GRAY)
        resized = cv2.resize(gray, (target_w, target_h), interpolation=cv2.INTER_LINEAR)
        if use_otsu:
            _, resized = cv2.threshold(resized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        arr = resized.astype(np.float32)
        if normalize_image:
            arr /= 255.0
        tensor = torch.from_numpy(arr[None, None, :, :].astype(np.float32))
        decoder_img = resized
    elif image_mode.lower() == "rgb":
        rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (target_w, target_h), interpolation=cv2.INTER_LINEAR)
        arr = resized.astype(np.float32)
        if normalize_image:
            arr /= 255.0
        tensor = torch.from_numpy(np.transpose(arr, (2, 0, 1))[None].astype(np.float32))
        decoder_img = resized
    else:
        raise ValueError(f"Unsupported image_mode={image_mode!r}")

    return original_bgr, decoder_img, tensor


def build_decoder_cfg(args) -> LADDecoderConfig:
    return LADDecoderConfig(
        node_extent_thresh=args.node_extent_thresh,
        node_center_thresh=args.node_center_thresh,
        min_node_area=args.min_node_area,
        head_thresh=args.head_thresh,
        tail_thresh=args.tail_thresh,
        shaft_thresh=args.shaft_thresh,
        skeleton_thresh=args.skeleton_thresh,
        support_thresh=args.support_thresh,
        endpoint_radius=args.endpoint_radius,
        node_attach_max_dist=args.node_attach_max_dist,
        max_pair_dist=args.max_pair_dist,
        max_expansions=args.max_expansions,
        min_arrow_confidence=args.min_arrow_confidence,
        allow_endpoint_reuse=args.allow_endpoint_reuse,
        branch_aware_endpoint_reuse=args.branch_aware_endpoint_reuse,
        branch_min_opposite_endpoint_dist=args.branch_min_opposite_endpoint_dist,
        branch_max_path_iou=args.branch_max_path_iou,
        branch_require_different_node_pair=args.branch_require_different_node_pair,
        keep_unattached_arrows=args.keep_unattached_arrows,
        use_image_ink_support=args.use_image_ink_support,
        enable_port_role_repair=args.enable_port_role_repair,
        port_repair_score_scale=args.port_repair_score_scale,
        port_repair_confidence_scale=args.port_repair_confidence_scale,
        port_repair_min_confidence=args.port_repair_min_confidence,
        port_repair_progress_cost_weight=args.port_repair_progress_cost_weight,
        port_repair_max_path_iou=args.port_repair_max_path_iou,
        port_repair_min_support_score=args.port_repair_min_support_score,
        port_repair_min_shaft_score=args.port_repair_min_shaft_score,
        port_repair_max_candidates=args.port_repair_max_candidates,
        auto_adapt=args.auto_adapt,
        enable_stroke_graph_candidates=args.enable_stroke_graph_candidates,
        stroke_graph_min_component_area=args.stroke_graph_min_component_area,
        stroke_graph_min_path_len=args.stroke_graph_min_path_len,
        stroke_graph_min_confidence=args.stroke_graph_min_confidence,
        stroke_graph_max_path_iou=args.stroke_graph_max_path_iou,
        stroke_graph_max_components=args.stroke_graph_max_components,
        stroke_graph_max_candidates=args.stroke_graph_max_candidates,
        stroke_graph_skip_self_loops=args.stroke_graph_skip_self_loops,
        enable_ink_bridge_candidates=args.enable_ink_bridge_candidates,
        ink_bridge_min_confidence=args.ink_bridge_min_confidence,
        ink_bridge_confidence_scale=args.ink_bridge_confidence_scale,
        ink_bridge_min_support_score=args.ink_bridge_min_support_score,
        ink_bridge_min_width=args.ink_bridge_min_width,
        ink_bridge_max_width=args.ink_bridge_max_width,
        ink_bridge_width_factor=args.ink_bridge_width_factor,
        ink_bridge_threshold=args.ink_bridge_threshold,
        ink_bridge_max_candidates=args.ink_bridge_max_candidates,
        ink_bridge_progress_cost_weight=args.ink_bridge_progress_cost_weight,
        ink_bridge_support_cost_weight=args.ink_bridge_support_cost_weight,
        ink_bridge_allow_self_loops=args.ink_bridge_allow_self_loops,
        verbose=args.verbose,
    )


@torch.no_grad()
def run_inference(args) -> None:
    dataset_root = Path(args.dataset_root) if args.dataset_root else None
    input_dir = Path(args.input_dir)
    output_dir = Path(args.output_dir)
    checkpoint_path = Path(args.checkpoint)
    output_dir.mkdir(parents=True, exist_ok=True)

    if dataset_root is not None and not dataset_root.exists():
        raise FileNotFoundError(f"Dataset root not found: {dataset_root}")
    if dataset_root is None and not input_dir.exists():
        raise FileNotFoundError(f"Input dir not found: {input_dir}")
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    families = _parse_csv_list(args.families)
    splits = _parse_csv_list(args.splits)
    require_gt = bool(args.evaluate)

    if dataset_root is not None:
        samples = discover_converted_dataset_samples(
            dataset_root=dataset_root,
            families=families,
            splits=splits,
            require_gt=require_gt,
        )
    else:
        gt_json_dir = Path(args.gt_json_dir) if args.gt_json_dir else None
        samples = discover_flat_samples(input_dir=input_dir, gt_json_dir=gt_json_dir, require_gt=require_gt)

    if not samples:
        print("No samples found.")
        return

    device = torch.device(args.device if args.device else ("cuda" if torch.cuda.is_available() else "cpu"))
    print(f"Loading checkpoint: {checkpoint_path}")
    print(f"Device: {device}")
    print(f"Found {len(samples)} samples")
    if dataset_root is not None:
        print(f"Dataset root: {dataset_root}")
        print(f"Families: {families or 'all'} | Splits: {splits or 'all'}")

    model, load_info = load_checkpoint_model(
        checkpoint_path=checkpoint_path,
        device=device,
        image_mode=args.image_mode,
        base_ch=args.base_ch,
        norm_type=args.norm_type,
        strict=not args.allow_partial_checkpoint,
    )

    cfg = build_decoder_cfg(args)
    save_json(output_dir / "run_config.json", vars(args))
    save_json(output_dir / "checkpoint_load_info.json", load_info)
    save_json(output_dir / "decoder_config.json", asdict(cfg))

    success_csv = output_dir / "success_log.csv"
    failure_csv = output_dir / "failure_log.csv"
    metric_rows: List[Dict[str, Any]] = []
    n_ok = 0
    n_fail = 0

    for idx, sample in enumerate(samples, start=1):
        image_path = sample.image_path
        stem = image_path.stem
        out_dir = sample_output_dir(output_dir, sample, preserve_structure=args.preserve_dataset_structure)
        out_dir.mkdir(parents=True, exist_ok=True)
        rel_label = f"{sample.family}/{sample.split}/{image_path.name}" if sample.family else image_path.name
        print(f"[{idx}/{len(samples)}] {rel_label}")
        try:
            original_bgr, decoder_img, image_tensor = load_image_for_model(
                image_path=image_path,
                target_w=args.target_w,
                target_h=args.target_h,
                image_mode=args.image_mode,
                normalize_image=args.normalize_image,
                use_otsu=args.use_otsu,
            )
            image_tensor = image_tensor.to(device)
            outputs = model(image_tensor)
            required = {
                "node_center", "node_extent", "shaft", "skeleton", "head", "tail", "dir",
                "progress", "head_offset", "tail_offset", "head_tangent", "tail_tangent",
            }
            missing = sorted(required - set(outputs.keys()))
            if missing:
                raise RuntimeError(f"Model did not return required heads: {missing}")

            pred = outputs_to_pred_maps(outputs)
            if args.save_pred_maps:
                save_prediction_maps(out_dir, stem, pred)

            orig_h, orig_w = original_bgr.shape[:2]
            result = decode_graph_learning_aligned(
                pred=pred,
                decoder_img=decoder_img,
                image_name=image_path.name,
                cfg=cfg,
                orig_size=(orig_w, orig_h) if args.save_original_space_json else None,
            )

            save_decode_artifacts(out_dir, stem, decoder_img, result, save_maps=args.save_stage_maps)
            save_json(out_dir / f"{stem}_meta.json", {
                "image_name": image_path.name,
                "image_path": str(image_path),
                "json_path": str(sample.json_path) if sample.json_path is not None else "",
                "family": sample.family,
                "split": sample.split,
                "orig_w": int(orig_w),
                "orig_h": int(orig_h),
                "model_w": int(args.target_w),
                "model_h": int(args.target_h),
                "image_mode": args.image_mode,
            })

            metric_row: Dict[str, Any] = {}
            if args.evaluate and sample.json_path is not None:
                with open(sample.json_path, "r", encoding="utf-8") as f:
                    gt_writer = json.load(f)
                pred_writer_for_eval = result.writer_style_original_json or result.writer_style_json
                metric_row = evaluate_graph_prediction(
                    pred_writer=pred_writer_for_eval,
                    gt_writer=gt_writer,
                    node_iou_thresh=args.eval_node_iou_thresh,
                    path_tolerance_px=args.eval_path_tolerance_px,
                )
                metric_row.update({
                    "image_name": image_path.name,
                    "sample_id": sample.sample_id,
                    "family": sample.family,
                    "split": sample.split,
                    "json_path": str(sample.json_path),
                })
                metric_rows.append(metric_row)
                save_json(out_dir / f"{stem}_metrics.json", metric_row)

            success_row = {
                "image_name": image_path.name,
                "sample_id": sample.sample_id,
                "family": sample.family,
                "split": sample.split,
                "num_nodes": len(result.nodes),
                "num_heads": len(result.heads),
                "num_tails": len(result.tails),
                "num_candidates": len(result.candidate_arrows),
                "num_arrows": len(result.arrows),
                "writer_model_json": str((out_dir / f"{stem}_writer_model.json").resolve()),
                "writer_original_json": str((out_dir / f"{stem}_writer_original.json").resolve()) if result.writer_style_original_json is not None else "",
                "debug_json": str((out_dir / f"{stem}_debug.json").resolve()),
                "final_viz": str((out_dir / f"{stem}_viz_final_graph.png").resolve()),
            }
            if metric_row:
                for k in ["node_f1", "edge_f1", "direction_proxy", "loop_back_recall", "decision_branch_recall", "path_mean_f1", "approx_nsee"]:
                    success_row[k] = metric_row.get(k, "")
            append_csv_row(
                success_csv,
                list(success_row.keys()),
                success_row,
            )
            n_ok += 1
        except Exception as e:
            tb = traceback.format_exc(limit=60)
            print(f"  FAILED: {rel_label} -> {e}")
            append_csv_row(
                failure_csv,
                ["image_name", "sample_id", "family", "split", "error", "traceback"],
                {
                    "image_name": image_path.name,
                    "sample_id": sample.sample_id,
                    "family": sample.family,
                    "split": sample.split,
                    "error": str(e),
                    "traceback": tb,
                },
            )
            with open(out_dir / f"{stem}_traceback.txt", "w", encoding="utf-8") as f:
                f.write(tb)
            n_fail += 1

    if metric_rows:
        write_csv_rows(output_dir / "metrics_per_image.csv", metric_rows)
        overall = aggregate_metric_rows(metric_rows, group_keys=[])[0]
        by_family = aggregate_metric_rows(metric_rows, group_keys=["family"])
        by_split = aggregate_metric_rows(metric_rows, group_keys=["split"])
        by_family_split = aggregate_metric_rows(metric_rows, group_keys=["family", "split"])
        write_csv_rows(output_dir / "metrics_by_family.csv", by_family)
        write_csv_rows(output_dir / "metrics_by_split.csv", by_split)
        write_csv_rows(output_dir / "metrics_by_family_split.csv", by_family_split)
        save_json(output_dir / "metrics_summary.json", {
            "overall": overall,
            "by_family": by_family,
            "by_split": by_split,
            "by_family_split": by_family_split,
            "metric_notes": {
                "node_f1": "Node detection after greedy bbox-IoU matching.",
                "edge_f1": "Directed graph edge F1 after mapping predicted nodes to matched GT node ids. Multi-edges are counted as multisets.",
                "direction_proxy": "Directed matches divided by undirected matches; measures whether connected node pairs have the correct orientation.",
                "loop_back_recall": "Recall over GT self-loop edges.",
                "decision_branch_recall": "Recall over GT outgoing edges from nodes with out-degree >= 2.",
                "path_mean_f1/path_mean_iou": "Raster overlap of matched arrow paths with tolerance thickness.",
                "approx_nsee": "Normalized structural edit proxy: (node FP+FN+edge FP+FN)/(GT nodes+GT edges).",
            },
        })
        print("\nEvaluation summary:")
        print(json.dumps(overall, indent=2))

    print("\nDone.")
    print(f"Output dir: {output_dir}")
    print(f"Success={n_ok}, Failed={n_fail}")


# -----------------------------------------------------------------------------
# Notebook-native inference defaults
# -----------------------------------------------------------------------------
# This unified hosted-runtime notebook intentionally has NO command-line interface.
# All user-facing settings live in the MASTER CONFIG cell.
#
# The object below only preserves the frozen decoder defaults from the original
# standalone script so the paper-compatible assembler can be reused without
# argparse or --flags.

def default_inference_args():
    return SimpleNamespace(
        # Dataset / output placeholders. The notebook runner overrides these
        # from the MASTER CONFIG when required.
        input_dir="",
        dataset_root=str(DATASET_ROOT),
        families=",".join(DATASET_FAMILIES),
        splits=",".join(INFERENCE_SPLITS),
        gt_json_dir=None,
        evaluate=False,
        eval_node_iou_thresh=0.50,
        eval_path_tolerance_px=3,
        preserve_dataset_structure=True,
        checkpoint=str(GRAPH_MODEL_CHECKPOINT),
        output_dir=str(OUTPUT_ROOT),

        # Model/input settings
        image_mode="grayscale",
        normalize_image=True,
        use_otsu=False,
        target_w=512,
        target_h=512,
        base_ch=0,
        norm_type="auto",
        device=None,
        allow_partial_checkpoint=False,

        # Decoder thresholds
        node_center_thresh=0.25,
        node_extent_thresh=0.20,
        min_node_area=35,
        head_thresh=0.25,
        tail_thresh=0.25,
        shaft_thresh=0.20,
        skeleton_thresh=0.20,
        support_thresh=0.20,
        endpoint_radius=8.0,
        node_attach_max_dist=28.0,
        max_pair_dist=1e9,
        max_expansions=90000,
        min_arrow_confidence=0.32,

        # Endpoint reuse / branch handling
        allow_endpoint_reuse=False,
        branch_aware_endpoint_reuse=True,
        branch_min_opposite_endpoint_dist=16.0,
        branch_max_path_iou=0.62,
        branch_require_different_node_pair=True,
        keep_unattached_arrows=True,
        use_image_ink_support=False,

        # Port-role repair
        enable_port_role_repair=True,
        port_repair_score_scale=0.62,
        port_repair_confidence_scale=0.82,
        port_repair_min_confidence=0.46,
        port_repair_progress_cost_weight=0.20,
        port_repair_max_path_iou=0.52,
        port_repair_min_support_score=0.72,
        port_repair_min_shaft_score=0.72,
        port_repair_max_candidates=12,

        # Adaptive / stroke-graph candidates
        auto_adapt=True,
        enable_stroke_graph_candidates=True,
        stroke_graph_min_component_area=18,
        stroke_graph_min_path_len=12,
        stroke_graph_min_confidence=0.34,
        stroke_graph_max_path_iou=0.78,
        stroke_graph_max_components=96,
        stroke_graph_max_candidates=80,
        stroke_graph_skip_self_loops=False,

        # Ink-bridge candidates
        enable_ink_bridge_candidates=True,
        ink_bridge_min_confidence=0.40,
        ink_bridge_confidence_scale=0.88,
        ink_bridge_min_support_score=0.42,
        ink_bridge_min_width=10,
        ink_bridge_max_width=32,
        ink_bridge_width_factor=0.10,
        ink_bridge_threshold=0.18,
        ink_bridge_max_candidates=80,
        ink_bridge_progress_cost_weight=0.28,
        ink_bridge_support_cost_weight=1.35,
        ink_bridge_allow_self_loops=True,

        # Saving / diagnostics
        save_pred_maps=False,
        save_stage_maps=False,
        save_original_space_json=True,
        verbose=False,
    )


In [ ]:
# ============================================================
# PLAIN RESNET34 U-NET NODE SEGMENTER
# ============================================================

try:
    import torchvision.models as tvm
except Exception as exc:
    tvm = None
    print("torchvision import failed:", exc)

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def node_conv_bn_relu(in_ch, out_ch, k=3, s=1, p=1):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class NodeUpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv1 = node_conv_bn_relu(out_ch + skip_ch, out_ch)
        self.conv2 = node_conv_bn_relu(out_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.size(-1) != skip.size(-1) or x.size(-2) != skip.size(-2):
            diff_y = skip.size(-2) - x.size(-2)
            diff_x = skip.size(-1) - x.size(-1)
            x = F.pad(
                x,
                [
                    diff_x // 2,
                    diff_x - diff_x // 2,
                    diff_y // 2,
                    diff_y - diff_y // 2,
                ],
            )
        x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x


class ResNet34NodeUNet(nn.Module):
    """Plain ResNet34 encoder with a U-Net-style node-region decoder."""

    def __init__(self, out_ch=1, pretrained=False):
        super().__init__()
        if tvm is None:
            raise ImportError("torchvision is required for ResNet34NodeUNet.")

        weights = tvm.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        r34 = tvm.resnet34(weights=weights)

        self.conv1 = r34.conv1
        self.bn1 = r34.bn1
        self.relu = r34.relu
        self.maxpool = r34.maxpool

        self.layer1 = r34.layer1
        self.layer2 = r34.layer2
        self.layer3 = r34.layer3
        self.layer4 = r34.layer4

        self.bot = node_conv_bn_relu(512, 512)
        self.up4 = NodeUpBlock(512, 256, 256)
        self.up3 = NodeUpBlock(256, 128, 128)
        self.up2 = NodeUpBlock(128, 64, 64)
        self.up1 = NodeUpBlock(64, 64, 64)
        self.head = nn.Conv2d(64, out_ch, kernel_size=1)

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.layer1(self.maxpool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)

        bottleneck = self.bot(x4)
        d4 = self.up4(bottleneck, x3)
        d3 = self.up3(d4, x2)
        d2 = self.up2(d3, x1)
        d1 = self.up1(d2, x0)

        logits = self.head(d1)
        return F.interpolate(
            logits,
            size=(x.size(2), x.size(3)),
            mode="bilinear",
            align_corners=False,
        )


def _extract_node_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ("model", "model_state_dict", "state_dict"):
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]
        return checkpoint
    raise TypeError("Unsupported node checkpoint format.")


def _strip_data_parallel_prefix(state):
    return {
        key[7:] if str(key).startswith("module.") else key: value
        for key, value in state.items()
    }


def load_node_segmentation_model(checkpoint_path, device):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"NODE_MODEL_CHECKPOINT was not found: {checkpoint_path}"
        )

    checkpoint = torch.load(str(checkpoint_path), map_location="cpu")
    state = _strip_data_parallel_prefix(_extract_node_state_dict(checkpoint))


    model = ResNet34NodeUNet(out_ch=1, pretrained=False)
    model.load_state_dict(state, strict=True)
    model = model.to(device).eval()

    return model, {
        "checkpoint": str(checkpoint_path),
        "architecture": "plain_resnet34_unet",
        "strict": True,
        "state_keys": len(state),
    }


def load_node_model_input(image_path, img_size=320):
    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f"Could not read image for node model: {image_path}")

    orig_h, orig_w = gray.shape[:2]
    rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    rgb = cv2.resize(rgb, (int(img_size), int(img_size)), interpolation=cv2.INTER_LINEAR)

    array = rgb.astype(np.float32) / 255.0
    array = (
        array - IMAGENET_MEAN[None, None, :]
    ) / IMAGENET_STD[None, None, :]
    tensor = torch.from_numpy(
        np.transpose(array, (2, 0, 1))
    ).unsqueeze(0).float()

    return gray, tensor, orig_w, orig_h


@torch.no_grad()
def predict_node_probability(model, image_path, device, img_size=320):
    gray, tensor, orig_w, orig_h = load_node_model_input(
        image_path, img_size=img_size
    )
    logits = model(tensor.to(device))
    probability = torch.sigmoid(logits)[0, 0].detach().cpu().numpy().astype(np.float32)
    probability = cv2.resize(
        probability,
        (orig_w, orig_h),
        interpolation=cv2.INTER_LINEAR,
    )
    return gray, probability


def extract_node_boxes(probability, threshold=0.46, min_area=35, close_kernel=3):
    binary = (probability >= float(threshold)).astype(np.uint8)

    if close_kernel and int(close_kernel) > 1:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (int(close_kernel), int(close_kernel)),
        )
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )

    nodes = []
    for component_id in range(1, count):
        x, y, w, h, area = stats[component_id].tolist()
        if int(area) < int(min_area):
            continue

        component = labels == component_id
        nodes.append({
            "id": f"N{len(nodes)}",
            "bbox": [int(x), int(y), int(w), int(h)],
            "area": int(area),
            "score": float(probability[component].mean()),
        })

    return nodes, binary

## 6. Evaluation core: submitted compatibility + reviewer-oriented revision metrics

Node matching and directed-link matching are shared. The revision layer does **not** change what counts as a correct directed link; it makes conditional geometry coverage and aggregation explicit.

In [ ]:
# ============================================================
# PAPER EVALUATION METRICS
# ============================================================

from collections import Counter, defaultdict
try:
    from skimage.morphology import skeletonize as _sk_skeletonize
except Exception:
    _sk_skeletonize = None


def metric_tolerance_px(width, height, ratio=METRIC_TOLERANCE_RATIO):
    return int(max(2, round(float(ratio) * max(int(width), int(height)))))


def bbox_xywh_to_xyxy(b):
    x, y, w, h = [float(v) for v in b]
    return x, y, x + w, y + h


def bbox_iou_xywh(a, b):
    ax1, ay1, ax2, ay2 = bbox_xywh_to_xyxy(a)
    bx1, by1, bx2, by2 = bbox_xywh_to_xyxy(b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    aa = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    bb = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    return float(inter / max(aa + bb - inter, 1e-9))


def bbox_center_xy(node):
    x, y, w, h = [float(v) for v in node.get("bbox", [0, 0, 0, 0])]
    return (x + w / 2.0, y + h / 2.0)


def tolerant_match_nodes(pred_nodes, gt_nodes, width, height, iou_accept=NODE_IOU_ACCEPT):
    tol = metric_tolerance_px(width, height)
    candidates = []
    for pi, pn in enumerate(pred_nodes):
        pc = bbox_center_xy(pn)
        for gi, gn in enumerate(gt_nodes):
            gc = bbox_center_xy(gn)
            iou = bbox_iou_xywh(pn.get("bbox", [0, 0, 0, 0]), gn.get("bbox", [0, 0, 0, 0]))
            cd = l2(pc, gc)
            gt_diag = math.hypot(float(gn.get("bbox", [0, 0, 1, 1])[2]), float(gn.get("bbox", [0, 0, 1, 1])[3]))
            center_tol = max(float(tol), 0.18 * gt_diag)
            if iou >= iou_accept or cd <= center_tol:
                # lower score is better
                score = (cd / max(center_tol, 1e-6)) + (1.0 - iou)
                candidates.append((score, -iou, cd, pi, gi, iou))
    candidates.sort()
    used_p, used_g = set(), set()
    matches = []
    pred_to_gt = {}
    for score, neg_iou, cd, pi, gi, iou in candidates:
        if pi in used_p or gi in used_g:
            continue
        used_p.add(pi); used_g.add(gi)
        pid = str(pred_nodes[pi].get("id", f"P{pi}"))
        gid = str(gt_nodes[gi].get("id", f"G{gi}"))
        pred_to_gt[pid] = gid
        matches.append((pid, gid, float(iou), float(cd)))
    return pred_to_gt, matches


def edge_pair(edge):
    return (edge.get("tail_node_id"), edge.get("head_node_id"))


def mapped_edge_pair(edge, pred_to_gt):
    return (pred_to_gt.get(str(edge.get("tail_node_id"))), pred_to_gt.get(str(edge.get("head_node_id"))))


def pair_counter(pairs):
    return Counter(tuple(p) for p in pairs)


def counter_intersection_count(a, b):
    return int(sum(min(v, b.get(k, 0)) for k, v in a.items()))


def path_points(edge):
    pts = edge.get("path", []) or []
    out = []
    for p in pts:
        if isinstance(p, dict) and "x" in p and "y" in p:
            out.append((float(p["x"]), float(p["y"])))
        elif isinstance(p, (list, tuple)) and len(p) >= 2:
            out.append((float(p[0]), float(p[1])))
    return out


def point_from_edge(edge, plain, hint):
    p = edge.get(plain)
    if isinstance(p, dict) and "x" in p and "y" in p:
        return (float(p["x"]), float(p["y"]))
    p = edge.get(hint)
    if isinstance(p, dict) and "x" in p and "y" in p:
        return (float(p["x"]), float(p["y"]))
    return None


def rasterize_writer_graph(writer, width, height, path_thickness=3, include_nodes=True, include_edges=True):
    mask = np.zeros((int(height), int(width)), dtype=np.uint8)
    if include_nodes:
        for node in writer.get("nodes", []) or []:
            x, y, w, h = [int(round(float(v))) for v in node.get("bbox", [0, 0, 0, 0])]
            x0, y0 = max(0, x), max(0, y)
            x1, y1 = min(int(width) - 1, x + w), min(int(height) - 1, y + h)
            if x1 >= x0 and y1 >= y0:
                cv2.rectangle(mask, (x0, y0), (x1, y1), 1, -1)
    if include_edges:
        for edge in writer.get("arrows", []) or []:
            pts = path_points(edge)
            if len(pts) >= 2:
                arr = np.array([[int(round(x)), int(round(y))] for x, y in pts], dtype=np.int32)
                cv2.polylines(mask, [arr], False, 1, int(path_thickness), lineType=cv2.LINE_AA)
            else:
                tail = point_from_edge(edge, "tail", "tail_hint")
                head = point_from_edge(edge, "head", "head_hint")
                if tail is not None and head is not None:
                    arr = np.array([[int(round(tail[0])), int(round(tail[1]))],
                                    [int(round(head[0])), int(round(head[1]))]], dtype=np.int32)
                    cv2.polylines(mask, [arr], False, 1, int(path_thickness), lineType=cv2.LINE_AA)
    return (mask > 0).astype(np.uint8)


def tolerant_mask_iou(pred_mask, gt_mask, tol_px):
    if tol_px > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * tol_px + 1, 2 * tol_px + 1))
        # symmetric relaxed IoU: allow small offsets without exploding false positives too badly
        p = cv2.dilate(pred_mask.astype(np.uint8), k, iterations=1)
        g = cv2.dilate(gt_mask.astype(np.uint8), k, iterations=1)
    else:
        p, g = pred_mask.astype(np.uint8), gt_mask.astype(np.uint8)
    inter = int(np.logical_and(p > 0, g > 0).sum())
    union = int(np.logical_or(p > 0, g > 0).sum())
    return _safe_div(inter, union)


def cldice_score(pred_mask, gt_mask, tol_px):
    pred = (pred_mask > 0).astype(np.uint8)
    gt = (gt_mask > 0).astype(np.uint8)
    if _sk_skeletonize is None:
        return 0.0
    sk_p = _sk_skeletonize(pred > 0).astype(np.uint8)
    sk_g = _sk_skeletonize(gt > 0).astype(np.uint8)
    if tol_px > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * tol_px + 1, 2 * tol_px + 1))
        gt_relaxed = cv2.dilate(gt, k, iterations=1)
        pred_relaxed = cv2.dilate(pred, k, iterations=1)
    else:
        gt_relaxed, pred_relaxed = gt, pred
    tprec = _safe_div(int(np.logical_and(sk_p > 0, gt_relaxed > 0).sum()), int((sk_p > 0).sum()))
    tsens = _safe_div(int(np.logical_and(sk_g > 0, pred_relaxed > 0).sum()), int((sk_g > 0).sum()))
    return _safe_div(2.0 * tprec * tsens, tprec + tsens)


def pair_matched_edges_for_geometry(pred_edges, gt_edges, pred_pairs_mapped, gt_pairs):
    gt_by_pair = defaultdict(list)
    for gi, pair in enumerate(gt_pairs):
        gt_by_pair[pair].append(gi)
    matched = []
    used_gt = set()
    for pred_edge, pair in zip(pred_edges, pred_pairs_mapped):
        choices = [i for i in gt_by_pair.get(pair, []) if i not in used_gt]
        if not choices:
            continue
        pt = point_from_edge(pred_edge, "tail", "tail_hint")
        ph = point_from_edge(pred_edge, "head", "head_hint")
        best_i, best_d = None, float("inf")
        for gi in choices:
            gt_edge = gt_edges[gi]
            gt = point_from_edge(gt_edge, "tail", "tail_hint")
            gh = point_from_edge(gt_edge, "head", "head_hint")
            d, c = 0.0, 0
            if pt is not None and gt is not None:
                d += l2(pt, gt); c += 1
            if ph is not None and gh is not None:
                d += l2(ph, gh); c += 1
            d = d / max(1, c)
            if d < best_d:
                best_d, best_i = d, gi
        if best_i is not None:
            used_gt.add(best_i)
            matched.append((pred_edge, gt_edges[best_i], best_d))
    return matched


def evaluate_graph_prediction_tolerant(pred_writer, gt_writer):
    gt_size = gt_writer.get("size", {}) or {}
    gt_w = int(gt_size.get("w", pred_writer.get("size", {}).get("w", 1)))
    gt_h = int(gt_size.get("h", pred_writer.get("size", {}).get("h", 1)))
    tol = metric_tolerance_px(gt_w, gt_h)
    pred_writer = _rescale_writer_to_size(pred_writer, gt_w, gt_h)

    pred_nodes = pred_writer.get("nodes", []) or []
    gt_nodes = gt_writer.get("nodes", []) or []
    pred_edges = pred_writer.get("arrows", []) or []
    gt_edges = gt_writer.get("arrows", []) or []

    pred_to_gt, node_matches = tolerant_match_nodes(pred_nodes, gt_nodes, gt_w, gt_h)
    node_tp = len(node_matches)
    node_fp = len(pred_nodes) - node_tp
    node_fn = len(gt_nodes) - node_tp
    node_precision, node_recall, node_f1 = _prf(node_tp, node_fp, node_fn)

    gt_pairs = [edge_pair(e) for e in gt_edges]
    pred_pairs_mapped = [mapped_edge_pair(e, pred_to_gt) for e in pred_edges]
    gt_counter = pair_counter(gt_pairs)
    pred_counter = pair_counter(pred_pairs_mapped)

    edge_tp = counter_intersection_count(pred_counter, gt_counter)
    edge_fp = len(pred_edges) - edge_tp
    edge_fn = len(gt_edges) - edge_tp
    edge_precision, edge_recall, edge_f1 = _prf(edge_tp, edge_fp, edge_fn)

    # Link_F1 is a paper-friendly alias for directed edge F1.
    link_precision, link_recall, link_f1 = edge_precision, edge_recall, edge_f1

    # endpoint error on matched directed edges only
    endpoint_errors = []
    path_ious = []
    path_f1s = []
    path_cldices = []
    matched_geom = pair_matched_edges_for_geometry(pred_edges, gt_edges, pred_pairs_mapped, gt_pairs)
    for pe, ge, _ in matched_geom:
        for plain, hint in [("tail", "tail_hint"), ("head", "head_hint")]:
            pp = point_from_edge(pe, plain, hint)
            gp = point_from_edge(ge, plain, hint)
            if pp is not None and gp is not None:
                endpoint_errors.append(l2(pp, gp))
        pred_path = path_points(pe)
        gt_path = path_points(ge)

        # Use endpoint-to-endpoint fallback lines only when an edge has no routed path.
        if len(pred_path) < 2:
            pred_tail = point_from_edge(pe, "tail", "tail_hint")
            pred_head = point_from_edge(pe, "head", "head_hint")
            pred_path = [pred_tail, pred_head] if pred_tail is not None and pred_head is not None else []
        if len(gt_path) < 2:
            gt_tail = point_from_edge(ge, "tail", "tail_hint")
            gt_head = point_from_edge(ge, "head", "head_hint")
            gt_path = [gt_tail, gt_head] if gt_tail is not None and gt_head is not None else []

        if len(pred_path) >= 2 and len(gt_path) >= 2:
            # Thick relaxed masks retain the historical path-overlap diagnostics.
            pm = np.zeros((gt_h, gt_w), dtype=np.uint8)
            gm = np.zeros((gt_h, gt_w), dtype=np.uint8)
            cv2.polylines(
                pm,
                [np.array([[int(round(x)), int(round(y))] for x, y in pred_path], dtype=np.int32)],
                False,
                1,
                max(PATH_BASE_TOLERANCE_PX, tol),
                lineType=cv2.LINE_AA,
            )
            cv2.polylines(
                gm,
                [np.array([[int(round(x)), int(round(y))] for x, y in gt_path], dtype=np.int32)],
                False,
                1,
                max(PATH_BASE_TOLERANCE_PX, tol),
                lineType=cv2.LINE_AA,
            )
            inter = int(np.logical_and(pm > 0, gm > 0).sum())
            pcount = int((pm > 0).sum())
            gcount = int((gm > 0).sum())
            prec = _safe_div(inter, pcount)
            rec = _safe_div(inter, gcount)
            path_f1s.append(_safe_div(2 * prec * rec, prec + rec))
            path_ious.append(
                _safe_div(inter, int(np.logical_or(pm > 0, gm > 0).sum()))
            )

            # Soft-clDice is isolated to matched arrow centerlines only.
            pred_centerline = np.zeros((gt_h, gt_w), dtype=np.uint8)
            gt_centerline = np.zeros((gt_h, gt_w), dtype=np.uint8)
            cv2.polylines(
                pred_centerline,
                [np.array([[int(round(x)), int(round(y))] for x, y in pred_path], dtype=np.int32)],
                False,
                1,
                max(1, PATH_BASE_TOLERANCE_PX),
                lineType=cv2.LINE_AA,
            )
            cv2.polylines(
                gt_centerline,
                [np.array([[int(round(x)), int(round(y))] for x, y in gt_path], dtype=np.int32)],
                False,
                1,
                max(1, PATH_BASE_TOLERANCE_PX),
                lineType=cv2.LINE_AA,
            )
            path_cldices.append(
                cldice_score(pred_centerline, gt_centerline, tol_px=tol)
            )

    # Matched Node Box IoU is computed only over one-to-one spatially matched nodes.
    node_ious = [m[2] for m in node_matches]
    node_center_errors = [m[3] for m in node_matches]
    matched_box_iou = float(np.mean(node_ious)) if node_ious else 0.0

    # Detection errors remain represented by Link F1; this geometry score is
    # averaged only across matched directed connectors.
    cldice = float(np.mean(path_cldices)) if path_cldices else 0.0

    gt_loop_counter = Counter(k for k in gt_pairs if k[0] is not None and k[0] == k[1])
    pred_loop_counter = Counter(k for k in pred_pairs_mapped if k[0] is not None and k[0] == k[1])
    matched_loop = counter_intersection_count(pred_loop_counter, gt_loop_counter)

    out_degree = defaultdict(int)
    for t, h in gt_pairs:
        if t is not None:
            out_degree[t] += 1
    branch_sources = {nid for nid, deg in out_degree.items() if deg >= 2}
    branch_counter = Counter(p for p in gt_pairs if p[0] in branch_sources)
    branch_matched = counter_intersection_count(pred_counter, branch_counter)

    # Legacy structural-error normalization (reported as nSEE in v2):
    # GED / (max(|V_pred|, |V_gt|) + max(|E_pred|, |E_gt|)).
    # This normalizes vertices and edges independently so dense graphs do not
    # eclipse sparse graphs through a denominator based only on one side.
    edit_cost = node_fp + node_fn + edge_fp + edge_fn
    normalizer = max(1, max(len(pred_nodes), len(gt_nodes)) + max(len(pred_edges), len(gt_edges)))
    nsee = _safe_div(edit_cost, normalizer)

    return {
        "gt_nodes": len(gt_nodes),
        "pred_nodes": len(pred_nodes),
        "node_tp": node_tp,
        "node_fp": node_fp,
        "node_fn": node_fn,
        "node_precision": node_precision,
        "node_recall": node_recall,
        "node_f1": node_f1,
        "node_mean_iou": float(np.mean(node_ious)) if node_ious else 0.0,
        "node_center_mae_px": float(np.mean(node_center_errors)) if node_center_errors else 0.0,

        "gt_edges": len(gt_edges),
        "pred_edges": len(pred_edges),
        "edge_tp": edge_tp,
        "edge_fp": edge_fp,
        "edge_fn": edge_fn,
        "edge_precision": edge_precision,
        "edge_recall": edge_recall,
        "edge_f1": edge_f1,

        "link_precision": link_precision,
        "link_recall": link_recall,
        "link_f1": link_f1,

        "matched_box_iou": matched_box_iou,
        "matched_node_box_iou": matched_box_iou,
        "cldice": cldice,
        "soft_cldice": cldice,
        "arrow_soft_cldice": cldice,
        "endpoint_l2": float(np.mean(endpoint_errors)) if endpoint_errors else 0.0,

        "path_mean_iou": float(np.mean(path_ious)) if path_ious else 0.0,
        "path_mean_f1": float(np.mean(path_f1s)) if path_f1s else 0.0,

        "gt_self_loops": int(sum(gt_loop_counter.values())),
        "matched_self_loops": int(matched_loop),
        "loop_back_recall": _safe_div(matched_loop, sum(gt_loop_counter.values())),

        "gt_branch_edges": int(sum(branch_counter.values())),
        "matched_branch_edges": int(branch_matched),
        "decision_branch_recall": _safe_div(branch_matched, sum(branch_counter.values())),

        "nsee": nsee,
                "approx_nsee": nsee,
        "structural_edit_cost": edit_cost,
        "metric_tolerance_px": tol,
    }


def aggregate_metric_rows_tolerant(rows, group_keys=()):
    groups = defaultdict(list)
    for r in rows:
        groups[tuple(r.get(k, "") for k in group_keys)].append(r)

    out = []
    for key, items in sorted(groups.items(), key=lambda kv: kv[0]):
        s = lambda name: sum(float(x.get(name, 0.0)) for x in items)
        node_p, node_r, node_f = _prf(int(s("node_tp")), int(s("node_fp")), int(s("node_fn")))
        edge_p, edge_r, edge_f = _prf(int(s("edge_tp")), int(s("edge_fp")), int(s("edge_fn")))

        row = {"num_images": len(items)}
        for k, v in zip(group_keys, key):
            row[k] = v

        row.update({
            "gt_nodes": int(s("gt_nodes")),
            "pred_nodes": int(s("pred_nodes")),
            "node_precision": node_p,
            "node_recall": node_r,
            "node_f1": node_f,
            "mean_node_f1": node_f,

            "mean_matched_box_iou": float(np.mean([float(x.get("matched_box_iou", 0.0)) for x in items])),
            "mean_cldice": float(np.mean([float(x.get("cldice", 0.0)) for x in items])),
            "mean_soft_cldice": float(np.mean([float(x.get("soft_cldice", x.get("cldice", 0.0))) for x in items])),

            "gt_edges": int(s("gt_edges")),
            "pred_edges": int(s("pred_edges")),
            "edge_precision": edge_p,
            "edge_recall": edge_r,
            "edge_f1": edge_f,
            "link_f1": edge_f,
            "mean_link_f1": edge_f,

            "mean_nsee": float(np.mean([float(x.get("nsee", 0.0)) for x in items])),
            "mean_endpoint_l2": float(np.mean([float(x.get("endpoint_l2", 0.0)) for x in items])),

            "loop_back_recall": _safe_div(s("matched_self_loops"), s("gt_self_loops")),
            "decision_branch_recall": _safe_div(s("matched_branch_edges"), s("gt_branch_edges")),
            "path_mean_iou": float(np.mean([float(x.get("path_mean_iou", 0.0)) for x in items])),
            "path_mean_f1": float(np.mean([float(x.get("path_mean_f1", 0.0)) for x in items])),
            "avg_metric_tolerance_px": float(np.mean([float(x.get("metric_tolerance_px", 0.0)) for x in items])),
        })
        out.append(row)
    return out

In [ ]:
# ============================================================
# REVIEWER-ORIENTED METRIC LAYER
# ============================================================

def _valid_edge_endpoint_count(edges):
    count = 0
    for e in edges:
        for plain, hint in [("tail", "tail_hint"), ("head", "head_hint")]:
            if point_from_edge(e, plain, hint) is not None:
                count += 1
    return count


def evaluate_graph_prediction_revision(pred_writer, gt_writer):
    """Evaluation with explicit conditional-geometry accounting.

    Structural matching is unchanged from the submitted evaluator.
    New fields expose:
      * matched-node bounding-box overlap (not a segmentation mask metric),
      * normalized FP/FN structural edit error (not general GED),
      * matched-link coverage for geometry,
      * pooled endpoint errors and matched-link Soft-clDice values.
    """
    gt_size = gt_writer.get("size", {}) or {}
    gt_w = int(gt_size.get("w", pred_writer.get("size", {}).get("w", 1)))
    gt_h = int(gt_size.get("h", pred_writer.get("size", {}).get("h", 1)))
    tol = metric_tolerance_px(gt_w, gt_h)
    pred_writer = _rescale_writer_to_size(pred_writer, gt_w, gt_h)

    pred_nodes = pred_writer.get("nodes", []) or []
    gt_nodes = gt_writer.get("nodes", []) or []
    pred_edges = pred_writer.get("arrows", []) or []
    gt_edges = gt_writer.get("arrows", []) or []

    pred_to_gt, node_matches = tolerant_match_nodes(pred_nodes, gt_nodes, gt_w, gt_h)
    node_tp = len(node_matches)
    node_fp = len(pred_nodes) - node_tp
    node_fn = len(gt_nodes) - node_tp
    node_precision, node_recall, node_f1 = _prf(node_tp, node_fp, node_fn)

    gt_pairs = [edge_pair(e) for e in gt_edges]
    pred_pairs_mapped = [mapped_edge_pair(e, pred_to_gt) for e in pred_edges]
    gt_counter = pair_counter(gt_pairs)
    pred_counter = pair_counter(pred_pairs_mapped)
    edge_tp = counter_intersection_count(pred_counter, gt_counter)
    edge_fp = len(pred_edges) - edge_tp
    edge_fn = len(gt_edges) - edge_tp
    edge_precision, edge_recall, edge_f1 = _prf(edge_tp, edge_fp, edge_fn)

    matched_geom = pair_matched_edges_for_geometry(pred_edges, gt_edges, pred_pairs_mapped, gt_pairs)
    endpoint_values = []
    cldice_values = []
    path_iou_values = []
    path_f1_values = []

    for pe, ge, _ in matched_geom:
        for plain, hint in [("tail", "tail_hint"), ("head", "head_hint")]:
            pp = point_from_edge(pe, plain, hint)
            gp = point_from_edge(ge, plain, hint)
            if pp is not None and gp is not None:
                endpoint_values.append(float(l2(pp, gp)))

        pred_path = path_points(pe)
        gt_path = path_points(ge)
        if len(pred_path) < 2:
            a=point_from_edge(pe,"tail","tail_hint"); b=point_from_edge(pe,"head","head_hint")
            pred_path=[a,b] if a is not None and b is not None else []
        if len(gt_path) < 2:
            a=point_from_edge(ge,"tail","tail_hint"); b=point_from_edge(ge,"head","head_hint")
            gt_path=[a,b] if a is not None and b is not None else []
        if len(pred_path) >= 2 and len(gt_path) >= 2:
            pm=np.zeros((gt_h,gt_w),np.uint8); gm=np.zeros((gt_h,gt_w),np.uint8)
            thick=max(PATH_BASE_TOLERANCE_PX,tol)
            cv2.polylines(pm,[np.asarray([[int(round(x)),int(round(y))] for x,y in pred_path],np.int32)],False,1,thick,lineType=cv2.LINE_AA)
            cv2.polylines(gm,[np.asarray([[int(round(x)),int(round(y))] for x,y in gt_path],np.int32)],False,1,thick,lineType=cv2.LINE_AA)
            inter=int(np.logical_and(pm>0,gm>0).sum()); pc=int((pm>0).sum()); gc=int((gm>0).sum())
            pr=_safe_div(inter,pc); rc=_safe_div(inter,gc)
            path_f1_values.append(float(_safe_div(2*pr*rc,pr+rc)))
            path_iou_values.append(float(_safe_div(inter,int(np.logical_or(pm>0,gm>0).sum()))))

            pcm=np.zeros((gt_h,gt_w),np.uint8); gcm=np.zeros((gt_h,gt_w),np.uint8)
            cth=max(1,PATH_BASE_TOLERANCE_PX)
            cv2.polylines(pcm,[np.asarray([[int(round(x)),int(round(y))] for x,y in pred_path],np.int32)],False,1,cth,lineType=cv2.LINE_AA)
            cv2.polylines(gcm,[np.asarray([[int(round(x)),int(round(y))] for x,y in gt_path],np.int32)],False,1,cth,lineType=cv2.LINE_AA)
            cldice_values.append(float(cldice_score(pcm,gcm,tol_px=tol)))

    node_ious=[float(m[2]) for m in node_matches]
    node_center_errors=[float(m[3]) for m in node_matches]

    gt_loop_counter=Counter(k for k in gt_pairs if k[0] is not None and k[0]==k[1])
    pred_loop_counter=Counter(k for k in pred_pairs_mapped if k[0] is not None and k[0]==k[1])
    matched_loop=counter_intersection_count(pred_loop_counter,gt_loop_counter)
    out_degree=defaultdict(int)
    for t,h in gt_pairs:
        if t is not None: out_degree[t]+=1
    branch_sources={nid for nid,deg in out_degree.items() if deg>=2}
    branch_counter=Counter(p for p in gt_pairs if p[0] in branch_sources)
    branch_matched=counter_intersection_count(pred_counter,branch_counter)

    edit_cost=node_fp+node_fn+edge_fp+edge_fn
    structural_normalizer=max(1,max(len(pred_nodes),len(gt_nodes))+max(len(pred_edges),len(gt_edges)))
    nsee=_safe_div(edit_cost,structural_normalizer)
    gt_valid_endpoints=_valid_edge_endpoint_count(gt_edges)

    return {
        "gt_nodes":len(gt_nodes), "pred_nodes":len(pred_nodes),
        "node_tp":node_tp, "node_fp":node_fp, "node_fn":node_fn,
        "node_precision":node_precision, "node_recall":node_recall, "node_f1":node_f1,
        "matched_box_iou":float(np.mean(node_ious)) if node_ious else 0.0,
        "matched_box_iou_values":node_ious,
        "node_center_mae_px":float(np.mean(node_center_errors)) if node_center_errors else 0.0,
        "gt_edges":len(gt_edges), "pred_edges":len(pred_edges),
        "edge_tp":edge_tp, "edge_fp":edge_fp, "edge_fn":edge_fn,
        "edge_precision":edge_precision, "edge_recall":edge_recall, "edge_f1":edge_f1,
        "link_precision":edge_precision, "link_recall":edge_recall, "link_f1":edge_f1,
        "matched_directed_links":len(matched_geom),
        "matched_link_coverage":_safe_div(len(matched_geom),len(gt_edges)),
        "soft_cldice":float(np.mean(cldice_values)) if cldice_values else float('nan'),
        "soft_cldice_values":cldice_values,
        "soft_cldice_evaluated_links":len(cldice_values),
        "soft_cldice_coverage":_safe_div(len(cldice_values),len(gt_edges)),
        "endpoint_l2":float(np.mean(endpoint_values)) if endpoint_values else float('nan'),
        "endpoint_l2_values":endpoint_values,
        "evaluated_endpoints":len(endpoint_values),
        "gt_valid_endpoints":gt_valid_endpoints,
        "endpoint_coverage":_safe_div(len(endpoint_values),gt_valid_endpoints),
        "path_mean_iou":float(np.mean(path_iou_values)) if path_iou_values else float('nan'),
        "path_mean_f1":float(np.mean(path_f1_values)) if path_f1_values else float('nan'),
        "gt_self_loops":int(sum(gt_loop_counter.values())), "matched_self_loops":int(matched_loop),
        "loop_back_recall":_safe_div(matched_loop,sum(gt_loop_counter.values())),
        "gt_branch_edges":int(sum(branch_counter.values())), "matched_branch_edges":int(branch_matched),
        "decision_branch_recall":_safe_div(branch_matched,sum(branch_counter.values())),
        "normalized_structural_edit_error":nsee,
        "nsee":nsee,
        "structural_edit_cost":edit_cost,
        "structural_edit_normalizer":structural_normalizer,
        "metric_tolerance_px":tol,
    }


def evaluate_prediction_by_profile(pred_writer, gt_writer, profile=None):
    profile=(profile or METRIC_PROFILE).lower().strip()
    if profile == 'submitted':
        return evaluate_graph_prediction_tolerant(pred_writer, gt_writer)
    if profile == 'revision':
        return evaluate_graph_prediction_revision(pred_writer, gt_writer)
    raise ValueError("METRIC_PROFILE must be 'submitted' or 'revision'.")


def _flatten_numeric(rows, key):
    out=[]
    for r in rows:
        vals=r.get(key,[]) or []
        for v in vals:
            try:
                fv=float(v)
                if math.isfinite(fv): out.append(fv)
            except Exception:
                pass
    return out


def aggregate_revision_metrics(rows, group_keys=()):
    groups=defaultdict(list)
    for r in rows:
        groups[tuple(r.get(k,'') for k in group_keys)].append(r)
    output=[]
    for key,items in sorted(groups.items(),key=lambda kv:kv[0]):
        s=lambda name: sum(float(x.get(name,0.0) or 0.0) for x in items)
        np_,nr,nf=_prf(int(s('node_tp')),int(s('node_fp')),int(s('node_fn')))
        ep_,er,ef=_prf(int(s('edge_tp')),int(s('edge_fp')),int(s('edge_fn')))
        box_vals=_flatten_numeric(items,'matched_box_iou_values')
        cl_vals=_flatten_numeric(items,'soft_cldice_values')
        end_vals=_flatten_numeric(items,'endpoint_l2_values')
        row={"num_images":len(items)}
        for k,v in zip(group_keys,key): row[k]=v
        row.update({
            "gt_nodes":int(s('gt_nodes')), "pred_nodes":int(s('pred_nodes')),
            "node_precision":np_, "node_recall":nr, "node_f1":nf,
            # Macro image mean keeps continuity with the submitted 82.71-style number.
            "mean_matched_box_iou":float(np.mean([float(x.get('matched_box_iou',0.0)) for x in items])),
            "pooled_matched_box_iou":float(np.mean(box_vals)) if box_vals else float('nan'),
            "matched_nodes_for_box_iou":len(box_vals),
            "gt_edges":int(s('gt_edges')), "pred_edges":int(s('pred_edges')),
            "edge_precision":ep_, "edge_recall":er, "edge_f1":ef, "link_f1":ef,
            "mean_normalized_structural_edit_error":float(np.mean([float(x.get('nsee',0.0)) for x in items])),
            "structural_edit_cost_total":int(s('structural_edit_cost')),
            "matched_directed_links":int(s('matched_directed_links')),
            "matched_link_coverage":_safe_div(s('matched_directed_links'),s('gt_edges')),
            "soft_cldice_matched_links":float(np.mean(cl_vals)) if cl_vals else float('nan'),
            "soft_cldice_median":float(np.median(cl_vals)) if cl_vals else float('nan'),
            "soft_cldice_evaluated_links":len(cl_vals),
            "soft_cldice_coverage":_safe_div(len(cl_vals),s('gt_edges')),
            "endpoint_l2_matched_endpoints_mean":float(np.mean(end_vals)) if end_vals else float('nan'),
            "endpoint_l2_matched_endpoints_median":float(np.median(end_vals)) if end_vals else float('nan'),
            "endpoint_l2_matched_endpoints_p90":float(np.percentile(end_vals,90)) if end_vals else float('nan'),
            "evaluated_endpoints":len(end_vals),
            "gt_valid_endpoints":int(s('gt_valid_endpoints')),
            "endpoint_coverage":_safe_div(len(end_vals),s('gt_valid_endpoints')),
            "loop_back_recall":_safe_div(s('matched_self_loops'),s('gt_self_loops')),
            "decision_branch_recall":_safe_div(s('matched_branch_edges'),s('gt_branch_edges')),
            "avg_metric_tolerance_px":float(np.mean([float(x.get('metric_tolerance_px',0.0)) for x in items])),
        })
        output.append(row)
    return output


In [ ]:
def save_json_local(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_json(data), f, indent=2, ensure_ascii=False)


def write_csv_rows_local(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    # stable union of keys
    keys = []
    seen = set()
    for r in rows:
        for k in r.keys():
            if k not in seen:
                keys.append(k); seen.add(k)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow(r)


def get_example_ids(samples, max_n=3, policy="first"):
    samples = list(samples)
    if max_n <= 0:
        return set()
    if policy == "random":
        rng = random.Random(RANDOM_SEED)
        selected = rng.sample(samples, min(max_n, len(samples)))
    else:
        selected = samples[:max_n]
    return {(s.family, s.split, s.sample_id) for s in selected}


def draw_overlay_from_writer(image_path, writer, out_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img is None:
        # fallback to declared writer size
        size = writer.get("size", {"w": 1024, "h": 1024})
        img = np.zeros((int(size.get("h", 1024)), int(size.get("w", 1024)), 3), dtype=np.uint8)
    size = writer.get("size", {})
    w = int(size.get("w", img.shape[1]))
    h = int(size.get("h", img.shape[0]))
    if (img.shape[1], img.shape[0]) != (w, h):
        img = cv2.resize(img, (w, h), interpolation=cv2.INTER_LINEAR)

    for node in writer.get("nodes", []) or []:
        x, y, bw, bh = [int(round(float(v))) for v in node.get("bbox", [0, 0, 0, 0])]
        cv2.rectangle(img, (x, y), (x + bw, y + bh), (0, 255, 255), 2)
        cv2.putText(img, str(node.get("id", "")), (x, max(0, y - 3)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1, cv2.LINE_AA)

    for edge in writer.get("arrows", []) or []:
        pts = path_points(edge)
        if len(pts) >= 2:
            arr = np.array([[int(round(x)), int(round(y))] for x, y in pts], dtype=np.int32)
            cv2.polylines(img, [arr], False, (255, 255, 0), 2, lineType=cv2.LINE_AA)
        tail = point_from_edge(edge, "tail", "tail_hint")
        head = point_from_edge(edge, "head", "head_hint")
        if tail is not None:
            cv2.drawMarker(img, (int(round(tail[0])), int(round(tail[1]))), (255, 0, 255),
                           cv2.MARKER_CROSS, 12, 2)
        if head is not None:
            cv2.drawMarker(img, (int(round(head[0])), int(round(head[1]))), (0, 255, 0),
                           cv2.MARKER_CROSS, 12, 2)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), img)


def make_decoder_args(device_str=None):
    class Args:
        pass
    args = Args()
    args.image_mode = IMAGE_MODE
    args.normalize_image = NORMALIZE_IMAGE
    args.use_otsu = bool(APPLY_OTSU_THRESHOLD)
    args.target_w = int(TARGET_W)
    args.target_h = int(TARGET_H)
    args.base_ch = int(GRAPH_BASE_CHANNELS)
    args.norm_type = NORM_TYPE
    args.allow_partial_checkpoint = False
    args.checkpoint = GRAPH_MODEL_CHECKPOINT
    args.device = device_str
    args.save_original_space_json = True
    args.save_stage_maps = False

    # frozen decoder defaults supplied directly by this notebook
    default_args = default_inference_args()
    for k, v in vars(default_args).items():
        if not hasattr(args, k):
            setattr(args, k, v)
    args.image_mode = IMAGE_MODE
    args.normalize_image = NORMALIZE_IMAGE
    args.use_otsu = bool(APPLY_OTSU_THRESHOLD)
    args.target_w = int(TARGET_W)
    args.target_h = int(TARGET_H)
    args.base_ch = int(GRAPH_BASE_CHANNELS)
    args.norm_type = NORM_TYPE
    args.checkpoint = GRAPH_MODEL_CHECKPOINT
    args.device = device_str
    return args


def nearest_node_id_by_boundary(point_xy, nodes):
    if not nodes:
        return None
    px, py = point_xy
    best_id, best_d = None, float("inf")
    for n in nodes:
        x, y, w, h = [float(v) for v in n.get("bbox", [0, 0, 0, 0])]
        left, top, right, bottom = x, y, x + w, y + h
        dx = max(left - px, 0.0, px - right)
        dy = max(top - py, 0.0, py - bottom)
        if dx > 0 or dy > 0:
            d = math.hypot(dx, dy)
        else:
            d = min(abs(px - left), abs(px - right), abs(py - top), abs(py - bottom))
        if d < best_d:
            best_d, best_id = d, n.get("id")
    return best_id


def load_assembler_checkpoint_safe(checkpoint_path, device, image_mode, base_ch=0, norm_type="auto", strict=True):
    """
    Same intent as load_checkpoint_model, but checkpoint is deserialized on CPU first
    and then model is moved to the selected device. This is safer for notebook
    parallel execution and avoids CUDA deserialization races.
    """
    t0 = time.time()
    if PRINT_TIMING_DIAGNOSTICS:
        print(f"[asm-load] torch.load CPU start on {device}: {checkpoint_path}")
    ckpt = torch.load(str(checkpoint_path), map_location="cpu")
    if PRINT_TIMING_DIAGNOSTICS:
        print(f"[asm-load] torch.load CPU done on {device} in {time.time() - t0:.2f}s")

    t1 = time.time()
    state = strip_module_prefix(extract_state_dict(ckpt))
    inferred = infer_arch_from_state(state)

    ckpt_base_ch = inferred.get("base_ch")
    ckpt_in_channels = inferred.get("in_channels")
    ckpt_norm_type = inferred.get("norm_type") or "group"

    if base_ch <= 0:
        if ckpt_base_ch is None:
            raise ValueError("Could not infer base_ch from checkpoint. Pass GRAPH_BASE_CHANNELS explicitly.")
        base_ch = ckpt_base_ch

    requested_in_channels = 1 if image_mode.lower() == "grayscale" else 3
    in_channels = ckpt_in_channels if ckpt_in_channels is not None else requested_in_channels
    if in_channels != requested_in_channels:
        raise ValueError(
            f"Checkpoint expects in_channels={in_channels}, but IMAGE_MODE={image_mode!r} gives "
            f"{requested_in_channels}. Use the same image mode used during training."
        )

    if norm_type == "auto":
        norm_type = ckpt_norm_type

    model = DiagramUNet(in_channels=in_channels, base_ch=base_ch, norm_type=norm_type)
    if PRINT_TIMING_DIAGNOSTICS:
        print(f"[asm-load] built DiagramUNet on CPU in {time.time() - t1:.2f}s")

    load_info = {
        "checkpoint_path": str(checkpoint_path),
        "base_ch": base_ch,
        "in_channels": in_channels,
        "norm_type": norm_type,
        "strict": strict,
        "inferred": inferred,
        "checkpoint_top_level_keys": list(ckpt.keys()) if isinstance(ckpt, dict) else [],
        "loaded_via": "cpu_then_to_device",
    }

    t2 = time.time()
    if strict:
        model.load_state_dict(state, strict=True)
        load_info["missing_keys"] = []
        load_info["unexpected_keys"] = []
    else:
        missing, unexpected = model.load_state_dict(state, strict=False)
        load_info["missing_keys"] = list(missing)
        load_info["unexpected_keys"] = list(unexpected)
    if PRINT_TIMING_DIAGNOSTICS:
        print(f"[asm-load] load_state_dict done on {device} in {time.time() - t2:.2f}s")

    t3 = time.time()
    model = model.to(device)
    if torch.cuda.is_available() and str(device).startswith("cuda"):
        torch.cuda.synchronize(device)
    model.eval()
    if PRINT_TIMING_DIAGNOSTICS:
        print(f"[asm-load] moved to {device} in {time.time() - t3:.2f}s; total={time.time() - t0:.2f}s")
    return model, load_info


def _writer_size(writer, fallback=(1024, 1024)):
    size = writer.get("size", {}) or {}
    return int(size.get("w", fallback[0])), int(size.get("h", fallback[1]))


def match_resnet_nodes_to_graph_nodes(graph_nodes, resnet_nodes, width, height):
    """
    Returns a one-to-one mapping from graph-model nodes to ResNet34 nodes.
    Matching uses relaxed center distance plus light IoU.
    """
    tol = max(2.0, float(NODE_REFINEMENT_CENTER_TOLERANCE_RATIO) * max(int(width), int(height)))
    candidates = []
    for vi, vn in enumerate(graph_nodes):
        vc = bbox_center_xy(vn)
        vdiag = math.hypot(float(vn.get("bbox", [0, 0, 1, 1])[2]), float(vn.get("bbox", [0, 0, 1, 1])[3]))
        center_tol = max(tol, 0.25 * vdiag)
        for ei, en in enumerate(resnet_nodes):
            ec = bbox_center_xy(en)
            iou = bbox_iou_xywh(vn.get("bbox", [0,0,0,0]), en.get("bbox", [0,0,0,0]))
            cd = l2(vc, ec)
            if iou >= float(NODE_REFINEMENT_IOU_ACCEPT) or cd <= center_tol:
                score = (cd / max(center_tol, 1e-6)) + (1.0 - iou)
                candidates.append((score, -iou, cd, vi, ei))
    candidates.sort()
    used_v, used_e = set(), set()
    mapping = {}
    for score, neg_iou, cd, vi, ei in candidates:
        if vi in used_v or ei in used_e:
            continue
        used_v.add(vi)
        used_e.add(ei)
        mapping[vi] = ei
    return mapping


def build_refined_node_writer(graph_writer, resnet_nodes):
    """
    Topology-preserving node-refinement writer:
    - Preserve graph-model node IDs.
    - Preserve arrows, endpoint attachments, and routed paths.
    - Replace node-box geometry with matched ResNet34 proposals when reliable.
    This improves node geometry without altering the selected graph topology.
    """
    writer = deepcopy(graph_writer)
    graph_nodes = graph_writer.get("nodes", []) or []
    w, h = _writer_size(graph_writer)

    mapping = match_resnet_nodes_to_graph_nodes(graph_nodes, resnet_nodes, w, h)
    refined_nodes = []

    for vi, vn in enumerate(graph_nodes):
        hn = deepcopy(vn)
        if vi in mapping:
            en = resnet_nodes[mapping[vi]]
            hn["bbox"] = [int(round(v)) for v in en.get("bbox", hn.get("bbox", [0,0,0,0]))]
            hn["refinement_source"] = "resnet34_matched"
            hn["node_seg_score"] = float(en.get("score", 0.0))
            hn["node_seg_area"] = int(en.get("area", 0))
        else:
            hn["refinement_source"] = "graph_model_unmatched"
        hn["id"] = vn.get("id", f"N{vi}")
        refined_nodes.append(hn)

    if not NODE_REFINEMENT_KEEP_UNMATCHED_GRAPH_NODES:
        refined_nodes = [n for n in refined_nodes if n.get("refinement_source") == "resnet34_matched"]

    writer["nodes"] = refined_nodes
    writer["arrows"] = deepcopy(graph_writer.get("arrows", []) or [])

    if NODE_REFINEMENT_REPAIR_EDGE_IDS:
        for arrow in writer.get("arrows", []) or []:
            tail = point_from_edge(arrow, "tail", "tail_hint")
            head = point_from_edge(arrow, "head", "head_hint")
            if tail is not None:
                arrow["tail_node_id"] = nearest_node_id_by_boundary(tail, refined_nodes)
            if head is not None:
                arrow["head_node_id"] = nearest_node_id_by_boundary(head, refined_nodes)

    writer.setdefault("metadata", {})
    writer["metadata"]["node_refinement"] = {
        "preserve_graph_topology": not NODE_REFINEMENT_REPAIR_EDGE_IDS,
        "matched_nodes": int(len(mapping)),
        "graph_nodes": int(len(graph_nodes)),
        "resnet_nodes": int(len(resnet_nodes)),
    }
    return writer


def samples_to_dicts(samples):
    return [{
        "sample_id": s.sample_id,
        "image_path": str(s.image_path),
        "json_path": str(s.json_path) if s.json_path else "",
        "family": s.family,
        "split": s.split,
    } for s in samples]


def split_evenly(items, n):
    return [items[i::n] for i in range(n)]


def read_jsonl(path):
    rows = []
    path = Path(path)
    if not path.exists():
        return rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

In [ ]:
FINALIZE_ATTACH_RELAX_MODEL_PX = 15.0


FINALIZE_ATTACH_MIN_MARGIN_MODEL_PX = 10.0


FINALIZE_ATTACH_BOTH_NULL_MIN_MARGIN = 24.0


FINALIZE_ATTACH_CONFIDENCE_MULTIPLIER = 1.22


FINALIZE_MAX_REPAIRED_ATTACHMENTS_PER_IMAGE = 8


FINALIZE_ENABLE_CANDIDATE_RESURRECTION = True


FINALIZE_MAX_RESURRECTED_PER_IMAGE = 2


FINALIZE_RESURRECT_MIN_CONFIDENCE = 0.56


FINALIZE_RESURRECT_MIN_SUPPORT = 0.72


FINALIZE_RESURRECT_MIN_SHAFT = 0.70


FINALIZE_RESURRECT_MAX_PATH_OVERLAP = 0.58


FINALIZE_DUPLICATE_STRONG_IOU = 0.82


FINALIZE_DUPLICATE_FALLBACK_IOU = 0.66


FINALIZE_DUPLICATE_ENDPOINT_DIST = 24.0


FINALIZE_PATH_MASK_THICKNESS = 5


FINALIZE_STRICT_FALLBACK_FAMILIES = {"fca"}


FINALIZE_INK_MIN_CONFIDENCE = 0.78


FINALIZE_INK_MIN_SUPPORT = 0.76


FINALIZE_INK_MIN_SHAFT = 0.72


FINALIZE_INK_MIN_DIRECTION = 0.62


FINALIZE_STROKE_MIN_CONFIDENCE = 0.60


FINALIZE_STROKE_MIN_SUPPORT = 0.70


FINALIZE_STROKE_MIN_SHAFT = 0.68


FINALIZE_STROKE_MIN_DIRECTION = 0.54


FINALIZE_NODE_MIN_MATCH_RATIO = 0.72


FINALIZE_NODE_MAX_COUNT_DELTA_RATIO = 0.24


FINALIZE_NODE_MIN_MATCHED = 2


FINALIZE_REATTACH_NULL_ON_REFINED_NODES = True


@dataclass
class AspectGeometry:
    """Aspect-correct geometry while the network still sees stretched 512x512 input.

    Coordinates passed to this object are in model space. Distances are measured in
    original-image geometry, then converted back to area-equivalent model pixels so
    the existing decoder thresholds retain roughly the same numerical meaning.
    """
    orig_w: int
    orig_h: int
    model_w: int = 512
    model_h: int = 512

    def __post_init__(self):
        self.sx = float(self.orig_w) / max(float(self.model_w), 1.0)
        self.sy = float(self.orig_h) / max(float(self.model_h), 1.0)
        self.ref = math.sqrt(max(self.sx * self.sy, 1e-12))
        self.aspect_ratio = max(float(self.orig_w) / max(self.orig_h, 1), float(self.orig_h) / max(self.orig_w, 1))

    def vector_to_original(self, v):
        return float(v[0]) * self.sx, float(v[1]) * self.sy

    def distance(self, p, q):
        dx = (float(p[0]) - float(q[0])) * self.sx
        dy = (float(p[1]) - float(q[1])) * self.sy
        return math.hypot(dx, dy) / self.ref

    def point_to_bbox_distance(self, point, bbox_xyxy):
        px, py = float(point[0]), float(point[1])
        x0, y0, x1, y1 = [float(v) for v in bbox_xyxy]
        dx_model = max(x0 - px, 0.0, px - x1)
        dy_model = max(y0 - py, 0.0, py - y1)
        return math.hypot(dx_model * self.sx, dy_model * self.sy) / self.ref

    def bbox_diag(self, bbox_xyxy):
        x0, y0, x1, y1 = [float(v) for v in bbox_xyxy]
        return math.hypot((x1 - x0) * self.sx, (y1 - y0) * self.sy) / self.ref

    def corrected_probe(self, ep, probe_dist):
        tx, ty = float(ep.tangent_xy[0]), float(ep.tangent_xy[1])
        ox, oy = tx * self.sx, ty * self.sy
        norm = math.hypot(ox, oy)
        if norm < 1e-8:
            ux, uy = 1.0, 0.0
        else:
            ux, uy = ox / norm, oy / norm
        physical_len = float(probe_dist) * self.ref
        dx_model = ux * physical_len / max(self.sx, 1e-8)
        dy_model = uy * physical_len / max(self.sy, 1e-8)
        px, py = ep.refined_xy
        if ep.kind == "tail":
            return float(px) - dx_model, float(py) - dy_model
        return float(px) + dx_model, float(py) + dy_model


def _arrow_source(arrow):
    meta = getattr(arrow, "meta", {}) or {}
    if meta.get("source"):
        return str(meta["source"])
    if meta.get("port_role_repair"):
        return "port_role_repair"
    return "endpoint_ast"


def _attached_count(arrow):
    return int(arrow.tail_node_id is not None) + int(arrow.head_node_id is not None)


def _arrow_quality(arrow):
    priority = {
        "endpoint_ast": 4.0,
        "port_role_repair": 3.4,
        "stroke_graph": 2.0,
        "ink_bridge": 1.0,
    }.get(_arrow_source(arrow), 1.5)
    return (
        _attached_count(arrow),
        priority,
        float(arrow.confidence),
        float(arrow.support_score),
        float(arrow.shaft_score),
        float(arrow.direction_score),
    )


def _node_by_id(nodes):
    return {n.node_id: n for n in nodes}


def aspect_attachment_rankings(ep, nodes, cfg, geom, top_k=3):
    probe = geom.corrected_probe(ep, cfg.tangent_probe_dist)
    ranked = []
    for n in nodes:
        d_point = geom.point_to_bbox_distance(ep.refined_xy, n.bbox_xyxy)
        d_probe = geom.point_to_bbox_distance(probe, n.bbox_xyxy)
        inside_bonus = -4.0 if point_inside_bbox(ep.refined_xy, n.bbox_xyxy) else 0.0
        score = 0.65 * d_point + 0.35 * d_probe + inside_bonus
        ranked.append({
            "node_id": n.node_id,
            "score": float(score),
            "endpoint_distance": float(d_point),
            "probe_distance": float(d_probe),
            "inside_bonus": float(inside_bonus),
            "bbox_xyxy": [int(v) for v in n.bbox_xyxy],
        })
    ranked.sort(key=lambda r: r["score"])
    margin = float(ranked[1]["score"] - ranked[0]["score"]) if len(ranked) > 1 else float("inf")
    return {
        "endpoint_id": ep.endpoint_id,
        "kind": ep.kind,
        "refined_xy": [float(ep.refined_xy[0]), float(ep.refined_xy[1])],
        "probe_xy": [float(probe[0]), float(probe[1])],
        "top_candidates": ranked[:max(1, int(top_k))],
        "margin_top1_top2": margin,
        "aspect_ratio": float(geom.aspect_ratio),
    }


def _allow_repaired_attachment(arrow, kind, node_id, nodes_by_id, geom):
    other_id = arrow.head_node_id if kind == "tail" else arrow.tail_node_id
    if other_id is None or node_id != other_id:
        return True
    # Avoid manufacturing a self-loop from a near-node attachment unless the path
    # clearly leaves the node region or the assembler already considered it a loop.
    if bool(arrow.self_loop):
        return True
    node = nodes_by_id.get(node_id)
    if node is None or len(arrow.path_xy) < 32:
        return False
    max_excursion = max(geom.point_to_bbox_distance(p, node.bbox_xyxy) for p in arrow.path_xy)
    return max_excursion >= max(8.0, 0.15 * geom.bbox_diag(node.bbox_xyxy))


def _repair_single_arrow_attachments(arrows, heads, tails, nodes, cfg, geom):
    heads_by_id = {e.endpoint_id: e for e in heads}
    tails_by_id = {e.endpoint_id: e for e in tails}
    nodes_by_id = _node_by_id(nodes)
    actions = []
    repaired = 0

    for arrow in sorted(arrows, key=lambda a: float(a.confidence), reverse=True):
        if repaired >= int(FINALIZE_MAX_REPAIRED_ATTACHMENTS_PER_IMAGE):
            break
        null_kinds = []
        if arrow.tail_node_id is None:
            null_kinds.append("tail")
        if arrow.head_node_id is None:
            null_kinds.append("head")
        if not null_kinds:
            continue

        pending = []
        for kind in null_kinds:
            ep = tails_by_id.get(arrow.tail_endpoint_id) if kind == "tail" else heads_by_id.get(arrow.head_endpoint_id)
            if ep is None:
                continue
            ranking = aspect_attachment_rankings(ep, nodes, cfg, geom, top_k=3)
            if not ranking["top_candidates"]:
                continue
            top = ranking["top_candidates"][0]
            max_score = float(cfg.node_attach_max_dist) + float(FINALIZE_ATTACH_RELAX_MODEL_PX)
            min_margin = float(FINALIZE_ATTACH_MIN_MARGIN_MODEL_PX)
            if len(null_kinds) == 2:
                min_margin = float(FINALIZE_ATTACH_BOTH_NULL_MIN_MARGIN)
            accepted = bool(top["score"] <= max_score and ranking["margin_top1_top2"] >= min_margin)
            if accepted and _allow_repaired_attachment(arrow, kind, top["node_id"], nodes_by_id, geom):
                pending.append((kind, top, ranking, ep))

        # If both endpoints were null, require both to be strongly resolved.
        if len(null_kinds) == 2 and len(pending) != 2:
            continue
        if len(null_kinds) == 1 and len(pending) != 1:
            continue

        before = {"tail_node_id": arrow.tail_node_id, "head_node_id": arrow.head_node_id, "confidence": float(arrow.confidence)}
        for kind, top, ranking, ep in pending:
            if kind == "tail":
                arrow.tail_node_id = top["node_id"]
                arrow.meta["tail_attach_score_post"] = float(top["score"])
            else:
                arrow.head_node_id = top["node_id"]
                arrow.meta["head_attach_score_post"] = float(top["score"])
            arrow.meta[f"{kind}_attachment_post_margin"] = float(ranking["margin_top1_top2"])
            arrow.meta[f"{kind}_attachment_aspect_corrected"] = True
            repaired += 1

        arrow.self_loop = bool(arrow.tail_node_id is not None and arrow.tail_node_id == arrow.head_node_id)
        if arrow.tail_node_id is not None and arrow.head_node_id is not None:
            arrow.confidence = float(min(1.0, arrow.confidence * FINALIZE_ATTACH_CONFIDENCE_MULTIPLIER))
        actions.append({
            "action": "complete_endpoint_attachment",
            "arrow_id": int(arrow.arrow_id),
            "source": _arrow_source(arrow),
            "before": before,
            "after": {"tail_node_id": arrow.tail_node_id, "head_node_id": arrow.head_node_id, "confidence": float(arrow.confidence)},
            "resolved": [
                {"kind": kind, "node_id": top["node_id"], "score": float(top["score"]), "margin": float(ranking["margin_top1_top2"])}
                for kind, top, ranking, _ in pending
            ],
        })
    return arrows, actions


def _arrow_mask(path_xy, shape_hw=(512, 512), thickness=5):
    h, w = int(shape_hw[0]), int(shape_hw[1])
    mask = np.zeros((h, w), np.uint8)
    if len(path_xy) >= 2:
        pts = np.asarray([[int(round(x)), int(round(y))] for x, y in path_xy], dtype=np.int32)
        cv2.polylines(mask, [pts], False, 1, int(thickness), cv2.LINE_8)
    return mask


def _mask_iou_arrays(ma, mb):
    inter = int(np.logical_and(ma > 0, mb > 0).sum())
    union = int(np.logical_or(ma > 0, mb > 0).sum())
    return float(inter / max(union, 1))


def _path_bounds(path_xy, pad=0):
    if not path_xy:
        return (0, 0, -1, -1)
    xs = [float(p[0]) for p in path_xy]
    ys = [float(p[1]) for p in path_xy]
    return (min(xs) - pad, min(ys) - pad, max(xs) + pad, max(ys) + pad)


def _bounds_overlap(a, b):
    return not (a[2] < b[0] or b[2] < a[0] or a[3] < b[1] or b[3] < a[1])


def _arrow_path_iou_model(a, b, shape_hw=(512, 512), thickness=5, mask_a=None, mask_b=None):
    pad = max(2, int(thickness))
    if not _bounds_overlap(_path_bounds(a.path_xy, pad), _path_bounds(b.path_xy, pad)):
        return 0.0
    ma = mask_a if mask_a is not None else _arrow_mask(a.path_xy, shape_hw, thickness)
    mb = mask_b if mask_b is not None else _arrow_mask(b.path_xy, shape_hw, thickness)
    return _mask_iou_arrays(ma, mb)


def _arrow_endpoint_distance(a, b, geom):
    direct = geom.distance(a.tail_xy, b.tail_xy) + geom.distance(a.head_xy, b.head_xy)
    reverse = geom.distance(a.tail_xy, b.head_xy) + geom.distance(a.head_xy, b.tail_xy)
    return min(direct, reverse) * 0.5


def _is_physical_duplicate(a, b, geom, shape_hw=(512, 512), mask_a=None, mask_b=None):
    iou = _arrow_path_iou_model(a, b, shape_hw, FINALIZE_PATH_MASK_THICKNESS, mask_a=mask_a, mask_b=mask_b)
    same_pair = a.tail_node_id == b.tail_node_id and a.head_node_id == b.head_node_id
    if same_pair and iou >= 0.45:
        return True, iou, "same_node_pair_path_overlap"
    if iou >= float(FINALIZE_DUPLICATE_STRONG_IOU):
        return True, iou, "strong_physical_path_overlap"
    fallback_involved = _arrow_source(a) in {"stroke_graph", "ink_bridge"} or _arrow_source(b) in {"stroke_graph", "ink_bridge"}
    null_involved = _attached_count(a) < 2 or _attached_count(b) < 2
    if (fallback_involved or null_involved) and iou >= float(FINALIZE_DUPLICATE_FALLBACK_IOU):
        if _arrow_endpoint_distance(a, b, geom) <= float(FINALIZE_DUPLICATE_ENDPOINT_DIST):
            return True, iou, "fallback_or_null_duplicate"
    return False, iou, ""


def _fallback_graph_need(arrow, base_arrows):
    if arrow.tail_node_id is None or arrow.head_node_id is None:
        return False
    outdeg, indeg = Counter(), Counter()
    for a in base_arrows:
        if a.tail_node_id is not None:
            outdeg[a.tail_node_id] += 1
        if a.head_node_id is not None:
            indeg[a.head_node_id] += 1
    return outdeg[arrow.tail_node_id] == 0 or indeg[arrow.head_node_id] == 0


def _gate_fallback_arrows(arrows, family):
    if str(family).lower() not in FINALIZE_STRICT_FALLBACK_FAMILIES:
        return arrows, []
    base = [a for a in arrows if _arrow_source(a) not in {"stroke_graph", "ink_bridge"} and _attached_count(a) == 2]
    kept, actions = [], []
    for a in arrows:
        src = _arrow_source(a)
        keep = True
        reasons = []
        if src == "ink_bridge":
            keep = bool(
                _attached_count(a) == 2
                and float(a.confidence) >= FINALIZE_INK_MIN_CONFIDENCE
                and float(a.support_score) >= FINALIZE_INK_MIN_SUPPORT
                and float(a.shaft_score) >= FINALIZE_INK_MIN_SHAFT
                and float(a.direction_score) >= FINALIZE_INK_MIN_DIRECTION
                and _fallback_graph_need(a, base)
            )
            reasons = ["strict_fca_ink_bridge_gate"]
        elif src == "stroke_graph":
            keep = bool(
                _attached_count(a) == 2
                and float(a.confidence) >= FINALIZE_STROKE_MIN_CONFIDENCE
                and float(a.support_score) >= FINALIZE_STROKE_MIN_SUPPORT
                and float(a.shaft_score) >= FINALIZE_STROKE_MIN_SHAFT
                and float(a.direction_score) >= FINALIZE_STROKE_MIN_DIRECTION
                and _fallback_graph_need(a, base)
            )
            reasons = ["strict_fca_stroke_graph_gate"]
        if keep:
            kept.append(a)
        else:
            actions.append({
                "action": "remove_fallback_candidate",
                "arrow_id": int(a.arrow_id),
                "source": src,
                "reason": reasons[0],
                "tail_node_id": a.tail_node_id,
                "head_node_id": a.head_node_id,
                "confidence": float(a.confidence),
                "support_score": float(a.support_score),
                "shaft_score": float(a.shaft_score),
                "direction_score": float(a.direction_score),
            })
    return kept, actions


def _suppress_physical_duplicates(arrows, geom, shape_hw=(512, 512)):
    kept, actions = [], []
    mask_cache = {}
    def get_mask(a):
        key = id(a)
        if key not in mask_cache:
            mask_cache[key] = _arrow_mask(a.path_xy, shape_hw, FINALIZE_PATH_MASK_THICKNESS)
        return mask_cache[key]
    for cand in sorted(arrows, key=_arrow_quality, reverse=True):
        duplicate = None
        cand_mask = None
        for old in kept:
            if not _bounds_overlap(_path_bounds(cand.path_xy, FINALIZE_PATH_MASK_THICKNESS), _path_bounds(old.path_xy, FINALIZE_PATH_MASK_THICKNESS)):
                continue
            if cand_mask is None:
                cand_mask = get_mask(cand)
            is_dup, iou, reason = _is_physical_duplicate(cand, old, geom, shape_hw, mask_a=cand_mask, mask_b=get_mask(old))
            if is_dup:
                duplicate = (old, iou, reason)
                break
        if duplicate is None:
            kept.append(cand)
        else:
            old, iou, reason = duplicate
            actions.append({
                "action": "suppress_physical_duplicate",
                "removed_arrow_id": int(cand.arrow_id),
                "kept_arrow_id": int(old.arrow_id),
                "removed_source": _arrow_source(cand),
                "kept_source": _arrow_source(old),
                "path_iou": float(iou),
                "reason": reason,
                "removed_pair": [cand.tail_node_id, cand.head_node_id],
                "kept_pair": [old.tail_node_id, old.head_node_id],
            })
    return kept, actions


def _candidate_already_selected(candidate, selected):
    for a in selected:
        if candidate.tail_endpoint_id == a.tail_endpoint_id and candidate.head_endpoint_id == a.head_endpoint_id:
            return True
    return False


def _resurrect_existing_candidates(selected, all_candidates, fixed_heads, fixed_tails, geom, family):
    if not FINALIZE_ENABLE_CANDIDATE_RESURRECTION:
        return selected, []
    heads = {e.endpoint_id: e for e in fixed_heads}
    tails = {e.endpoint_id: e for e in fixed_tails}
    outdeg, indeg = Counter(), Counter()
    pair_set = set()
    for a in selected:
        if a.tail_node_id is not None:
            outdeg[a.tail_node_id] += 1
        if a.head_node_id is not None:
            indeg[a.head_node_id] += 1
        pair_set.add((a.tail_node_id, a.head_node_id))

    proposals = []
    for raw in all_candidates:
        if _candidate_already_selected(raw, selected):
            continue
        src = _arrow_source(raw)
        if src == "ink_bridge":
            continue
        cand = deepcopy(raw)
        te = tails.get(cand.tail_endpoint_id)
        he = heads.get(cand.head_endpoint_id)
        if te is not None:
            cand.tail_node_id = te.attached_node_id
        if he is not None:
            cand.head_node_id = he.attached_node_id
        cand.self_loop = bool(cand.tail_node_id is not None and cand.tail_node_id == cand.head_node_id)
        if _attached_count(cand) != 2 or cand.self_loop:
            continue
        if (cand.tail_node_id, cand.head_node_id) in pair_set:
            continue
        graph_need = outdeg[cand.tail_node_id] == 0 or indeg[cand.head_node_id] == 0 or bool(cand.meta.get("port_role_repair"))
        if not graph_need:
            continue
        if float(cand.confidence) < FINALIZE_RESURRECT_MIN_CONFIDENCE:
            continue
        if float(cand.support_score) < FINALIZE_RESURRECT_MIN_SUPPORT or float(cand.shaft_score) < FINALIZE_RESURRECT_MIN_SHAFT:
            continue
        cand_bounds = _path_bounds(cand.path_xy, FINALIZE_PATH_MASK_THICKNESS)
        cand_mask = None
        max_overlap = 0.0
        for old in selected:
            if not _bounds_overlap(cand_bounds, _path_bounds(old.path_xy, FINALIZE_PATH_MASK_THICKNESS)):
                continue
            if cand_mask is None:
                cand_mask = _arrow_mask(cand.path_xy, (TARGET_H, TARGET_W), FINALIZE_PATH_MASK_THICKNESS)
            old_mask = _arrow_mask(old.path_xy, (TARGET_H, TARGET_W), FINALIZE_PATH_MASK_THICKNESS)
            max_overlap = max(max_overlap, _mask_iou_arrays(cand_mask, old_mask))
            if max_overlap > FINALIZE_RESURRECT_MAX_PATH_OVERLAP:
                break
        if max_overlap > FINALIZE_RESURRECT_MAX_PATH_OVERLAP:
            continue
        proposals.append((
            float(cand.confidence) + 0.20 * float(cand.support_score) + 0.15 * float(cand.shaft_score),
            max_overlap,
            cand,
        ))

    actions = []
    for _, max_overlap, cand in sorted(proposals, key=lambda x: x[0], reverse=True)[:int(FINALIZE_MAX_RESURRECTED_PER_IMAGE)]:
        cand.meta["Pass-II finalization_resurrected"] = True
        selected.append(cand)
        pair_set.add((cand.tail_node_id, cand.head_node_id))
        actions.append({
            "action": "resurrect_existing_candidate",
            "arrow_id": int(cand.arrow_id),
            "source": _arrow_source(cand),
            "tail_node_id": cand.tail_node_id,
            "head_node_id": cand.head_node_id,
            "confidence": float(cand.confidence),
            "support_score": float(cand.support_score),
            "shaft_score": float(cand.shaft_score),
            "max_path_overlap": float(max_overlap),
        })
    return selected, actions


def _propagate_fixed_endpoint_nodes(arrows, heads, tails):
    heads_by_id = {e.endpoint_id: e for e in heads}
    tails_by_id = {e.endpoint_id: e for e in tails}
    for a in arrows:
        te = tails_by_id.get(a.tail_endpoint_id)
        he = heads_by_id.get(a.head_endpoint_id)
        if te is not None and a.tail_node_id is None:
            a.tail_node_id = te.attached_node_id
        if he is not None and a.head_node_id is None:
            a.head_node_id = he.attached_node_id
        a.self_loop = bool(a.tail_node_id is not None and a.tail_node_id == a.head_node_id)
    return arrows


def _repair_endpoint_objects(endpoints, nodes, cfg, geom):
    fixed = deepcopy(endpoints)
    audits = []
    for ep in fixed:
        if ep.attached_node_id is not None:
            continue
        ranking = aspect_attachment_rankings(ep, nodes, cfg, geom, top_k=3)
        if not ranking["top_candidates"]:
            continue
        top = ranking["top_candidates"][0]
        accepted = bool(
            top["score"] <= float(cfg.node_attach_max_dist) + FINALIZE_ATTACH_RELAX_MODEL_PX
            and ranking["margin_top1_top2"] >= FINALIZE_ATTACH_MIN_MARGIN_MODEL_PX
        )
        audits.append({
            "endpoint_id": ep.endpoint_id,
            "kind": ep.kind,
            "accepted": accepted,
            "top_node_id": top["node_id"],
            "top_score": float(top["score"]),
            "margin": float(ranking["margin_top1_top2"]),
        })
        if accepted:
            ep.attached_node_id = top["node_id"]
            ep.attach_score = float(top["score"])
    return fixed, audits


def _bbox_distance_original(point_xy, bbox_xywh):
    px, py = float(point_xy[0]), float(point_xy[1])
    x, y, w, h = [float(v) for v in bbox_xywh]
    x1, y1 = x + w, y + h
    dx = max(x - px, 0.0, px - x1)
    dy = max(y - py, 0.0, py - y1)
    if dx > 0.0 or dy > 0.0:
        return math.hypot(dx, dy)
    return min(abs(px - x), abs(px - x1), abs(py - y), abs(py - y1))


def _rank_writer_nodes(point_xy, nodes):
    rows = []
    for n in nodes:
        rows.append({"node_id": n.get("id"), "distance": float(_bbox_distance_original(point_xy, n.get("bbox", [0, 0, 0, 0])))})
    rows.sort(key=lambda r: r["distance"])
    margin = rows[1]["distance"] - rows[0]["distance"] if len(rows) > 1 else float("inf")
    return rows, float(margin)

In [ ]:
def refine_nodes_after_topology(structural_writer, resnet_nodes):
    """Reliability-gated node refinement after topology is fixed.

    This agreement-gated refinement is executed only after graph topology is fixed. It cannot
    perturb A*, candidate generation, endpoint reuse, or branch selection.
    """
    graph_nodes = structural_writer.get("nodes", []) or []
    width, height = _writer_size(structural_writer)
    mapping = match_resnet_nodes_to_graph_nodes(graph_nodes, resnet_nodes, width, height)
    denom = max(len(graph_nodes), len(resnet_nodes), 1)
    match_ratio = float(len(mapping) / denom)
    count_delta_ratio = float(
        abs(len(graph_nodes) - len(resnet_nodes)) / max(len(graph_nodes), 1)
    )
    reliable = bool(
        len(mapping) >= min(int(FINALIZE_NODE_MIN_MATCHED), max(1, len(graph_nodes)))
        and match_ratio >= float(FINALIZE_NODE_MIN_MATCH_RATIO)
        and count_delta_ratio <= float(FINALIZE_NODE_MAX_COUNT_DELTA_RATIO)
    )

    if ENABLE_NODE_REFINEMENT and reliable:
        writer = build_refined_node_writer(structural_writer, resnet_nodes)
        decision = "agreement_gated_node_refinement_after_topology"
    else:
        writer = deepcopy(structural_writer)
        decision = (
            "node_finalization_disabled"
            if not ENABLE_NODE_REFINEMENT
            else "retain_structural_topology_due_to_low_node_agreement"
        )

    reattach_actions = []
    if ENABLE_NODE_REFINEMENT and FINALIZE_REATTACH_NULL_ON_REFINED_NODES:
        nodes = writer.get("nodes", []) or []
        diag = math.hypot(width, height)
        max_dist = max(4.0, 0.018 * diag)
        min_margin = max(3.0, 0.004 * diag)
        for edge in writer.get("arrows", []) or []:
            for kind, id_key, hint_key in [
                ("tail", "tail_node_id", "tail_hint"),
                ("head", "head_node_id", "head_hint"),
            ]:
                if edge.get(id_key) is not None:
                    continue
                p = point_from_edge(edge, kind, hint_key)
                if p is None or not nodes:
                    continue
                ranked, margin = _rank_writer_nodes(p, nodes)
                if ranked and ranked[0]["distance"] <= max_dist and margin >= min_margin:
                    edge[id_key] = ranked[0]["node_id"]
                    reattach_actions.append({
                        "action": "attach_null_endpoint_after_node_refinement",
                        "edge_id": edge.get("id"),
                        "kind": kind,
                        "node_id": ranked[0]["node_id"],
                        "distance": float(ranked[0]["distance"]),
                        "margin": float(margin),
                    })

    debug = {
        "decision": decision,
        "reliable": bool(reliable),
        "matched_nodes": int(len(mapping)),
        "graph_nodes": int(len(graph_nodes)),
        "resnet_nodes": int(len(resnet_nodes)),
        "match_ratio": float(match_ratio),
        "count_delta_ratio": float(count_delta_ratio),
        "mapping": {str(k): int(v) for k, v in mapping.items()},
        "reattach_actions": reattach_actions,
    }
    return writer, debug


def finalize_two_pass_graph(
    first_pass_result, image_name, orig_size, family, resnet_nodes_orig=None,
):
    """Pass 2: structural finalization using the already-selected graph.

    The stage order intentionally mirrors the validated successful behaviour:
      1. repair endpoint objects;
      2. propagate repairs to selected arrows;
      3. complete clear selected-arrow attachments;
      4. gate selected fallbacks using the actual graph;
      5. suppress duplicates among selected paths;
      6. resurrect only strong already-generated candidates when graph need remains;
      7. suppress duplicates again;
      8. refine node geometry after topology is frozen.
    """
    orig_w, orig_h = int(orig_size[0]), int(orig_size[1])
    cfg_used = LADDecoderConfig(**first_pass_result.decoder_debug_json["config"])
    geom = AspectGeometry(
        orig_w=orig_w, orig_h=orig_h,
        model_w=int(TARGET_W), model_h=int(TARGET_H),
    )

    fixed_heads, head_audits = _repair_endpoint_objects(
        first_pass_result.heads, first_pass_result.nodes, cfg_used, geom
    )
    fixed_tails, tail_audits = _repair_endpoint_objects(
        first_pass_result.tails, first_pass_result.nodes, cfg_used, geom
    )

    arrows = deepcopy(first_pass_result.arrows)
    arrows = _propagate_fixed_endpoint_nodes(arrows, fixed_heads, fixed_tails)
    arrows, attach_actions = _repair_single_arrow_attachments(
        arrows, fixed_heads, fixed_tails,
        first_pass_result.nodes, cfg_used, geom,
    )
    arrows, fallback_actions = _gate_fallback_arrows(arrows, family)
    arrows, duplicate_actions_1 = _suppress_physical_duplicates(
        arrows, geom, shape_hw=(int(TARGET_H), int(TARGET_W))
    )
    arrows, resurrect_actions = _resurrect_existing_candidates(
        arrows, first_pass_result.candidate_arrows,
        fixed_heads, fixed_tails, geom, family,
    )
    arrows, duplicate_actions_2 = _suppress_physical_duplicates(
        arrows, geom, shape_hw=(int(TARGET_H), int(TARGET_W))
    )

    arrows = sorted(
        arrows, key=lambda a: (_arrow_quality(a), -int(a.arrow_id)), reverse=True
    )
    for idx, arrow in enumerate(arrows, start=1):
        arrow.arrow_id = int(idx)

    writer_model = build_writer_json(
        image_name, int(TARGET_W), int(TARGET_H),
        first_pass_result.nodes, arrows,
    )
    writer_model["coordinate_space"] = "model"
    structural_writer = rescale_writer_json(writer_model, orig_w, orig_h)

    final_writer, node_debug = refine_nodes_after_topology(
        structural_writer, resnet_nodes_orig or []
    )

    aspect_debug = {
        "orig_w": orig_w,
        "orig_h": orig_h,
        "model_w": int(TARGET_W),
        "model_h": int(TARGET_H),
        "scale_x": float(geom.sx),
        "scale_y": float(geom.sy),
        "area_equivalent_scale": float(geom.ref),
        "aspect_ratio": float(geom.aspect_ratio),
        "extreme_aspect": bool(geom.aspect_ratio > 4.0),
    }
    counts = {
        "raw_candidates": int(len(first_pass_result.candidate_arrows)),
        "first_pass_selected_arrows": int(len(first_pass_result.arrows)),
        "endpoint_object_repairs": int(
            sum(int(a["accepted"]) for a in head_audits + tail_audits)
        ),
        "selected_attachment_repairs": int(len(attach_actions)),
        "selected_fallback_removals": int(len(fallback_actions)),
        "selected_duplicate_removals": int(
            len(duplicate_actions_1) + len(duplicate_actions_2)
        ),
        "candidate_resurrections": int(len(resurrect_actions)),
        "structurally_finalized_arrows": int(len(arrows)),
        "final_writer_arrows": int(len(final_writer.get("arrows", []) or [])),
        "matched_node_refinements": int(node_debug["matched_nodes"]),
        "node_reattach_actions": int(len(node_debug["reattach_actions"])),
    }
    debug = {
        "version": "two_pass_evidence_aware_assembler_paper_v1",
        "family": family,
        "pass_1": {
            "decoder": "unchanged_learning_aligned_assembler",
            "candidate_count": int(len(first_pass_result.candidate_arrows)),
            "selected_count": int(len(first_pass_result.arrows)),
        },
        "pass_2": {
            "stage_order": [
                "endpoint_object_repair",
                "selected_arrow_attachment_completion",
                "selected_fallback_gating",
                "selected_physical_duplicate_suppression",
                "limited_candidate_resurrection",
                "final_duplicate_suppression",
                "node_refinement_after_topology",
            ],
            "aspect_geometry": aspect_debug,
            "endpoint_object_audits": {
                "heads": head_audits,
                "tails": tail_audits,
            },
            "actions": {
                "attachment": attach_actions,
                "fallback_gate": fallback_actions,
                "duplicate_suppression_first": duplicate_actions_1,
                "candidate_resurrection": resurrect_actions,
                "duplicate_suppression_final": duplicate_actions_2,
            },
            "node_finalization": node_debug,
        },
        "candidate_counts": {
            "raw_total": int(len(first_pass_result.candidate_arrows)),
            "first_pass_selected": int(len(first_pass_result.arrows)),
            "selected": int(len(final_writer.get("arrows", []) or [])),
        },
        "counts": counts,
        "final_structural_arrows_model": [safe_json(asdict(a)) for a in arrows],
        "final_writer_arrows_original": safe_json(final_writer.get("arrows", []) or []),
        "config": safe_json(asdict(cfg_used)),
    }

    final_writer.setdefault("metadata", {})["assembler"] = {
        "name": "two_pass_evidence_aware_structural_assembler",
        "version": "paper-v1",
        "counts": counts,
        "aspect_geometry": aspect_debug,
        "node_finalization": {
            "decision": node_debug["decision"],
            "reliable": node_debug["reliable"],
            "matched_nodes": node_debug["matched_nodes"],
        },
    }

    return DecodeResult(
        nodes=first_pass_result.nodes,
        heads=fixed_heads,
        tails=fixed_tails,
        candidate_arrows=first_pass_result.candidate_arrows,
        arrows=arrows,
        writer_style_json=writer_model,
        writer_style_original_json=final_writer,
        decoder_debug_json=debug,
        debug_maps=first_pass_result.debug_maps,
        nx_graph=build_networkx_graph(first_pass_result.nodes, arrows),
    )


def decode_two_pass_graph(
    pred, decoder_img, image_name, cfg, orig_size, family,
    resnet_nodes_orig=None, first_pass_result=None,
):
    """Canonical two-pass decode.

    `first_pass_result` should be supplied by the worker so the unchanged assembler
    is run only once. The fallback call is retained for standalone use.
    """
    if first_pass_result is None:
        first_pass_result = decode_graph_learning_aligned(
            pred, decoder_img, image_name, cfg, orig_size=orig_size
        )
    return finalize_two_pass_graph(
        first_pass_result=first_pass_result,
        image_name=image_name,
        orig_size=orig_size,
        family=family,
        resnet_nodes_orig=resnet_nodes_orig,
    )

## 7A. Reviewer 1 parameter-sensitivity experiment

This section implements the agreed **two-stage robustness strategy** without changing the frozen paper configuration.

### Stage 1 — completed group screen

The deterministic assembler parameters are grouped into five functional modules:

1. **G1 — Evidence admission**
2. **G2 — Geometry and attachment**
3. **G3 — Path tracing and recovery**
4. **G4 — Conflict, branch, and duplicate resolution**
5. **G5 — Graph-context finalization and node refinement**

Stage 1 applies a **20% behavioural relaxation** and **20% behavioural tightening** to each group, plus coordinated all-relaxed/all-strict stress tests. The previous 450-image run flagged **G1 — Evidence admission** for Stage 2.

### Stage 2 — implemented in this notebook

Stage 2 decomposes the flagged G1 group into its eight active parameters:

- node extent threshold;
- node center threshold;
- head threshold;
- tail threshold;
- shaft threshold;
- skeleton threshold;
- fused support threshold;
- minimum arrow confidence.

Each parameter is perturbed **individually** while every other assembler parameter remains at the submitted configuration. By default, both relaxed and strict ±20% conditions are evaluated, giving **17 configurations** including the baseline. Image-adaptive thresholds are perturbed only after adaptation, so the experiment changes the effective runtime threshold rather than an overwritten default.

Neural predictions are computed **once per image** and reused across all Stage-2 variants. The final report ranks individual parameters by Link-F1 sensitivity and reports precision/recall, nSEE, loop/branch recall, family-wise effects, and assembler action counts.


In [ ]:

# ============================================================
# REVIEWER 1 — FIVE-GROUP BEHAVIOURAL PARAMETER SENSITIVITY
# ============================================================

import threading
from contextlib import contextmanager

SENSITIVITY_FINALIZER_LOCK = threading.RLock()

# Parameter scopes:
#   direct    -> LADDecoderConfig field changed before Pass I
#   effective -> factor applied *after* image-adaptive threshold/geometry computation
#   global    -> Pass-II/finalizer global changed only inside a thread-safe context
#
# direction:
#   lower_is_relaxed  -> relaxed factor = 1-p, strict = 1+p
#   higher_is_relaxed -> relaxed factor = 1+p, strict = 1-p
#
# This explicitly perturbs BEHAVIOUR rather than blindly multiplying all raw numbers.

SENSITIVITY_GROUP_LABELS = {
    "G1": "Evidence admission",
    "G2": "Geometry and attachment",
    "G3": "Path tracing and recovery",
    "G4": "Conflict, branch, and duplicate resolution",
    "G5": "Graph-context finalization and node refinement",
}

SENSITIVITY_PARAMETER_SPECS = {
    "G1": [
        # Effective map thresholds: applied after auto_adapt.
        {"scope":"effective","name":"node_extent_thresh","direction":"lower_is_relaxed","bounds":(0.01,0.98),
         "why":"Node-region evidence threshold after adaptive calibration."},
        {"scope":"effective","name":"node_center_thresh","direction":"lower_is_relaxed","bounds":(0.01,0.98),
         "why":"Node-center evidence threshold after adaptive calibration."},
        {"scope":"effective","name":"head_thresh","direction":"lower_is_relaxed","bounds":(0.01,0.98),
         "why":"Arrow-head heatmap admission threshold."},
        {"scope":"effective","name":"tail_thresh","direction":"lower_is_relaxed","bounds":(0.01,0.98),
         "why":"Arrow-tail heatmap admission threshold."},
        {"scope":"effective","name":"shaft_thresh","direction":"lower_is_relaxed","bounds":(0.01,0.98),
         "why":"Shaft evidence threshold used to create walkable support."},
        {"scope":"effective","name":"skeleton_thresh","direction":"lower_is_relaxed","bounds":(0.01,0.98),
         "why":"Skeleton evidence threshold used to create walkable support."},
        {"scope":"effective","name":"support_thresh","direction":"lower_is_relaxed","bounds":(0.01,0.98),
         "why":"Fused support threshold after adaptive calibration."},
        {"scope":"direct","name":"min_arrow_confidence","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"Minimum confidence for first-pass candidate selection."},
    ],
    "G2": [
        # Post-adaptive geometry. Note that endpoint NMS has the opposite numerical
        # direction to an attachment tolerance.
        {"scope":"effective","name":"endpoint_nms_dist","direction":"lower_is_relaxed","bounds":(1.0,24.0),
         "why":"Smaller NMS radius preserves more nearby endpoint hypotheses."},
        {"scope":"effective","name":"node_attach_max_dist","direction":"higher_is_relaxed","bounds":(4.0,90.0),
         "why":"Maximum endpoint-to-node attachment tolerance."},
        {"scope":"direct","name":"goal_radius","direction":"higher_is_relaxed","bounds":(1,12),"integer":True,
         "why":"A* goal acceptance radius."},
    ],
    "G3": [
        {"scope":"effective","name":"nearest_walkable_radius","direction":"higher_is_relaxed","bounds":(2,36),"integer":True,
         "why":"Search radius for a nearby walkable start/goal pixel."},
        {"scope":"direct","name":"min_path_len","direction":"lower_is_relaxed","bounds":(2,20),"integer":True,
         "why":"Minimum traced path length."},
        {"scope":"direct","name":"support_cost_weight","direction":"lower_is_relaxed","bounds":(0.05,10.0),
         "why":"Penalty for traversing weak support."},
        {"scope":"direct","name":"direction_cost_weight","direction":"lower_is_relaxed","bounds":(0.05,5.0),
         "why":"Penalty for disagreement with learned direction."},
        {"scope":"direct","name":"progress_cost_weight","direction":"lower_is_relaxed","bounds":(0.01,5.0),
         "why":"Penalty for inconsistent path progress."},
        {"scope":"direct","name":"curvature_cost_weight","direction":"lower_is_relaxed","bounds":(0.01,3.0),
         "why":"Penalty for abrupt path curvature."},
        {"scope":"direct","name":"node_core_cost_weight","direction":"lower_is_relaxed","bounds":(0.05,8.0),
         "why":"Penalty for crossing node interiors."},
        {"scope":"direct","name":"stroke_graph_min_component_area","direction":"lower_is_relaxed","bounds":(4,80),"integer":True,
         "why":"Minimum stroke-component size for fallback graph candidates."},
        {"scope":"direct","name":"stroke_graph_min_path_len","direction":"lower_is_relaxed","bounds":(4,60),"integer":True,
         "why":"Minimum stroke-graph path length."},
        {"scope":"direct","name":"stroke_graph_min_confidence","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"Minimum confidence for a stroke-graph fallback candidate."},
        {"scope":"direct","name":"stroke_graph_node_contact_factor","direction":"higher_is_relaxed","bounds":(0.10,1.20),
         "why":"Node-contact radius as a fraction of median node diagonal."},
        {"scope":"direct","name":"ink_bridge_min_confidence","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"Minimum confidence for an ink-bridge candidate."},
        {"scope":"direct","name":"ink_bridge_min_support_score","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"Minimum learned support for ink-bridge retention."},
        {"scope":"direct","name":"ink_bridge_threshold","direction":"lower_is_relaxed","bounds":(0.05,0.60),
         "why":"Raw-ink threshold floor inside the bridge corridor."},
        {"scope":"direct","name":"ink_bridge_width_factor","direction":"higher_is_relaxed","bounds":(0.02,0.40),
         "why":"Width of the local endpoint-to-endpoint bridge corridor."},
        {"scope":"direct","name":"ink_bridge_progress_cost_weight","direction":"lower_is_relaxed","bounds":(0.01,2.0),
         "why":"Progress penalty inside ink-bridge A*."},
        {"scope":"direct","name":"ink_bridge_support_cost_weight","direction":"lower_is_relaxed","bounds":(0.05,5.0),
         "why":"Support penalty inside ink-bridge A*."},
    ],
    "G4": [
        {"scope":"effective","name":"branch_min_opposite_endpoint_dist","direction":"lower_is_relaxed","bounds":(2.0,60.0),
         "why":"Minimum separation required before endpoint reuse is accepted as a branch."},
        {"scope":"direct","name":"branch_max_path_iou","direction":"higher_is_relaxed","bounds":(0.05,0.98),
         "why":"Maximum path overlap allowed for branch endpoint reuse."},
        {"scope":"effective","name":"duplicate_endpoint_dist","direction":"lower_is_relaxed","bounds":(1.0,30.0),
         "why":"Endpoint-distance criterion for first-pass duplicate suppression."},
        {"scope":"direct","name":"duplicate_path_iou","direction":"higher_is_relaxed","bounds":(0.05,0.98),
         "why":"Path-IoU threshold for first-pass duplicate suppression."},
        {"scope":"direct","name":"port_repair_min_confidence","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"Minimum confidence for crowded-port role repair."},
        {"scope":"direct","name":"port_repair_min_support_score","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"Minimum support for crowded-port role repair."},
        {"scope":"direct","name":"port_repair_min_shaft_score","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"Minimum shaft evidence for crowded-port role repair."},
        {"scope":"direct","name":"port_repair_max_path_iou","direction":"higher_is_relaxed","bounds":(0.05,0.98),
         "why":"Maximum overlap with an already-selected path for port repair."},
        {"scope":"direct","name":"stroke_graph_max_path_iou","direction":"higher_is_relaxed","bounds":(0.05,0.98),
         "why":"Duplicate-overlap threshold among stroke-graph candidates."},
    ],
    "G5": [
        {"scope":"global","name":"FINALIZE_ATTACH_RELAX_MODEL_PX","direction":"higher_is_relaxed","bounds":(1.0,60.0),
         "why":"Additional Pass-II attachment-distance allowance."},
        {"scope":"global","name":"FINALIZE_ATTACH_MIN_MARGIN_MODEL_PX","direction":"lower_is_relaxed","bounds":(1.0,40.0),
         "why":"Minimum top-1 versus top-2 node margin for attachment completion."},
        {"scope":"global","name":"FINALIZE_ATTACH_BOTH_NULL_MIN_MARGIN","direction":"lower_is_relaxed","bounds":(1.0,60.0),
         "why":"Stronger attachment margin required when both endpoints are unresolved."},
        {"scope":"global","name":"FINALIZE_RESURRECT_MIN_CONFIDENCE","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"Minimum confidence for conservative candidate resurrection."},
        {"scope":"global","name":"FINALIZE_RESURRECT_MIN_SUPPORT","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"Minimum support for candidate resurrection."},
        {"scope":"global","name":"FINALIZE_RESURRECT_MIN_SHAFT","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"Minimum shaft evidence for candidate resurrection."},
        {"scope":"global","name":"FINALIZE_RESURRECT_MAX_PATH_OVERLAP","direction":"higher_is_relaxed","bounds":(0.05,0.98),
         "why":"Maximum overlap tolerated when resurrecting a missing structural candidate."},
        {"scope":"global","name":"FINALIZE_DUPLICATE_STRONG_IOU","direction":"higher_is_relaxed","bounds":(0.05,0.98),
         "why":"Higher value makes final physical-duplicate suppression less aggressive."},
        {"scope":"global","name":"FINALIZE_DUPLICATE_FALLBACK_IOU","direction":"higher_is_relaxed","bounds":(0.05,0.98),
         "why":"Overlap threshold for fallback/null-path duplicate suppression."},
        {"scope":"global","name":"FINALIZE_DUPLICATE_ENDPOINT_DIST","direction":"lower_is_relaxed","bounds":(1.0,80.0),
         "why":"Endpoint-distance support for final fallback/null duplicate suppression."},
        {"scope":"global","name":"FINALIZE_INK_MIN_CONFIDENCE","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"FCA Pass-II confidence gate for ink-bridge fallbacks."},
        {"scope":"global","name":"FINALIZE_INK_MIN_SUPPORT","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"FCA Pass-II support gate for ink-bridge fallbacks."},
        {"scope":"global","name":"FINALIZE_INK_MIN_SHAFT","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"FCA Pass-II shaft gate for ink-bridge fallbacks."},
        {"scope":"global","name":"FINALIZE_INK_MIN_DIRECTION","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"FCA Pass-II direction gate for ink-bridge fallbacks."},
        {"scope":"global","name":"FINALIZE_STROKE_MIN_CONFIDENCE","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"FCA Pass-II confidence gate for stroke-graph fallbacks."},
        {"scope":"global","name":"FINALIZE_STROKE_MIN_SUPPORT","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"FCA Pass-II support gate for stroke-graph fallbacks."},
        {"scope":"global","name":"FINALIZE_STROKE_MIN_SHAFT","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"FCA Pass-II shaft gate for stroke-graph fallbacks."},
        {"scope":"global","name":"FINALIZE_STROKE_MIN_DIRECTION","direction":"lower_is_relaxed","bounds":(0.01,0.99),
         "why":"FCA Pass-II direction gate for stroke-graph fallbacks."},
        {"scope":"global","name":"FINALIZE_NODE_MIN_MATCH_RATIO","direction":"lower_is_relaxed","bounds":(0.05,0.99),
         "why":"Minimum graph/ResNet node agreement ratio for node refinement."},
        {"scope":"global","name":"FINALIZE_NODE_MAX_COUNT_DELTA_RATIO","direction":"higher_is_relaxed","bounds":(0.01,0.90),
         "why":"Maximum node-count disagreement tolerated before refinement is rejected."},
        {"scope":"global","name":"NODE_REFINEMENT_IOU_ACCEPT","direction":"lower_is_relaxed","bounds":(0.01,0.90),
         "why":"IoU acceptance threshold when matching ResNet node proposals to graph nodes."},
        {"scope":"global","name":"NODE_REFINEMENT_CENTER_TOLERANCE_RATIO","direction":"higher_is_relaxed","bounds":(0.002,0.10),
         "why":"Image-relative center-distance tolerance for node refinement matching."},
    ],
}

# Freeze nominal Pass-II values once, before any sensitivity context can modify them.
SENSITIVITY_FINALIZER_PARAMETER_NAMES = sorted({
    spec["name"]
    for specs in SENSITIVITY_PARAMETER_SPECS.values()
    for spec in specs
    if spec["scope"] == "global"
})
SENSITIVITY_FINALIZER_BASELINE_VALUES = {
    name: deepcopy(globals()[name])
    for name in SENSITIVITY_FINALIZER_PARAMETER_NAMES
}


def _sens_factor(direction, mode, p=None):
    p = float(SENSITIVITY_STRENGTH if p is None else p)
    if mode == "baseline":
        return 1.0
    if direction == "lower_is_relaxed":
        return (1.0 - p) if mode == "relaxed" else (1.0 + p)
    if direction == "higher_is_relaxed":
        return (1.0 + p) if mode == "relaxed" else (1.0 - p)
    raise ValueError(f"Unknown sensitivity direction: {direction}")


def _sens_scale(value, spec, mode):
    factor = _sens_factor(spec["direction"], mode)
    lo, hi = spec.get("bounds", (-float("inf"), float("inf")))
    scaled = float(value) * factor
    scaled = float(np.clip(scaled, lo, hi))
    if spec.get("integer", False):
        return int(round(scaled))
    return scaled


def sensitivity_stage1_variants():
    variants = [{
        "variant_id": "baseline",
        "group": "BASE",
        "group_label": "Submitted configuration",
        "mode": "baseline",
        "groups": [],
        "is_global_stress_test": False,
    }]
    for gid, label in SENSITIVITY_GROUP_LABELS.items():
        for mode in ("relaxed", "strict"):
            variants.append({
                "variant_id": f"{gid.lower()}_{mode}",
                "group": gid,
                "group_label": label,
                "mode": mode,
                "groups": [gid],
                "is_global_stress_test": False,
            })
    for mode in ("relaxed", "strict"):
        variants.append({
            "variant_id": f"all_{mode}",
            "group": "ALL",
            "group_label": "All five groups",
            "mode": mode,
            "groups": list(SENSITIVITY_GROUP_LABELS.keys()),
            "is_global_stress_test": True,
        })
    return variants


SENSITIVITY_STAGE1_VARIANTS = sensitivity_stage1_variants()


def sensitivity_stage2_variants(group_id=None, modes=None):
    """Individual-parameter variants for the Stage-1 flagged group."""
    gid = str(group_id or SENSITIVITY_STAGE2_GROUP).upper()
    if gid not in SENSITIVITY_PARAMETER_SPECS:
        raise ValueError(f"Unknown Stage-2 sensitivity group: {gid}")
    selected_modes = tuple(modes or SENSITIVITY_STAGE2_MODES)
    bad_modes = [m for m in selected_modes if m not in {"relaxed", "strict"}]
    if bad_modes:
        raise ValueError(f"Stage-2 modes must be relaxed/strict, got: {bad_modes}")

    variants = [{
        "variant_id": "baseline",
        "group": "BASE",
        "group_label": "Submitted configuration",
        "mode": "baseline",
        "groups": [],
        "parameter_names": [],
        "parameter_name": "baseline",
        "parameter_label": "Submitted configuration",
        "is_global_stress_test": False,
        "is_stage2": True,
    }]
    for spec in SENSITIVITY_PARAMETER_SPECS[gid]:
        pname = spec["name"]
        for mode in selected_modes:
            variants.append({
                "variant_id": f"{gid.lower()}_{pname}_{mode}",
                "group": gid,
                "group_label": SENSITIVITY_GROUP_LABELS[gid],
                "mode": mode,
                "groups": [gid],
                "parameter_names": [pname],
                "parameter_name": pname,
                "parameter_label": spec.get("why", pname),
                "is_global_stress_test": False,
                "is_stage2": True,
            })
    return variants


SENSITIVITY_STAGE2_VARIANTS = sensitivity_stage2_variants()


def active_sensitivity_variants():
    stage = str(SENSITIVITY_STAGE).strip().lower()
    if stage == "stage1":
        return SENSITIVITY_STAGE1_VARIANTS
    if stage == "stage2":
        return SENSITIVITY_STAGE2_VARIANTS
    raise ValueError('SENSITIVITY_STAGE must be "stage1" or "stage2".')


ACTIVE_SENSITIVITY_VARIANTS = active_sensitivity_variants()


def _specs_for_variant(variant):
    specs = []
    selected_names = set(variant.get("parameter_names", []) or [])
    for gid in variant.get("groups", []):
        for spec in SENSITIVITY_PARAMETER_SPECS[gid]:
            if selected_names and spec["name"] not in selected_names:
                continue
            row = dict(spec)
            row["group"] = gid
            row["group_label"] = SENSITIVITY_GROUP_LABELS[gid]
            specs.append(row)
    return specs


def build_sensitivity_decoder_cfg(base_cfg, variant):
    cfg = deepcopy(base_cfg)
    cfg.sensitivity_effective_factors = {}
    mode = variant["mode"]
    if mode == "baseline":
        return cfg

    for spec in _specs_for_variant(variant):
        if spec["scope"] == "direct":
            if not hasattr(cfg, spec["name"]):
                raise AttributeError(f"Decoder config has no field {spec['name']}")
            setattr(cfg, spec["name"], _sens_scale(getattr(cfg, spec["name"]), spec, mode))
        elif spec["scope"] == "effective":
            cfg.sensitivity_effective_factors[spec["name"]] = _sens_factor(spec["direction"], mode)
    return cfg


def build_sensitivity_finalizer_overrides(variant):
    overrides = {}
    mode = variant["mode"]
    if mode == "baseline":
        return overrides
    for spec in _specs_for_variant(variant):
        if spec["scope"] != "global":
            continue
        name = spec["name"]
        if name not in SENSITIVITY_FINALIZER_BASELINE_VALUES:
            raise KeyError(f"Finalizer sensitivity parameter is not defined: {name}")
        overrides[name] = _sens_scale(
            SENSITIVITY_FINALIZER_BASELINE_VALUES[name], spec, mode
        )
    return overrides


@contextmanager
def sensitivity_finalizer_context(overrides):
    """Thread-safe temporary Pass-II overrides.

    Every finalizer call in a sensitivity-enabled run uses this lock, including
    the baseline finalizer, so two hosted-runtime GPU worker threads cannot observe
    different temporary global values.
    """
    with SENSITIVITY_FINALIZER_LOCK:
        old = {}
        try:
            for name, value in (overrides or {}).items():
                old[name] = globals()[name]
                globals()[name] = value
            yield
        finally:
            for name, value in old.items():
                globals()[name] = value


def sensitivity_finalize_two_pass(
    first_pass_result, image_name, orig_size, family, resnet_nodes_orig, variant
):
    overrides = build_sensitivity_finalizer_overrides(variant)
    with sensitivity_finalizer_context(overrides):
        return finalize_two_pass_graph(
            first_pass_result=first_pass_result,
            image_name=image_name,
            orig_size=orig_size,
            family=family,
            resnet_nodes_orig=resnet_nodes_orig,
        )


def _sensitivity_action_counts(final_result, prediction):
    counts = final_result.decoder_debug_json.get("counts", {}) or {}
    return {
        "first_pass_candidates": int(counts.get("raw_candidates", len(final_result.candidate_arrows))),
        "first_pass_selected_edges": int(counts.get("first_pass_selected_arrows", 0)),
        "final_selected_edges": int(len(prediction.get("arrows", []) or [])),
        "endpoint_repairs": int(
            counts.get("endpoint_object_repairs", 0)
            + counts.get("selected_attachment_repairs", 0)
        ),
        "fallback_removals": int(counts.get("selected_fallback_removals", 0)),
        "duplicate_removals": int(counts.get("selected_duplicate_removals", 0)),
        "candidate_resurrections": int(counts.get("candidate_resurrections", 0)),
        "matched_node_refinements": int(counts.get("matched_node_refinements", 0)),
    }


def run_sensitivity_variants_for_sample(
    sample,
    evidence_maps,
    decoder_image,
    original_size,
    resnet_nodes,
    ground_truth,
    base_decoder_config,
    baseline_first_pass,
    baseline_final_result,
):
    """Run the active sensitivity configurations without repeating neural inference."""
    rows, failures = [], []

    for variant in ACTIVE_SENSITIVITY_VARIANTS:
        try:
            t0 = time.time()

            if variant["variant_id"] == "baseline":
                first_pass_variant = baseline_first_pass
                final_variant = baseline_final_result
            else:
                cfg_variant = build_sensitivity_decoder_cfg(base_decoder_config, variant)

                # G5-only perturbations do not alter Pass I, so reuse it exactly.
                if set(variant["groups"]) == {"G5"}:
                    first_pass_variant = baseline_first_pass
                else:
                    first_pass_variant = decode_graph_learning_aligned(
                        pred=evidence_maps,
                        decoder_img=decoder_image,
                        image_name=sample.image_path.name,
                        cfg=cfg_variant,
                        orig_size=original_size,
                    )

                final_variant = sensitivity_finalize_two_pass(
                    first_pass_result=first_pass_variant,
                    image_name=sample.image_path.name,
                    orig_size=original_size,
                    family=sample.family,
                    resnet_nodes_orig=resnet_nodes,
                    variant=variant,
                )

            prediction = (
                final_variant.writer_style_original_json
                or final_variant.writer_style_json
            )

            metric_row = evaluate_prediction_by_profile(
                prediction,
                ground_truth,
                profile=SENSITIVITY_METRIC_PROFILE,
            )
            metric_row.update({
                "variant_id": variant["variant_id"],
                "sensitivity_group": variant["group"],
                "sensitivity_group_label": variant["group_label"],
                "sensitivity_mode": variant["mode"],
                "sensitivity_stage": str(SENSITIVITY_STAGE),
                "sensitivity_parameter": variant.get("parameter_name", ""),
                "sensitivity_parameter_label": variant.get("parameter_label", ""),
                "is_global_stress_test": bool(variant["is_global_stress_test"]),
                "sample_id": sample.sample_id,
                "image_name": sample.image_path.name,
                "family": sample.family,
                "split": sample.split,
                "sensitivity_strength": float(SENSITIVITY_STRENGTH),
                "elapsed_seconds_variant": float(time.time() - t0),
            })
            metric_row.update(_sensitivity_action_counts(final_variant, prediction))
            rows.append(metric_row)

            if SENSITIVITY_SAVE_VARIANT_JSON:
                out_path = (
                    Path(OUTPUT_ROOT)
                    / str(SENSITIVITY_OUTPUT_SUBDIR)
                    / "variant_predictions"
                    / variant["variant_id"]
                    / sample.family
                    / sample.split
                    / f"{sample.sample_id}.json"
                )
                save_json_local(out_path, prediction)

            if SENSITIVITY_PRINT_PER_VARIANT:
                print(
                    f"    [sens] {sample.sample_id} {variant['variant_id']}: "
                    f"LinkF1={100.0*float(metric_row.get('link_f1',0.0)):.2f}"
                )

        except Exception as exc:
            failures.append({
                "variant_id": variant["variant_id"],
                "sensitivity_group": variant["group"],
                "sensitivity_mode": variant["mode"],
                "sample_id": sample.sample_id,
                "family": sample.family,
                "split": sample.split,
                "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc(limit=60),
            })

    return rows, failures


def _collect_sensitivity_rows(pattern):
    rows = []
    shard_dir = Path(OUTPUT_ROOT) / "shards"
    for p in sorted(shard_dir.glob(pattern)):
        rows.extend(read_jsonl(p))
    return rows


def _sens_pct(x):
    try:
        return 100.0 * float(x)
    except Exception:
        return float("nan")


def _sens_markdown_table(rows, columns):
    if not rows:
        return "_No rows._"
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for r in rows:
        vals = []
        for c in columns:
            v = r.get(c, "")
            if isinstance(v, float):
                if math.isnan(v):
                    vals.append("NA")
                else:
                    vals.append(f"{v:.3f}")
            else:
                vals.append(str(v))
        body.append("| " + " | ".join(vals) + " |")
    return "\n".join([header, sep] + body)


def sensitivity_parameter_manifest(base_cfg):
    rows = []
    for gid, specs in SENSITIVITY_PARAMETER_SPECS.items():
        for spec in specs:
            if spec["scope"] == "direct":
                nominal = getattr(base_cfg, spec["name"])
            elif spec["scope"] == "global":
                nominal = SENSITIVITY_FINALIZER_BASELINE_VALUES.get(spec["name"], "MISSING")
            else:
                nominal = "adaptive per image"
            rows.append({
                "group": gid,
                "group_label": SENSITIVITY_GROUP_LABELS[gid],
                "scope": spec["scope"],
                "parameter": spec["name"],
                "nominal": nominal,
                "relaxed_factor": _sens_factor(spec["direction"], "relaxed"),
                "strict_factor": _sens_factor(spec["direction"], "strict"),
                "behaviour_rule": spec["direction"],
                "rationale": spec["why"],
            })
    return rows


def finalize_parameter_sensitivity_stage1():
    """Aggregate Stage-1 results, print them, and write a paste-ready report."""
    rows = _collect_sensitivity_rows("worker_*_sensitivity.jsonl")
    failures = _collect_sensitivity_rows("worker_*_sensitivity_failures.jsonl")
    if not rows:
        raise RuntimeError(
            "No parameter-sensitivity rows were found. "
            "Run full inference with RUN_PARAMETER_SENSITIVITY=True."
        )

    out_dir = Path(OUTPUT_ROOT) / str(SENSITIVITY_OUTPUT_SUBDIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    variants = {v["variant_id"]: v for v in SENSITIVITY_STAGE1_VARIANTS}
    overall_rows = []
    family_rows = []
    action_rows = []

    for variant_id in [v["variant_id"] for v in SENSITIVITY_STAGE1_VARIANTS]:
        items = [r for r in rows if r.get("variant_id") == variant_id]
        if not items:
            continue
        meta = variants[variant_id]
        agg = aggregate_revision_metrics(items, group_keys=[])[0]
        family = aggregate_revision_metrics(items, group_keys=["family"])

        overall_rows.append({
            "variant_id": variant_id,
            "group": meta["group"],
            "group_label": meta["group_label"],
            "mode": meta["mode"],
            "images": int(agg["num_images"]),
            "node_f1": _sens_pct(agg["node_f1"]),
            "edge_precision": _sens_pct(agg["edge_precision"]),
            "edge_recall": _sens_pct(agg["edge_recall"]),
            "link_f1": _sens_pct(agg["link_f1"]),
            "nsee": float(agg["mean_normalized_structural_edit_error"]),
            "loop_recall": _sens_pct(agg["loop_back_recall"]),
            "branch_recall": _sens_pct(agg["decision_branch_recall"]),
            "matched_link_coverage": _sens_pct(agg["matched_link_coverage"]),
        })

        for fr in family:
            family_rows.append({
                "variant_id": variant_id,
                "group": meta["group"],
                "group_label": meta["group_label"],
                "mode": meta["mode"],
                "family": str(fr.get("family", "")).upper(),
                "images": int(fr["num_images"]),
                "edge_precision": _sens_pct(fr["edge_precision"]),
                "edge_recall": _sens_pct(fr["edge_recall"]),
                "link_f1": _sens_pct(fr["link_f1"]),
                "nsee": float(fr["mean_normalized_structural_edit_error"]),
                "loop_recall": _sens_pct(fr["loop_back_recall"]),
                "branch_recall": _sens_pct(fr["decision_branch_recall"]),
            })

        action_names = [
            "first_pass_candidates",
            "first_pass_selected_edges",
            "final_selected_edges",
            "endpoint_repairs",
            "fallback_removals",
            "duplicate_removals",
            "candidate_resurrections",
            "matched_node_refinements",
        ]
        arow = {
            "variant_id": variant_id,
            "group": meta["group"],
            "mode": meta["mode"],
            "images": len(items),
        }
        for name in action_names:
            vals = [float(r.get(name, 0.0) or 0.0) for r in items]
            arow[f"{name}_total"] = int(round(sum(vals)))
            arow[f"{name}_per_image"] = float(np.mean(vals)) if vals else float("nan")
        action_rows.append(arow)

    baseline = next((r for r in overall_rows if r["variant_id"] == "baseline"), None)
    if baseline is None:
        raise RuntimeError("Baseline sensitivity row is missing.")

    for r in overall_rows:
        r["delta_link_f1_pp"] = float(r["link_f1"] - baseline["link_f1"])
        r["delta_edge_precision_pp"] = float(r["edge_precision"] - baseline["edge_precision"])
        r["delta_edge_recall_pp"] = float(r["edge_recall"] - baseline["edge_recall"])
        r["delta_nsee"] = float(r["nsee"] - baseline["nsee"])
        r["delta_loop_recall_pp"] = float(r["loop_recall"] - baseline["loop_recall"])
        r["delta_branch_recall_pp"] = float(r["branch_recall"] - baseline["branch_recall"])

    baseline_family = {
        r["family"]: r for r in family_rows if r["variant_id"] == "baseline"
    }
    for r in family_rows:
        b = baseline_family.get(r["family"])
        r["delta_link_f1_pp"] = float(r["link_f1"] - b["link_f1"]) if b else float("nan")
        r["delta_nsee"] = float(r["nsee"] - b["nsee"]) if b else float("nan")

    screening_rows = []
    for gid, label in SENSITIVITY_GROUP_LABELS.items():
        candidates = [r for r in overall_rows if r["group"] == gid]
        if not candidates:
            continue
        worst_abs = max(candidates, key=lambda r: abs(r["delta_link_f1_pp"]))
        worst_drop = min(candidates, key=lambda r: r["delta_link_f1_pp"])
        screening_rows.append({
            "group": gid,
            "group_label": label,
            "largest_abs_delta_link_f1_pp": abs(float(worst_abs["delta_link_f1_pp"])),
            "condition_for_largest_abs_delta": worst_abs["mode"],
            "worst_drop_link_f1_pp": float(worst_drop["delta_link_f1_pp"]),
            "worst_drop_condition": worst_drop["mode"],
            "max_nsee_increase": max(float(r["delta_nsee"]) for r in candidates),
            "stage2_flag": bool(
                abs(float(worst_abs["delta_link_f1_pp"]))
                >= float(SENSITIVITY_STAGE2_TRIGGER_PP)
            ),
        })
    screening_rows.sort(
        key=lambda r: r["largest_abs_delta_link_f1_pp"], reverse=True
    )

    # Save machine-readable results.
    write_rows_csv_clean(out_dir / "stage1_overall.csv", overall_rows)
    write_rows_csv_clean(out_dir / "stage1_by_family.csv", family_rows)
    write_rows_csv_clean(out_dir / "stage1_action_counts.csv", action_rows)
    write_rows_csv_clean(out_dir / "stage2_screening.csv", screening_rows)
    write_rows_csv_clean(out_dir / "stage1_failures.csv", failures)

    base_cfg = build_decoder_cfg(make_decoder_args(device_str=None))
    manifest = sensitivity_parameter_manifest(base_cfg)
    write_rows_csv_clean(out_dir / "parameter_manifest.csv", manifest)

    result_json = {
        "strength": float(SENSITIVITY_STRENGTH),
        "metric_profile": str(SENSITIVITY_METRIC_PROFILE),
        "stage2_trigger_link_f1_pp": float(SENSITIVITY_STAGE2_TRIGGER_PP),
        "overall": overall_rows,
        "by_family": family_rows,
        "action_counts": action_rows,
        "stage2_screening": screening_rows,
        "failures": failures,
        "parameter_manifest": manifest,
    }
    save_json_local(out_dir / "stage1_results.json", result_json)

    # Compact plot: overall Link F1 deviation from the submitted configuration.
    try:
        import matplotlib.pyplot as plt
        plot_rows = [r for r in overall_rows if r["variant_id"] != "baseline"]
        fig, ax = plt.subplots(figsize=(12, 5))
        labels = [r["variant_id"] for r in plot_rows]
        values = [r["delta_link_f1_pp"] for r in plot_rows]
        ax.bar(range(len(labels)), values)
        ax.axhline(0.0, linewidth=1)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha="right")
        ax.set_ylabel("Δ Link F1 (percentage points)")
        ax.set_title("Reviewer 1 sensitivity screen: deviation from baseline")
        fig.tight_layout()
        fig.savefig(out_dir / "stage1_link_f1_delta.png", dpi=180, bbox_inches="tight")
        plt.close(fig)
    except Exception as exc:
        print("Sensitivity plot skipped:", exc)

    group_only = [
        r for r in overall_rows
        if r["group"] in SENSITIVITY_GROUP_LABELS
    ]
    group_link_values = [float(r["link_f1"]) for r in group_only]
    min_group_link = min(group_link_values) if group_link_values else float("nan")
    max_group_link = max(group_link_values) if group_link_values else float("nan")
    max_abs_group_delta = max(
        (abs(float(r["delta_link_f1_pp"])) for r in group_only),
        default=float("nan"),
    )
    most_sensitive = screening_rows[0] if screening_rows else None

    global_relaxed = next(
        (r for r in overall_rows if r["variant_id"] == "all_relaxed"), None
    )
    global_strict = next(
        (r for r in overall_rows if r["variant_id"] == "all_strict"), None
    )

    stage2_flagged = [r for r in screening_rows if r["stage2_flag"]]

    report = []
    report.append("# Reviewer 1 — Parameter Sensitivity Stage-1 Report")
    report.append("")
    report.append(
        f"**Perturbation:** ±{100*SENSITIVITY_STRENGTH:.0f}% in behavioural space.  "
        f"**Metric profile:** `{SENSITIVITY_METRIC_PROFILE}`.  "
        f"**Stage-2 screening trigger:** |Δ Link F1| ≥ {SENSITIVITY_STAGE2_TRIGGER_PP:.2f} percentage points "
        "(an operational screening rule, not a statistical significance threshold)."
    )
    report.append("")
    report.append("## 1. Experimental design")
    report.append("")
    report.append(
        "Continuous decision parameters were grouped by function rather than swept combinatorially. "
        "For each group, one relaxed and one strict configuration was created. Minimum evidence "
        "requirements were lowered under relaxation, geometric tolerances were widened, and duplicate "
        "suppression was made less aggressive; the strict condition applied the opposite behavioural "
        "change. Image-adaptive values were perturbed only after their per-image adaptation, ensuring "
        "that the experiment modifies effective runtime behaviour rather than inactive defaults."
    )
    report.append("")
    report.append(
        "The graph-evidence network and ResNet34 node model are identical for every condition. "
        "Their predictions are computed once per image and reused across the 13 deterministic assembly "
        "configurations. Thus the experiment isolates sensitivity of graph assembly."
    )
    report.append("")
    report.append("## 2. Five parameter groups")
    report.append("")
    group_desc = [
        {"Group":"G1","Meaning":"Evidence admission","Question":"How much learned evidence is admitted into graph construction?"},
        {"Group":"G2","Meaning":"Geometry and attachment","Question":"How tolerant are endpoint/node attachment and local search geometry?"},
        {"Group":"G3","Meaning":"Path tracing and recovery","Question":"How permissive are A* tracing and fallback path-generation mechanisms?"},
        {"Group":"G4","Meaning":"Conflict, branch, and duplicate resolution","Question":"How aggressively are competing, branched, repaired, or overlapping hypotheses retained?"},
        {"Group":"G5","Meaning":"Graph-context finalization and node refinement","Question":"How aggressively may Pass II repair, recover, suppress, and refine the provisional graph?"},
    ]
    report.append(_sens_markdown_table(group_desc, ["Group","Meaning","Question"]))
    report.append("")
    report.append("## 3. Overall Stage-1 results")
    report.append("")
    pretty_overall = []
    for r in overall_rows:
        pretty_overall.append({
            "Variant": r["variant_id"],
            "Group": r["group"],
            "Link F1": round(r["link_f1"], 3),
            "Δ Link F1": round(r["delta_link_f1_pp"], 3),
            "Precision": round(r["edge_precision"], 3),
            "Recall": round(r["edge_recall"], 3),
            "nSEE": round(r["nsee"], 4),
            "Δ nSEE": round(r["delta_nsee"], 4),
            "Loop Rec.": round(r["loop_recall"], 3),
            "Branch Rec.": round(r["branch_recall"], 3),
        })
    report.append(_sens_markdown_table(
        pretty_overall,
        ["Variant","Group","Link F1","Δ Link F1","Precision","Recall","nSEE","Δ nSEE","Loop Rec.","Branch Rec."]
    ))
    report.append("")
    report.append("## 4. Family-wise Link F1")
    report.append("")
    pretty_family = []
    for r in family_rows:
        pretty_family.append({
            "Variant": r["variant_id"],
            "Family": r["family"],
            "Link F1": round(r["link_f1"], 3),
            "Δ Link F1": round(r["delta_link_f1_pp"], 3),
            "nSEE": round(r["nsee"], 4),
        })
    report.append(_sens_markdown_table(
        pretty_family, ["Variant","Family","Link F1","Δ Link F1","nSEE"]
    ))
    report.append("")
    report.append("## 5. Stage-2 screening")
    report.append("")
    report.append(_sens_markdown_table(
        screening_rows,
        ["group","group_label","largest_abs_delta_link_f1_pp",
         "condition_for_largest_abs_delta","worst_drop_link_f1_pp",
         "worst_drop_condition","max_nsee_increase","stage2_flag"]
    ))
    report.append("")
    if stage2_flagged:
        report.append(
            "**Groups flagged for Stage 2:** "
            + ", ".join(f"{r['group']} ({r['group_label']})" for r in stage2_flagged)
            + ". These groups should be decomposed into individual active parameters in the next experiment."
        )
    else:
        report.append(
            "**No group crosses the configured Stage-2 screening trigger.** "
            "The full table should still be inspected before deciding whether a parameter-level drill-down is unnecessary."
        )
    report.append("")
    report.append("## 6. Mechanistic action counts")
    report.append("")
    report.append(
        "The CSV `stage1_action_counts.csv` records, for every variant, the number of first-pass "
        "candidates, selected edges, endpoint repairs, fallback removals, duplicate removals, "
        "candidate resurrections, and matched node refinements. Use these counts to explain *why* "
        "precision or recall moved rather than reporting only the final score."
    )
    report.append("")
    report.append("## 7. Robustness envelope")
    report.append("")
    report.append(
        f"Baseline Link F1 = **{baseline['link_f1']:.3f}%**. Across the ten one-group "
        f"±20% perturbations, Link F1 spans **{min_group_link:.3f}% to {max_group_link:.3f}%**, "
        f"with a maximum absolute deviation of **{max_abs_group_delta:.3f} percentage points**."
    )
    if most_sensitive:
        report.append(
            f"The largest group-level change is observed for **{most_sensitive['group']} — "
            f"{most_sensitive['group_label']}**, with |Δ Link F1| = "
            f"**{most_sensitive['largest_abs_delta_link_f1_pp']:.3f} p.p.** "
            f"under the {most_sensitive['condition_for_largest_abs_delta']} condition."
        )
    if global_relaxed and global_strict:
        report.append(
            f"The coordinated all-groups stress tests produce Link F1 = "
            f"**{global_relaxed['link_f1']:.3f}%** (relaxed; Δ {global_relaxed['delta_link_f1_pp']:+.3f} p.p.) "
            f"and **{global_strict['link_f1']:.3f}%** (strict; Δ {global_strict['delta_link_f1_pp']:+.3f} p.p.)."
        )
    report.append("")
    report.append("## 8. Paper-ready methods wording")
    report.append("")
    report.append(
        "To assess robustness to deterministic assembly parameters, continuous decision thresholds "
        "were organized into five functional groups: evidence admission, geometry and attachment, "
        "path tracing and recovery, conflict/branch/duplicate resolution, and graph-context finalization. "
        "Each group was perturbed by ±20% in behavioural space while all remaining groups were held at "
        "the submitted configuration. A relaxed condition lowered minimum evidence requirements and "
        "widened admissible tolerances, whereas a strict condition applied the converse changes. "
        "Image-adaptive thresholds and distances were perturbed after per-image adaptation. In addition, "
        "coordinated all-relaxed and all-strict configurations were evaluated as stress tests. Neural "
        "predictions were shared across conditions so that only deterministic graph assembly changed."
    )
    report.append("")
    report.append("## 9. Paper-ready result wording (generated from this run)")
    report.append("")
    if most_sensitive:
        result_sentence = (
            f"Across the ten group-wise ±20% perturbations, directed Link F1 ranged from "
            f"{min_group_link:.2f}% to {max_group_link:.2f}% around the baseline of "
            f"{baseline['link_f1']:.2f}%. The largest absolute group-level change was "
            f"{most_sensitive['largest_abs_delta_link_f1_pp']:.2f} percentage points for "
            f"{most_sensitive['group']} ({most_sensitive['group_label']})."
        )
    else:
        result_sentence = "No group-wise results were available."
    report.append(result_sentence)
    if global_relaxed and global_strict:
        report.append(
            f"The coordinated stress tests yielded {global_relaxed['link_f1']:.2f}% Link F1 "
            f"for the relaxed setting and {global_strict['link_f1']:.2f}% for the strict setting. "
            "Family-wise results and precision/recall decomposition are reported separately to expose "
            "whether any sensitivity is concentrated in one diagram family or arises from over- versus under-generation."
        )
    report.append("")
    report.append("## 10. Files produced")
    report.append("")
    report.append(
        "- `stage1_overall.csv` — headline metrics and deltas for all 13 configurations\n"
        "- `stage1_by_family.csv` — FA/FCA/FCB sensitivity results\n"
        "- `stage1_action_counts.csv` — assembler mechanism counts\n"
        "- `stage2_screening.csv` — group ranking and Stage-2 flags\n"
        "- `parameter_manifest.csv` — every perturbed parameter and direction\n"
        "- `stage1_results.json` — complete machine-readable report\n"
        "- `stage1_link_f1_delta.png` — overall sensitivity visualization\n"
        "- `stage1_report.md` — this paste-ready report"
    )

    report_text = "\n".join(report) + "\n"
    (out_dir / "stage1_report.md").write_text(report_text, encoding="utf-8")

    print("\n" + "="*88)
    print("REVIEWER 1 PARAMETER SENSITIVITY — STAGE 1")
    print("="*88)
    print(report_text)

    return result_json

# ============================================================
# STAGE 2 — INDIVIDUAL-PARAMETER DRILL-DOWN FOR FLAGGED GROUP
# ============================================================

def _stage2_parameter_spec_map(group_id):
    return {spec["name"]: spec for spec in SENSITIVITY_PARAMETER_SPECS[group_id]}


def finalize_parameter_sensitivity_stage2():
    """Aggregate individual-parameter Stage-2 results and write a paste-ready report."""
    rows = _collect_sensitivity_rows("worker_*_sensitivity.jsonl")
    failures = _collect_sensitivity_rows("worker_*_sensitivity_failures.jsonl")
    if not rows:
        raise RuntimeError(
            "No Stage-2 sensitivity rows were found. Run full inference with "
            "RUN_PARAMETER_SENSITIVITY=True and SENSITIVITY_STAGE='stage2'."
        )

    out_dir = Path(OUTPUT_ROOT) / str(SENSITIVITY_OUTPUT_SUBDIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    gid = str(SENSITIVITY_STAGE2_GROUP).upper()
    variants = {v["variant_id"]: v for v in SENSITIVITY_STAGE2_VARIANTS}
    ordered_ids = [v["variant_id"] for v in SENSITIVITY_STAGE2_VARIANTS]
    overall_rows, family_rows, action_rows = [], [], []

    for variant_id in ordered_ids:
        items = [r for r in rows if r.get("variant_id") == variant_id]
        if not items:
            continue
        meta = variants[variant_id]
        agg = aggregate_revision_metrics(items, group_keys=[])[0]
        family = aggregate_revision_metrics(items, group_keys=["family"])

        overall_rows.append({
            "variant_id": variant_id,
            "group": meta["group"],
            "group_label": meta["group_label"],
            "parameter": meta.get("parameter_name", "baseline"),
            "parameter_label": meta.get("parameter_label", ""),
            "mode": meta["mode"],
            "images": int(agg["num_images"]),
            "node_f1": _sens_pct(agg["node_f1"]),
            "edge_precision": _sens_pct(agg["edge_precision"]),
            "edge_recall": _sens_pct(agg["edge_recall"]),
            "link_f1": _sens_pct(agg["link_f1"]),
            "nsee": float(agg["mean_normalized_structural_edit_error"]),
            "loop_recall": _sens_pct(agg["loop_back_recall"]),
            "branch_recall": _sens_pct(agg["decision_branch_recall"]),
            "matched_link_coverage": _sens_pct(agg["matched_link_coverage"]),
        })

        for fr in family:
            family_rows.append({
                "variant_id": variant_id,
                "group": meta["group"],
                "parameter": meta.get("parameter_name", "baseline"),
                "parameter_label": meta.get("parameter_label", ""),
                "mode": meta["mode"],
                "family": str(fr.get("family", "")).upper(),
                "images": int(fr["num_images"]),
                "edge_precision": _sens_pct(fr["edge_precision"]),
                "edge_recall": _sens_pct(fr["edge_recall"]),
                "link_f1": _sens_pct(fr["link_f1"]),
                "nsee": float(fr["mean_normalized_structural_edit_error"]),
                "loop_recall": _sens_pct(fr["loop_back_recall"]),
                "branch_recall": _sens_pct(fr["decision_branch_recall"]),
            })

        action_names = [
            "first_pass_candidates", "first_pass_selected_edges", "final_selected_edges",
            "endpoint_repairs", "fallback_removals", "duplicate_removals",
            "candidate_resurrections", "matched_node_refinements",
        ]
        arow = {
            "variant_id": variant_id,
            "parameter": meta.get("parameter_name", "baseline"),
            "mode": meta["mode"],
            "images": len(items),
        }
        for name in action_names:
            vals = [float(r.get(name, 0.0)) for r in items]
            arow[f"total_{name}"] = int(round(sum(vals)))
            arow[f"mean_{name}_per_image"] = float(np.mean(vals)) if vals else 0.0
        action_rows.append(arow)

    baseline = next((r for r in overall_rows if r["variant_id"] == "baseline"), None)
    if baseline is None:
        raise RuntimeError("Stage-2 baseline row is missing.")

    for r in overall_rows:
        r["delta_link_f1_pp"] = float(r["link_f1"] - baseline["link_f1"])
        r["delta_edge_precision_pp"] = float(r["edge_precision"] - baseline["edge_precision"])
        r["delta_edge_recall_pp"] = float(r["edge_recall"] - baseline["edge_recall"])
        r["delta_nsee"] = float(r["nsee"] - baseline["nsee"])
        r["delta_loop_recall_pp"] = float(r["loop_recall"] - baseline["loop_recall"])
        r["delta_branch_recall_pp"] = float(r["branch_recall"] - baseline["branch_recall"])

    baseline_family = {r["family"]: r for r in family_rows if r["variant_id"] == "baseline"}
    for r in family_rows:
        b = baseline_family.get(r["family"])
        r["delta_link_f1_pp"] = float(r["link_f1"] - b["link_f1"]) if b else float("nan")
        r["delta_nsee"] = float(r["nsee"] - b["nsee"]) if b else float("nan")

    spec_map = _stage2_parameter_spec_map(gid)
    ranking_rows = []
    for pname, spec in spec_map.items():
        candidates = [r for r in overall_rows if r.get("parameter") == pname]
        if not candidates:
            continue
        max_abs = max(candidates, key=lambda r: abs(r["delta_link_f1_pp"]))
        worst = min(candidates, key=lambda r: r["delta_link_f1_pp"])
        best = max(candidates, key=lambda r: r["delta_link_f1_pp"])
        relaxed = next((r for r in candidates if r["mode"] == "relaxed"), None)
        strict = next((r for r in candidates if r["mode"] == "strict"), None)
        ranking_rows.append({
            "parameter": pname,
            "scope": spec["scope"],
            "behaviour_rule": spec["direction"],
            "largest_abs_delta_link_f1_pp": abs(float(max_abs["delta_link_f1_pp"])),
            "condition_for_largest_abs_delta": max_abs["mode"],
            "worst_delta_link_f1_pp": float(worst["delta_link_f1_pp"]),
            "worst_condition": worst["mode"],
            "best_delta_link_f1_pp": float(best["delta_link_f1_pp"]),
            "best_condition": best["mode"],
            "strict_delta_link_f1_pp": float(strict["delta_link_f1_pp"]) if strict else float("nan"),
            "relaxed_delta_link_f1_pp": float(relaxed["delta_link_f1_pp"]) if relaxed else float("nan"),
            "max_nsee_increase": max(float(r["delta_nsee"]) for r in candidates),
            "rationale": spec["why"],
        })
    ranking_rows.sort(key=lambda r: r["largest_abs_delta_link_f1_pp"], reverse=True)

    # Stage-2 manifest restricted to the drilled-down group.
    base_cfg = build_decoder_cfg(make_decoder_args(device_str=None))
    manifest = [r for r in sensitivity_parameter_manifest(base_cfg) if r["group"] == gid]

    write_rows_csv_clean(out_dir / "stage2_overall.csv", overall_rows)
    write_rows_csv_clean(out_dir / "stage2_by_family.csv", family_rows)
    write_rows_csv_clean(out_dir / "stage2_action_counts.csv", action_rows)
    write_rows_csv_clean(out_dir / "stage2_parameter_ranking.csv", ranking_rows)
    write_rows_csv_clean(out_dir / "stage2_parameter_manifest.csv", manifest)
    write_rows_csv_clean(out_dir / "stage2_failures.csv", failures)

    result_json = {
        "stage": "stage2",
        "group": gid,
        "group_label": SENSITIVITY_GROUP_LABELS[gid],
        "strength": float(SENSITIVITY_STRENGTH),
        "modes": list(SENSITIVITY_STAGE2_MODES),
        "metric_profile": str(SENSITIVITY_METRIC_PROFILE),
        "overall": overall_rows,
        "by_family": family_rows,
        "action_counts": action_rows,
        "parameter_ranking": ranking_rows,
        "parameter_manifest": manifest,
        "failures": failures,
    }
    save_json_local(out_dir / "stage2_results.json", result_json)

    try:
        import matplotlib.pyplot as plt
        plot_rows = [r for r in overall_rows if r["variant_id"] != "baseline"]
        fig, ax = plt.subplots(figsize=(14, 6))
        labels = [f"{r['parameter']}\
{r['mode']}" for r in plot_rows]
        values = [r["delta_link_f1_pp"] for r in plot_rows]
        ax.bar(range(len(labels)), values)
        ax.axhline(0.0, linewidth=1)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=55, ha="right")
        ax.set_ylabel("Δ Link F1 (percentage points)")
        ax.set_title(f"Reviewer 1 Stage-2 sensitivity: individual {gid} parameters")
        fig.tight_layout()
        fig.savefig(out_dir / "stage2_link_f1_delta.png", dpi=180, bbox_inches="tight")
        plt.close(fig)
    except Exception as exc:
        print("Stage-2 sensitivity plot skipped:", exc)

    # Build report.
    report = []
    report.append("# Reviewer 1 — Parameter Sensitivity Stage-2 Report")
    report.append("")
    report.append(
        f"**Drilled-down group:** {gid} — {SENSITIVITY_GROUP_LABELS[gid]}.  "
        f"**Perturbation:** ±{100*SENSITIVITY_STRENGTH:.0f}% in behavioural space.  "
        f"**Modes run:** {', '.join(SENSITIVITY_STAGE2_MODES)}.  "
        f"**Metric profile:** `{SENSITIVITY_METRIC_PROFILE}`."
    )
    report.append("")
    report.append("## 1. Experimental design")
    report.append("")
    report.append(
        "Stage 1 identified evidence admission as the only functional group requiring parameter-level "
        "drill-down. Stage 2 therefore perturbs one G1 parameter at a time while all other learned "
        "predictions and deterministic assembler parameters remain at the submitted configuration. "
        "For minimum evidence thresholds, relaxation lowers the threshold and strictness raises it. "
        "Image-adaptive thresholds are perturbed only after their per-image adaptive value is computed. "
        "The graph-evidence and ResNet34 forward passes are shared across all variants for each image."
    )
    report.append("")
    report.append("## 2. G1 parameters tested")
    report.append("")
    pretty_manifest = [{
        "Parameter": r["parameter"],
        "Scope": r["scope"],
        "Nominal": r["nominal"],
        "Relaxed factor": r["relaxed_factor"],
        "Strict factor": r["strict_factor"],
        "Role": r["rationale"],
    } for r in manifest]
    report.append(_sens_markdown_table(
        pretty_manifest, ["Parameter","Scope","Nominal","Relaxed factor","Strict factor","Role"]
    ))
    report.append("")
    report.append("## 3. Overall Stage-2 results")
    report.append("")
    pretty_overall = []
    for r in overall_rows:
        pretty_overall.append({
            "Parameter": r["parameter"],
            "Mode": r["mode"],
            "Link F1": round(r["link_f1"], 3),
            "Δ Link F1": round(r["delta_link_f1_pp"], 3),
            "Precision": round(r["edge_precision"], 3),
            "Recall": round(r["edge_recall"], 3),
            "nSEE": round(r["nsee"], 4),
            "Δ nSEE": round(r["delta_nsee"], 4),
            "Loop Rec.": round(r["loop_recall"], 3),
            "Branch Rec.": round(r["branch_recall"], 3),
        })
    report.append(_sens_markdown_table(
        pretty_overall,
        ["Parameter","Mode","Link F1","Δ Link F1","Precision","Recall","nSEE","Δ nSEE","Loop Rec.","Branch Rec."]
    ))
    report.append("")
    report.append("## 4. Parameter ranking")
    report.append("")
    report.append(_sens_markdown_table(
        ranking_rows,
        ["parameter","largest_abs_delta_link_f1_pp","condition_for_largest_abs_delta",
         "strict_delta_link_f1_pp","relaxed_delta_link_f1_pp","max_nsee_increase"]
    ))
    report.append("")
    report.append("## 5. Family-wise Link F1")
    report.append("")
    pretty_family = [{
        "Parameter": r["parameter"],
        "Mode": r["mode"],
        "Family": r["family"],
        "Link F1": round(r["link_f1"], 3),
        "Δ Link F1": round(r["delta_link_f1_pp"], 3),
        "nSEE": round(r["nsee"], 4),
    } for r in family_rows]
    report.append(_sens_markdown_table(
        pretty_family, ["Parameter","Mode","Family","Link F1","Δ Link F1","nSEE"]
    ))
    report.append("")
    report.append("## 6. Mechanistic action counts")
    report.append("")
    report.append(
        "The CSV `stage2_action_counts.csv` records candidate generation, Pass-I selection, final edge "
        "count, endpoint repairs, fallback removals, duplicate removals, candidate resurrections, and "
        "node refinements for each individual threshold perturbation. These counts should be used to "
        "explain whether a sensitive parameter acts by suppressing candidate generation, changing edge "
        "selection, or increasing downstream repair."
    )
    report.append("")
    report.append("## 7. Stage-2 interpretation")
    report.append("")
    if ranking_rows:
        top = ranking_rows[0]
        report.append(
            f"The largest individual-parameter response is produced by **{top['parameter']}**, with "
            f"|Δ Link F1| = **{top['largest_abs_delta_link_f1_pp']:.3f} p.p.** under the "
            f"**{top['condition_for_largest_abs_delta']}** condition. Its worst observed change is "
            f"{top['worst_delta_link_f1_pp']:+.3f} p.p. and its maximum nSEE increase is "
            f"{top['max_nsee_increase']:+.4f}."
        )
        strict_rows = [r for r in ranking_rows if not math.isnan(float(r.get("strict_delta_link_f1_pp", float("nan"))))]
        if strict_rows:
            strict_sorted = sorted(strict_rows, key=lambda r: r["strict_delta_link_f1_pp"])
            top_strict = strict_sorted[0]
            report.append(
                f"On the strict side specifically, **{top_strict['parameter']}** causes the largest "
                f"Link-F1 decrease ({top_strict['strict_delta_link_f1_pp']:+.3f} p.p.)."
            )
    report.append("")
    report.append("## 8. Paper-ready methods wording")
    report.append("")
    report.append(
        "Because the group-level screen localized sensitivity to evidence admission, a second-stage "
        "analysis perturbed each evidence-admission parameter independently by ±20% in behavioural "
        "space while holding all remaining parameters fixed. The tested quantities were the adaptive "
        "node-extent, node-center, head, tail, shaft, skeleton, and fused-support thresholds together "
        "with the minimum arrow-confidence threshold. Adaptive thresholds were modified only after "
        "per-image calibration, and neural predictions were shared across all conditions."
    )
    report.append("")
    report.append("## 9. Paper-ready result wording (generated from this run)")
    report.append("")
    if ranking_rows:
        top = ranking_rows[0]
        report.append(
            f"Parameter-level analysis within evidence admission showed that the largest individual "
            f"change was associated with {top['parameter']}, for which the maximum absolute Link-F1 "
            f"deviation was {top['largest_abs_delta_link_f1_pp']:.2f} percentage points under the "
            f"{top['condition_for_largest_abs_delta']} perturbation. The remaining parameters are "
            "reported separately together with precision/recall and family-wise results, allowing the "
            "group-level sensitivity to be localized rather than attributed uniformly to all thresholds."
        )
    report.append("")
    report.append("## 10. Files produced")
    report.append("")
    report.append(
        "- `stage2_overall.csv` — baseline and individual-parameter headline metrics\
"
        "- `stage2_by_family.csv` — FA/FCA/FCB parameter-level results\
"
        "- `stage2_action_counts.csv` — mechanism counts for each parameter variant\
"
        "- `stage2_parameter_ranking.csv` — parameters ranked by Link-F1 sensitivity\
"
        "- `stage2_parameter_manifest.csv` — exact behavioural perturbation directions\
"
        "- `stage2_results.json` — complete machine-readable report\
"
        "- `stage2_link_f1_delta.png` — parameter-level sensitivity visualization\
"
        "- `stage2_report.md` — paste-ready report"
    )

    report_text = "\
".join(report) + "\
"
    (out_dir / "stage2_report.md").write_text(report_text, encoding="utf-8")

    print("\
" + "="*88)
    print("REVIEWER 1 PARAMETER SENSITIVITY — STAGE 2")
    print("="*88)
    print(report_text)
    return result_json


def finalize_parameter_sensitivity():
    stage = str(SENSITIVITY_STAGE).strip().lower()
    if stage == "stage1":
        return finalize_parameter_sensitivity_stage1()
    if stage == "stage2":
        return finalize_parameter_sensitivity_stage2()
    raise ValueError('SENSITIVITY_STAGE must be "stage1" or "stage2".')



## 7. Inference runner

Each GPU owns one model copy and processes an independent shard. This avoids funneling deterministic assembly work through GPU 0. Predictions are saved as compact graph JSON; no dense maps are retained.

In [ ]:
# ============================================================
# CANONICAL INFERENCE, OUTPUT, AND METRIC RUNNER
# ============================================================

import hashlib


def require_filled_path(value, variable_name, must_exist=True):
    value = str(value).strip()
    if not value:
        raise ValueError(
            f"{variable_name} is blank. Fill it in the USER SETTINGS cell."
        )
    path = Path(value).expanduser().resolve()
    if must_exist and not path.exists():
        raise FileNotFoundError(f"{variable_name} does not exist: {path}")
    return path


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def validate_run_configuration():
    dataset_root = require_filled_path(DATASET_ROOT, "DATASET_ROOT")
    graph_checkpoint = require_filled_path(
        GRAPH_MODEL_CHECKPOINT, "GRAPH_MODEL_CHECKPOINT"
    )
    node_checkpoint = require_filled_path(
        NODE_MODEL_CHECKPOINT, "NODE_MODEL_CHECKPOINT"
    )
    output_root = require_filled_path(
        OUTPUT_ROOT, "OUTPUT_ROOT", must_exist=False
    )

    if output_root in {dataset_root, graph_checkpoint, node_checkpoint}:
        raise ValueError("OUTPUT_ROOT must be a separate output directory.")

    if _sk_skeletonize is None:
        raise ImportError(
            "scikit-image is required to calculate arrow-only Soft-clDice."
        )

    if tvm is None:
        raise ImportError(
            "torchvision is required to load the plain ResNet34 node model."
        )

    if int(GRAPH_BASE_CHANNELS) != 16:
        raise ValueError(
            "The paper graph-evidence checkpoint uses GRAPH_BASE_CHANNELS = 16."
        )

    if CLEAR_OUTPUT_BEFORE_RUN and output_root.exists():
        shutil.rmtree(output_root)

    output_root.mkdir(parents=True, exist_ok=True)
    cv2.setNumThreads(1)
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    return dataset_root, graph_checkpoint, node_checkpoint, output_root


def discover_paper_samples():
    samples = discover_converted_dataset_samples(
        dataset_root=Path(DATASET_ROOT),
        families=list(DATASET_FAMILIES),
        splits=list(INFERENCE_SPLITS),
        require_gt=True,
    )

    if QUICK_TEST:
        rng = random.Random(RANDOM_SEED)
        samples = rng.sample(samples, min(int(QUICK_TEST_N), len(samples)))

    if not samples:
        raise RuntimeError("No image/annotation pairs were discovered.")

    print(f"Discovered {len(samples)} inference samples.")
    return samples


def process_sample_list_worker(
    worker_id,
    sample_dicts,
    device_str,
    example_keys,
    output_root,
):
    output_root = Path(output_root)
    device = torch.device(
        device_str if device_str else (
            "cuda" if torch.cuda.is_available() else "cpu"
        )
    )

    decoder_args = make_decoder_args(device_str=device_str)

    print(f"[worker {worker_id}] loading graph-evidence model on {device}")
    graph_model, graph_load_info = load_assembler_checkpoint_safe(
        checkpoint_path=Path(GRAPH_MODEL_CHECKPOINT),
        device=device,
        image_mode=IMAGE_MODE,
        base_ch=int(GRAPH_BASE_CHANNELS),
        norm_type=NORM_TYPE,
        strict=True,
    )

    inferred_base_channels = int(
        graph_load_info.get("base_ch", GRAPH_BASE_CHANNELS)
    )
    if inferred_base_channels != int(GRAPH_BASE_CHANNELS):
        raise ValueError(
            "Graph checkpoint architecture mismatch: "
            f"checkpoint base channels={inferred_base_channels}, "
            f"configured={GRAPH_BASE_CHANNELS}."
        )

    decoder_config = build_decoder_cfg(decoder_args)

    print(f"[worker {worker_id}] loading plain ResNet34 node model on {device}")
    node_model, node_load_info = load_node_segmentation_model(
        NODE_MODEL_CHECKPOINT, device
    )

    metric_rows = []
    failures = []
    sensitivity_rows = []
    sensitivity_failures = []

    for index, sample_dict in enumerate(sample_dicts, start=1):
        sample = InferenceSample(
            sample_id=sample_dict["sample_id"],
            image_path=Path(sample_dict["image_path"]),
            json_path=Path(sample_dict["json_path"]),
            family=sample_dict.get("family", ""),
            split=sample_dict.get("split", ""),
        )

        try:
            start_time = time.time()

            if PRINT_TIMING_DIAGNOSTICS and (
                index <= 3 or index % int(PROGRESS_EVERY_N) == 0
            ):
                print(
                    f"[worker {worker_id}] {index}/{len(sample_dicts)} "
                    f"{sample.family}/{sample.split}/{sample.image_path.name}"
                )

            original_bgr, decoder_image, image_tensor = load_image_for_model(
                image_path=sample.image_path,
                target_w=int(TARGET_W),
                target_h=int(TARGET_H),
                image_mode=IMAGE_MODE,
                normalize_image=NORMALIZE_IMAGE,
                use_otsu=bool(APPLY_OTSU_THRESHOLD),
            )

            with torch.no_grad():
                graph_outputs = graph_model(image_tensor.to(device))
            evidence_maps = outputs_to_pred_maps(graph_outputs)

            _, node_probability = predict_node_probability(
                node_model,
                sample.image_path,
                device,
                img_size=int(NODE_SEG_INPUT_SIZE),
            )
            resnet_nodes, node_binary = extract_node_boxes(
                node_probability,
                threshold=float(NODE_SEG_THRESHOLD),
                min_area=int(NODE_SEG_MIN_AREA),
                close_kernel=int(NODE_SEG_CLOSE_KERNEL),
            )

            original_height, original_width = original_bgr.shape[:2]

            # Pass I: physical graph hypothesis construction.
            first_pass = decode_graph_learning_aligned(
                pred=evidence_maps,
                decoder_img=decoder_image,
                image_name=sample.image_path.name,
                cfg=decoder_config,
                orig_size=(original_width, original_height),
            )

            # Pass II: graph-context structural finalization followed by
            # agreement-gated node-region refinement.
            if RUN_PARAMETER_SENSITIVITY:
                # Every Pass-II call is placed behind the same lock because G5
                # sensitivity variants temporarily override finalizer globals.
                final_result = sensitivity_finalize_two_pass(
                    first_pass_result=first_pass,
                    image_name=sample.image_path.name,
                    orig_size=(original_width, original_height),
                    family=sample.family,
                    resnet_nodes_orig=resnet_nodes,
                    variant=ACTIVE_SENSITIVITY_VARIANTS[0],  # baseline
                )
            else:
                final_result = decode_two_pass_graph(
                    pred=evidence_maps,
                    decoder_img=decoder_image,
                    image_name=sample.image_path.name,
                    cfg=decoder_config,
                    orig_size=(original_width, original_height),
                    family=sample.family,
                    resnet_nodes_orig=resnet_nodes,
                    first_pass_result=first_pass,
                )
            prediction = (
                final_result.writer_style_original_json
                or final_result.writer_style_json
            )

            with open(sample.json_path, "r", encoding="utf-8") as handle:
                ground_truth = json.load(handle)

            row = evaluate_prediction_by_profile(
                prediction, ground_truth, profile=METRIC_PROFILE
            )
            final_counts = (
                final_result.decoder_debug_json.get("counts", {}) or {}
            )
            row.update({
                "sample_id": sample.sample_id,
                "image_name": sample.image_path.name,
                "family": sample.family,
                "split": sample.split,
                "image_path": str(sample.image_path),
                "json_path": str(sample.json_path),
                "worker_id": worker_id,
                "first_pass_candidates": int(
                    len(first_pass.candidate_arrows)
                ),
                "first_pass_selected_edges": int(
                    len(first_pass.arrows)
                ),
                "final_selected_edges": int(
                    len(prediction.get("arrows", []) or [])
                ),
                "endpoint_repairs": int(
                    final_counts.get("endpoint_object_repairs", 0)
                    + final_counts.get("selected_attachment_repairs", 0)
                ),
                "fallback_removals": int(
                    final_counts.get("selected_fallback_removals", 0)
                ),
                "duplicate_removals": int(
                    final_counts.get("selected_duplicate_removals", 0)
                ),
                "candidate_resurrections": int(
                    final_counts.get("candidate_resurrections", 0)
                ),
                "matched_node_refinements": int(
                    final_counts.get("matched_node_refinements", 0)
                ),
                "elapsed_seconds": float(time.time() - start_time),
            })
            metric_rows.append(row)

            if RUN_PARAMETER_SENSITIVITY:
                sens_rows_i, sens_fail_i = run_sensitivity_variants_for_sample(
                    sample=sample,
                    evidence_maps=evidence_maps,
                    decoder_image=decoder_image,
                    original_size=(original_width, original_height),
                    resnet_nodes=resnet_nodes,
                    ground_truth=ground_truth,
                    base_decoder_config=decoder_config,
                    baseline_first_pass=first_pass,
                    baseline_final_result=final_result,
                )
                sensitivity_rows.extend(sens_rows_i)
                sensitivity_failures.extend(sens_fail_i)

            if SAVE_ALL_PREDICTION_JSON:
                prediction_path = (
                    output_root
                    / "predictions"
                    / sample.family
                    / sample.split
                    / "json"
                    / f"{sample.sample_id}.json"
                )
                save_json_local(prediction_path, prediction)

            sample_key = (
                sample.family,
                sample.split,
                sample.sample_id,
            )
            if SAVE_DEBUG_EXAMPLES and sample_key in example_keys:
                example_dir = (
                    output_root
                    / "examples"
                    / f"{sample.family}_{sample.split}_{sample.sample_id}"
                )
                example_dir.mkdir(parents=True, exist_ok=True)

                save_json_local(
                    example_dir / "prediction.json", prediction
                )
                save_json_local(
                    example_dir / "ground_truth.json", ground_truth
                )
                save_json_local(
                    example_dir / "metrics.json", row
                )

                if SAVE_DEBUG_ASSEMBLER_JSON:
                    save_json_local(
                        example_dir / "two_pass_assembler_debug.json",
                        final_result.decoder_debug_json,
                    )

                if SAVE_DEBUG_OVERLAYS:
                    draw_overlay_from_writer(
                        sample.image_path,
                        prediction,
                        example_dir / "prediction_overlay.png",
                    )
                    draw_overlay_from_writer(
                        sample.image_path,
                        ground_truth,
                        example_dir / "ground_truth_overlay.png",
                    )

            if PRINT_TIMING_DIAGNOSTICS and (
                index <= 3 or index % int(PROGRESS_EVERY_N) == 0
            ):
                print(
                    f"[worker {worker_id}] completed in "
                    f"{time.time() - start_time:.2f}s"
                )

        except Exception as exc:
            failures.append({
                "sample_id": sample_dict.get("sample_id", ""),
                "family": sample_dict.get("family", ""),
                "split": sample_dict.get("split", ""),
                "image_path": sample_dict.get("image_path", ""),
                "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc(limit=60),
                "worker_id": worker_id,
            })

    shard_dir = output_root / "shards"
    shard_dir.mkdir(parents=True, exist_ok=True)

    with open(
        shard_dir / f"worker_{worker_id}_metrics.jsonl",
        "w",
        encoding="utf-8",
    ) as handle:
        for row in metric_rows:
            handle.write(
                json.dumps(safe_json(row), ensure_ascii=False) + "\n"
            )

    with open(
        shard_dir / f"worker_{worker_id}_failures.jsonl",
        "w",
        encoding="utf-8",
    ) as handle:
        for row in failures:
            handle.write(
                json.dumps(safe_json(row), ensure_ascii=False) + "\n"
            )

    if RUN_PARAMETER_SENSITIVITY:
        with open(
            shard_dir / f"worker_{worker_id}_sensitivity.jsonl",
            "w",
            encoding="utf-8",
        ) as handle:
            for row in sensitivity_rows:
                handle.write(
                    json.dumps(safe_json(row), ensure_ascii=False) + "\n"
                )

        with open(
            shard_dir / f"worker_{worker_id}_sensitivity_failures.jsonl",
            "w",
            encoding="utf-8",
        ) as handle:
            for row in sensitivity_failures:
                handle.write(
                    json.dumps(safe_json(row), ensure_ascii=False) + "\n"
                )

    save_json_local(
        shard_dir / f"worker_{worker_id}_load_info.json",
        {
            "device": str(device),
            "num_samples": len(sample_dicts),
            "num_successful": len(metric_rows),
            "num_failures": len(failures),
            "graph_model": graph_load_info,
            "node_model": node_load_info,
        },
    )

    return True


def run_parallel_or_single(samples):
    output_root = Path(OUTPUT_ROOT)
    output_root.mkdir(parents=True, exist_ok=True)

    example_keys = get_example_ids(
        samples,
        max_n=int(MAX_DEBUG_EXAMPLES),
        policy=DEBUG_EXAMPLE_POLICY,
    )
    sample_dicts = samples_to_dicts(samples)

    cuda_count = (
        torch.cuda.device_count() if torch.cuda.is_available() else 0
    )
    worker_count = (
        min(int(MAX_GPU_WORKERS), max(1, cuda_count))
        if PARALLEL_INFERENCE
        else 1
    )

    if (
        not PARALLEL_INFERENCE
        or str(PARALLEL_BACKEND).lower() == "single"
        or worker_count <= 1
    ):
        device_str = "cuda:0" if cuda_count else "cpu"
        process_sample_list_worker(
            0,
            sample_dicts,
            device_str,
            example_keys,
            output_root,
        )
        return

    if str(PARALLEL_BACKEND).lower() != "threads":
        raise ValueError(
            "PARALLEL_BACKEND must be 'threads' or 'single'. "
            "Forked CUDA multiprocessing is intentionally not used."
        )

    from concurrent.futures import ThreadPoolExecutor, as_completed

    shards = split_evenly(sample_dicts, worker_count)
    futures = []

    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        for worker_id, shard in enumerate(shards):
            if not shard:
                continue
            device_str = f"cuda:{worker_id % cuda_count}"
            futures.append(
                executor.submit(
                    process_sample_list_worker,
                    worker_id,
                    shard,
                    device_str,
                    example_keys,
                    output_root,
                )
            )

        for future in as_completed(futures):
            future.result()


def collect_worker_rows(pattern):
    rows = []
    shard_dir = Path(OUTPUT_ROOT) / "shards"
    for path in sorted(shard_dir.glob(pattern)):
        rows.extend(read_jsonl(path))
    return rows


def percent(value):
    return 100.0 * float(value)


def build_table4_row(overall):
    return {
        "Method": "Proposed framework",
        "Node F1": round(percent(overall.get("mean_node_f1", 0.0)), 2),
        "Matched Node Box IoU": round(percent(overall.get("mean_matched_box_iou", 0.0)), 2),
        "Soft-clDice": round(
            percent(overall.get("mean_soft_cldice", 0.0)), 2
        ),
        "End. L2": round(float(overall.get("mean_endpoint_l2", 0.0)), 2),
        "Link F1": round(percent(overall.get("mean_link_f1", 0.0)), 2),
        "nSEE": round(float(overall.get("mean_nsee", 0.0)), 3),
        "Loop Rec.": round(
            percent(overall.get("loop_back_recall", 0.0)), 2
        ),
        "Branch Rec.": round(
            percent(overall.get("decision_branch_recall", 0.0)), 2
        ),
    }


def build_table5_rows(by_family):
    rows = []
    for family_row in by_family:
        rows.append({
            "Family": str(family_row.get("family", "")).upper(),
            "Images": int(family_row.get("num_images", 0)),
            "Link F1": round(
                percent(family_row.get("mean_link_f1", 0.0)), 2
            ),
            "nSEE": round(float(family_row.get("mean_nsee", 0.0)), 3),
            "End. L2": round(
                float(family_row.get("mean_endpoint_l2", 0.0)), 2
            ),
        })
    return rows


def finalize_paper_metrics():
    output_root = Path(OUTPUT_ROOT)
    metrics_dir = output_root / "metrics"
    metrics_dir.mkdir(parents=True, exist_ok=True)

    rows = collect_worker_rows("worker_*_metrics.jsonl")
    failures = collect_worker_rows("worker_*_failures.jsonl")

    if not rows:
        raise RuntimeError(
            "No successful metric rows were produced. Inspect failures.csv."
        )

    write_csv_rows_local(metrics_dir / "metrics_per_image.csv", rows)
    write_csv_rows_local(output_root / "failures.csv", failures)

    overall = aggregate_metric_rows_tolerant(
        rows, group_keys=[]
    )[0]
    by_family = aggregate_metric_rows_tolerant(
        rows, group_keys=["family"]
    )
    by_split = aggregate_metric_rows_tolerant(
        rows, group_keys=["split"]
    )
    by_family_split = aggregate_metric_rows_tolerant(
        rows, group_keys=["family", "split"]
    )

    table4 = [build_table4_row(overall)]
    table5 = build_table5_rows(by_family)

    write_csv_rows_local(metrics_dir / "metrics_by_family.csv", by_family)
    write_csv_rows_local(metrics_dir / "metrics_by_split.csv", by_split)
    write_csv_rows_local(
        metrics_dir / "metrics_by_family_split.csv",
        by_family_split,
    )
    write_csv_rows_local(metrics_dir / "table4_proposed_row.csv", table4)
    write_csv_rows_local(metrics_dir / "table5_family_results.csv", table5)

    summary = {
        "overall": overall,
        "by_family": by_family,
        "by_split": by_split,
        "by_family_split": by_family_split,
        "paper_table_4_proposed_row": table4[0],
        "paper_table_5_rows": table5,
        "successful_images": len(rows),
        "failed_images": len(failures),
        "metric_definitions": {
            "node_f1": (
                "One-to-one tolerant node matching using IoU or "
                "center-distance acceptance."
            ),
            "matched_node_box_iou": (
                "Mean bounding-box IoU over one-to-one spatially matched predicted and ground-truth nodes."
            ),
            "soft_cldice": (
                "Tolerance-aware centerline Dice between predicted and "
                "ground-truth arrow-path rasters only."
            ),
            "endpoint_l2": (
                "Mean endpoint Euclidean error on matched directed edges."
            ),
            "link_f1": "Directed edge F1 after node matching.",
            "nsee": (
                "Normalized structural edit error: (node FP + node FN + edge FP + edge FN) divided by "
                "max(|V_pred|,|V_gt|)+max(|E_pred|,|E_gt|)."
            ),
        },
    }
    save_json_local(metrics_dir / "metrics_summary.json", summary)

    checkpoint_manifest = {
        "graph_model": {
            "path": str(Path(GRAPH_MODEL_CHECKPOINT).resolve()),
            "sha256": sha256_file(GRAPH_MODEL_CHECKPOINT),
        },
        "node_model": {
            "path": str(Path(NODE_MODEL_CHECKPOINT).resolve()),
            "sha256": sha256_file(NODE_MODEL_CHECKPOINT),
        },
        "dataset_root": str(Path(DATASET_ROOT).resolve()),
        "families": list(DATASET_FAMILIES),
        "splits": list(INFERENCE_SPLITS),
        "successful_images": len(rows),
        "failed_images": len(failures),
    }
    save_json_local(
        output_root / "reproduction_manifest.json",
        checkpoint_manifest,
    )

    print("\n" + "=" * 72)
    print("TABLE 4 — PROPOSED FRAMEWORK ROW")
    print("=" * 72)
    print(json.dumps(table4[0], indent=2))

    print("\n" + "=" * 72)
    print("TABLE 5 — FAMILY-WISE RESULTS")
    print("=" * 72)
    for row in table5:
        print(row)

    print(f"\nSaved all outputs to: {output_root}")
    if failures:
        print(
            f"Warning: {len(failures)} samples failed. "
            "Inspect failures.csv before using the table values."
        )

    return summary

In [ ]:
# ============================================================
# MODULAR FINALIZATION / RE-EVALUATION / COMPRESSION
# ============================================================

def _json_safe_for_csv(v):
    if isinstance(v,(list,dict,tuple)):
        return json.dumps(v, ensure_ascii=False)
    return v


def write_rows_csv_clean(path, rows):
    clean=[{k:_json_safe_for_csv(v) for k,v in r.items()} for r in rows]
    write_csv_rows_local(path, clean)


def build_revision_table_row(overall):
    return {
        "Method":"Proposed framework",
        "Node F1":round(100*overall.get('node_f1',0.0),2),
        "Matched Node Box IoU":round(100*overall.get('mean_matched_box_iou',0.0),2),
        "Soft-clDice (matched links)":round(100*overall.get('soft_cldice_matched_links',0.0),2),
        "Soft-clDice coverage":round(100*overall.get('soft_cldice_coverage',0.0),2),
        "Endpoint L2 mean":round(float(overall.get('endpoint_l2_matched_endpoints_mean',float('nan'))),2),
        "Endpoint coverage":round(100*overall.get('endpoint_coverage',0.0),2),
        "Link F1":round(100*overall.get('link_f1',0.0),2),
        "nSEE":round(float(overall.get('mean_normalized_structural_edit_error',0.0)),3),
        "Loop Rec.":round(100*overall.get('loop_back_recall',0.0),2),
        "Branch Rec.":round(100*overall.get('decision_branch_recall',0.0),2),
    }


def finalize_metrics_modular(rows=None, failures=None, profile=None):
    profile=(profile or METRIC_PROFILE).lower()
    out=Path(OUTPUT_ROOT); mdir=out/'metrics'; mdir.mkdir(parents=True,exist_ok=True)
    if rows is None: rows=collect_worker_rows('worker_*_metrics.jsonl')
    if failures is None: failures=collect_worker_rows('worker_*_failures.jsonl')
    if not rows: raise RuntimeError('No successful metric rows were produced.')
    write_rows_csv_clean(mdir/'metrics_per_image.csv',rows)
    write_rows_csv_clean(out/'failures.csv',failures or [])
    if profile=='submitted':
        overall=aggregate_metric_rows_tolerant(rows,group_keys=[])[0]
        by_family=aggregate_metric_rows_tolerant(rows,group_keys=['family'])
        by_split=aggregate_metric_rows_tolerant(rows,group_keys=['split'])
        table=[build_table4_row(overall)]
        definitions={
            'Matched Node Box IoU':'Mean bounding-box IoU over one-to-one spatially matched predicted and ground-truth nodes.',
            'nSEE':'(node FP + node FN + edge FP + edge FN) / [max(|V_pred|,|V_gt|)+max(|E_pred|,|E_gt|)].',
            'Soft-clDice':'Conditional on matched directed links.',
            'Endpoint L2':'Conditional on matched directed links/endpoints.',
        }
    else:
        overall=aggregate_revision_metrics(rows,group_keys=[])[0]
        by_family=aggregate_revision_metrics(rows,group_keys=['family'])
        by_split=aggregate_revision_metrics(rows,group_keys=['split'])
        table=[build_revision_table_row(overall)]
        definitions={
            'Matched Node Box IoU':'Bounding-box IoU on one-to-one spatially matched nodes. The table keeps the macro image aggregation for continuity; pooled matched-node IoU is also exported.',
            'nSEE':'(node FP + node FN + edge FP + edge FN) / [max(|V_pred|,|V_gt|)+max(|E_pred|,|E_gt|)]. This is a normalized structural edit error, not a general GED solver.',
            'Soft-clDice (matched links)':'Pooled mean across matched directed links with valid path geometry. Coverage is reported as evaluated links / GT links.',
            'Endpoint L2':'Pooled over valid head/tail endpoint pairs belonging to matched directed links; mean, median, P90 and endpoint coverage are exported.',
            'matching tolerance':'Image-relative tolerance tau=max(2, round(METRIC_TOLERANCE_RATIO * max(width,height))).',
        }
    write_rows_csv_clean(mdir/'metrics_by_family.csv',by_family)
    write_rows_csv_clean(mdir/'metrics_by_split.csv',by_split)
    write_rows_csv_clean(mdir/'headline_table.csv',table)
    summary={'profile':profile,'overall':overall,'by_family':by_family,'by_split':by_split,'headline_table':table[0],
             'successful_images':len(rows),'failed_images':len(failures or []),'metric_definitions':definitions}
    save_json_local(mdir/'metrics_summary.json',summary)
    print('\n'+'='*72); print(f'HEADLINE METRICS — {profile.upper()} PROFILE'); print('='*72)
    print(json.dumps(table[0],indent=2,allow_nan=True))
    return summary


def prediction_folder_available():
    p=Path(OUTPUT_ROOT)/'predictions'
    return p.is_dir() and any(p.rglob('*.json'))


def prediction_zip_available():
    return Path(PREDICTIONS_ZIP).is_file()


def read_prediction_for_sample(sample):
    rel=Path('predictions')/sample.family/sample.split/'json'/f'{sample.sample_id}.json'
    folder_path=Path(OUTPUT_ROOT)/rel.relative_to('predictions') if str(rel).startswith('predictions') else Path(OUTPUT_ROOT)/rel
    # Actual inference folder is OUTPUT_ROOT/predictions/...
    folder_path=Path(OUTPUT_ROOT)/rel
    if folder_path.is_file():
        with open(folder_path,'r',encoding='utf-8') as f: return json.load(f)
    zpath=Path(PREDICTIONS_ZIP)
    if zpath.is_file():
        with zipfile.ZipFile(zpath,'r') as zf:
            name=str(rel).replace('\\','/')
            if name in zf.namelist():
                return json.loads(zf.read(name).decode('utf-8'))
    raise FileNotFoundError(f'Prediction missing for {sample.family}/{sample.split}/{sample.sample_id}')


def evaluate_existing_predictions(profile=None):
    profile=(profile or METRIC_PROFILE).lower()
    samples=discover_paper_samples()
    rows=[]; failures=[]
    for i,sample in enumerate(samples,1):
        try:
            pred=read_prediction_for_sample(sample)
            with open(sample.json_path,'r',encoding='utf-8') as f: gt=json.load(f)
            row=evaluate_prediction_by_profile(pred,gt,profile=profile)
            row.update({'sample_id':sample.sample_id,'image_name':sample.image_path.name,'family':sample.family,'split':sample.split,
                        'image_path':str(sample.image_path),'json_path':str(sample.json_path),'worker_id':'evaluation_only'})
            rows.append(row)
            if i<=3 or i%PROGRESS_EVERY_N==0: print(f'[metrics] {i}/{len(samples)}')
        except Exception as exc:
            failures.append({'sample_id':sample.sample_id,'family':sample.family,'split':sample.split,'error':f'{type(exc).__name__}: {exc}'})
    return finalize_metrics_modular(rows,failures,profile=profile)


def compress_predictions(delete_source=None):
    delete_source=DELETE_UNCOMPRESSED_PREDICTIONS_AFTER_ZIP if delete_source is None else bool(delete_source)
    src=Path(OUTPUT_ROOT)/'predictions'; zpath=Path(PREDICTIONS_ZIP)
    if not src.is_dir():
        print('No uncompressed predictions folder to compress.')
        return zpath if zpath.exists() else None
    zpath.parent.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(zpath,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=9) as zf:
        for p in src.rglob('*'):
            if p.is_file(): zf.write(p,arcname=str(Path('predictions')/p.relative_to(src)).replace('\\','/'))
    print(f'Compressed predictions: {zpath} ({zpath.stat().st_size/1024**2:.2f} MB)')
    if delete_source:
        shutil.rmtree(src,ignore_errors=True)
        print('Removed uncompressed prediction folder after successful ZIP creation.')
    return zpath


## 8. Run controller

`auto` first checks for predictions created in the current hosted-runtime run. If none exist, it uses the trained checkpoints for full inference. Training is never silently enabled.

In [ ]:
# ============================================================
# RUN CONTROLLER
# ============================================================

if RESOLVED_DATASET_ROOT is None:
    if SHOULD_PREPARE_DATASET:
        # dataset-preparation cells above should have populated this by now
        RESOLVED_DATASET_ROOT=Path(DATASET_ROOT) if Path(DATASET_ROOT).exists() else None
    if RESOLVED_DATASET_ROOT is None:
        raise FileNotFoundError('No valid converted dataset is available. Fix DATASET_ROOT or enable dataset preparation.')
DATASET_ROOT=str(RESOLVED_DATASET_ROOT)

if RESOLVED_GRAPH_CHECKPOINT is not None: GRAPH_MODEL_CHECKPOINT=str(RESOLVED_GRAPH_CHECKPOINT)
if RESOLVED_NODE_CHECKPOINT is not None: NODE_MODEL_CHECKPOINT=str(RESOLVED_NODE_CHECKPOINT)

mode=RUN_MODE.lower().strip()
if mode=='auto':
    # Parameter sensitivity requires fresh dense evidence maps and Pass-I objects;
    # final prediction JSON alone is insufficient. Therefore auto mode deliberately
    # performs full inference when the sensitivity study is enabled.
    if RUN_PARAMETER_SENSITIVITY and Path(str(GRAPH_MODEL_CHECKPOINT)).is_file() and Path(str(NODE_MODEL_CHECKPOINT)).is_file():
        mode='full_inference'
    elif REUSE_EXISTING_PREDICTIONS and (prediction_folder_available() or prediction_zip_available()):
        mode='evaluate_existing'
    elif Path(str(GRAPH_MODEL_CHECKPOINT)).is_file() and Path(str(NODE_MODEL_CHECKPOINT)).is_file():
        mode='full_inference'
    else:
        missing=[]
        if not Path(str(GRAPH_MODEL_CHECKPOINT)).is_file(): missing.append('graph checkpoint')
        if not Path(str(NODE_MODEL_CHECKPOINT)).is_file(): missing.append('node checkpoint')
        raise FileNotFoundError('Auto mode cannot start inference because the following are unavailable: '+', '.join(missing)+'. Enable the corresponding training switch or set the checkpoint path.')

print('Resolved run mode:',mode)
PAPER_RESULTS=None
SENSITIVITY_RESULTS=None

if mode=='prepare_dataset_only':
    print('Dataset preparation/validation complete:', DATASET_ROOT)
elif mode=='evaluate_existing':
    PAPER_RESULTS=evaluate_existing_predictions(profile=METRIC_PROFILE)
    if RUN_PARAMETER_SENSITIVITY:
        print(
            'NOTE: parameter sensitivity was not run because evaluate_existing '
            'contains only final JSON predictions. Set RUN_MODE="full_inference" '
            'or RUN_MODE="auto" with checkpoints available.'
        )
elif mode=='full_inference':
    if not Path(str(GRAPH_MODEL_CHECKPOINT)).is_file(): raise FileNotFoundError('GRAPH_MODEL_CHECKPOINT not found.')
    if not Path(str(NODE_MODEL_CHECKPOINT)).is_file(): raise FileNotFoundError('NODE_MODEL_CHECKPOINT not found.')
    dataset_root, graph_checkpoint, node_checkpoint, output_root = validate_run_configuration()
    samples=discover_paper_samples()

    if RUN_PARAMETER_SENSITIVITY:
        shard_dir = Path(OUTPUT_ROOT) / 'shards'
        if shard_dir.exists():
            for stale in shard_dir.glob('worker_*_sensitivity*.jsonl'):
                stale.unlink(missing_ok=True)

    t0=time.time(); run_parallel_or_single(samples)
    PAPER_RESULTS=finalize_metrics_modular(profile=METRIC_PROFILE)
    if RUN_PARAMETER_SENSITIVITY:
        SENSITIVITY_RESULTS=finalize_parameter_sensitivity()
    print(f'Completed {len(samples)} samples in {(time.time()-t0)/60.0:.2f} minutes.')
    if COMPRESS_PREDICTIONS_AT_END:
        compress_predictions()
else:
    raise ValueError("RUN_MODE must be auto, full_inference, evaluate_existing, or prepare_dataset_only")


## 9. Revision workspace

Use this final area for reviewer experiments without modifying the frozen inference pipeline.

### Reviewer 1 sensitivity workflow

The master-config variable `SENSITIVITY_STAGE` selects the experiment:

- `"stage1"` — five-group screen;
- `"stage2"` — individual-parameter drill-down inside `SENSITIVITY_STAGE2_GROUP` (currently `G1`).

For Stage 2, keep `RUN_PARAMETER_SENSITIVITY=True` and use `RUN_MODE="auto"` or `"full_inference"`. Neural predictions are computed once per image and reused. The default `SENSITIVITY_STAGE2_MODES=("relaxed", "strict")` evaluates both ±20% directions. If you only want the faster strict-side diagnostic first, change it to `("strict",)`.

The full Stage-2 run prints and saves:

`OUTPUT_ROOT/parameter_sensitivity/stage2_report.md`

Paste that report back into the discussion. The report also produces a parameter ranking, family-wise results, action counts, a plot, and machine-readable CSV/JSON files.


In [ ]:
# Reviewer 1 Stage-2 outputs after a full run:
# report_path = Path(OUTPUT_ROOT) / SENSITIVITY_OUTPUT_SUBDIR / "stage2_report.md"
# print(report_path.read_text(encoding="utf-8"))

# To rerun the original group screen instead:
# SENSITIVITY_STAGE = "stage1"

# To run only the strict-side individual G1 drill-down (faster):
# SENSITIVITY_STAGE = "stage2"
# SENSITIVITY_STAGE2_MODES = ("strict",)
